# RR CLUTCH | Performance Under Pressure

Code only portfolio copy of the original research notebook for Rajasthan Royals' SupeRR Selector Season 2. The competition data, execution outputs, player level data tables and source files are intentionally excluded.

This notebook cannot run without authorised access to the original data archive. Place your privately held archive at `data/rajasthanroyals_dataset.zip` to run it locally. Never commit the data archive, any extracted CSVs or generated result files.

The notebook preserves the original exploratory analysis sequence; some cells are investigations and revisions rather than a single minimal pipeline.


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
from IPython.display import display

ZIP_PATH = Path("data/rajasthanroyals_dataset.zip")

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH.resolve()}"

print("Found:", ZIP_PATH.resolve())

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    file_list = z.namelist()

print("Total files inside ZIP:", len(file_list))
                                                                                                                
print("\nFiles found:")
for file in file_list:
    print(" -", file)

In [ ]:
# Load all CSV datasets directly from ZIP

datasets = {}

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    csv_files = [
        file for file in z.namelist()
        if file.lower().endswith(".csv")
    ]

    for file in csv_files:
        with z.open(file) as f:
            df = pd.read_csv(f, low_memory=False)

        dataset_name = Path(file).stem
        datasets[dataset_name] = df

        print(f"{dataset_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nTotal datasets loaded:", len(datasets))

In [ ]:
# Inspect the structure of all six datasets

for name, df in datasets.items():

    print("\n" + "=" * 70)
    print("DATASET:", name.upper())
    print("=" * 70)

    print("\nShape:", df.shape)

    print("\nColumn Names and Data Types:")
    print(df.dtypes.to_string())

    print("\nFirst 3 Rows:")
    display(df.head(3))

In [ ]:
audit_results = []

for name, df in datasets.items():

    total_rows = len(df)
    total_columns = len(df.columns)

    missing_values = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()

    audit_results.append({
        "Dataset": name,
        "Rows": total_rows,
        "Columns": total_columns,
        "Missing Values": missing_values,
        "Duplicate Rows": duplicate_rows,
        "Missing %": round(
            (missing_values / df.size) * 100, 2
        ) if df.size > 0 else 0
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

In [ ]:
for name, df in datasets.items():

    print("\n" + "=" * 70)
    print("DATASET:", name.upper())
    print("=" * 70)

    missing_summary = pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing %": (df.isnull().mean() * 100).round(3)
    })

    missing_summary = missing_summary.loc[
        missing_summary["Missing Count"] > 0
    ]

    missing_summary = missing_summary.sort_values(
        by="Missing %",
        ascending=False
    )

    if missing_summary.empty:
        print("No missing values found.")
    else:
        display(missing_summary)

In [ ]:
balls = datasets["ball_by_ball"]

# Identify deliveries with mismatched dismissal information
missing_type = balls["dismissal_type"].isna()
missing_player = balls["dismissed_player_id"].isna()

mismatch = balls[
    missing_type != missing_player
].copy()

print("Total deliveries:", len(balls))

print(
    "Dismissal type recorded:",
    balls["dismissal_type"].notna().sum()
)

print(
    "Dismissed player recorded:",
    balls["dismissed_player_id"].notna().sum()
)

print(
    "\nTotal mismatched dismissal records:",
    len(mismatch)
)

print("\nMismatch categories:")

print(
    "Player recorded, dismissal type missing:",
    (missing_type & ~missing_player).sum()
)

print(
    "Dismissal type recorded, player missing:",
    (~missing_type & missing_player).sum()
)

print("\nMismatched records:")
display(mismatch)

In [ ]:
# CELL 8 — Dismissal Types and Wicket Attribution

balls = datasets["ball_by_ball"]

print("=" * 70)
print("DISMISSAL TYPE DISTRIBUTION")
print("=" * 70)

dismissal_summary = (
    balls.groupby(
        ["dismissal_type_id", "dismissal_type"],
        dropna=False
    )
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

display(dismissal_summary)

print("\n" + "=" * 70)
print("INCONSISTENT DISMISSAL RECORDS")
print("=" * 70)

columns_to_show = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "striker_name",
    "dismissed_player_id",
    "dismissal_type_id",
    "dismissal_type"
]

display(mismatch[columns_to_show])

In [ ]:
# Display all recorded dismissal types

display(
    dismissal_summary[
        dismissal_summary["dismissal_type"].notna()
    ]
)

In [ ]:
# CELL 9 — Delivery Scoring and Extras Investigation

balls = datasets["ball_by_ball"]

print("=" * 70)
print("RUNS AND EXTRAS COLUMNS")
print("=" * 70)

scoring_columns = [
    col for col in balls.columns
    if any(
        keyword in col.lower()
        for keyword in ["run", "extra", "wide", "noball", "no_ball"]
    )
]

print("\nScoring-related columns:")
print(scoring_columns)

print("\n" + "=" * 70)
print("EXTRAS TYPE DISTRIBUTION")
print("=" * 70)

display(
    balls["extras_type"]
    .value_counts(dropna=False)
    .reset_index()
)

print("\n" + "=" * 70)
print("SCORING COLUMN SUMMARY")
print("=" * 70)

display(
    balls[scoring_columns].describe(include="all").T
)

In [ ]:
# CELL 10 — Investigate Scoring Conventions

balls = datasets["ball_by_ball"]

print("=" * 70)
print("RUNS DISTRIBUTION")
print("=" * 70)

display(
    balls["runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("runs")
    .reset_index(name="deliveries")
)

print("\n" + "=" * 70)
print("EXTRAS1 RUNS DISTRIBUTION")
print("=" * 70)

display(
    balls["extras1_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("extras1_runs")
    .reset_index(name="deliveries")
)

print("\n" + "=" * 70)
print("SCORING PATTERNS BY EXTRAS TYPE")
print("=" * 70)

display(
    balls.groupby("extras_type", dropna=False)[
        ["runs", "extras1_runs", "extras_2_runs"]
    ].agg(["count", "mean", "min", "max"])
)

print("\n" + "=" * 70)
print("SAMPLE DELIVERIES BY EXTRAS TYPE")
print("=" * 70)

sample_columns = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "runs",
    "extras_type",
    "extras1_runs",
    "extras_2_runs"
]

for extras_type, group in balls.groupby(
    "extras_type", dropna=False
):
    print("\nExtras type:", extras_type)
    display(group[sample_columns].head(5))

In [ ]:
# CELL 11 — Run Accounting and Delivery Numbering

balls = datasets["ball_by_ball"]

print("=" * 70)
print("COMPLETE BALL-BY-BALL COLUMN LIST")
print("=" * 70)

for i, column in enumerate(balls.columns, start=1):
    print(f"{i:02d}. {column}")

print("\n" + "=" * 70)
print("FIRST 25 DELIVERIES OF ONE MATCH")
print("=" * 70)

match_id = balls["match_id"].iloc[0]

print("Match ID:", match_id)

display(
    balls.loc[
        balls["match_id"] == match_id,
        [
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].head(25)
)

print("\n" + "=" * 70)
print("EXTRAS CODE CONSISTENCY")
print("=" * 70)

display(
    pd.crosstab(
        balls["extras1_runs"],
        balls["extras_type"].fillna("No extras")
    )
)

In [ ]:
# CELL 12 — Legal Delivery and Ball Numbering Validation

balls = datasets["ball_by_ball"]

print("=" * 70)
print("LEGAL VS ILLEGAL DELIVERY COUNTS")
print("=" * 70)

illegal_delivery = balls["extras_type"].isin(
    ["wide", "Overstep"]
)

print("Total delivery records:", len(balls))
print("Wides:", (balls["extras_type"] == "wide").sum())
print("No-balls:", (balls["extras_type"] == "Overstep").sum())
print("Provisional legal deliveries:", (~illegal_delivery).sum())

print("\n" + "=" * 70)
print("SAMPLE OVERS CONTAINING WIDES OR NO-BALLS")
print("=" * 70)

sample_overs = (
    balls.loc[
        illegal_delivery,
        ["match_id", "innings", "overid"]
    ]
    .drop_duplicates()
    .head(3)
)

for _, over in sample_overs.iterrows():

    print(
        f"\nMatch: {over['match_id']}, "
        f"Innings: {over['innings']}, "
        f"Over: {over['overid']}"
    )

    over_data = balls.loc[
        (balls["match_id"] == over["match_id"]) &
        (balls["innings"] == over["innings"]) &
        (balls["overid"] == over["overid"])
    ]

    display(
        over_data[
            [
                "over_ball_num",
                "innings_ball_num",
                "runs",
                "extras_type",
                "extras_2_runs"
            ]
        ]
    )

print("\n" + "=" * 70)
print("DELIVERY NUMBERING ANOMALIES")
print("=" * 70)

print(
    "Records with over_ball_num > 6:",
    (balls["over_ball_num"] > 6).sum()
)

print(
    "Records with over_ball_num < 1:",
    (balls["over_ball_num"] < 1).sum()
)

In [ ]:
# CELL 13 — Reconstruct Legal-Ball Counts

balls = datasets["ball_by_ball"]

# Create a separate working copy
balls_work = balls.copy()

# Preserve original row order
balls_work["original_row"] = np.arange(len(balls_work))

# Identify legal deliveries
balls_work["is_legal_ball"] = (
    ~balls_work["extras_type"].isin(["wide", "Overstep"])
).astype(int)

# Count legal deliveries within each innings
balls_work["legal_balls_bowled"] = (
    balls_work.groupby(
        ["match_id", "innings"],
        sort=False
    )["is_legal_ball"].cumsum()
)

# Calculate legal deliveries before each event
balls_work["legal_balls_before"] = (
    balls_work["legal_balls_bowled"]
    - balls_work["is_legal_ball"]
)

print("=" * 70)
print("LEGAL DELIVERY VALIDATION")
print("=" * 70)

print("Total records:", len(balls_work))
print("Legal deliveries:", balls_work["is_legal_ball"].sum())
print("Illegal deliveries:", (balls_work["is_legal_ball"] == 0).sum())

print("\n" + "=" * 70)
print("MATCH 13511 — INNINGS 1 — OVER 8")
print("=" * 70)

display(
    balls_work.loc[
        (balls_work["match_id"] == 13511) &
        (balls_work["innings"] == 1) &
        (balls_work["overid"] == 8),
        [
            "over_ball_num",
            "innings_ball_num",
            "extras_type",
            "is_legal_ball",
            "legal_balls_before",
            "legal_balls_bowled"
        ]
    ]
)

print("\n" + "=" * 70)
print("LEGAL BALLS PER OVER — DISTRIBUTION")
print("=" * 70)

over_summary = (
    balls_work.groupby(
        ["match_id", "innings", "overid"]
    )["is_legal_ball"]
    .sum()
)

display(
    over_summary.value_counts()
    .sort_index()
    .rename_axis("legal_balls")
    .reset_index(name="number_of_overs")
)

In [ ]:
# CELL 14 — Investigate Legal-Ball Anomalies

print("=" * 70)
print("OVERS WITH MORE THAN SIX LEGAL BALLS")
print("=" * 70)

suspicious_overs = (
    balls_work.groupby(
        ["match_id", "innings", "overid"]
    )["is_legal_ball"]
    .sum()
    .reset_index(name="legal_balls")
)

suspicious_overs = suspicious_overs[
    suspicious_overs["legal_balls"] > 6
]

print("Suspicious overs:", len(suspicious_overs))

display(suspicious_overs.head(10))

print("\n" + "=" * 70)
print("DETAILED INSPECTION — FIRST THREE OVERS")
print("=" * 70)

for _, row in suspicious_overs.head(3).iterrows():

    match_id = row["match_id"]
    innings = row["innings"]
    overid = row["overid"]

    print(
        f"\nMatch: {match_id}, "
        f"Innings: {innings}, "
        f"Over: {overid}"
    )

    display(
        balls_work.loc[
            (balls_work["match_id"] == match_id) &
            (balls_work["innings"] == innings) &
            (balls_work["overid"] == overid),
            [
                "over_ball_num",
                "innings_ball_num",
                "runs",
                "extras_type",
                "extras1_runs",
                "extras_2_runs",
                "is_legal_ball"
            ]
        ]
    )

In [ ]:
# CELL 15 — Investigate "other" Extras

balls = datasets["ball_by_ball"]

print("=" * 70)
print("OTHER EXTRAS INVESTIGATION")
print("=" * 70)

print(
    "Total 'other' deliveries:",
    (balls["extras_type"] == "other").sum()
)

# Recalculate legal balls, provisionally excluding "other"
balls_work["is_legal_ball_v2"] = (
    ~balls_work["extras_type"].isin(
        ["wide", "Overstep", "other"]
    )
).astype(int)

over_validation = (
    balls_work.groupby(
        ["match_id", "innings", "overid"]
    )["is_legal_ball_v2"]
    .sum()
    .reset_index(name="legal_balls")
)

print("\n" + "=" * 70)
print("LEGAL BALLS PER OVER — REVISED DISTRIBUTION")
print("=" * 70)

display(
    over_validation["legal_balls"]
    .value_counts()
    .sort_index()
    .rename_axis("legal_balls")
    .reset_index(name="number_of_overs")
)

print("\n" + "=" * 70)
print("REMAINING OVERS WITH MORE THAN SIX LEGAL BALLS")
print("=" * 70)

remaining_anomalies = over_validation[
    over_validation["legal_balls"] > 6
]

print("Remaining anomalous overs:", len(remaining_anomalies))

display(remaining_anomalies.head(10))

print("\n" + "=" * 70)
print("OTHER DELIVERIES — RUN DISTRIBUTION")
print("=" * 70)

display(
    balls.loc[
        balls["extras_type"] == "other",
        "runs"
    ].value_counts().sort_index()
)

In [ ]:
# CELL 16 — Investigate Final Legal-Ball Anomaly

print("=" * 70)
print("FINAL ANOMALOUS OVER INVESTIGATION")
print("=" * 70)

match_id = 41509
innings = 1
overid = 12

anomaly = balls_work.loc[
    (balls_work["match_id"] == match_id) &
    (balls_work["innings"] == innings) &
    (balls_work["overid"] == overid)
].copy()

columns_to_show = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "striker_name",
    "bowler_name",
    "runs",
    "extras_type",
    "extras1_runs",
    "extras_2_runs",
    "is_wicket",
    "dismissal_type",
    "is_legal_ball_v2"
]

display(anomaly[columns_to_show])

print("\n" + "=" * 70)
print("DELIVERY COUNTS")
print("=" * 70)

print("Total delivery events:", len(anomaly))
print("Provisional legal deliveries:", anomaly["is_legal_ball_v2"].sum())

print("\nExtras distribution:")
print(anomaly["extras_type"].value_counts(dropna=False))

In [ ]:
# CELL 17 — Innings-Level Legal Delivery Validation

print("=" * 70)
print("INNINGS-LEVEL LEGAL DELIVERY VALIDATION")
print("=" * 70)

innings_summary = (
    balls_work.groupby(
        ["match_id", "innings"]
    )
    .agg(
        total_events=("is_legal_ball_v2", "size"),
        legal_balls=("is_legal_ball_v2", "sum"),
        wides=("extras_type", lambda x: (x == "wide").sum()),
        no_balls=("extras_type", lambda x: (x == "Overstep").sum()),
        other=("extras_type", lambda x: (x == "other").sum())
    )
    .reset_index()
)

print("Total innings:", len(innings_summary))

print("\n" + "=" * 70)
print("LEGAL BALLS PER INNINGS — SUMMARY")
print("=" * 70)

display(
    innings_summary["legal_balls"].describe()
)

print("\n" + "=" * 70)
print("INNINGS EXCEEDING 120 LEGAL BALLS")
print("=" * 70)

long_innings = innings_summary[
    innings_summary["legal_balls"] > 120
]

print("Number of innings:", len(long_innings))

display(long_innings.head(20))

print("\n" + "=" * 70)
print("INNINGS WITH ZERO LEGAL BALLS")
print("=" * 70)

zero_ball_innings = innings_summary[
    innings_summary["legal_balls"] == 0
]

print("Number of innings:", len(zero_ball_innings))

display(zero_ball_innings)

In [ ]:
# CELL 18 — Match Metadata Validation

matches = datasets["matches"]

print("=" * 70)
print("MATCH METADATA COLUMNS")
print("=" * 70)

print(matches.columns.tolist())

print("\n" + "=" * 70)
print("MATCH DATA SAMPLE")
print("=" * 70)

display(matches.head(5))

print("\n" + "=" * 70)
print("INNINGS LENGTH DISTRIBUTION")
print("=" * 70)

innings_summary["length_category"] = pd.cut(
    innings_summary["legal_balls"],
    bins=[0, 30, 60, 90, 119, 120],
    labels=[
        "1-30 balls",
        "31-60 balls",
        "61-90 balls",
        "91-119 balls",
        "120 balls"
    ]
)

display(
    innings_summary["length_category"]
    .value_counts()
    .sort_index()
    .reset_index()
)

print("\n" + "=" * 70)
print("SHORTEST INNINGS")
print("=" * 70)

display(
    innings_summary.sort_values(
        "legal_balls"
    ).head(15)
)

In [ ]:
# CELL 19 — Match-Level Innings Validation

print("=" * 70)
print("INNINGS NUMBER DISTRIBUTION")
print("=" * 70)

display(
    innings_summary["innings"]
    .value_counts()
    .sort_index()
    .rename_axis("innings")
    .reset_index(name="count")
)

print("\n" + "=" * 70)
print("REVISED OVERS — MATCH METADATA")
print("=" * 70)

revised_matches = matches.loc[
    matches[
        [
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].notna().any(axis=1)
].copy()

print("Matches with revision information:", len(revised_matches))

display(
    revised_matches[
        [
            "match_id",
            "match_date",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ].head(20)
)

print("\n" + "=" * 70)
print("SHORTEST REGULAR INNINGS WITH MATCH CONTEXT")
print("=" * 70)

short_regular = innings_summary.loc[
    (innings_summary["innings"].isin([1, 2])) &
    (innings_summary["legal_balls"] < 60)
].copy()

short_regular = short_regular.merge(
    matches[
        [
            "match_id",
            "match_date",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

display(
    short_regular.sort_values(
        "legal_balls"
    ).head(20)
)

In [ ]:
# CELL 20 — Match-State Classification Audit

print("=" * 70)
print("MATCH-STATE CLASSIFICATION AUDIT")
print("=" * 70)

match_audit = matches.copy()

# Identify matches with recorded Super Over innings
super_over_matches = set(
    innings_summary.loc[
        innings_summary["innings"] >= 3,
        "match_id"
    ]
)

# Identify matches with revised-over or target metadata
revision_columns = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

match_audit["has_revision"] = (
    match_audit[revision_columns]
    .notna()
    .any(axis=1)
)

match_audit["has_super_over"] = (
    match_audit["match_id"].isin(super_over_matches)
)

# Identify abandoned / no-result matches
result_text = (
    match_audit["result"]
    .fillna("")
    .astype(str)
    .str.lower()
)

match_audit["is_incomplete_result"] = (
    result_text.str.contains(
        r"no result|abandon",
        regex=True
    )
)

# Assign preliminary categories
conditions = [
    match_audit["is_incomplete_result"],
    match_audit["has_revision"],
    match_audit["has_super_over"]
]

choices = [
    "No Result / Abandoned",
    "Revised Match",
    "Super Over Match"
]

match_audit["match_category"] = np.select(
    conditions,
    choices,
    default="Standard Match"
)

print("\nMATCH CATEGORY DISTRIBUTION")
display(
    match_audit["match_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="matches")
)

print("\n" + "=" * 70)
print("OVERLAPPING MATCH CONDITIONS")
print("=" * 70)

display(
    pd.crosstab(
        match_audit["has_revision"],
        [
            match_audit["has_super_over"],
            match_audit["is_incomplete_result"]
        ]
    )
)

print("\n" + "=" * 70)
print("MATCHES WITH INCOMPLETE RESULTS")
print("=" * 70)

display(
    match_audit.loc[
        match_audit["is_incomplete_result"],
        [
            "match_id",
            "match_date",
            "result",
            "has_revision",
            "has_super_over"
        ]
    ].head(20)
)

print("\n" + "=" * 70)
print("SUPER OVER MATCHES")
print("=" * 70)

display(
    match_audit.loc[
        match_audit["has_super_over"],
        [
            "match_id",
            "match_date",
            "result",
            "has_revision"
        ]
    ]
)

In [ ]:
# CELL 21 — Match and Delivery Coverage Validation

print("=" * 70)
print("MATCH AND DELIVERY COVERAGE VALIDATION")
print("=" * 70)

metadata_ids = set(matches["match_id"].dropna())
delivery_ids = set(balls_work["match_id"].dropna())

print("Matches in metadata:", len(metadata_ids))
print("Matches with deliveries:", len(delivery_ids))

print(
    "Metadata matches without deliveries:",
    len(metadata_ids - delivery_ids)
)

print(
    "Delivery matches missing metadata:",
    len(delivery_ids - metadata_ids)
)

print("\n" + "=" * 70)
print("INNINGS COVERAGE BY MATCH CATEGORY")
print("=" * 70)

coverage = innings_summary.merge(
    match_audit[
        [
            "match_id",
            "match_category",
            "has_revision",
            "has_super_over",
            "is_incomplete_result"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

display(
    coverage.groupby(
        ["match_category", "innings"],
        dropna=False
    ).agg(
        innings_count=("match_id", "size"),
        total_legal_balls=("legal_balls", "sum"),
        min_legal_balls=("legal_balls", "min"),
        median_legal_balls=("legal_balls", "median"),
        max_legal_balls=("legal_balls", "max")
    ).reset_index()
)

print("\n" + "=" * 70)
print("NUMBER OF REGULAR INNINGS PER MATCH")
print("=" * 70)

regular_innings_count = (
    coverage.loc[coverage["innings"].isin([1, 2])]
    .groupby("match_id")["innings"]
    .nunique()
)

display(
    regular_innings_count
    .value_counts()
    .sort_index()
    .rename_axis("regular_innings")
    .reset_index(name="matches")
)

print("\n" + "=" * 70)
print("MATCHES WITH ONLY ONE REGULAR INNINGS")
print("=" * 70)

one_innings_ids = regular_innings_count[
    regular_innings_count == 1
].index

display(
    match_audit.loc[
        match_audit["match_id"].isin(one_innings_ids),
        [
            "match_id",
            "match_date",
            "match_category",
            "result"
        ]
    ].head(20)
)

In [ ]:
# CELL 22 — Investigate Matches Without Delivery Records

print("=" * 70)
print("MATCHES WITHOUT BALL-BY-BALL RECORDS")
print("=" * 70)

metadata_ids = set(matches["match_id"].dropna())
delivery_ids = set(balls_work["match_id"].dropna())

missing_delivery_ids = metadata_ids - delivery_ids
orphan_delivery_ids = delivery_ids - metadata_ids

print("Metadata matches:", len(metadata_ids))
print("Matches with deliveries:", len(delivery_ids))
print("Matches without deliveries:", len(missing_delivery_ids))
print("Delivery matches without metadata:", len(orphan_delivery_ids))

missing_matches = match_audit.loc[
    match_audit["match_id"].isin(missing_delivery_ids),
    [
        "match_id",
        "match_date",
        "match_category",
        "result",
        "has_revision"
    ]
].sort_values("match_date")

display(missing_matches)

print("\n" + "=" * 70)
print("MISSING MATCHES BY CATEGORY")
print("=" * 70)

display(
    missing_matches["match_category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="matches")
)

print("\n" + "=" * 70)
print("ORPHAN DELIVERY MATCH IDS")
print("=" * 70)

print(sorted(orphan_delivery_ids))

In [ ]:
# CELL 23 — Chronological Delivery Sequence Audit

print("=" * 70)
print("CHRONOLOGICAL DELIVERY SEQUENCE AUDIT")
print("=" * 70)

sequence = balls_work.copy()

# Preserve existing source order
sequence["source_position"] = np.arange(len(sequence))

group_cols = ["match_id", "innings"]

# Detect innings split across non-contiguous source rows
innings_key = list(
    zip(sequence["match_id"], sequence["innings"])
)

segment_start = pd.Series(innings_key).ne(
    pd.Series(innings_key).shift()
)

segment_count = (
    pd.DataFrame({
        "match_id": sequence["match_id"].to_numpy(),
        "innings": sequence["innings"].to_numpy(),
        "segment_start": segment_start.to_numpy()
    })
    .groupby(group_cols)["segment_start"]
    .sum()
)

print(
    "Innings split across source segments:",
    int((segment_count > 1).sum())
)

# Check delivery-event numbering
sequence["previous_innings_ball"] = (
    sequence.groupby(group_cols, sort=False)[
        "innings_ball_num"
    ].shift()
)

sequence["event_number_gap"] = (
    sequence["innings_ball_num"]
    - sequence["previous_innings_ball"]
)

numbering_anomalies = sequence.loc[
    sequence["previous_innings_ball"].notna()
    & sequence["event_number_gap"].ne(1)
]

print(
    "Non-consecutive innings-ball transitions:",
    len(numbering_anomalies)
)

display(
    numbering_anomalies[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "previous_innings_ball",
            "event_number_gap"
        ]
    ].head(15)
)

print("\n" + "=" * 70)
print("OVER NUMBER SEQUENCE CHECK")
print("=" * 70)

sequence["previous_over"] = (
    sequence.groupby(group_cols, sort=False)[
        "overid"
    ].shift()
)

over_reversals = sequence.loc[
    sequence["previous_over"].notna()
    & sequence["overid"].lt(sequence["previous_over"])
]

print("Over-number reversals:", len(over_reversals))

display(
    over_reversals[
        [
            "match_id",
            "innings",
            "overid",
            "previous_over",
            "innings_ball_num"
        ]
    ].head(15)
)

print("\n" + "=" * 70)
print("ORPHAN DELIVERY MATCH IDS")
print("=" * 70)

print(sorted(orphan_delivery_ids))

In [ ]:
# CELL 24 — Run Accounting Investigation

print("=" * 70)
print("RUN ACCOUNTING INVESTIGATION")
print("=" * 70)

# Examine run values by extras category
run_audit = balls_work.copy()

run_audit["extras_category"] = (
    run_audit["extras_type"].fillna("No Extras")
)

print("\n1. RUNS BY EXTRAS CATEGORY")
print("=" * 70)

display(
    run_audit.groupby("extras_category")
    .agg(
        deliveries=("runs", "size"),
        total_runs_field=("runs", "sum"),
        mean_runs=("runs", "mean"),
        total_extras_2=("extras_2_runs", "sum"),
        nonzero_extras_2=(
            "extras_2_runs",
            lambda x: (x.fillna(0) != 0).sum()
        )
    )
    .reset_index()
)

print("\n2. RUNS DISTRIBUTION BY EXTRAS CATEGORY")
print("=" * 70)

display(
    pd.crosstab(
        run_audit["extras_category"],
        run_audit["runs"]
    )
)

print("\n3. EXTRAS_2_RUNS DISTRIBUTION")
print("=" * 70)

display(
    run_audit["extras_2_runs"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n4. SAMPLE EXTRAS DELIVERIES")
print("=" * 70)

display(
    run_audit.loc[
        run_audit["extras_type"].notna(),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].groupby(
        "extras_type",
        group_keys=False
    ).head(5)
)

In [ ]:
# CELL 25 — Extras Scoring Structure Investigation

print("=" * 70)
print("EXTRAS SCORING STRUCTURE INVESTIGATION")
print("=" * 70)

extras_rows = balls_work.loc[
    balls_work["extras_type"].notna()
].copy()

print("\n1. EXTRAS TYPE × EXTRAS_2_RUNS")
print("=" * 70)

display(
    pd.crosstab(
        extras_rows["extras_type"],
        extras_rows["extras_2_runs"],
        dropna=False
    )
)

print("\n2. NONZERO EXTRAS_2_RUNS — ALL RECORDS")
print("=" * 70)

display(
    balls_work.loc[
        balls_work["extras_2_runs"].fillna(0).ne(0),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ]
)

print("\n3. POTENTIAL SCORING-RELATED COLUMNS")
print("=" * 70)

keywords = [
    "run", "score", "extra",
    "total", "target", "wide",
    "bye", "penalty"
]

for name, df in datasets.items():
    relevant = [
        col for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in keywords
        )
    ]

    print(f"\n{name}:")
    print(relevant)

print("\n4. EXTRAS CATEGORY COUNTS")
print("=" * 70)

display(
    extras_rows["extras_type"]
    .value_counts(dropna=False)
    .rename_axis("extras_type")
    .reset_index(name="deliveries")
)

In [ ]:
# CELL 26 — Team-Score Reconstruction Feasibility Audit

print("=" * 70)
print("TEAM-SCORE RECONSTRUCTION FEASIBILITY")
print("=" * 70)

score_audit = balls_work.copy()

# Runs explicitly recorded for the batter
score_audit["recorded_batter_runs"] = (
    pd.to_numeric(
        score_audit["runs"],
        errors="coerce"
    )
)

# Extras categories whose run amounts are not explicitly known
uncertain_categories = [
    "wide",
    "byes",
    "legbyes",
    "Overstep",
    "other",
    "Penalty"
]

score_audit["has_uncertain_extras"] = (
    score_audit["extras_type"].isin(
        uncertain_categories
    )
)

print("\n1. DELIVERY-LEVEL SCORING COVERAGE")
print("=" * 70)

total_deliveries = len(score_audit)

uncertain_deliveries = int(
    score_audit["has_uncertain_extras"].sum()
)

print("Total deliveries:", total_deliveries)
print("Deliveries with extras:", uncertain_deliveries)

print(
    "Percentage involving extras:",
    round(
        100 * uncertain_deliveries / total_deliveries,
        2
    )
)

print("\n2. INNINGS AFFECTED BY UNCERTAIN EXTRAS")
print("=" * 70)

innings_score_audit = (
    score_audit.groupby(
        ["match_id", "innings"]
    )
    .agg(
        deliveries=("runs", "size"),
        recorded_batter_runs=(
            "recorded_batter_runs", "sum"
        ),
        uncertain_extra_events=(
            "has_uncertain_extras", "sum"
        )
    )
    .reset_index()
)

innings_score_audit["has_uncertainty"] = (
    innings_score_audit[
        "uncertain_extra_events"
    ] > 0
)

display(
    innings_score_audit["has_uncertainty"]
    .value_counts()
    .rename_axis("has_uncertainty")
    .reset_index(name="innings")
)

print("\n3. UNCERTAIN EXTRAS PER INNINGS")
print("=" * 70)

display(
    innings_score_audit[
        "uncertain_extra_events"
    ].describe()
)

print("\n4. INNINGS WITH NO RECORDED EXTRAS")
print("=" * 70)

display(
    innings_score_audit.loc[
        ~innings_score_audit["has_uncertainty"]
    ].head(15)
)

print("\n5. INNINGS WITH MOST UNCERTAIN EXTRA EVENTS")
print("=" * 70)

display(
    innings_score_audit.sort_values(
        "uncertain_extra_events",
        ascending=False
    ).head(15)
)


In [ ]:
# CELL 27 — Extras and Strike Rotation Investigation

print("=" * 70)
print("EXTRAS AND STRIKE ROTATION INVESTIGATION")
print("=" * 70)

rotation = balls_work.copy()

group_cols = ["match_id", "innings"]

# Identify the next recorded delivery
rotation["next_striker_id"] = (
    rotation.groupby(group_cols, sort=False)[
        "striker_id"
    ].shift(-1)
)

rotation["next_non_striker_id"] = (
    rotation.groupby(group_cols, sort=False)[
        "non_striker_id"
    ].shift(-1)
)

rotation["next_overid"] = (
    rotation.groupby(group_cols, sort=False)[
        "overid"
    ].shift(-1)
)

# Compare only consecutive deliveries within the same over
rotation["same_over_next"] = (
    rotation["overid"] == rotation["next_overid"]
)

rotation["striker_changed"] = (
    rotation["striker_id"] != rotation["next_striker_id"]
)

rotation["batters_swapped"] = (
    (rotation["striker_id"] == rotation["next_non_striker_id"]) &
    (rotation["non_striker_id"] == rotation["next_striker_id"])
)

extras_rotation = rotation.loc[
    rotation["extras_type"].notna() &
    rotation["same_over_next"]
].copy()

print("\n1. STRIKE ROTATION BY EXTRAS TYPE")
print("=" * 70)

display(
    extras_rotation.groupby("extras_type")
    .agg(
        deliveries=("match_id", "size"),
        striker_changed=("striker_changed", "sum"),
        batters_swapped=("batters_swapped", "sum")
    )
    .reset_index()
)

print("\n2. BATTER SWAPS BY EXTRAS TYPE")
print("=" * 70)

display(
    pd.crosstab(
        extras_rotation["extras_type"],
        extras_rotation["batters_swapped"]
    )
)

print("\n3. SAMPLE BYES AND LEG-BYES")
print("=" * 70)

display(
    extras_rotation.loc[
        extras_rotation["extras_type"].isin(
            ["byes", "legbyes"]
        ),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "runs",
            "extras_type",
            "striker_changed",
            "batters_swapped"
        ]
    ].head(20)
)

print("\n4. SAMPLE WIDES")
print("=" * 70)

display(
    extras_rotation.loc[
        extras_rotation["extras_type"] == "wide",
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "runs",
            "striker_changed",
            "batters_swapped"
        ]
    ].head(20)
)

In [ ]:
# CELL 28 — Extras Scoring Sensitivity Audit

print("=" * 70)
print("EXTRAS SCORING SENSITIVITY AUDIT")
print("=" * 70)

scoring = rotation.copy()

scoring["batter_runs"] = pd.to_numeric(
    scoring["runs"],
    errors="coerce"
)

# Minimum plausible extras contribution
minimum_extras = {
    "wide": 1,
    "Overstep": 1,
    "other": 0,
    "byes": 0,
    "legbyes": 0,
    "Penalty": 0
}

scoring["minimum_extras"] = (
    scoring["extras_type"]
    .map(minimum_extras)
    .fillna(0)
)

scoring["minimum_team_runs"] = (
    scoring["batter_runs"] +
    scoring["minimum_extras"]
)

# Identify swaps within the same over
scoring["confirmed_swap"] = (
    scoring["same_over_next"] &
    scoring["batters_swapped"]
)

# Exploratory scenario: minimum odd-run contribution
scoring["rotation_adjustment"] = 0

bye_or_legbye_swap = (
    scoring["extras_type"].isin(
        ["byes", "legbyes"]
    ) &
    scoring["confirmed_swap"]
)

wide_swap = (
    scoring["extras_type"].eq("wide") &
    scoring["confirmed_swap"]
)

scoring.loc[
    bye_or_legbye_swap,
    "rotation_adjustment"
] = 1

scoring.loc[
    wide_swap,
    "rotation_adjustment"
] = 1

scoring["rotation_scenario_runs"] = (
    scoring["minimum_team_runs"] +
    scoring["rotation_adjustment"]
)

print("\n1. SCORING SCENARIO TOTALS")
print("=" * 70)

print(
    "Recorded batter runs:",
    scoring["batter_runs"].sum()
)

print(
    "Minimum scenario team runs:",
    scoring["minimum_team_runs"].sum()
)

print(
    "Rotation scenario team runs:",
    scoring["rotation_scenario_runs"].sum()
)

print("\n2. SCENARIO DIFFERENCES BY EXTRAS TYPE")
print("=" * 70)

display(
    scoring.groupby(
        "extras_type",
        dropna=False
    ).agg(
        deliveries=("runs", "size"),
        minimum_runs=("minimum_team_runs", "sum"),
        rotation_runs=("rotation_scenario_runs", "sum"),
        rotation_adjustments=(
            "rotation_adjustment", "sum"
        )
    ).reset_index()
)

print("\n3. INNINGS-LEVEL SCENARIO DIFFERENCES")
print("=" * 70)

scenario_innings = (
    scoring.groupby(
        ["match_id", "innings"]
    ).agg(
        minimum_score=("minimum_team_runs", "sum"),
        rotation_score=(
            "rotation_scenario_runs", "sum"
        )
    )
    .reset_index()
)

scenario_innings["difference"] = (
    scenario_innings["rotation_score"] -
    scenario_innings["minimum_score"]
)

display(
    scenario_innings["difference"].describe()
)

print("\n4. LARGEST SCENARIO DIFFERENCES")
print("=" * 70)

display(
    scenario_innings.sort_values(
        "difference",
        ascending=False
    ).head(15)
)

In [ ]:
# CELL 29 — Validate Strike Rotation Against Batter Runs

print("=" * 70)
print("STRIKE ROTATION RELIABILITY AUDIT")
print("=" * 70)

rotation_validation = rotation.copy()

# Only ordinary deliveries without extras or wickets
ordinary_mask = (
    rotation_validation["extras_type"].isna() &
    rotation_validation["same_over_next"] &
    rotation_validation["is_wicket"].fillna(0).eq(0) &
    rotation_validation["runs"].isin([0, 1, 2, 3, 4, 6])
)

ordinary = rotation_validation.loc[
    ordinary_mask
].copy()

# Expected rotation from batter runs
ordinary["expected_swap"] = (
    ordinary["runs"] % 2 == 1
)

ordinary["rotation_matches"] = (
    ordinary["expected_swap"] ==
    ordinary["batters_swapped"]
)

print("\n1. OVERALL ROTATION CONSISTENCY")
print("=" * 70)

print("Deliveries tested:", len(ordinary))

print(
    "Consistent deliveries:",
    int(ordinary["rotation_matches"].sum())
)

print(
    "Consistency percentage:",
    round(
        100 * ordinary["rotation_matches"].mean(),
        2
    )
)

print("\n2. CONSISTENCY BY RECORDED RUNS")
print("=" * 70)

display(
    ordinary.groupby("runs")
    .agg(
        deliveries=("runs", "size"),
        observed_swaps=("batters_swapped", "sum"),
        consistent=("rotation_matches", "sum"),
        consistency_rate=("rotation_matches", "mean")
    )
    .reset_index()
)

print("\n3. EXPECTED VS OBSERVED ROTATION")
print("=" * 70)

display(
    pd.crosstab(
        ordinary["expected_swap"],
        ordinary["batters_swapped"],
        margins=True
    )
)

print("\n4. SAMPLE ROTATION MISMATCHES")
print("=" * 70)

display(
    ordinary.loc[
        ~ordinary["rotation_matches"],
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "non_striker_name",
            "runs",
            "is_wicket",
            "expected_swap",
            "batters_swapped"
        ]
    ].head(20)
)

In [ ]:
# CELL 30 — Validate extras_2_runs Against Strike Rotation

print("=" * 70)
print("EXTRAS_2_RUNS ROTATION VALIDATION")
print("=" * 70)

extras2_audit = rotation.copy()

extras2_audit["extras2_numeric"] = pd.to_numeric(
    extras2_audit["extras_2_runs"],
    errors="coerce"
).fillna(0)

extras2_audit["batter_runs_numeric"] = pd.to_numeric(
    extras2_audit["runs"],
    errors="coerce"
)

# Focus on deliveries with nonzero extras_2_runs
extras2_nonzero = extras2_audit.loc[
    extras2_audit["extras2_numeric"].ne(0)
].copy()

extras2_nonzero["extras2_odd"] = (
    extras2_nonzero["extras2_numeric"] % 2 == 1
)

print("\n1. NONZERO EXTRAS_2_RUNS SUMMARY")
print("=" * 70)

display(
    extras2_nonzero.groupby(
        ["extras_type", "extras2_numeric"],
        dropna=False
    )
    .agg(
        deliveries=("match_id", "size"),
        same_over_next=("same_over_next", "sum"),
        batter_swaps=("batters_swapped", "sum")
    )
    .reset_index()
)

print("\n2. EXTRAS_2_RUNS PARITY VS BATTER SWAPS")
print("=" * 70)

display(
    pd.crosstab(
        extras2_nonzero.loc[
            extras2_nonzero["same_over_next"],
            "extras2_odd"
        ],
        extras2_nonzero.loc[
            extras2_nonzero["same_over_next"],
            "batters_swapped"
        ],
        margins=True
    )
)

print("\n3. ALL NONZERO EXTRAS_2_RUNS RECORDS")
print("=" * 70)

display(
    extras2_nonzero[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "non_striker_name",
            "runs",
            "extras_type",
            "extras_2_runs",
            "same_over_next",
            "batters_swapped"
        ]
    ]
)

print("\n4. ORDINARY ROTATION MISMATCHES WITH EXTRAS_2_RUNS")
print("=" * 70)

display(
    ordinary.loc[
        ~ordinary["rotation_matches"],
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "runs",
            "extras_type",
            "extras_2_runs",
            "expected_swap",
            "batters_swapped"
        ]
    ]
)

In [ ]:
# CELL 31 — Combined Run Fields Rotation Validation

print("=" * 70)
print("COMBINED RUN FIELDS VALIDATION")
print("=" * 70)

combined_audit = rotation.copy()

combined_audit["batter_runs"] = pd.to_numeric(
    combined_audit["runs"],
    errors="coerce"
)

combined_audit["additional_runs"] = pd.to_numeric(
    combined_audit["extras_2_runs"],
    errors="coerce"
).fillna(0)

combined_audit["combined_runs"] = (
    combined_audit["batter_runs"] +
    combined_audit["additional_runs"]
)

# Ordinary deliveries, same over, no recorded wicket
test_mask = (
    combined_audit["extras_type"].isna() &
    combined_audit["same_over_next"] &
    combined_audit["is_wicket"].fillna(0).eq(0) &
    combined_audit["batter_runs"].notna() &
    combined_audit["combined_runs"].notna()
)

combined_test = combined_audit.loc[
    test_mask
].copy()

combined_test["expected_swap_original"] = (
    combined_test["batter_runs"] % 2 == 1
)

combined_test["expected_swap_combined"] = (
    combined_test["combined_runs"] % 2 == 1
)

combined_test["original_matches"] = (
    combined_test["expected_swap_original"] ==
    combined_test["batters_swapped"]
)

combined_test["combined_matches"] = (
    combined_test["expected_swap_combined"] ==
    combined_test["batters_swapped"]
)

print("\n1. ORIGINAL VS COMBINED CONSISTENCY")
print("=" * 70)

print("Deliveries tested:", len(combined_test))

print(
    "Original mismatches:",
    int((~combined_test["original_matches"]).sum())
)

print(
    "Combined mismatches:",
    int((~combined_test["combined_matches"]).sum())
)

print(
    "Original consistency (%):",
    round(
        100 * combined_test["original_matches"].mean(),
        4
    )
)

print(
    "Combined consistency (%):",
    round(
        100 * combined_test["combined_matches"].mean(),
        4
    )
)

print("\n2. CHANGES IN ROTATION CLASSIFICATION")
print("=" * 70)

display(
    pd.crosstab(
        combined_test["original_matches"],
        combined_test["combined_matches"],
        margins=True
    )
)

print("\n3. REMAINING COMBINED MISMATCHES")
print("=" * 70)

display(
    combined_test.loc[
        ~combined_test["combined_matches"],
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "runs",
            "extras_type",
            "extras_2_runs",
            "combined_runs",
            "batters_swapped"
        ]
    ].head(40)
)

print("\n4. TOTAL ADDITIONAL RECORDED RUNS")
print("=" * 70)

print(
    "Sum of runs:",
    combined_audit["batter_runs"].sum()
)

print(
    "Sum of extras_2_runs:",
    combined_audit["additional_runs"].sum()
)

print(
    "Sum of combined runs:",
    combined_audit["combined_runs"].sum()
)

In [ ]:
# CELL 32 — Validate "other" Extras Classification

print("=" * 70)
print('VALIDATING "OTHER" EXTRAS CLASSIFICATION')
print("=" * 70)

other_audit = rotation.copy()

other_audit["is_other"] = (
    other_audit["extras_type"].eq("other")
)

print("\n1. OTHER EVENTS BY RECORDED RUNS")
print("=" * 70)

display(
    pd.crosstab(
        other_audit.loc[
            other_audit["is_other"],
            "runs"
        ],
        other_audit.loc[
            other_audit["is_other"],
            "extras_2_runs"
        ]
    )
)

print("\n2. OTHER EVENTS AND BATTER ROTATION")
print("=" * 70)

other_same_over = other_audit.loc[
    other_audit["is_other"] &
    other_audit["same_over_next"]
].copy()

other_same_over["combined_runs"] = (
    pd.to_numeric(
        other_same_over["runs"],
        errors="coerce"
    ) +
    pd.to_numeric(
        other_same_over["extras_2_runs"],
        errors="coerce"
    ).fillna(0)
)

other_same_over["expected_swap"] = (
    other_same_over["combined_runs"] % 2 == 1
)

display(
    pd.crosstab(
        other_same_over["expected_swap"],
        other_same_over["batters_swapped"],
        margins=True
    )
)

print("\n3. OVERS CONTAINING OTHER EVENTS")
print("=" * 70)

over_other_audit = (
    other_audit.groupby(
        ["match_id", "innings", "overid"]
    )
    .agg(
        total_events=("match_id", "size"),
        other_events=("is_other", "sum"),
        wides=(
            "extras_type",
            lambda x: (x == "wide").sum()
        ),
        no_balls=(
            "extras_type",
            lambda x: (x == "Overstep").sum()
        )
    )
    .reset_index()
)

over_other_audit["legal_if_other_legal"] = (
    over_other_audit["total_events"] -
    over_other_audit["wides"] -
    over_other_audit["no_balls"]
)

over_other_audit["legal_if_other_illegal"] = (
    over_other_audit["legal_if_other_legal"] -
    over_other_audit["other_events"]
)

affected_overs = over_other_audit.loc[
    over_other_audit["other_events"] > 0
].copy()

display(
    affected_overs[
        [
            "legal_if_other_legal",
            "legal_if_other_illegal"
        ]
    ].describe()
)

print("\n4. LEGAL-BALL COUNTS IN AFFECTED OVERS")
print("=" * 70)

display(
    pd.crosstab(
        affected_overs["legal_if_other_legal"],
        affected_overs["legal_if_other_illegal"]
    )
)

print("\n5. EXAMPLE OVERS CONTAINING OTHER EVENTS")
print("=" * 70)

display(
    affected_overs.head(20)
)

In [ ]:
# CELL 33 — Finalize Legal-Ball Progression

print("=" * 70)
print("LEGAL-BALL PROGRESSION AUDIT")
print("=" * 70)

pressure_base = balls_work.copy()

# Working classification supported by previous audits
illegal_categories = [
    "wide",
    "Overstep",
    "other"
]

pressure_base["is_legal_delivery"] = (
    ~pressure_base["extras_type"].isin(
        illegal_categories
    )
).astype(int)

group_cols = ["match_id", "innings"]

# Legal balls completed through the current event
pressure_base["legal_balls_after"] = (
    pressure_base.groupby(
        group_cols,
        sort=False
    )["is_legal_delivery"].cumsum()
)

# Legal balls completed before the current event
pressure_base["legal_balls_before"] = (
    pressure_base["legal_balls_after"] -
    pressure_base["is_legal_delivery"]
)

# Ball progression within each over
pressure_base["legal_balls_in_over"] = (
    pressure_base.groupby(
        ["match_id", "innings", "overid"],
        sort=False
    )["is_legal_delivery"].cumsum()
)

print("\n1. OVERALL LEGAL DELIVERY COUNTS")
print("=" * 70)

display(
    pressure_base["is_legal_delivery"]
    .value_counts()
    .rename_axis("is_legal_delivery")
    .reset_index(name="deliveries")
)

print("\n2. INNINGS LEGAL-BALL TOTALS")
print("=" * 70)

innings_progression = (
    pressure_base.groupby(group_cols)
    .agg(
        delivery_events=("match_id", "size"),
        legal_balls=("is_legal_delivery", "sum")
    )
    .reset_index()
)

display(
    innings_progression["legal_balls"].describe()
)

print("\n3. OVERS WITH MORE THAN SIX LEGAL BALLS")
print("=" * 70)

over_progression = (
    pressure_base.groupby(
        ["match_id", "innings", "overid"]
    )
    .agg(
        legal_balls=("is_legal_delivery", "sum"),
        events=("match_id", "size")
    )
    .reset_index()
)

anomalous_overs = over_progression.loc[
    over_progression["legal_balls"] > 6
].copy()

display(anomalous_overs)

print("\n4. PROGRESSION CONSISTENCY")
print("=" * 70)

pressure_base["previous_legal_after"] = (
    pressure_base.groupby(
        group_cols,
        sort=False
    )["legal_balls_after"].shift(1)
)

pressure_base["progression_consistent"] = (
    pressure_base["previous_legal_after"].isna() |
    (
        pressure_base["legal_balls_before"] ==
        pressure_base["previous_legal_after"]
    )
)

print(
    "Progression mismatches:",
    int(
        (~pressure_base["progression_consistent"]).sum()
    )
)

print("\n5. SAMPLE LEGAL-BALL TIMELINE")
print("=" * 70)

sample_match = pressure_base["match_id"].iloc[0]

display(
    pressure_base.loc[
        (pressure_base["match_id"] == sample_match) &
        (pressure_base["innings"] == 1),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "extras_type",
            "is_legal_delivery",
            "legal_balls_before",
            "legal_balls_after"
        ]
    ].head(30)
)

In [ ]:
# CELL 34 — Wicket Accounting and Match-State Validation

print("=" * 70)
print("WICKET ACCOUNTING AND MATCH-STATE VALIDATION")
print("=" * 70)

wicket_audit = pressure_base.copy()

# Examine the original wicket flag
print("\n1. ORIGINAL WICKET FLAG DISTRIBUTION")
print("=" * 70)

print("Original dtype:", wicket_audit["is_wicket"].dtype)

display(
    wicket_audit["is_wicket"].value_counts(dropna=False)
)

# Create a working wicket-event flag
wicket_audit["wicket_flag"] = (
    wicket_audit["is_wicket"]
    .isin([True, 1, "True", "true", "1"])
)

wicket_audit["has_dismissal_type"] = (
    wicket_audit["dismissal_type"].notna()
)

wicket_audit["has_dismissed_player"] = (
    wicket_audit["dismissed_player_id"].notna()
)

wicket_audit["has_dismissal_record"] = (
    wicket_audit["has_dismissal_type"] |
    wicket_audit["has_dismissed_player"]
)

print("\n2. WICKET FLAG VS DISMISSAL RECORDS")
print("=" * 70)

display(
    pd.crosstab(
        wicket_audit["wicket_flag"],
        wicket_audit["has_dismissal_record"],
        margins=True
    )
)

print("\n3. WICKET RECORD INCONSISTENCIES")
print("=" * 70)

wicket_audit["wicket_record_mismatch"] = (
    wicket_audit["wicket_flag"] !=
    wicket_audit["has_dismissal_record"]
)

print(
    "Total mismatches:",
    int(wicket_audit["wicket_record_mismatch"].sum())
)

display(
    wicket_audit.loc[
        wicket_audit["wicket_record_mismatch"],
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "is_wicket",
            "dismissal_type",
            "dismissed_player_id",
            "extras_type"
        ]
    ].head(25)
)

print("\n4. WICKETS PER INNINGS")
print("=" * 70)

innings_wicket_audit = (
    wicket_audit.groupby(
        ["match_id", "innings"]
    )
    .agg(
        flagged_wickets=("wicket_flag", "sum"),
        dismissal_records=("has_dismissal_record", "sum")
    )
    .reset_index()
)

display(
    innings_wicket_audit["flagged_wickets"]
    .value_counts()
    .sort_index()
    .rename_axis("wickets")
    .reset_index(name="innings")
)

print("\n5. INNINGS WITH MORE THAN TEN FLAGGED WICKETS")
print("=" * 70)

display(
    innings_wicket_audit.loc[
        innings_wicket_audit["flagged_wickets"] > 10
    ]
)

In [ ]:
# CELL 35 — Investigate Wicket Record Discrepancies

print("=" * 70)
print("WICKET DISCREPANCY INVESTIGATION")
print("=" * 70)

wicket_followup = wicket_audit.copy()

group_cols = ["match_id", "innings"]

# Examine the next recorded delivery in each innings
for col in ["striker_id", "non_striker_id", "overid"]:
    wicket_followup[f"next_{col}"] = (
        wicket_followup.groupby(
            group_cols, sort=False
        )[col].shift(-1)
    )

print("\n1. WICKET FLAG VS DISMISSAL TYPE")
print("=" * 70)

display(
    pd.crosstab(
        wicket_followup["wicket_flag"],
        wicket_followup["has_dismissal_type"],
        margins=True
    )
)

print("\n2. EXAMINE THE 12 DISCREPANCIES")
print("=" * 70)

suspected_wickets = wicket_followup.loc[
    wicket_followup["wicket_record_mismatch"]
].copy()

suspected_wickets["next_delivery_exists"] = (
    suspected_wickets["next_striker_id"].notna() &
    suspected_wickets["next_non_striker_id"].notna()
)

suspected_wickets["dismissed_was_batting"] = (
    suspected_wickets["dismissed_player_id"].eq(
        suspected_wickets["striker_id"]
    ) |
    suspected_wickets["dismissed_player_id"].eq(
        suspected_wickets["non_striker_id"]
    )
)

suspected_wickets["dismissed_present_next"] = (
    suspected_wickets["dismissed_player_id"].eq(
        suspected_wickets["next_striker_id"]
    ) |
    suspected_wickets["dismissed_player_id"].eq(
        suspected_wickets["next_non_striker_id"]
    )
)

suspected_wickets["new_batter_next"] = (
    (
        ~suspected_wickets["next_striker_id"].eq(
            suspected_wickets["striker_id"]
        ) &
        ~suspected_wickets["next_striker_id"].eq(
            suspected_wickets["non_striker_id"]
        )
    ) |
    (
        ~suspected_wickets["next_non_striker_id"].eq(
            suspected_wickets["striker_id"]
        ) &
        ~suspected_wickets["next_non_striker_id"].eq(
            suspected_wickets["non_striker_id"]
        )
    )
)

# Do not infer continuity where no next delivery exists
suspected_wickets.loc[
    ~suspected_wickets["next_delivery_exists"],
    ["dismissed_present_next", "new_batter_next"]
] = pd.NA

display(
    suspected_wickets[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "dismissed_player_id",
            "dismissed_was_batting",
            "next_delivery_exists",
            "dismissed_present_next",
            "new_batter_next"
        ]
    ]
)

print("\n3. DISCREPANCY CONTINUITY SUMMARY")
print("=" * 70)

for col in [
    "dismissed_was_batting",
    "next_delivery_exists",
    "dismissed_present_next",
    "new_batter_next"
]:
    print(f"\n{col}:")
    display(
        suspected_wickets[col].value_counts(
            dropna=False
        )
    )

print("\n4. POSSIBLE MISSING WICKETS")
print("=" * 70)

possible_missing_wickets = suspected_wickets.loc[
    suspected_wickets["dismissed_was_batting"].eq(True) &
    suspected_wickets["next_delivery_exists"].eq(True) &
    suspected_wickets["dismissed_present_next"].eq(False) &
    suspected_wickets["new_batter_next"].eq(True)
]

display(
    possible_missing_wickets[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "dismissed_player_id",
            "striker_id",
            "non_striker_id",
            "next_striker_id",
            "next_non_striker_id",
            "next_overid"
        ]
    ]
)

In [ ]:
# CELL 36 — Verify Suspected Missing Wickets

print("=" * 70)
print("SUSPECTED MISSING WICKET VERIFICATION")
print("=" * 70)

wicket_sequence = wicket_audit.copy()

wicket_sequence["event_position"] = np.arange(
    len(wicket_sequence)
)

candidates = possible_missing_wickets.copy()

candidates["event_position"] = (
    wicket_sequence.loc[
        candidates.index,
        "event_position"
    ].to_numpy()
)

print("\n1. CHECK WHETHER DISMISSED PLAYERS RETURNED")
print("=" * 70)

return_results = []

for _, row in candidates.iterrows():

    later = wicket_sequence.loc[
        (wicket_sequence["match_id"] == row["match_id"]) &
        (wicket_sequence["innings"] == row["innings"]) &
        (
            wicket_sequence["event_position"] >
            row["event_position"]
        )
    ]

    returned = (
        later["striker_id"].eq(
            row["dismissed_player_id"]
        ).any() |
        later["non_striker_id"].eq(
            row["dismissed_player_id"]
        ).any()
    )

    return_results.append({
        "match_id": row["match_id"],
        "innings": row["innings"],
        "overid": row["overid"],
        "over_ball_num": row["over_ball_num"],
        "striker_name": row["striker_name"],
        "dismissed_player_id": row["dismissed_player_id"],
        "later_deliveries": len(later),
        "player_returned": bool(returned)
    })

return_audit = pd.DataFrame(return_results)

display(return_audit)

print("\n2. RETURN STATUS SUMMARY")
print("=" * 70)

display(
    return_audit["player_returned"]
    .value_counts(dropna=False)
    .rename_axis("player_returned")
    .reset_index(name="records")
)

print("\n3. EFFECT ON INNINGS WICKET TOTALS")
print("=" * 70)

base_wickets = (
    wicket_sequence.groupby(
        ["match_id", "innings"]
    )
    .agg(
        recorded_wickets=("wicket_flag", "sum")
    )
    .reset_index()
)

candidate_counts = (
    candidates.groupby(
        ["match_id", "innings"]
    )
    .size()
    .reset_index(name="suspected_missing_wickets")
)

wicket_impact = base_wickets.merge(
    candidate_counts,
    on=["match_id", "innings"],
    how="left",
    validate="one_to_one"
)

wicket_impact["suspected_missing_wickets"] = (
    wicket_impact["suspected_missing_wickets"]
    .fillna(0)
    .astype(int)
)

wicket_impact["wickets_if_added"] = (
    wicket_impact["recorded_wickets"] +
    wicket_impact["suspected_missing_wickets"]
)

display(
    wicket_impact.loc[
        wicket_impact["suspected_missing_wickets"] > 0
    ]
)

print("\n4. POSSIBLE WICKET COUNT VIOLATIONS")
print("=" * 70)

print(
    "Innings exceeding 10 wickets:",
    int(
        (
            wicket_impact["wickets_if_added"] > 10
        ).sum()
    )
)

display(
    wicket_impact.loc[
        wicket_impact["wickets_if_added"] > 10
    ]
)

In [ ]:
# CELL 38 — Construct Wicket Match-State Timeline

print("=" * 70)
print("WICKET MATCH-STATE TIMELINE")
print("=" * 70)

wicket_state = pressure_base.copy()

group_cols = ["match_id", "innings"]

# Use original wicket flag; do not count unflagged departures
wicket_state["is_confirmed_wicket"] = (
    wicket_state["is_wicket"].eq(True).astype(int)
)

# Wickets after the current delivery
wicket_state["wickets_after"] = (
    wicket_state.groupby(
        group_cols,
        sort=False
    )["is_confirmed_wicket"].cumsum()
)

# Wickets before the current delivery
wicket_state["wickets_before"] = (
    wicket_state["wickets_after"] -
    wicket_state["is_confirmed_wicket"]
)

wicket_state["wickets_remaining_before"] = (
    10 - wicket_state["wickets_before"]
)

wicket_state["wickets_remaining_after"] = (
    10 - wicket_state["wickets_after"]
)

# Preserve departure anomalies without converting them to wickets
wicket_state["unflagged_departure"] = (
    wicket_state["dismissed_player_id"].notna() &
    wicket_state["dismissal_type"].isna() &
    ~wicket_state["is_wicket"]
)

print("\n1. OVERALL WICKET ACCOUNTING")
print("=" * 70)

print(
    "Confirmed wicket events:",
    int(wicket_state["is_confirmed_wicket"].sum())
)

print(
    "Unflagged departure records:",
    int(wicket_state["unflagged_departure"].sum())
)

print("\n2. WICKET-STATE VALIDITY")
print("=" * 70)

print(
    "Negative wickets remaining:",
    int(
        (
            wicket_state["wickets_remaining_after"] < 0
        ).sum()
    )
)

print(
    "Wickets above 10:",
    int(
        (
            wicket_state["wickets_after"] > 10
        ).sum()
    )
)

print(
    "Wicket count below zero:",
    int(
        (
            wicket_state["wickets_before"] < 0
        ).sum()
    )
)

print("\n3. INNINGS-LEVEL WICKET DISTRIBUTION")
print("=" * 70)

innings_wicket_state = (
    wicket_state.groupby(group_cols)
    .agg(
        wickets=("is_confirmed_wicket", "sum"),
        unflagged_departures=("unflagged_departure", "sum")
    )
    .reset_index()
)

display(
    innings_wicket_state["wickets"]
    .value_counts()
    .sort_index()
    .rename_axis("wickets")
    .reset_index(name="innings")
)

print("\n4. WICKET PROGRESSION CONSISTENCY")
print("=" * 70)

wicket_state["previous_wickets_after"] = (
    wicket_state.groupby(
        group_cols,
        sort=False
    )["wickets_after"].shift(1)
)

progression_valid = (
    wicket_state["previous_wickets_after"].isna() |
    wicket_state["wickets_before"].eq(
        wicket_state["previous_wickets_after"]
    )
)

print(
    "Progression mismatches:",
    int((~progression_valid).sum())
)

print("\n5. KNOWN RETIRED-HURT EXAMPLE")
print("=" * 70)

display(
    wicket_state.loc[
        (wicket_state["match_id"] == 20880) &
        (wicket_state["innings"] == 2) &
        (wicket_state["overid"] == 11),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "is_confirmed_wicket",
            "unflagged_departure",
            "wickets_before",
            "wickets_after",
            "wickets_remaining_before"
        ]
    ]
)

In [ ]:
# CELL 39 — Build the Master Match-State Dataset

print("=" * 70)
print("MASTER MATCH-STATE DATASET")
print("=" * 70)

# Preserve the verified legal-ball and wicket timelines
match_state = wicket_state.copy()

# Add match classifications
match_state = match_state.merge(
    match_audit[
        [
            "match_id",
            "match_category",
            "has_revision",
            "has_super_over",
            "is_incomplete_result"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Flag regular innings
match_state["is_regular_innings"] = (
    match_state["innings"].isin([1, 2])
)

# Preliminary standard-match eligibility
match_state["is_standard_analysis"] = (
    match_state["is_regular_innings"] &
    match_state["match_category"].eq("Standard Match")
)

# Flag entire innings containing over-length anomalies
anomaly_innings = (
    anomalous_overs[
        ["match_id", "innings"]
    ]
    .drop_duplicates()
    .assign(has_over_anomaly=True)
)

match_state = match_state.merge(
    anomaly_innings,
    on=["match_id", "innings"],
    how="left",
    validate="many_to_one"
)

match_state["has_over_anomaly"] = (
    match_state["has_over_anomaly"]
    .fillna(False)
    .astype(bool)
)

print("\n1. MASTER DATASET SIZE")
print("=" * 70)

print("Rows:", len(match_state))
print("Columns:", len(match_state.columns))
print("Unique matches:", match_state["match_id"].nunique())

print("\n2. MATCH CATEGORY COVERAGE")
print("=" * 70)

display(
    match_state.groupby(
        "match_category",
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        deliveries=("match_id", "size"),
        wickets=("is_confirmed_wicket", "sum")
    )
    .reset_index()
)

print("\n3. PRELIMINARY ANALYSIS ELIGIBILITY")
print("=" * 70)

display(
    match_state["is_standard_analysis"]
    .value_counts()
    .rename_axis("is_standard_analysis")
    .reset_index(name="deliveries")
)

print("\n4. MATCH-STATE VALIDITY CHECK")
print("=" * 70)

checks = {
    "missing_match_category": match_state[
        "match_category"
    ].isna().sum(),

    "negative_legal_balls_before": (
        match_state["legal_balls_before"] < 0
    ).sum(),

    "legal_balls_above_120": (
        match_state["legal_balls_after"] > 120
    ).sum(),

    "negative_wickets_remaining": (
        match_state["wickets_remaining_before"] < 0
    ).sum(),

    "wickets_above_10": (
        match_state["wickets_after"] > 10
    ).sum(),

    "innings_with_over_anomaly": (
        match_state.loc[
            match_state["has_over_anomaly"],
            ["match_id", "innings"]
        ].drop_duplicates().shape[0]
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

print("\n5. SAMPLE MASTER MATCH-STATE RECORDS")
print("=" * 70)

display(
    match_state[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "legal_balls_before",
            "wickets_before",
            "wickets_remaining_before",
            "match_category",
            "is_standard_analysis",
            "has_over_anomaly"
        ]
    ].head(20)
)

In [ ]:
# CELL 40 — Match Phases and Balls Remaining

print("=" * 70)
print("MATCH PHASE AND BALLS-REMAINING ENGINE")
print("=" * 70)

# Identify matches with known over-length anomalies
anomaly_match_ids = match_state.loc[
    match_state["has_over_anomaly"],
    "match_id"
].unique()

# Standard T20 innings, excluding known anomaly matches
standard_state = match_state.loc[
    match_state["is_standard_analysis"] &
    ~match_state["match_id"].isin(anomaly_match_ids)
].copy()

# Standard T20 scheduled innings length
standard_state["scheduled_balls"] = 120

# Calculate balls remaining before and after each delivery
standard_state["balls_remaining_before"] = (
    standard_state["scheduled_balls"] -
    standard_state["legal_balls_before"]
)

standard_state["balls_remaining_after"] = (
    standard_state["scheduled_balls"] -
    standard_state["legal_balls_after"]
)

# Calculate completed overs before each delivery
standard_state["overs_completed_before"] = (
    standard_state["legal_balls_before"] / 6
)

# Classify deliveries by match phase
phase_conditions = [
    standard_state["legal_balls_before"] < 36,

    standard_state["legal_balls_before"].between(
        36, 95
    ),

    standard_state["legal_balls_before"].between(
        96, 119
    )
]

phase_labels = [
    "Powerplay",
    "Middle Overs",
    "Death Overs"
]

standard_state["match_phase"] = np.select(
    phase_conditions,
    phase_labels,
    default="Outside Standard Range"
)


# ============================================================
# 1. ELIGIBLE ANALYSIS DATASET
# ============================================================

print("\n1. ELIGIBLE ANALYSIS DATASET")
print("=" * 70)

print(
    "Eligible matches:",
    standard_state["match_id"].nunique()
)

print(
    "Eligible innings:",
    standard_state[
        ["match_id", "innings"]
    ].drop_duplicates().shape[0]
)

print(
    "Eligible deliveries:",
    len(standard_state)
)


# ============================================================
# 2. MATCH PHASE DISTRIBUTION
# ============================================================

print("\n2. MATCH PHASE DISTRIBUTION")
print("=" * 70)

display(
    standard_state.groupby("match_phase")
    .agg(
        deliveries=("match_id", "size"),
        legal_deliveries=("is_legal_delivery", "sum"),
        wickets=("is_confirmed_wicket", "sum")
    )
    .reindex(
        [
            "Powerplay",
            "Middle Overs",
            "Death Overs",
            "Outside Standard Range"
        ]
    )
    .dropna(how="all")
    .reset_index()
)


# ============================================================
# 3. BALLS-REMAINING VALIDATION
# ============================================================

print("\n3. BALLS-REMAINING VALIDATION")
print("=" * 70)

validation = {
    "negative_balls_remaining_before": int(
        (
            standard_state["balls_remaining_before"] < 0
        ).sum()
    ),

    "negative_balls_remaining_after": int(
        (
            standard_state["balls_remaining_after"] < 0
        ).sum()
    ),

    "outside_standard_phase": int(
        (
            standard_state["match_phase"] ==
            "Outside Standard Range"
        ).sum()
    ),

    "wickets_remaining_outside_0_to_10": int(
        (
            ~standard_state[
                "wickets_remaining_before"
            ].between(0, 10)
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(validation.items()),
        columns=["check", "count"]
    )
)


# ============================================================
# 4. MATCH PHASE BOUNDARY EXAMPLES
# ============================================================

print("\n4. MATCH PHASE BOUNDARY EXAMPLES")
print("=" * 70)

boundary_values = [
    0, 35, 36, 95, 96, 119
]

display(
    standard_state.loc[
        standard_state["legal_balls_before"].isin(
            boundary_values
        ),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "legal_balls_before",
            "balls_remaining_before",
            "match_phase",
            "wickets_remaining_before"
        ]
    ]
    .groupby(
        "legal_balls_before",
        group_keys=False
    )
    .head(2)
)


# ============================================================
# 5. SAMPLE INNINGS MATCH-STATE TIMELINE
# ============================================================

print("\n5. SAMPLE INNINGS MATCH-STATE TIMELINE")
print("=" * 70)

sample_match = standard_state["match_id"].iloc[0]

display(
    standard_state.loc[
        (standard_state["match_id"] == sample_match) &
        (standard_state["innings"] == 1),
        [
            "overid",
            "over_ball_num",
            "is_legal_delivery",
            "legal_balls_before",
            "balls_remaining_before",
            "wickets_before",
            "wickets_remaining_before",
            "match_phase"
        ]
    ].tail(20)
)

In [ ]:
# CELL 41 — Recent-Wicket Pressure Features

print("=" * 70)
print("RECENT-WICKET PRESSURE FEATURE ENGINE")
print("=" * 70)

pressure_features = standard_state.copy()

group_cols = ["match_id", "innings"]

# Initialize arrays for recent-wicket counts
recent_6 = np.zeros(len(pressure_features), dtype=int)
recent_12 = np.zeros(len(pressure_features), dtype=int)

# Use existing chronological order within each innings
ball_positions = pressure_features[
    "legal_balls_before"
].to_numpy()

wicket_positions = pressure_features[
    "wickets_before"
].to_numpy()

for _, positions in pressure_features.groupby(
    group_cols, sort=False
).indices.items():

    positions = np.asarray(positions)

    balls = ball_positions[positions]
    wickets = wicket_positions[positions]

    # Ensure each innings remains chronological
    assert np.all(np.diff(balls) >= 0), (
        "Legal-ball order is not chronological."
    )

    # Locate the start of each historical window
    start_6 = np.searchsorted(
        balls,
        np.maximum(0, balls - 6),
        side="left"
    )

    start_12 = np.searchsorted(
        balls,
        np.maximum(0, balls - 12),
        side="left"
    )

    # Wickets lost before the current delivery
    recent_6[positions] = (
        wickets - wickets[start_6]
    )

    recent_12[positions] = (
        wickets - wickets[start_12]
    )

pressure_features["wickets_last_6_balls"] = recent_6
pressure_features["wickets_last_12_balls"] = recent_12

print("\n1. FEATURE DATASET SIZE")
print("=" * 70)

print("Deliveries:", len(pressure_features))

print(
    "Unique matches:",
    pressure_features["match_id"].nunique()
)

print("\n2. WICKETS IN PREVIOUS 6 LEGAL BALLS")
print("=" * 70)

display(
    pressure_features["wickets_last_6_balls"]
    .value_counts()
    .sort_index()
    .rename_axis("recent_wickets")
    .reset_index(name="deliveries")
)

print("\n3. WICKETS IN PREVIOUS 12 LEGAL BALLS")
print("=" * 70)

display(
    pressure_features["wickets_last_12_balls"]
    .value_counts()
    .sort_index()
    .rename_axis("recent_wickets")
    .reset_index(name="deliveries")
)

print("\n4. RECENT WICKET PRESSURE BY MATCH PHASE")
print("=" * 70)

display(
    pressure_features.groupby("match_phase")
    .agg(
        deliveries=("match_id", "size"),
        avg_wickets_last_6=(
            "wickets_last_6_balls", "mean"
        ),
        avg_wickets_last_12=(
            "wickets_last_12_balls", "mean"
        ),
        max_wickets_last_12=(
            "wickets_last_12_balls", "max"
        )
    )
    .reset_index()
)

print("\n5. FEATURE CONSISTENCY CHECKS")
print("=" * 70)

checks = {
    "negative_recent_6": int(
        (pressure_features["wickets_last_6_balls"] < 0).sum()
    ),
    "negative_recent_12": int(
        (pressure_features["wickets_last_12_balls"] < 0).sum()
    ),
    "recent_6_exceeds_recent_12": int(
        (
            pressure_features["wickets_last_6_balls"] >
            pressure_features["wickets_last_12_balls"]
        ).sum()
    ),
    "recent_12_exceeds_total_wickets_before": int(
        (
            pressure_features["wickets_last_12_balls"] >
            pressure_features["wickets_before"]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

print("\n6. HIGH RECENT-WICKET PRESSURE EXAMPLES")
print("=" * 70)

display(
    pressure_features.loc[
        pressure_features["wickets_last_12_balls"] >= 3,
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "legal_balls_before",
            "wickets_before",
            "wickets_last_6_balls",
            "wickets_last_12_balls",
            "match_phase"
        ]
    ].head(20)
)

In [ ]:
# CELL 42 — Phase-Adjusted Wicket Pressure

print("=" * 70)
print("PHASE-ADJUSTED WICKET PRESSURE")
print("=" * 70)

# Preserve existing pressure features
phase_pressure = pressure_features.copy()

# Use legal deliveries for phase reference distributions
phase_reference = phase_pressure.loc[
    phase_pressure["is_legal_delivery"].eq(1)
].copy()


# ============================================================
# 1. PHASE-SPECIFIC WICKET BENCHMARKS
# ============================================================

print("\n1. PHASE-SPECIFIC WICKET BENCHMARKS")
print("=" * 70)

phase_wicket_baselines = (
    phase_reference.groupby("match_phase")
    .agg(
        reference_deliveries=("match_id", "size"),

        mean_recent_6=(
            "wickets_last_6_balls", "mean"
        ),

        std_recent_6=(
            "wickets_last_6_balls",
            lambda x: x.std(ddof=0)
        ),

        mean_recent_12=(
            "wickets_last_12_balls", "mean"
        ),

        std_recent_12=(
            "wickets_last_12_balls",
            lambda x: x.std(ddof=0)
        )
    )
    .reset_index()
)

display(phase_wicket_baselines)


# ============================================================
# 2. ADD PHASE BENCHMARKS TO EACH DELIVERY
# ============================================================

print("\n2. APPLY PHASE BENCHMARKS")
print("=" * 70)

phase_pressure = phase_pressure.merge(
    phase_wicket_baselines,
    on="match_phase",
    how="left",
    validate="many_to_one"
)

# Avoid division by zero if any phase has zero variation
phase_pressure["wicket_shock_6"] = (
    (
        phase_pressure["wickets_last_6_balls"] -
        phase_pressure["mean_recent_6"]
    ) /
    phase_pressure["std_recent_6"].replace(0, np.nan)
)

phase_pressure["wicket_shock_12"] = (
    (
        phase_pressure["wickets_last_12_balls"] -
        phase_pressure["mean_recent_12"]
    ) /
    phase_pressure["std_recent_12"].replace(0, np.nan)
)

print("Deliveries:", len(phase_pressure))

print(
    "Missing 6-ball shock scores:",
    int(phase_pressure["wicket_shock_6"].isna().sum())
)

print(
    "Missing 12-ball shock scores:",
    int(phase_pressure["wicket_shock_12"].isna().sum())
)


# ============================================================
# 3. STANDARDIZED PRESSURE DISTRIBUTION
# ============================================================

print("\n3. STANDARDIZED PRESSURE DISTRIBUTION")
print("=" * 70)

display(
    phase_pressure[
        ["wicket_shock_6", "wicket_shock_12"]
    ].describe()
)


# ============================================================
# 4. COMPARE TWO-WICKET SITUATIONS ACROSS PHASES
# ============================================================

print("\n4. TWO-WICKET SITUATIONS ACROSS PHASES")
print("=" * 70)

display(
    phase_pressure.loc[
        phase_pressure["wickets_last_12_balls"].eq(2)
    ]
    .groupby("match_phase")
    .agg(
        deliveries=("match_id", "size"),
        average_shock_12=("wicket_shock_12", "mean")
    )
    .reset_index()
)


# ============================================================
# 5. HIGH WICKET-SHOCK EXAMPLES
# ============================================================

print("\n5. HIGH WICKET-SHOCK EXAMPLES")
print("=" * 70)

display(
    phase_pressure.sort_values(
        "wicket_shock_12",
        ascending=False
    )[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "match_phase",
            "wickets_before",
            "wickets_last_6_balls",
            "wickets_last_12_balls",
            "wicket_shock_12"
        ]
    ].head(20)
)


# ============================================================
# 6. FEATURE VALIDATION
# ============================================================

print("\n6. FEATURE VALIDATION")
print("=" * 70)

validation_checks = {
    "missing_phase_baselines": int(
        phase_pressure["mean_recent_12"].isna().sum()
    ),

    "missing_shock_6": int(
        phase_pressure["wicket_shock_6"].isna().sum()
    ),

    "missing_shock_12": int(
        phase_pressure["wicket_shock_12"].isna().sum()
    ),

    "row_count_difference": abs(
        len(phase_pressure) - len(pressure_features)
    ),

    "duplicate_source_rows": int(
        phase_pressure.index.duplicated().sum()
    )
}

display(
    pd.DataFrame(
        list(validation_checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 43 — Stage-Adjusted Wicket Depletion

print("=" * 70)
print("STAGE-ADJUSTED WICKET DEPLETION")
print("=" * 70)

# Preserve existing validated pressure features
stage_pressure = phase_pressure.copy()

# Completed legal overs: 0 to 19
stage_pressure["legal_over_index"] = (
    stage_pressure["legal_balls_before"] // 6
).astype("int64")

# Use legal deliveries for reference distributions
stage_reference = stage_pressure.loc[
    stage_pressure["is_legal_delivery"].eq(1)
].copy()


# ============================================================
# 1. STAGE-SPECIFIC WICKET BASELINES
# ============================================================

print("\n1. STAGE-SPECIFIC WICKET BASELINES")
print("=" * 70)

stage_baselines = (
    stage_reference.groupby("legal_over_index")
    .agg(
        reference_deliveries=("match_id", "size"),

        mean_wickets_before=(
            "wickets_before", "mean"
        ),

        std_wickets_before=(
            "wickets_before",
            lambda x: x.std(ddof=0)
        ),

        median_wickets_before=(
            "wickets_before", "median"
        )
    )
    .reset_index()
)

# FIX: Ensure both merge keys have identical integer dtypes
stage_pressure["legal_over_index"] = (
    stage_pressure["legal_over_index"].astype("int64")
)

stage_baselines["legal_over_index"] = (
    stage_baselines["legal_over_index"].astype("int64")
)

display(stage_baselines)


# ============================================================
# 2. CALCULATE WICKET DEPLETION Z-SCORES
# ============================================================

print("\n2. APPLY STAGE BENCHMARKS")
print("=" * 70)

stage_pressure = stage_pressure.merge(
    stage_baselines,
    on="legal_over_index",
    how="left",
    validate="many_to_one"
)

stage_pressure["wicket_depletion_z"] = (
    (
        stage_pressure["wickets_before"] -
        stage_pressure["mean_wickets_before"]
    ) /
    stage_pressure[
        "std_wickets_before"
    ].replace(0, np.nan)
)

print("Deliveries:", len(stage_pressure))

print(
    "Missing depletion scores:",
    int(
        stage_pressure["wicket_depletion_z"]
        .isna().sum()
    )
)

display(
    stage_pressure["wicket_depletion_z"].describe()
)


# ============================================================
# 3. IDENTIFY ELEVATED WICKET PRESSURE
# ============================================================

print("\n3. ELEVATED WICKET PRESSURE DISTRIBUTION")
print("=" * 70)

# Exploratory thresholds only — not final pressure labels
stage_pressure["elevated_depletion"] = (
    stage_pressure["wicket_depletion_z"] >= 2
)

stage_pressure["elevated_recent_shock"] = (
    stage_pressure["wicket_shock_12"] >= 2
)

display(
    pd.crosstab(
        stage_pressure["elevated_depletion"],
        stage_pressure["elevated_recent_shock"],
        margins=True
    )
)


# ============================================================
# 4. PRESSURE CONDITIONS BY MATCH PHASE
# ============================================================

print("\n4. PRESSURE CONDITIONS BY MATCH PHASE")
print("=" * 70)

display(
    stage_pressure.groupby("match_phase")
    .agg(
        deliveries=("match_id", "size"),

        avg_depletion_z=(
            "wicket_depletion_z", "mean"
        ),

        elevated_depletion_rate=(
            "elevated_depletion", "mean"
        ),

        elevated_recent_shock_rate=(
            "elevated_recent_shock", "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 5. HIGH WICKET-DEPLETION EXAMPLES
# ============================================================

print("\n5. HIGH WICKET-DEPLETION EXAMPLES")
print("=" * 70)

display(
    stage_pressure.sort_values(
        "wicket_depletion_z",
        ascending=False
    )[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "match_phase",
            "legal_balls_before",
            "wickets_before",
            "wickets_remaining_before",
            "wickets_last_12_balls",
            "wicket_shock_12",
            "wicket_depletion_z"
        ]
    ].head(20)
)


# ============================================================
# 6. FEATURE VALIDATION
# ============================================================

print("\n6. FEATURE VALIDATION")
print("=" * 70)

checks = {
    "missing_stage_baselines": int(
        stage_pressure[
            "mean_wickets_before"
        ].isna().sum()
    ),

    "missing_depletion_scores": int(
        stage_pressure[
            "wicket_depletion_z"
        ].isna().sum()
    ),

    "invalid_over_index": int(
        (
            ~stage_pressure[
                "legal_over_index"
            ].between(0, 19)
        ).sum()
    ),

    "row_count_difference": abs(
        len(stage_pressure) -
        len(phase_pressure)
    ),

    "duplicate_delivery_keys": int(
        stage_pressure.duplicated(
            subset=[
                "match_id",
                "innings",
                "innings_ball_num"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 44 — Score Reconstruction vs Match Result Margins

print("=" * 70)
print("SCORE RECONSTRUCTION — MATCH RESULT VALIDATION")
print("=" * 70)

# ============================================================
# 1. IDENTIFY MATCHES WON BY RUNS
# ============================================================

print("\n1. OFFICIAL RUN-MARGIN RESULTS")
print("=" * 70)

margin_metadata = matches.loc[
    matches["match_id"].isin(
        standard_state["match_id"].unique()
    )
].copy()

# Extract winning margin from result text
margin_metadata["reported_run_margin"] = (
    pd.to_numeric(
        margin_metadata["result"]
        .fillna("")
        .astype(str)
        .str.extract(
            r"(?i)\bwon\s+by\s+(\d+)\s+runs?\b",
            expand=False
        ),
        errors="coerce"
    )
)

run_win_metadata = margin_metadata.loc[
    margin_metadata["reported_run_margin"].notna()
].copy()

print(
    "Standard matches:",
    margin_metadata["match_id"].nunique()
)

print(
    "Matches with reported run margins:",
    len(run_win_metadata)
)

display(
    run_win_metadata[
        ["match_id", "result", "reported_run_margin"]
    ].head(15)
)


# ============================================================
# 2. CONSTRUCT TWO SCORING SCENARIOS
# ============================================================

print("\n2. SCORING SCENARIOS")
print("=" * 70)

score_comparison = standard_state.loc[
    standard_state["innings"].isin([1, 2])
].copy()

score_comparison["batter_runs"] = pd.to_numeric(
    score_comparison["runs"],
    errors="coerce"
)

score_comparison["extra_field_runs"] = (
    pd.to_numeric(
        score_comparison["extras_2_runs"],
        errors="coerce"
    ).fillna(0)
)

# Scenario A: directly recorded run fields only
score_comparison["observed_field_runs"] = (
    score_comparison["batter_runs"] +
    score_comparison["extra_field_runs"]
)

# Scenario B: exploratory one-run contribution
# for each ordinary extras category.
# This is an assumption, not a verified score.
assumed_extras_categories = [
    "wide",
    "Overstep",
    "other",
    "byes",
    "legbyes"
]

score_comparison["assumed_extra_runs"] = (
    score_comparison["extras_type"]
    .isin(assumed_extras_categories)
    .astype(int)
)

score_comparison["candidate_team_runs"] = (
    score_comparison["observed_field_runs"] +
    score_comparison["assumed_extra_runs"]
)

print(
    "Observed field runs:",
    score_comparison["observed_field_runs"].sum()
)

print(
    "Candidate team runs:",
    score_comparison["candidate_team_runs"].sum()
)

print(
    "Penalty events:",
    int(
        score_comparison["extras_type"]
        .eq("Penalty").sum()
    )
)


# ============================================================
# 3. AGGREGATE INNINGS SCORES
# ============================================================

print("\n3. RECONSTRUCTED INNINGS TOTALS")
print("=" * 70)

innings_score_comparison = (
    score_comparison.groupby(
        ["match_id", "innings"]
    )
    .agg(
        observed_field_runs=(
            "observed_field_runs", "sum"
        ),
        candidate_team_runs=(
            "candidate_team_runs", "sum"
        )
    )
    .reset_index()
)

score_pivot = innings_score_comparison.pivot(
    index="match_id",
    columns="innings",
    values=[
        "observed_field_runs",
        "candidate_team_runs"
    ]
)

score_pivot.columns = [
    f"{metric}_inn{int(innings)}"
    for metric, innings in score_pivot.columns
]

score_pivot = score_pivot.reset_index()

display(score_pivot.head(10))


# ============================================================
# 4. COMPARE AGAINST REPORTED WINNING MARGINS
# ============================================================

print("\n4. MARGIN ACCURACY COMPARISON")
print("=" * 70)

margin_validation = run_win_metadata[
    ["match_id", "result", "reported_run_margin"]
].merge(
    score_pivot,
    on="match_id",
    how="inner",
    validate="one_to_one"
)

required_columns = [
    "observed_field_runs_inn1",
    "observed_field_runs_inn2",
    "candidate_team_runs_inn1",
    "candidate_team_runs_inn2"
]

margin_validation = margin_validation.dropna(
    subset=required_columns
).copy()

# Penalty-run direction is not yet established.
# Exclude matches containing penalty events.
penalty_match_ids = score_comparison.loc[
    score_comparison["extras_type"].eq("Penalty"),
    "match_id"
].unique()

margin_validation = margin_validation.loc[
    ~margin_validation["match_id"].isin(
        penalty_match_ids
    )
].copy()

# A completed run victory normally means
# the first-innings team won.
margin_validation["observed_margin"] = (
    margin_validation["observed_field_runs_inn1"] -
    margin_validation["observed_field_runs_inn2"]
)

margin_validation["candidate_margin"] = (
    margin_validation["candidate_team_runs_inn1"] -
    margin_validation["candidate_team_runs_inn2"]
)

margin_validation["observed_error"] = (
    margin_validation["observed_margin"] -
    margin_validation["reported_run_margin"]
)

margin_validation["candidate_error"] = (
    margin_validation["candidate_margin"] -
    margin_validation["reported_run_margin"]
)

metrics = []

for scenario, error_column in [
    ("Observed fields", "observed_error"),
    ("One-run extras assumption", "candidate_error")
]:
    errors = margin_validation[error_column]

    metrics.append({
        "scenario": scenario,
        "matches": len(errors),
        "exact_matches": int(errors.eq(0).sum()),
        "within_1_run": int(errors.abs().le(1).sum()),
        "within_3_runs": int(errors.abs().le(3).sum()),
        "mean_absolute_error": round(
            errors.abs().mean(), 3
        ),
        "maximum_absolute_error": errors.abs().max()
    })

display(pd.DataFrame(metrics))


# ============================================================
# 5. LARGEST MARGIN DISCREPANCIES
# ============================================================

print("\n5. LARGEST MARGIN DISCREPANCIES")
print("=" * 70)

display(
    margin_validation.assign(
        absolute_error=lambda df:
            df["candidate_error"].abs()
    )
    .sort_values(
        "absolute_error",
        ascending=False
    )[
        [
            "match_id",
            "result",
            "reported_run_margin",
            "observed_margin",
            "candidate_margin",
            "observed_error",
            "candidate_error"
        ]
    ].head(20)
)


# ============================================================
# 6. ERROR DISTRIBUTION
# ============================================================

print("\n6. MARGIN ERROR DISTRIBUTION")
print("=" * 70)

display(
    margin_validation[
        ["observed_error", "candidate_error"]
    ].describe()
)

In [ ]:
# CELL 45 — Clean Run-Margin Validation Sample

print("=" * 70)
print("CLEAN RUN-MARGIN VALIDATION")
print("=" * 70)

# Preserve all results from Cell 44
clean_margin_audit = margin_validation.copy()

# Add available revision metadata
revision_columns = [
    "revised_target",
    "first_inn_revised_overs",
    "second_inn_revised_overs"
]

clean_margin_audit = clean_margin_audit.merge(
    matches[
        ["match_id", "match_date"] + revision_columns
    ],
    on="match_id",
    how="left",
    validate="one_to_one"
)

# ============================================================
# 1. IDENTIFY DLS AND REVISED RESULTS
# ============================================================

print("\n1. IDENTIFY DLS AND REVISED RESULTS")
print("=" * 70)

dls_pattern = (
    r"(?i)\bDLS\b|D\s*/\s*L\b|"
    r"duckworth|lewis|\bVJD\b"
)

clean_margin_audit["has_dls_result"] = (
    clean_margin_audit["result"]
    .fillna("")
    .astype(str)
    .str.contains(
        dls_pattern,
        regex=True
    )
)

clean_margin_audit["has_revision_fields"] = (
    clean_margin_audit[
        revision_columns
    ].notna().any(axis=1)
)

clean_margin_audit["is_clean_run_margin"] = (
    ~clean_margin_audit["has_dls_result"] &
    ~clean_margin_audit["has_revision_fields"]
)

display(
    clean_margin_audit.groupby(
        [
            "has_dls_result",
            "has_revision_fields",
            "is_clean_run_margin"
        ]
    )
    .size()
    .reset_index(name="matches")
)


# ============================================================
# 2. EXAMINE EXCLUDED MATCHES
# ============================================================

print("\n2. DLS / REVISED MATCHES")
print("=" * 70)

display(
    clean_margin_audit.loc[
        ~clean_margin_audit["is_clean_run_margin"],
        [
            "match_id",
            "match_date",
            "result",
            "reported_run_margin",
            "observed_margin",
            "candidate_margin",
            "candidate_error",
            "has_dls_result",
            "has_revision_fields"
        ]
    ].sort_values(
        "candidate_error",
        key=lambda x: x.abs(),
        ascending=False
    )
)


# ============================================================
# 3. RECOMPUTE SCORING ACCURACY
# ============================================================

print("\n3. CLEAN VS ORIGINAL MARGIN ACCURACY")
print("=" * 70)

clean_only = clean_margin_audit.loc[
    clean_margin_audit["is_clean_run_margin"]
].copy()

comparison_sets = [
    ("Original sample", clean_margin_audit),
    ("Clean sample", clean_only)
]

accuracy_results = []

for sample_name, df in comparison_sets:

    for scenario, error_column in [
        ("Observed fields", "observed_error"),
        ("One-run extras", "candidate_error")
    ]:

        errors = df[error_column]

        accuracy_results.append({
            "sample": sample_name,
            "scenario": scenario,
            "matches": len(errors),
            "exact_matches": int(
                errors.eq(0).sum()
            ),
            "within_1_run": int(
                errors.abs().le(1).sum()
            ),
            "within_3_runs": int(
                errors.abs().le(3).sum()
            ),
            "mean_absolute_error": round(
                errors.abs().mean(), 3
            ),
            "max_absolute_error": (
                errors.abs().max()
            )
        })

display(
    pd.DataFrame(accuracy_results)
)


# ============================================================
# 4. REMAINING LARGE DISCREPANCIES
# ============================================================

print("\n4. LARGEST ERRORS IN CLEAN MATCHES")
print("=" * 70)

display(
    clean_only.assign(
        absolute_error=lambda df:
            df["candidate_error"].abs()
    )
    .sort_values(
        "absolute_error",
        ascending=False
    )[
        [
            "match_id",
            "match_date",
            "result",
            "reported_run_margin",
            "observed_margin",
            "candidate_margin",
            "observed_error",
            "candidate_error"
        ]
    ].head(20)
)


# ============================================================
# 5. DOES THE EXTRAS ASSUMPTION HELP?
# ============================================================

print("\n5. SCORING SCENARIO IMPROVEMENT")
print("=" * 70)

clean_only["observed_absolute_error"] = (
    clean_only["observed_error"].abs()
)

clean_only["candidate_absolute_error"] = (
    clean_only["candidate_error"].abs()
)

clean_only["scenario_outcome"] = np.select(
    [
        clean_only["candidate_absolute_error"] <
        clean_only["observed_absolute_error"],

        clean_only["candidate_absolute_error"] >
        clean_only["observed_absolute_error"]
    ],
    [
        "Improved",
        "Worsened"
    ],
    default="Unchanged"
)

display(
    clean_only["scenario_outcome"]
    .value_counts()
    .rename_axis("scenario_outcome")
    .reset_index(name="matches")
)

print("\n6. REMAINING ERROR DISTRIBUTION")
print("=" * 70)

display(
    clean_only[
        ["observed_error", "candidate_error"]
    ].describe()
)

In [ ]:
# CELL 46 — Parity-Informed Extras Reconstruction

print("=" * 70)
print("PARITY-INFORMED EXTRAS RECONSTRUCTION")
print("=" * 70)

# Preserve Cell 44 scoring calculations
parity_score = score_comparison.copy()

group_cols = ["match_id", "innings"]

# ============================================================
# 1. IDENTIFY RELIABLE STRIKE ROTATION
# ============================================================

print("\n1. STRIKE ROTATION COVERAGE")
print("=" * 70)

for col in ["striker_id", "non_striker_id", "overid"]:
    parity_score[f"next_{col}"] = (
        parity_score.groupby(
            group_cols, sort=False
        )[col].shift(-1)
    )

parity_score["same_over_next"] = (
    parity_score["overid"] ==
    parity_score["next_overid"]
)

parity_score["pair_unchanged"] = (
    parity_score["striker_id"].eq(
        parity_score["next_striker_id"]
    ) &
    parity_score["non_striker_id"].eq(
        parity_score["next_non_striker_id"]
    )
)

parity_score["pair_swapped"] = (
    parity_score["striker_id"].eq(
        parity_score["next_non_striker_id"]
    ) &
    parity_score["non_striker_id"].eq(
        parity_score["next_striker_id"]
    )
)

parity_score["reliable_rotation"] = (
    parity_score["same_over_next"] &
    ~parity_score["is_wicket"] &
    (
        parity_score["pair_unchanged"] |
        parity_score["pair_swapped"]
    ) &
    parity_score["runs"].eq(0) &
    parity_score["extras_2_runs"].fillna(0).eq(0)
)

display(
    parity_score.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        reliable_rotation=("reliable_rotation", "sum")
    )
    .reset_index()
)


# ============================================================
# 2. CALCULATE PARITY-BASED ADJUSTMENTS
# ============================================================

print("\n2. PARITY-BASED EXTRAS ADJUSTMENTS")
print("=" * 70)

bye_types = ["byes", "legbyes"]

# For byes/leg-byes with an unchanged pair,
# two completed runs is the minimum positive even total.
even_byes = (
    parity_score["extras_type"].isin(bye_types) &
    parity_score["reliable_rotation"] &
    parity_score["pair_unchanged"]
)

# For a wide followed by a batter swap,
# add one completed run to the initial wide penalty.
wide_with_swap = (
    parity_score["extras_type"].eq("wide") &
    parity_score["reliable_rotation"] &
    parity_score["pair_swapped"]
)

parity_score["parity_adjustment"] = (
    even_byes.astype(int) +
    wide_with_swap.astype(int)
)

# Preserve the original one-run extras scenario
parity_score["parity_candidate_runs"] = (
    parity_score["candidate_team_runs"] +
    parity_score["parity_adjustment"]
)

print("Even bye/leg-bye adjustments:", int(even_byes.sum()))

print("Wide rotation adjustments:", int(wide_with_swap.sum()))

print(
    "Total additional runs:",
    int(parity_score["parity_adjustment"].sum())
)

display(
    parity_score.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        original_candidate_runs=("candidate_team_runs", "sum"),
        parity_candidate_runs=("parity_candidate_runs", "sum"),
        added_runs=("parity_adjustment", "sum")
    )
    .reset_index()
)


# ============================================================
# 3. RECONSTRUCT PARITY-ADJUSTED INNINGS TOTALS
# ============================================================

print("\n3. PARITY-ADJUSTED INNINGS TOTALS")
print("=" * 70)

parity_innings = (
    parity_score.groupby(group_cols)
    .agg(
        parity_score=("parity_candidate_runs", "sum")
    )
    .reset_index()
)

parity_pivot = parity_innings.pivot(
    index="match_id",
    columns="innings",
    values="parity_score"
)

parity_pivot = parity_pivot.rename(
    columns={
        1: "parity_score_inn1",
        2: "parity_score_inn2"
    }
).reset_index()

display(parity_pivot.head(10))


# ============================================================
# 4. COMPARE WITH CLEAN OFFICIAL RUN MARGINS
# ============================================================

print("\n4. SCORING ACCURACY COMPARISON")
print("=" * 70)

parity_validation = clean_only[
    [
        "match_id",
        "result",
        "reported_run_margin",
        "observed_error",
        "candidate_error"
    ]
].merge(
    parity_pivot,
    on="match_id",
    how="inner",
    validate="one_to_one"
)

parity_validation = parity_validation.dropna(
    subset=[
        "parity_score_inn1",
        "parity_score_inn2"
    ]
).copy()

parity_validation["parity_margin"] = (
    parity_validation["parity_score_inn1"] -
    parity_validation["parity_score_inn2"]
)

parity_validation["parity_error"] = (
    parity_validation["parity_margin"] -
    parity_validation["reported_run_margin"]
)

scenario_metrics = []

for scenario, error_col in [
    ("Recorded fields", "observed_error"),
    ("One-run extras", "candidate_error"),
    ("Parity-informed extras", "parity_error")
]:

    errors = parity_validation[error_col]

    scenario_metrics.append({
        "scenario": scenario,
        "matches": len(errors),
        "exact_matches": int(errors.eq(0).sum()),
        "within_1_run": int(errors.abs().le(1).sum()),
        "within_3_runs": int(errors.abs().le(3).sum()),
        "mean_absolute_error": round(
            errors.abs().mean(), 3
        ),
        "max_absolute_error": errors.abs().max()
    })

display(pd.DataFrame(scenario_metrics))


# ============================================================
# 5. DOES PARITY IMPROVE THE SCORING ESTIMATE?
# ============================================================

print("\n5. PARITY SCENARIO IMPROVEMENT")
print("=" * 70)

parity_validation["previous_absolute_error"] = (
    parity_validation["candidate_error"].abs()
)

parity_validation["parity_absolute_error"] = (
    parity_validation["parity_error"].abs()
)

parity_validation["parity_outcome"] = np.select(
    [
        parity_validation["parity_absolute_error"] <
        parity_validation["previous_absolute_error"],

        parity_validation["parity_absolute_error"] >
        parity_validation["previous_absolute_error"]
    ],
    ["Improved", "Worsened"],
    default="Unchanged"
)

display(
    parity_validation["parity_outcome"]
    .value_counts()
    .rename_axis("outcome")
    .reset_index(name="matches")
)


# ============================================================
# 6. REMAINING LARGE SCORING ERRORS
# ============================================================

print("\n6. LARGEST REMAINING ERRORS")
print("=" * 70)

display(
    parity_validation.sort_values(
        "parity_absolute_error",
        ascending=False
    )[
        [
            "match_id",
            "result",
            "reported_run_margin",
            "parity_margin",
            "candidate_error",
            "parity_error"
        ]
    ].head(20)
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "row_count_difference": abs(
        len(parity_score) - len(score_comparison)
    ),

    "negative_adjustments": int(
        (parity_score["parity_adjustment"] < 0).sum()
    ),

    "unexpected_adjustment_values": int(
        (~parity_score["parity_adjustment"].isin([0, 1])).sum()
    ),

    "negative_candidate_runs": int(
        (parity_score["parity_candidate_runs"] < 0).sum()
    ),

    "margin_matches_lost": int(
        len(clean_only) - len(parity_validation)
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 47 — Provisional Score Timeline and Uncertainty Audit

print("=" * 70)
print("PROVISIONAL SCORE TIMELINE AND UNCERTAINTY AUDIT")
print("=" * 70)

# ============================================================
# 1. COMBINE PRESSURE FEATURES AND PARITY SCORING
# ============================================================

print("\n1. MERGE COVERAGE")
print("=" * 70)

score_state = stage_pressure.copy()

score_columns = [
    "match_id",
    "innings",
    "innings_ball_num",
    "parity_candidate_runs",
    "parity_adjustment",
    "reliable_rotation"
]

score_state = score_state.merge(
    parity_score[score_columns],
    on=["match_id", "innings", "innings_ball_num"],
    how="left",
    validate="one_to_one",
    sort=False
)

missing_scoring = int(
    score_state["parity_candidate_runs"].isna().sum()
)

print("Rows:", len(score_state))
print("Missing scoring records:", missing_scoring)

assert missing_scoring == 0, (
    "Scoring merge incomplete — investigate before continuing."
)

# Ensure chronological progression
score_state = score_state.sort_values(
    ["match_id", "innings", "innings_ball_num"],
    kind="stable"
).reset_index(drop=True)


# ============================================================
# 2. CONSTRUCT ESTIMATED TEAM SCORE
# ============================================================

print("\n2. ESTIMATED TEAM SCORE TIMELINE")
print("=" * 70)

group_cols = ["match_id", "innings"]

score_state["estimated_score_after"] = (
    score_state.groupby(
        group_cols, sort=False
    )["parity_candidate_runs"].cumsum()
)

score_state["estimated_score_before"] = (
    score_state["estimated_score_after"] -
    score_state["parity_candidate_runs"]
)

print(
    "Total provisional runs:",
    score_state["parity_candidate_runs"].sum()
)

print(
    "Maximum provisional innings score:",
    score_state.groupby(group_cols)[
        "estimated_score_after"
    ].max().max()
)


# ============================================================
# 3. TRACK EXTRAS-RELATED SCORING UNCERTAINTY
# ============================================================

print("\n3. SCORING UNCERTAINTY COVERAGE")
print("=" * 70)

# Extras amounts are estimated, not independently verified
score_state["assumption_dependent_event"] = (
    score_state["extras_type"].notna()
).astype(int)

score_state["assumption_events_after"] = (
    score_state.groupby(
        group_cols, sort=False
    )["assumption_dependent_event"].cumsum()
)

score_state["assumption_events_before"] = (
    score_state["assumption_events_after"] -
    score_state["assumption_dependent_event"]
)

score_state["score_has_prior_assumptions"] = (
    score_state["assumption_events_before"] > 0
)

print(
    "Extras events using assumptions:",
    int(score_state["assumption_dependent_event"].sum())
)

print(
    "Deliveries with prior extras assumptions:",
    int(score_state["score_has_prior_assumptions"].sum())
)

display(
    score_state["assumption_events_before"].describe()
)


# ============================================================
# 4. FLAG DLS-AFFECTED RESULTS
# ============================================================

print("\n4. DLS RESULT FLAGGING")
print("=" * 70)

match_results = matches[
    ["match_id", "result"]
].rename(
    columns={"result": "match_result"}
)

score_state = score_state.merge(
    match_results,
    on="match_id",
    how="left",
    validate="many_to_one"
)

dls_pattern = (
    r"(?i)\bDLS\b|D\s*/\s*L\b|"
    r"duckworth|lewis|\bVJD\b"
)

score_state["has_dls_result"] = (
    score_state["match_result"]
    .fillna("")
    .astype(str)
    .str.contains(dls_pattern, regex=True)
)

print(
    "DLS-flagged matches:",
    score_state.loc[
        score_state["has_dls_result"],
        "match_id"
    ].nunique()
)

print(
    "DLS-flagged deliveries:",
    int(score_state["has_dls_result"].sum())
)

display(
    score_state.loc[
        score_state["has_dls_result"],
        ["match_id", "match_result"]
    ].drop_duplicates()
)


# ============================================================
# 5. VALIDATE AGAINST CELL 46 INNINGS TOTALS
# ============================================================

print("\n5. INNINGS TOTAL RECONCILIATION")
print("=" * 70)

timeline_totals = (
    score_state.groupby(group_cols)
    .agg(
        timeline_total=("estimated_score_after", "max")
    )
    .reset_index()
)

timeline_reconciliation = timeline_totals.merge(
    parity_innings[
        ["match_id", "innings", "parity_score"]
    ],
    on=group_cols,
    how="left",
    validate="one_to_one"
)

timeline_reconciliation["difference"] = (
    timeline_reconciliation["timeline_total"] -
    timeline_reconciliation["parity_score"]
)

print(
    "Innings reconciled:",
    len(timeline_reconciliation)
)

print(
    "Innings total mismatches:",
    int(
        timeline_reconciliation["difference"]
        .ne(0).sum()
    )
)

display(timeline_reconciliation.head(10))


# ============================================================
# 6. FINAL DATA INTEGRITY CHECKS
# ============================================================

print("\n6. DATA INTEGRITY CHECKS")
print("=" * 70)

first_deliveries = (
    score_state.groupby(group_cols, sort=False)
    .head(1)
)

checks = {
    "row_count_difference": abs(
        len(score_state) - len(stage_pressure)
    ),

    "missing_estimated_scores": int(
        score_state[
            ["estimated_score_before", "estimated_score_after"]
        ].isna().any(axis=1).sum()
    ),

    "negative_estimated_scores": int(
        (
            (score_state["estimated_score_before"] < 0) |
            (score_state["estimated_score_after"] < 0)
        ).sum()
    ),

    "nonzero_starting_scores": int(
        first_deliveries[
            "estimated_score_before"
        ].ne(0).sum()
    ),

    "innings_total_mismatches": int(
        timeline_reconciliation["difference"].ne(0).sum()
    ),

    "duplicate_delivery_keys": int(
        score_state.duplicated(
            ["match_id", "innings", "innings_ball_num"]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)


# ============================================================
# 7. EXAMPLE — SCORE PROGRESSION AROUND AN EXTRAS EVENT
# ============================================================

print("\n7. SAMPLE SCORE PROGRESSION")
print("=" * 70)

first_extras_index = score_state.index[
    score_state["assumption_dependent_event"].eq(1)
][0]

display(
    score_state.iloc[
        max(0, first_extras_index - 3):
        first_extras_index + 5
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "runs",
            "extras_type",
            "parity_candidate_runs",
            "estimated_score_before",
            "estimated_score_after",
            "assumption_events_before"
        ]
    ]
)

In [ ]:
# CELL 48 — Chase Pressure and Scoring Sensitivity

print("=" * 70)
print("CHASE PRESSURE AND SCORING SENSITIVITY")
print("=" * 70)

# ============================================================
# 1. COMBINE THREE SCORING SCENARIOS
# ============================================================

print("\n1. SCORING SCENARIO COVERAGE")
print("=" * 70)

chase_base = score_state.copy()

scenario_fields = score_comparison[
    [
        "match_id",
        "innings",
        "innings_ball_num",
        "observed_field_runs",
        "candidate_team_runs"
    ]
].copy()

merge_keys = [
    "match_id",
    "innings",
    "innings_ball_num"
]

# Use matching key types to avoid pandas merge errors
for key in merge_keys:
    chase_base[key] = (
        pd.to_numeric(chase_base[key]).astype("int64")
    )

    scenario_fields[key] = (
        pd.to_numeric(scenario_fields[key]).astype("int64")
    )

chase_base = chase_base.merge(
    scenario_fields,
    on=merge_keys,
    how="left",
    validate="one_to_one",
    sort=False
)

print("Rows:", len(chase_base))

print(
    "Missing scenario records:",
    int(
        chase_base[
            [
                "observed_field_runs",
                "candidate_team_runs"
            ]
        ].isna().any(axis=1).sum()
    )
)

# Preserve chronological delivery order
chase_base = chase_base.sort_values(
    merge_keys,
    kind="stable"
).reset_index(drop=True)


# ============================================================
# 2. BUILD THREE SCORE TIMELINES
# ============================================================

print("\n2. SCORING SCENARIO TIMELINES")
print("=" * 70)

group_cols = ["match_id", "innings"]

scenarios = {
    "observed": "observed_field_runs",
    "one_run": "candidate_team_runs",
    "parity": "parity_candidate_runs"
}

for scenario, runs_col in scenarios.items():

    chase_base[f"{scenario}_score_after"] = (
        chase_base.groupby(
            group_cols, sort=False
        )[runs_col].cumsum()
    )

    chase_base[f"{scenario}_score_before"] = (
        chase_base[f"{scenario}_score_after"] -
        chase_base[runs_col]
    )

print(
    "Parity timeline mismatches:",
    int(
        (
            chase_base["parity_score_before"] !=
            chase_base["estimated_score_before"]
        ).sum()
    )
)

display(
    chase_base[
        [
            "observed_score_before",
            "one_run_score_before",
            "parity_score_before"
        ]
    ].describe()
)


# ============================================================
# 3. ESTIMATE FIRST-INNINGS CHASE TARGETS
# ============================================================

print("\n3. FIRST-INNINGS TARGET ESTIMATES")
print("=" * 70)

first_innings_totals = (
    chase_base.loc[
        chase_base["innings"].eq(1)
    ]
    .groupby("match_id")
    .agg(
        first_observed=("observed_field_runs", "sum"),
        first_one_run=("candidate_team_runs", "sum"),
        first_parity=("parity_candidate_runs", "sum")
    )
    .reset_index()
)

# A normal chase target is the first-innings total + 1
for scenario, first_col in [
    ("observed", "first_observed"),
    ("one_run", "first_one_run"),
    ("parity", "first_parity")
]:
    first_innings_totals[
        f"target_{scenario}"
    ] = first_innings_totals[first_col] + 1

display(
    first_innings_totals[
        [
            "target_observed",
            "target_one_run",
            "target_parity"
        ]
    ].describe()
)


# ============================================================
# 4. ISOLATE NON-DLS SECOND INNINGS
# ============================================================

print("\n4. ELIGIBLE CHASE COVERAGE")
print("=" * 70)

chase_state = chase_base.loc[
    chase_base["innings"].eq(2) &
    ~chase_base["has_dls_result"]
].copy()

chase_state = chase_state.merge(
    first_innings_totals[
        [
            "match_id",
            "target_observed",
            "target_one_run",
            "target_parity"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

print(
    "Eligible chase matches:",
    chase_state["match_id"].nunique()
)

print(
    "Eligible chase deliveries:",
    len(chase_state)
)

print(
    "DLS deliveries remaining:",
    int(chase_state["has_dls_result"].sum())
)


# ============================================================
# 5. CALCULATE REQUIRED RUN RATE
# ============================================================

print("\n5. REQUIRED RUN RATE BY SCENARIO")
print("=" * 70)

for scenario in scenarios:

    chase_state[f"runs_needed_{scenario}"] = (
        chase_state[f"target_{scenario}"] -
        chase_state[f"{scenario}_score_before"]
    ).clip(lower=0)

    chase_state[f"rrr_{scenario}"] = np.where(
        chase_state["balls_remaining_before"] > 0,
        (
            chase_state[f"runs_needed_{scenario}"] * 6
        ) / chase_state["balls_remaining_before"],
        np.nan
    )

display(
    chase_state[
        [
            "rrr_observed",
            "rrr_one_run",
            "rrr_parity"
        ]
    ].describe()
)


# ============================================================
# 6. MEASURE SCORING SENSITIVITY
# ============================================================

print("\n6. CHASE PRESSURE SENSITIVITY")
print("=" * 70)

chase_state["rrr_scenario_spread"] = (
    chase_state[
        [
            "rrr_observed",
            "rrr_one_run",
            "rrr_parity"
        ]
    ].max(axis=1) -
    chase_state[
        [
            "rrr_observed",
            "rrr_one_run",
            "rrr_parity"
        ]
    ].min(axis=1)
)

display(
    chase_state[
        "rrr_scenario_spread"
    ].describe()
)

# Exploratory high-RRR threshold, not final pressure label
rrr_threshold = 12

for scenario in scenarios:
    chase_state[f"high_rrr_{scenario}"] = (
        chase_state[f"rrr_{scenario}"] >= rrr_threshold
    )

display(
    pd.DataFrame(
        [
            {
                "scenario": scenario,
                "high_rrr_deliveries": int(
                    chase_state[
                        f"high_rrr_{scenario}"
                    ].sum()
                )
            }
            for scenario in scenarios
        ]
    )
)

# Identify where the scenarios disagree
high_rrr_columns = [
    "high_rrr_observed",
    "high_rrr_one_run",
    "high_rrr_parity"
]

chase_state["high_rrr_disagreement"] = (
    chase_state[high_rrr_columns].nunique(
        axis=1
    ) > 1
)

print(
    "High-RRR classification disagreements:",
    int(
        chase_state[
            "high_rrr_disagreement"
        ].sum()
    )
)


# ============================================================
# 7. MOST SENSITIVE CHASE SITUATIONS
# ============================================================

print("\n7. MOST SENSITIVE CHASE SITUATIONS")
print("=" * 70)

display(
    chase_state.sort_values(
        "rrr_scenario_spread",
        ascending=False
    )[
        [
            "match_id",
            "overid",
            "over_ball_num",
            "striker_name",
            "balls_remaining_before",
            "wickets_remaining_before",
            "rrr_observed",
            "rrr_one_run",
            "rrr_parity",
            "rrr_scenario_spread",
            "assumption_events_before"
        ]
    ].head(20)
)


# ============================================================
# 8. DATA INTEGRITY CHECKS
# ============================================================

print("\n8. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "row_count_difference": abs(
        len(chase_base) - len(score_state)
    ),

    "missing_chase_targets": int(
        chase_state[
            [
                "target_observed",
                "target_one_run",
                "target_parity"
            ]
        ].isna().any(axis=1).sum()
    ),

    "missing_required_run_rates": int(
        chase_state[
            [
                "rrr_observed",
                "rrr_one_run",
                "rrr_parity"
            ]
        ].isna().any(axis=1).sum()
    ),

    "negative_runs_needed": int(
        (
            chase_state[
                [
                    "runs_needed_observed",
                    "runs_needed_one_run",
                    "runs_needed_parity"
                ]
            ] < 0
        ).any(axis=1).sum()
    ),

    "parity_timeline_mismatches": int(
        (
            chase_base["parity_score_before"] !=
            chase_base["estimated_score_before"]
        ).sum()
    ),

    "dls_deliveries_in_chase": int(
        chase_state["has_dls_result"].sum()
    ),

    "duplicate_delivery_keys": int(
        chase_state.duplicated(
            merge_keys
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 49 — Chase Outcome Validation and Endgame Sensitivity

print("=" * 70)
print("CHASE OUTCOME AND ENDGAME SENSITIVITY AUDIT")
print("=" * 70)

chase_audit = chase_state.copy()

# ============================================================
# 1. CLASSIFY CHASES BY BALLS REMAINING
# ============================================================

print("\n1. BALLS-REMAINING WINDOWS")
print("=" * 70)

chase_audit["balls_window"] = pd.cut(
    chase_audit["balls_remaining_before"],
    bins=[0, 6, 12, 24, 60, 120],
    labels=[
        "1-6 balls",
        "7-12 balls",
        "13-24 balls",
        "25-60 balls",
        "61-120 balls"
    ],
    include_lowest=True
)

display(
    chase_audit["balls_window"]
    .value_counts()
    .sort_index()
    .rename_axis("balls_window")
    .reset_index(name="deliveries")
)


# ============================================================
# 2. RRR SENSITIVITY BY BALLS REMAINING
# ============================================================

print("\n2. RRR SENSITIVITY BY BALLS REMAINING")
print("=" * 70)

sensitivity_by_window = (
    chase_audit.groupby(
        "balls_window",
        observed=True
    )
    .agg(
        deliveries=("match_id", "size"),

        median_rrr_spread=(
            "rrr_scenario_spread", "median"
        ),

        mean_rrr_spread=(
            "rrr_scenario_spread", "mean"
        ),

        p95_rrr_spread=(
            "rrr_scenario_spread",
            lambda x: x.quantile(0.95)
        ),

        max_rrr_spread=(
            "rrr_scenario_spread", "max"
        ),

        high_rrr_disagreement_rate=(
            "high_rrr_disagreement", "mean"
        )
    )
    .reset_index()
)

display(sensitivity_by_window)


# ============================================================
# 3. EXAMINE ALREADY-REACHED ESTIMATED TARGETS
# ============================================================

print("\n3. ESTIMATED TARGETS ALREADY REACHED")
print("=" * 70)

target_status = []

for scenario in ["observed", "one_run", "parity"]:

    target_status.append({
        "scenario": scenario,

        "deliveries_with_zero_runs_needed": int(
            chase_audit[
                f"runs_needed_{scenario}"
            ].eq(0).sum()
        ),

        "deliveries_with_rrr_above_36": int(
            chase_audit[
                f"rrr_{scenario}"
            ].gt(36).sum()
        ),

        "maximum_rrr": (
            chase_audit[
                f"rrr_{scenario}"
            ].max()
        )
    })

display(pd.DataFrame(target_status))


# ============================================================
# 4. GET FINAL CHASE TOTALS AND OFFICIAL RESULTS
# ============================================================

print("\n4. OFFICIAL CHASE OUTCOME COVERAGE")
print("=" * 70)

final_chases = (
    chase_audit.sort_values(
        ["match_id", "innings_ball_num"],
        kind="stable"
    )
    .groupby("match_id", sort=False)
    .tail(1)
    .copy()
)

final_chases = final_chases.merge(
    matches[
        ["match_id", "win_team_id", "result"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one"
)

final_chases["actual_chase_win"] = (
    pd.to_numeric(
        final_chases["batting_team_id"],
        errors="coerce"
    ).eq(
        pd.to_numeric(
            final_chases["win_team_id"],
            errors="coerce"
        )
    )
)

# Only compare matches with a known winner and
# an explicit winning result
final_chases["known_winner"] = (
    final_chases["win_team_id"].notna() &
    final_chases["result"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"\bwon\b|\bbeat\b|\bdefeated\b",
        case=False,
        regex=True
    )
)

print("Final chase records:", len(final_chases))

print(
    "Matches with known winning team:",
    int(final_chases["known_winner"].sum())
)

display(
    final_chases["actual_chase_win"]
    .value_counts()
    .rename_axis("actual_chase_win")
    .reset_index(name="matches")
)


# ============================================================
# 5. COMPARE PREDICTED VS ACTUAL MATCH WINNERS
# ============================================================

print("\n5. CHASE OUTCOME AGREEMENT BY SCENARIO")
print("=" * 70)

outcome_sample = final_chases.loc[
    final_chases["known_winner"]
].copy()

outcome_metrics = []

for scenario in ["observed", "one_run", "parity"]:

    predicted_col = f"predicted_win_{scenario}"

    outcome_sample[predicted_col] = (
        outcome_sample[
            f"{scenario}_score_after"
        ] >= outcome_sample[
            f"target_{scenario}"
        ]
    )

    correct = (
        outcome_sample[predicted_col] ==
        outcome_sample["actual_chase_win"]
    )

    false_negative = (
        outcome_sample["actual_chase_win"] &
        ~outcome_sample[predicted_col]
    )

    false_positive = (
        ~outcome_sample["actual_chase_win"] &
        outcome_sample[predicted_col]
    )

    outcome_metrics.append({
        "scenario": scenario,
        "matches": len(outcome_sample),
        "correct_outcomes": int(correct.sum()),
        "agreement_rate": round(
            100 * correct.mean(), 2
        ),
        "missed_chase_wins": int(
            false_negative.sum()
        ),
        "incorrect_chase_wins": int(
            false_positive.sum()
        )
    })

display(pd.DataFrame(outcome_metrics))

print("\n6. PARITY SCENARIO OUTCOME CONFUSION MATRIX")
print("=" * 70)

display(
    pd.crosstab(
        outcome_sample["actual_chase_win"],
        outcome_sample["predicted_win_parity"],
        margins=True
    )
)


# ============================================================
# 7. EXAMINE OUTCOME DISAGREEMENTS
# ============================================================

print("\n7. PARITY SCENARIO OUTCOME DISAGREEMENTS")
print("=" * 70)

outcome_sample["parity_outcome_mismatch"] = (
    outcome_sample["actual_chase_win"] !=
    outcome_sample["predicted_win_parity"]
)

display(
    outcome_sample.loc[
        outcome_sample["parity_outcome_mismatch"],
        [
            "match_id",
            "result",
            "batting_team_id",
            "win_team_id",
            "balls_remaining_after",
            "parity_score_after",
            "target_parity",
            "actual_chase_win",
            "predicted_win_parity"
        ]
    ].head(25)
)


# ============================================================
# 8. FINAL INTEGRITY CHECKS
# ============================================================

print("\n8. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "missing_ball_windows": int(
        chase_audit["balls_window"].isna().sum()
    ),

    "final_chase_count_difference": abs(
        len(final_chases) -
        chase_audit["match_id"].nunique()
    ),

    "duplicate_final_chases": int(
        final_chases["match_id"].duplicated().sum()
    ),

    "negative_rrr_spread": int(
        (
            chase_audit["rrr_scenario_spread"] < 0
        ).sum()
    ),

    "outcome_sample_exceeds_chases": int(
        len(outcome_sample) > len(final_chases)
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)


In [ ]:
# CELL 50 — Diagnose Chase Outcome Errors

print("=" * 70)
print("CHASE OUTCOME ERROR DIAGNOSIS")
print("=" * 70)

# Preserve the validated Cell 49 outcome data
terminal_audit = outcome_sample.copy()


# ============================================================
# 1. CLASSIFY OUTCOME ERRORS
# ============================================================

print("\n1. CHASE OUTCOME CLASSIFICATION")
print("=" * 70)

terminal_audit["terminal_gap_parity"] = (
    terminal_audit["target_parity"] -
    terminal_audit["parity_score_after"]
)

terminal_audit["outcome_class"] = np.select(
    [
        terminal_audit["actual_chase_win"] &
        ~terminal_audit["predicted_win_parity"],

        ~terminal_audit["actual_chase_win"] &
        terminal_audit["predicted_win_parity"],

        terminal_audit["actual_chase_win"] &
        terminal_audit["predicted_win_parity"]
    ],
    [
        "Missed Chase Win",
        "Incorrect Chase Win",
        "Correct Chase Win"
    ],
    default="Correct Chase Loss"
)

display(
    terminal_audit["outcome_class"]
    .value_counts()
    .rename_axis("outcome_class")
    .reset_index(name="matches")
)


# ============================================================
# 2. TERMINAL SCORE-GAP DISTRIBUTION
# ============================================================

print("\n2. TERMINAL SCORE GAPS")
print("=" * 70)

display(
    terminal_audit.groupby("outcome_class")
    .agg(
        matches=("match_id", "size"),
        average_gap=("terminal_gap_parity", "mean"),
        median_gap=("terminal_gap_parity", "median"),
        minimum_gap=("terminal_gap_parity", "min"),
        maximum_gap=("terminal_gap_parity", "max")
    )
    .reset_index()
)


# ============================================================
# 3. INVESTIGATE MISSED CHASE VICTORIES
# ============================================================

print("\n3. MISSED CHASE VICTORY GAPS")
print("=" * 70)

missed_wins = terminal_audit.loc[
    terminal_audit["outcome_class"].eq(
        "Missed Chase Win"
    )
].copy()

print("Missed chase victories:", len(missed_wins))

display(
    missed_wins["terminal_gap_parity"].describe()
)

display(
    missed_wins.sort_values(
        "terminal_gap_parity",
        ascending=False
    )[
        [
            "match_id",
            "result",
            "balls_remaining_after",
            "parity_score_after",
            "target_parity",
            "terminal_gap_parity"
        ]
    ].head(20)
)


# ============================================================
# 4. EXTRAS CONTEXT FOR OUTCOME ERRORS
# ============================================================

print("\n4. EXTRAS CONTEXT BY OUTCOME CLASS")
print("=" * 70)

extras_by_innings = (
    score_state.groupby(
        ["match_id", "innings"]
    )
    .agg(
        extras_events=(
            "assumption_dependent_event", "sum"
        ),
        parity_added_runs=(
            "parity_adjustment", "sum"
        )
    )
    .reset_index()
)

extras_pivot = extras_by_innings.pivot(
    index="match_id",
    columns="innings",
    values=[
        "extras_events",
        "parity_added_runs"
    ]
)

extras_pivot.columns = [
    f"{metric}_inn{int(innings)}"
    for metric, innings in extras_pivot.columns
]

extras_pivot = extras_pivot.reset_index()

terminal_extras_audit = terminal_audit.merge(
    extras_pivot,
    on="match_id",
    how="left",
    validate="one_to_one"
)

display(
    terminal_extras_audit.groupby("outcome_class")
    .agg(
        matches=("match_id", "size"),
        avg_extras_inn1=("extras_events_inn1", "mean"),
        avg_extras_inn2=("extras_events_inn2", "mean"),
        avg_parity_added_inn1=(
            "parity_added_runs_inn1", "mean"
        ),
        avg_parity_added_inn2=(
            "parity_added_runs_inn2", "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 5. CHECK PREMATURELY REACHED TARGETS
# ============================================================

print("\n5. PREMATURE TARGET CROSSINGS")
print("=" * 70)

premature_results = []

for scenario in ["observed", "one_run", "parity"]:

    premature_mask = (
        chase_audit[f"{scenario}_score_before"] >=
        chase_audit[f"target_{scenario}"]
    )

    premature_results.append({
        "scenario": scenario,
        "deliveries": int(premature_mask.sum()),
        "matches": chase_audit.loc[
            premature_mask, "match_id"
        ].nunique()
    })

display(pd.DataFrame(premature_results))

premature_parity = chase_audit.loc[
    chase_audit["parity_score_before"] >=
    chase_audit["target_parity"]
].copy()

display(
    premature_parity[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "parity_score_before",
            "target_parity",
            "balls_remaining_before",
            "assumption_events_before"
        ]
    ].groupby(
        "match_id",
        group_keys=False
    ).head(1).head(20)
)


# ============================================================
# 6. FINAL DATA INTEGRITY CHECKS
# ============================================================

print("\n6. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "outcome_records_lost": (
        len(outcome_sample) - len(terminal_audit)
    ),

    "unclassified_outcomes": int(
        terminal_audit["outcome_class"].isna().sum()
    ),

    "missing_terminal_gaps": int(
        terminal_audit[
            "terminal_gap_parity"
        ].isna().sum()
    ),

    "missing_extras_context": int(
        terminal_extras_audit[
            [
                "extras_events_inn1",
                "extras_events_inn2"
            ]
        ].isna().any(axis=1).sum()
    ),

    "duplicate_match_ids": int(
        terminal_audit["match_id"]
        .duplicated().sum()
    ),

    "outcome_classification_total_error": abs(
        len(terminal_audit) -
        terminal_audit["outcome_class"]
        .value_counts().sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 51 — Pressure Classification Reliability Audit
# CORRECTED VERSION

print("=" * 70)
print("PRESSURE CLASSIFICATION RELIABILITY AUDIT")
print("=" * 70)

# Preserve previous chase analysis
chase_quality = chase_audit.copy()

scenarios = ["observed", "one_run", "parity"]

rrr_threshold = 12


# ============================================================
# 1. IDENTIFY TARGET-CROSSING CONFLICTS
# ============================================================

print("\n1. TARGET-CROSSING CONFLICTS")
print("=" * 70)

for scenario in scenarios:

    chase_quality[
        f"target_reached_before_{scenario}"
    ] = (
        chase_quality[
            f"{scenario}_score_before"
        ] >= chase_quality[
            f"target_{scenario}"
        ]
    )

crossing_columns = [
    f"target_reached_before_{scenario}"
    for scenario in scenarios
]

chase_quality["any_target_crossing_conflict"] = (
    chase_quality[crossing_columns].any(axis=1)
)

display(
    pd.DataFrame([
        {
            "scenario": scenario,
            "deliveries_after_estimated_target": int(
                chase_quality[
                    f"target_reached_before_{scenario}"
                ].sum()
            ),
            "affected_matches": (
                chase_quality.loc[
                    chase_quality[
                        f"target_reached_before_{scenario}"
                    ],
                    "match_id"
                ].nunique()
            )
        }
        for scenario in scenarios
    ])
)

print(
    "Deliveries with conflict in any scenario:",
    int(
        chase_quality[
            "any_target_crossing_conflict"
        ].sum()
    )
)


# ============================================================
# 2. CHECK HIGH-RRR AGREEMENT
# ============================================================

print("\n2. HIGH-RRR CLASSIFICATION AGREEMENT")
print("=" * 70)

high_rrr_columns = [
    f"high_rrr_{scenario}"
    for scenario in scenarios
]

chase_quality["high_rrr_votes"] = (
    chase_quality[high_rrr_columns]
    .astype(int)
    .sum(axis=1)
)

display(
    chase_quality["high_rrr_votes"]
    .value_counts()
    .sort_index()
    .rename_axis("scenarios_classifying_high_rrr")
    .reset_index(name="deliveries")
)


# ============================================================
# 3. CREATE DIAGNOSTIC RELIABILITY GROUPS
# ============================================================

print("\n3. PRESSURE RELIABILITY GROUPS")
print("=" * 70)

chase_quality["pressure_reliability_group"] = (
    np.select(
        [
            chase_quality[
                "any_target_crossing_conflict"
            ],

            chase_quality[
                "high_rrr_votes"
            ].isin([1, 2]),

            chase_quality[
                "high_rrr_votes"
            ].eq(3)
        ],
        [
            "Target-Crossing Conflict",
            "Scenario Disagreement",
            "Consistent High RRR"
        ],
        default="Consistent Below Threshold"
    )
)

display(
    chase_quality[
        "pressure_reliability_group"
    ]
    .value_counts()
    .rename_axis("reliability_group")
    .reset_index(name="deliveries")
)


# ============================================================
# 4. RELIABILITY BY BALLS REMAINING
# ============================================================

print("\n4. RELIABILITY BY BALLS REMAINING")
print("=" * 70)

display(
    pd.crosstab(
        chase_quality["balls_window"],
        chase_quality["pressure_reliability_group"]
    )
)

display(
    chase_quality.groupby(
        "balls_window",
        observed=True
    )
    .agg(
        deliveries=("match_id", "size"),

        target_conflict_rate=(
            "any_target_crossing_conflict",
            "mean"
        ),

        classification_disagreement_rate=(
            "high_rrr_disagreement",
            "mean"
        ),

        median_rrr_spread=(
            "rrr_scenario_spread",
            "median"
        )
    )
    .reset_index()
)


# ============================================================
# 5. COMPARE WICKET PRESSURE ACROSS GROUPS
# ============================================================

print("\n5. WICKET PRESSURE ACROSS RELIABILITY GROUPS")
print("=" * 70)

display(
    chase_quality.groupby(
        "pressure_reliability_group"
    )
    .agg(
        deliveries=("match_id", "size"),

        avg_wickets_remaining=(
            "wickets_remaining_before",
            "mean"
        ),

        avg_recent_wicket_shock=(
            "wicket_shock_12",
            "mean"
        ),

        avg_wicket_depletion=(
            "wicket_depletion_z",
            "mean"
        ),

        median_rrr_parity=(
            "rrr_parity",
            "median"
        ),

        median_rrr_spread=(
            "rrr_scenario_spread",
            "median"
        )
    )
    .reset_index()
)


# ============================================================
# 6. SAMPLE UNRELIABLE PRESSURE SITUATIONS
# ============================================================

print("\n6. SAMPLE UNCERTAIN PRESSURE SITUATIONS")
print("=" * 70)

display(
    chase_quality.loc[
        chase_quality[
            "pressure_reliability_group"
        ].isin([
            "Target-Crossing Conflict",
            "Scenario Disagreement"
        ]),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "balls_remaining_before",
            "wickets_remaining_before",
            "rrr_observed",
            "rrr_one_run",
            "rrr_parity",
            "rrr_scenario_spread",
            "pressure_reliability_group"
        ]
    ].head(25)
)


# ============================================================
# 7. DATA INTEGRITY CHECKS — CORRECTED
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

valid_groups = [
    "Target-Crossing Conflict",
    "Scenario Disagreement",
    "Consistent High RRR",
    "Consistent Below Threshold"
]

checks = {
    "row_count_difference": abs(
        len(chase_quality) - len(chase_audit)
    ),

    "missing_reliability_groups": int(
        chase_quality[
            "pressure_reliability_group"
        ].isna().sum()
    ),

    "invalid_reliability_groups": int(
        (
            ~chase_quality[
                "pressure_reliability_group"
            ].isin(valid_groups)
        ).sum()
    ),

    "invalid_high_rrr_votes": int(
        (
            ~chase_quality[
                "high_rrr_votes"
            ].between(0, 3)
        ).sum()
    ),

    "missing_rrr_spreads": int(
        chase_quality[
            "rrr_scenario_spread"
        ].isna().sum()
    ),

    "duplicate_delivery_keys": int(
        chase_quality.duplicated(
            [
                "match_id",
                "innings",
                "innings_ball_num"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 52 — Player-Level Scouting Foundation

print("=" * 70)
print("PLAYER-LEVEL SCOUTING FOUNDATION")
print("=" * 70)

# ============================================================
# 1. COMBINE MATCH-STATE AND CHASE CONTEXT
# ============================================================

print("\n1. PLAYER ANALYSIS COVERAGE")
print("=" * 70)

scout_base = score_state.copy()

player_keys = [
    "match_id",
    "innings",
    "innings_ball_num"
]

chase_context_columns = [
    "pressure_reliability_group",
    "rrr_parity",
    "rrr_scenario_spread",
    "high_rrr_votes"
]

chase_context = chase_quality[
    player_keys + chase_context_columns
].copy()

# Match key data types before joining
for key in player_keys:
    scout_base[key] = pd.to_numeric(
        scout_base[key], errors="raise"
    ).astype("int64")

    chase_context[key] = pd.to_numeric(
        chase_context[key], errors="raise"
    ).astype("int64")

scout_base = scout_base.merge(
    chase_context,
    on=player_keys,
    how="left",
    validate="one_to_one",
    sort=False
)

# Exclude every DLS-flagged match from the
# initial role-specific scouting sample.
scout_base["eligible_pressure_match"] = (
    ~scout_base["has_dls_result"]
)

player_events = scout_base.loc[
    scout_base["eligible_pressure_match"]
].copy()

print("Original standard-state deliveries:", len(score_state))
print("Player-analysis deliveries:", len(player_events))

print(
    "Eligible matches:",
    player_events["match_id"].nunique()
)

print(
    "Excluded DLS matches:",
    scout_base.loc[
        ~scout_base["eligible_pressure_match"],
        "match_id"
    ].nunique()
)

print(
    "Eligible chase deliveries with RRR context:",
    int(
        player_events["pressure_reliability_group"]
        .notna().sum()
    )
)


# ============================================================
# 2. DEFINE ROLE-SPECIFIC EVENT FLAGS
# ============================================================

print("\n2. ROLE-SPECIFIC EVENT DEFINITIONS")
print("=" * 70)

player_events["batter_runs"] = pd.to_numeric(
    player_events["runs"],
    errors="coerce"
)

# Provisional balls-faced definition:
# wides do not count as balls faced;
# "other" extras remain ambiguous and are excluded.
# Overstep/no-ball deliveries can count as balls faced.
player_events["batter_ball_proxy"] = (
    ~player_events["extras_type"].isin(
        ["wide", "other"]
    )
)

# Official legal-ball classification established earlier
player_events["bowler_legal_ball"] = (
    player_events["is_legal_delivery"].eq(1)
)

# Only dismissal types normally credited to the bowler
bowler_credited_types = {
    "caught",
    "caught and bowled",
    "bowled",
    "lbw",
    "stump",
    "stumped",
    "hit wicket"
}

dismissal_clean = (
    player_events["dismissal_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

player_events["bowler_credited_wicket_proxy"] = (
    player_events["is_wicket"].eq(True) &
    dismissal_clean.isin(bowler_credited_types)
)

# Diagnostic batting-side wicket pressure only.
# These are exploratory thresholds, not final labels.
player_events["batting_wicket_burden"] = (
    player_events["wicket_depletion_z"].ge(2) |
    player_events["wicket_shock_12"].ge(2)
)

player_events["chase_consensus_high_rrr"] = (
    player_events["pressure_reliability_group"].eq(
        "Consistent High RRR"
    )
)

display(
    pd.DataFrame(
        [
            ["Batter deliveries", len(player_events)],
            [
                "Provisional balls faced",
                int(player_events["batter_ball_proxy"].sum())
            ],
            [
                "Legal bowling deliveries",
                int(player_events["bowler_legal_ball"].sum())
            ],
            [
                "Confirmed wicket events",
                int(player_events["is_wicket"].eq(True).sum())
            ],
            [
                "Bowler-credited wicket proxy",
                int(
                    player_events[
                        "bowler_credited_wicket_proxy"
                    ].sum()
                )
            ],
            [
                "Elevated batting wicket context",
                int(
                    player_events[
                        "batting_wicket_burden"
                    ].sum()
                )
            ],
            [
                "Consistent high-RRR chase events",
                int(
                    player_events[
                        "chase_consensus_high_rrr"
                    ].sum()
                )
            ]
        ],
        columns=["metric", "count"]
    )
)


# ============================================================
# 3. BATTER-LEVEL OPPORTUNITY TABLE
# ============================================================

print("\n3. BATTER SAMPLE DISTRIBUTION")
print("=" * 70)

batting_exposure = (
    player_events.groupby("striker_id")
    .agg(
        player_name=("striker_name", "first"),
        matches=("match_id", "nunique"),
        delivery_events=("match_id", "size"),
        provisional_balls_faced=("batter_ball_proxy", "sum"),
        recorded_batter_runs=("batter_runs", "sum"),
        wicket_pressure_events=("batting_wicket_burden", "sum"),
        high_rrr_chase_events=("chase_consensus_high_rrr", "sum")
    )
    .reset_index()
)

print("Unique batters:", len(batting_exposure))

display(
    batting_exposure[
        [
            "matches",
            "delivery_events",
            "provisional_balls_faced",
            "recorded_batter_runs",
            "wicket_pressure_events",
            "high_rrr_chase_events"
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95]
    ).T
)


# ============================================================
# 4. BOWLER-LEVEL OPPORTUNITY TABLE
# ============================================================

print("\n4. BOWLER SAMPLE DISTRIBUTION")
print("=" * 70)

bowling_exposure = (
    player_events.groupby("bowler_id")
    .agg(
        player_name=("bowler_name", "first"),
        matches=("match_id", "nunique"),
        delivery_events=("match_id", "size"),
        legal_balls_bowled=("bowler_legal_ball", "sum"),
        credited_wickets=(
            "bowler_credited_wicket_proxy", "sum"
        ),
        high_rrr_chase_events=(
            "chase_consensus_high_rrr", "sum"
        )
    )
    .reset_index()
)

print("Unique bowlers:", len(bowling_exposure))

display(
    bowling_exposure[
        [
            "matches",
            "delivery_events",
            "legal_balls_bowled",
            "credited_wickets",
            "high_rrr_chase_events"
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95]
    ).T
)


# ============================================================
# 5. ROLE OPPORTUNITY BY INNINGS AND PHASE
# ============================================================

print("\n5. ROLE OPPORTUNITY BY INNINGS AND PHASE")
print("=" * 70)

display(
    player_events.groupby(
        ["innings", "match_phase"]
    )
    .agg(
        deliveries=("match_id", "size"),
        unique_batters=("striker_id", "nunique"),
        unique_bowlers=("bowler_id", "nunique"),
        provisional_balls_faced=("batter_ball_proxy", "sum"),
        legal_balls_bowled=("bowler_legal_ball", "sum"),
        wicket_pressure_events=("batting_wicket_burden", "sum"),
        high_rrr_chase_events=("chase_consensus_high_rrr", "sum")
    )
    .reset_index()
)


# ============================================================
# 6. SAMPLE PLAYER COVERAGE — NOT RANKINGS
# ============================================================

print("\n6. SAMPLE PLAYERS WITH HIGHEST EXPOSURE")
print("=" * 70)

print("\nBatters — most provisional balls faced")

display(
    batting_exposure.sort_values(
        "provisional_balls_faced",
        ascending=False
    ).head(12)
)

print("\nBowlers — most legal balls bowled")

display(
    bowling_exposure.sort_values(
        "legal_balls_bowled",
        ascending=False
    ).head(12)
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "merge_row_count_difference": abs(
        len(scout_base) - len(score_state)
    ),

    "duplicate_delivery_keys": int(
        scout_base.duplicated(player_keys).sum()
    ),

    "missing_role_identifiers": int(
        player_events[
            ["striker_id", "bowler_id"]
        ].isna().any(axis=1).sum()
    ),

    "missing_batter_runs": int(
        player_events["batter_runs"].isna().sum()
    ),

    "missing_eligible_chase_context": int(
        player_events.loc[
            player_events["innings"].eq(2),
            "pressure_reliability_group"
        ].isna().sum()
    ),

    "unexpected_first_innings_chase_context": int(
        player_events.loc[
            player_events["innings"].eq(1),
            "pressure_reliability_group"
        ].notna().sum()
    ),

    "ineligible_matches_in_player_sample": int(
        (
            ~player_events[
                "eligible_pressure_match"
            ]
        ).sum()
    ),

    "credited_wickets_without_wicket_flag": int(
        (
            player_events[
                "bowler_credited_wicket_proxy"
            ] &
            ~player_events["is_wicket"].eq(True)
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 53 — Batter Personal Baseline Feasibility

print("=" * 70)
print("BATTER PERSONAL BASELINE FEASIBILITY")
print("=" * 70)

# ============================================================
# 1. ESTABLISH BATTER OPPORTUNITY SAMPLE
# ============================================================

print("\n1. BATTER SAMPLE COVERAGE")
print("=" * 70)

# Retain provisional balls faced only
batting_scope = player_events.loc[
    player_events["batter_ball_proxy"] &
    player_events["striker_id"].notna()
].copy()

# For innings 2, exclude uncertain RRR classification
# from this exploratory comparison.
# All innings 1 deliveries remain eligible.
reliable_chase_groups = [
    "Consistent High RRR",
    "Consistent Below Threshold"
]

batting_scope["usable_pressure_context"] = (
    batting_scope["innings"].eq(1) |
    batting_scope["pressure_reliability_group"].isin(
        reliable_chase_groups
    )
)

batting_eval = batting_scope.loc[
    batting_scope["usable_pressure_context"]
].copy()

print("Original provisional batting balls:", len(batting_scope))
print("Usable comparison balls:", len(batting_eval))

print(
    "Excluded uncertain chase balls:",
    len(batting_scope) - len(batting_eval)
)

print(
    "Batters represented:",
    batting_eval["striker_id"].nunique()
)


# ============================================================
# 2. DEFINE EXPLORATORY BATTING PRESSURE
# ============================================================

print("\n2. PRESSURE CONTEXT BREAKDOWN")
print("=" * 70)

wicket_context = (
    batting_eval["batting_wicket_burden"]
)

high_rrr_context = (
    batting_eval["chase_consensus_high_rrr"]
)

batting_eval["batting_pressure_type"] = np.select(
    [
        wicket_context & high_rrr_context,
        wicket_context,
        high_rrr_context
    ],
    [
        "Both",
        "Wicket Pressure Only",
        "High Chase RRR Only"
    ],
    default="Reference Context"
)

batting_eval["is_pressure"] = (
    batting_eval["batting_pressure_type"] !=
    "Reference Context"
)

display(
    batting_eval.groupby("batting_pressure_type")
    .agg(
        balls=("striker_id", "size"),
        matches=("match_id", "nunique"),
        unique_batters=("striker_id", "nunique"),
        batter_runs=("batter_runs", "sum")
    )
    .reset_index()
)


# ============================================================
# 3. PRESSURE AND REFERENCE OPPORTUNITY BY PHASE
# ============================================================

print("\n3. OPPORTUNITY BY INNINGS AND PHASE")
print("=" * 70)

display(
    batting_eval.groupby(
        ["innings", "match_phase", "is_pressure"]
    )
    .agg(
        balls=("striker_id", "size"),
        batters=("striker_id", "nunique"),
        matches=("match_id", "nunique"),
        runs=("batter_runs", "sum")
    )
    .reset_index()
)


# ============================================================
# 4. BUILD PLAYER-SPECIFIC COMPARISON STRATA
# ============================================================

print("\n4. PLAYER-LEVEL MATCHED STRATA")
print("=" * 70)

stratum_keys = [
    "striker_id",
    "innings",
    "match_phase"
]

stratum_summary = (
    batting_eval.groupby(
        stratum_keys + ["is_pressure"]
    )
    .agg(
        player_name=("striker_name", "first"),
        balls=("striker_id", "size"),
        runs=("batter_runs", "sum"),
        matches=("match_id", "nunique"),
        dot_balls=(
            "batter_runs",
            lambda x: x.eq(0).sum()
        ),
        boundary_balls=(
            "batter_runs",
            lambda x: x.ge(4).sum()
        )
    )
    .reset_index()
)

# Reference: same player, innings, and phase,
# without the highlighted pressure conditions
reference_strata = (
    stratum_summary.loc[
        ~stratum_summary["is_pressure"]
    ]
    .drop(columns=["is_pressure", "player_name"])
    .rename(
        columns={
            "balls": "reference_balls",
            "runs": "reference_runs",
            "matches": "reference_matches",
            "dot_balls": "reference_dots",
            "boundary_balls": "reference_boundaries"
        }
    )
)

pressure_strata = (
    stratum_summary.loc[
        stratum_summary["is_pressure"]
    ]
    .drop(columns=["is_pressure"])
    .rename(
        columns={
            "balls": "pressure_balls",
            "runs": "pressure_runs",
            "matches": "pressure_matches",
            "dot_balls": "pressure_dots",
            "boundary_balls": "pressure_boundaries"
        }
    )
)

batter_comparisons = pressure_strata.merge(
    reference_strata,
    on=stratum_keys,
    how="left",
    validate="one_to_one"
)

reference_count_columns = [
    "reference_balls",
    "reference_runs",
    "reference_matches",
    "reference_dots",
    "reference_boundaries"
]

batter_comparisons[reference_count_columns] = (
    batter_comparisons[
        reference_count_columns
    ].fillna(0)
)

print(
    "Player-phase-innings strata with pressure:",
    len(batter_comparisons)
)

print(
    "Strata also having reference deliveries:",
    int(
        batter_comparisons["reference_balls"]
        .gt(0).sum()
    )
)

print(
    "Pressure strata without reference deliveries:",
    int(
        batter_comparisons["reference_balls"]
        .eq(0).sum()
    )
)


# ============================================================
# 5. CALCULATE PRELIMINARY PERSONAL BASELINES
# ============================================================

print("\n5. PERSONAL BASELINE DISTRIBUTION")
print("=" * 70)

batter_comparisons["pressure_strike_rate"] = (
    100 *
    batter_comparisons["pressure_runs"].div(
        batter_comparisons[
            "pressure_balls"
        ].replace(0, np.nan)
    )
)

batter_comparisons["reference_strike_rate"] = (
    100 *
    batter_comparisons["reference_runs"].div(
        batter_comparisons[
            "reference_balls"
        ].replace(0, np.nan)
    )
)

batter_comparisons["strike_rate_difference"] = (
    batter_comparisons["pressure_strike_rate"] -
    batter_comparisons["reference_strike_rate"]
)

display(
    batter_comparisons[
        [
            "pressure_balls",
            "reference_balls",
            "pressure_matches",
            "reference_matches",
            "pressure_strike_rate",
            "reference_strike_rate",
            "strike_rate_difference"
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90]
    ).T
)


# ============================================================
# 6. EVALUATE MINIMUM SAMPLE THRESHOLDS
# ============================================================

print("\n6. SAMPLE SUFFICIENCY SENSITIVITY")
print("=" * 70)

sample_thresholds = [
    (5, 20),
    (10, 30),
    (15, 50),
    (20, 75)
]

threshold_results = []

for min_pressure, min_reference in sample_thresholds:

    sufficient = (
        batter_comparisons["pressure_balls"].ge(
            min_pressure
        ) &
        batter_comparisons["reference_balls"].ge(
            min_reference
        ) &
        batter_comparisons["pressure_matches"].ge(2) &
        batter_comparisons["reference_matches"].ge(2)
    )

    threshold_results.append({
        "minimum_pressure_balls": min_pressure,
        "minimum_reference_balls": min_reference,
        "minimum_matches_each_context": 2,
        "eligible_strata": int(sufficient.sum()),
        "eligible_unique_batters": (
            batter_comparisons.loc[
                sufficient, "striker_id"
            ].nunique()
        )
    })

display(pd.DataFrame(threshold_results))


# ============================================================
# 7. SAMPLE HIGH-EXPOSURE BATTER COMPARISONS
# ============================================================

print("\n7. EXAMPLE COMPARABLE BATTERS — NOT RANKINGS")
print("=" * 70)

display(
    batter_comparisons.loc[
        batter_comparisons["reference_balls"].gt(0)
    ]
    .sort_values(
        ["pressure_balls", "reference_balls"],
        ascending=False
    )[
        [
            "striker_id",
            "player_name",
            "innings",
            "match_phase",
            "pressure_balls",
            "reference_balls",
            "pressure_matches",
            "reference_matches",
            "pressure_strike_rate",
            "reference_strike_rate",
            "strike_rate_difference"
        ]
    ].head(20)
)


# ============================================================
# 8. DATA INTEGRITY CHECKS
# ============================================================

print("\n8. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "pressure_label_accounting_error": abs(
        batting_eval[
            "batting_pressure_type"
        ].value_counts().sum() -
        len(batting_eval)
    ),

    "pressure_strata_accounting_error": abs(
        int(
            batter_comparisons[
                "pressure_balls"
            ].sum()
        ) -
        int(
            batting_eval[
                "is_pressure"
            ].sum()
        )
    ),

    "duplicate_player_strata": int(
        batter_comparisons.duplicated(
            stratum_keys
        ).sum()
    ),

    "missing_pressure_strike_rates": int(
        batter_comparisons[
            "pressure_strike_rate"
        ].isna().sum()
    ),

    "invalid_negative_ball_counts": int(
        (
            (batter_comparisons[
                "pressure_balls"
            ] < 0) |
            (batter_comparisons[
                "reference_balls"
            ] < 0)
        ).sum()
    ),

    "uncertain_chase_context_retained": int(
        (
            batting_eval["innings"].eq(2) &
            ~batting_eval[
                "pressure_reliability_group"
            ].isin(reliable_chase_groups)
        ).sum()
    ),

    "duplicate_delivery_keys": int(
        batting_eval.duplicated(
            [
                "match_id",
                "innings",
                "innings_ball_num"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 54 — Pressure-Type-Specific Batting Performance

print("=" * 70)
print("PRESSURE-TYPE-SPECIFIC BATTING PERFORMANCE")
print("=" * 70)

# Preserve the validated Cell 53 dataset
type_batting = batting_eval.copy()

reference_label = "Reference Context"

pressure_labels = [
    "Wicket Pressure Only",
    "High Chase RRR Only",
    "Both"
]

stratum_keys = [
    "striker_id",
    "innings",
    "match_phase"
]

context_keys = stratum_keys + [
    "batting_pressure_type"
]


# ============================================================
# 1. PRESSURE TYPE COVERAGE
# ============================================================

print("\n1. PRESSURE TYPE COVERAGE")
print("=" * 70)

display(
    type_batting.groupby("batting_pressure_type")
    .agg(
        balls=("striker_id", "size"),
        players=("striker_id", "nunique"),
        matches=("match_id", "nunique"),
        runs=("batter_runs", "sum")
    )
    .reset_index()
)


# ============================================================
# 2. BUILD REFERENCE AND PRESSURE STRATA
# ============================================================

print("\n2. BUILD PRESSURE-TYPE MATCHED STRATA")
print("=" * 70)

reference_type_strata = (
    type_batting.loc[
        type_batting["batting_pressure_type"].eq(
            reference_label
        )
    ]
    .groupby(stratum_keys)
    .agg(
        reference_balls=("batter_runs", "size"),
        reference_runs=("batter_runs", "sum"),
        reference_matches=("match_id", "nunique")
    )
    .reset_index()
)

pressure_type_strata = (
    type_batting.loc[
        type_batting["batting_pressure_type"].isin(
            pressure_labels
        )
    ]
    .groupby(context_keys)
    .agg(
        player_name=("striker_name", "first"),
        pressure_balls=("batter_runs", "size"),
        pressure_runs=("batter_runs", "sum"),
        pressure_matches=("match_id", "nunique")
    )
    .reset_index()
)

type_comparisons = pressure_type_strata.merge(
    reference_type_strata,
    on=stratum_keys,
    how="left",
    validate="many_to_one"
)

for col in [
    "reference_balls",
    "reference_runs",
    "reference_matches"
]:
    type_comparisons[col] = (
        type_comparisons[col].fillna(0)
    )

print("Pressure-type strata:", len(type_comparisons))

print(
    "Strata with reference observations:",
    int(type_comparisons["reference_balls"].gt(0).sum())
)

print(
    "Strata missing reference observations:",
    int(type_comparisons["reference_balls"].eq(0).sum())
)


# ============================================================
# 3. MATCHED REFERENCE COVERAGE BY PRESSURE TYPE
# ============================================================

print("\n3. REFERENCE COVERAGE BY PRESSURE TYPE")
print("=" * 70)

type_comparisons["matched_pressure_balls"] = (
    np.where(
        type_comparisons["reference_balls"].gt(0),
        type_comparisons["pressure_balls"],
        0
    )
)

display(
    type_comparisons.groupby("batting_pressure_type")
    .agg(
        pressure_strata=("striker_id", "size"),
        pressure_balls=("pressure_balls", "sum"),
        matched_pressure_balls=(
            "matched_pressure_balls", "sum"
        ),
        players=("striker_id", "nunique"),
        matched_strata=(
            "reference_balls",
            lambda x: int(x.gt(0).sum())
        )
    )
    .reset_index()
)


# ============================================================
# 4. SAMPLE SUFFICIENCY BY PRESSURE TYPE
# ============================================================

print("\n4. SAMPLE SUFFICIENCY BY PRESSURE TYPE")
print("=" * 70)

thresholds = [
    (5, 20),
    (10, 30),
    (15, 50),
    (20, 75)
]

type_threshold_results = []

for pressure_type, df in type_comparisons.groupby(
    "batting_pressure_type"
):

    for min_pressure, min_reference in thresholds:

        sufficient = (
            df["pressure_balls"].ge(min_pressure) &
            df["reference_balls"].ge(min_reference) &
            df["pressure_matches"].ge(2) &
            df["reference_matches"].ge(2)
        )

        type_threshold_results.append({
            "pressure_type": pressure_type,
            "min_pressure_balls": min_pressure,
            "min_reference_balls": min_reference,
            "min_matches_each": 2,
            "eligible_strata": int(sufficient.sum()),
            "eligible_batters": int(
                df.loc[sufficient, "striker_id"].nunique()
            ),
            "covered_pressure_balls": int(
                df.loc[sufficient, "pressure_balls"].sum()
            )
        })

display(
    pd.DataFrame(type_threshold_results)
)


# ============================================================
# 5. CALCULATE PERSONAL EXPECTED RUNS
# ============================================================

print("\n5. CALCULATE MATCHED PERSONAL EXPECTATIONS")
print("=" * 70)

# Exploratory threshold — not the final selection rule
type_comparisons["qualified_stratum"] = (
    type_comparisons["pressure_balls"].ge(10) &
    type_comparisons["reference_balls"].ge(30) &
    type_comparisons["pressure_matches"].ge(2) &
    type_comparisons["reference_matches"].ge(2)
)

qualified_type_strata = type_comparisons.loc[
    type_comparisons["qualified_stratum"]
].copy()

qualified_type_strata["reference_runs_per_ball"] = (
    qualified_type_strata["reference_runs"] /
    qualified_type_strata["reference_balls"]
)

qualified_type_strata["expected_pressure_runs"] = (
    qualified_type_strata["pressure_balls"] *
    qualified_type_strata["reference_runs_per_ball"]
)

qualified_type_strata["runs_above_baseline"] = (
    qualified_type_strata["pressure_runs"] -
    qualified_type_strata["expected_pressure_runs"]
)

qualified_type_strata["strike_rate_uplift"] = (
    100 *
    qualified_type_strata["runs_above_baseline"] /
    qualified_type_strata["pressure_balls"]
)

print("Qualified strata:", len(qualified_type_strata))

print(
    "Unique qualified batters:",
    qualified_type_strata["striker_id"].nunique()
)

display(
    qualified_type_strata[
        [
            "pressure_balls",
            "reference_balls",
            "expected_pressure_runs",
            "runs_above_baseline",
            "strike_rate_uplift"
        ]
    ].describe().T
)


# ============================================================
# 6. AGGREGATE PERSONAL BASELINES BY PRESSURE TYPE
# ============================================================

print("\n6. PLAYER-LEVEL PRESSURE TYPE PROFILES")
print("=" * 70)

player_context_keys = [
    "striker_id",
    "batting_pressure_type"
]

batter_type_profiles = (
    qualified_type_strata.groupby(
        player_context_keys
    )
    .agg(
        player_name=("player_name", "first"),
        qualified_strata=("match_phase", "size"),
        pressure_balls=("pressure_balls", "sum"),
        reference_balls=("reference_balls", "sum"),
        actual_pressure_runs=("pressure_runs", "sum"),
        expected_pressure_runs=(
            "expected_pressure_runs", "sum"
        )
    )
    .reset_index()
)

# Calculate distinct matches correctly across all
# qualified strata, without double-counting a match.
qualified_keys = qualified_type_strata[
    context_keys
].drop_duplicates()

qualified_pressure_events = (
    type_batting.loc[
        type_batting["batting_pressure_type"].isin(
            pressure_labels
        )
    ]
    .merge(
        qualified_keys,
        on=context_keys,
        how="inner",
        validate="many_to_one"
    )
)

distinct_pressure_matches = (
    qualified_pressure_events.groupby(
        player_context_keys
    )
    .agg(
        distinct_pressure_matches=(
            "match_id", "nunique"
        )
    )
    .reset_index()
)

batter_type_profiles = batter_type_profiles.merge(
    distinct_pressure_matches,
    on=player_context_keys,
    how="left",
    validate="one_to_one"
)

batter_type_profiles["actual_pressure_sr"] = (
    100 *
    batter_type_profiles["actual_pressure_runs"] /
    batter_type_profiles["pressure_balls"]
)

batter_type_profiles["matched_baseline_sr"] = (
    100 *
    batter_type_profiles["expected_pressure_runs"] /
    batter_type_profiles["pressure_balls"]
)

batter_type_profiles["runs_above_baseline"] = (
    batter_type_profiles["actual_pressure_runs"] -
    batter_type_profiles["expected_pressure_runs"]
)

batter_type_profiles["matched_sr_uplift"] = (
    100 *
    batter_type_profiles["runs_above_baseline"] /
    batter_type_profiles["pressure_balls"]
)

# Additional exploratory player-level volume criterion
batter_type_profiles["diagnostic_volume_pass"] = (
    batter_type_profiles["pressure_balls"].ge(30) &
    batter_type_profiles["distinct_pressure_matches"].ge(3)
)

display(
    batter_type_profiles.groupby(
        "batting_pressure_type"
    )
    .agg(
        players=("striker_id", "nunique"),
        qualified_pressure_balls=(
            "pressure_balls", "sum"
        ),
        players_with_30_balls_3_matches=(
            "diagnostic_volume_pass", "sum"
        ),
        median_matched_sr_uplift=(
            "matched_sr_uplift", "median"
        )
    )
    .reset_index()
)


# ============================================================
# 7. HIGH-EXPOSURE PLAYER EXAMPLES — NOT RANKINGS
# ============================================================

print("\n7. HIGH-EXPOSURE COMPARISONS — NOT RANKINGS")
print("=" * 70)

profile_columns = [
    "player_name",
    "batting_pressure_type",
    "qualified_strata",
    "pressure_balls",
    "distinct_pressure_matches",
    "actual_pressure_sr",
    "matched_baseline_sr",
    "matched_sr_uplift",
    "runs_above_baseline"
]

for pressure_type in pressure_labels:

    print("\nPressure type:", pressure_type)

    sample = batter_type_profiles.loc[
        batter_type_profiles[
            "batting_pressure_type"
        ].eq(pressure_type)
    ]

    display(
        sample.sort_values(
            "pressure_balls",
            ascending=False
        )[profile_columns].head(8)
    )


# ============================================================
# 8. DATA INTEGRITY CHECKS
# ============================================================

print("\n8. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "pressure_ball_accounting_error": abs(
        int(type_comparisons["pressure_balls"].sum()) -
        int(type_batting["is_pressure"].sum())
    ),

    "reference_ball_accounting_error": abs(
        int(reference_type_strata["reference_balls"].sum()) -
        int(
            type_batting[
                "batting_pressure_type"
            ].eq(reference_label).sum()
        )
    ),

    "qualified_ball_reconciliation_error": abs(
        int(qualified_type_strata["pressure_balls"].sum()) -
        len(qualified_pressure_events)
    ),

    "duplicate_pressure_type_strata": int(
        type_comparisons.duplicated(
            context_keys
        ).sum()
    ),

    "duplicate_player_context_profiles": int(
        batter_type_profiles.duplicated(
            player_context_keys
        ).sum()
    ),

    "missing_expected_runs": int(
        qualified_type_strata[
            "expected_pressure_runs"
        ].isna().sum()
    ),

    "negative_expected_runs": int(
        qualified_type_strata[
            "expected_pressure_runs"
        ].lt(0).sum()
    ),

    "missing_distinct_match_counts": int(
        batter_type_profiles[
            "distinct_pressure_matches"
        ].isna().sum()
    ),

    "high_chase_pressure_in_first_innings": int(
        (
            type_batting["batting_pressure_type"].isin(
                ["High Chase RRR Only", "Both"]
            ) &
            type_batting["innings"].ne(2)
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 55 — Match-Cluster Bootstrap for Batting Performance

print("=" * 70)
print("MATCH-CLUSTER BOOTSTRAP — BATTING PRESSURE")
print("=" * 70)

rng = np.random.default_rng(2027)
n_bootstrap = 400

stratum_keys = [
    "striker_id",
    "innings",
    "match_phase"
]

context_keys = stratum_keys + [
    "batting_pressure_type"
]

profile_keys = [
    "striker_id",
    "batting_pressure_type"
]

match_keys = profile_keys + [
    "match_id",
    "innings",
    "match_phase"
]


# ============================================================
# 1. ISOLATE QUALIFIED PRESSURE AND REFERENCE EVENTS
# ============================================================

print("\n1. QUALIFIED MATCHED EVENT COVERAGE")
print("=" * 70)

qualified_keys = (
    qualified_type_strata[context_keys]
    .drop_duplicates()
    .copy()
)

qualified_pressure_events_boot = (
    type_batting.loc[
        type_batting["is_pressure"]
    ]
    .merge(
        qualified_keys,
        on=context_keys,
        how="inner",
        validate="many_to_one"
    )
)

# A reference stratum may legitimately support more than
# one pressure type. Replication across those types is
# intentional, not accidental double-counting.
reference_source = type_batting.loc[
    ~type_batting["is_pressure"],
    [
        "striker_id",
        "match_id",
        "innings",
        "match_phase",
        "batter_runs"
    ]
].copy()

qualified_reference_events_boot = (
    reference_source.merge(
        qualified_keys,
        on=stratum_keys,
        how="inner",
        validate="many_to_many"
    )
)

print(
    "Qualified pressure observations:",
    len(qualified_pressure_events_boot)
)

print(
    "Matched reference observations:",
    len(qualified_reference_events_boot)
)

print(
    "Unique qualified batters:",
    qualified_keys["striker_id"].nunique()
)

print(
    "Qualified player-pressure profiles:",
    qualified_keys[profile_keys]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# 2. AGGREGATE OBSERVATIONS TO THE MATCH LEVEL
# ============================================================

print("\n2. MATCH-LEVEL AGGREGATION")
print("=" * 70)

pressure_by_match = (
    qualified_pressure_events_boot.groupby(match_keys)
    .agg(
        pressure_balls=("batter_runs", "size"),
        pressure_runs=("batter_runs", "sum")
    )
    .reset_index()
)

reference_by_match = (
    qualified_reference_events_boot.groupby(match_keys)
    .agg(
        reference_balls=("batter_runs", "size"),
        reference_runs=("batter_runs", "sum")
    )
    .reset_index()
)

bootstrap_match_data = pressure_by_match.merge(
    reference_by_match,
    on=match_keys,
    how="outer",
    validate="one_to_one"
)

measure_cols = [
    "pressure_balls",
    "pressure_runs",
    "reference_balls",
    "reference_runs"
]

bootstrap_match_data[measure_cols] = (
    bootstrap_match_data[measure_cols].fillna(0)
)

print(
    "Player-context-match-stratum records:",
    len(bootstrap_match_data)
)

print(
    "Distinct player-context profiles:",
    bootstrap_match_data[profile_keys]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# 3. RESAMPLE MATCHES AND RECOMPUTE PERSONAL BASELINES
# ============================================================

print("\n3. MATCH-CLUSTER BOOTSTRAP")
print("=" * 70)

# Preserve the exact strata qualified in Cell 54.
# Qualification is not recalculated during resampling.
strata_lookup = {
    key: list(
        zip(
            group["innings"].tolist(),
            group["match_phase"].tolist()
        )
    )
    for key, group in qualified_type_strata.groupby(
        profile_keys
    )
}

bootstrap_results = []

for key, group in bootstrap_match_data.groupby(
    profile_keys, sort=False
):

    player_id, pressure_type = key

    strata = strata_lookup[key]

    match_ids = group["match_id"].unique().tolist()

    match_index = {
        match_id: i
        for i, match_id in enumerate(match_ids)
    }

    stratum_index = {
        stratum: j
        for j, stratum in enumerate(strata)
    }

    m = len(match_ids)
    s = len(strata)

    p_balls = np.zeros((m, s), dtype=float)
    p_runs = np.zeros((m, s), dtype=float)
    r_balls = np.zeros((m, s), dtype=float)
    r_runs = np.zeros((m, s), dtype=float)

    for record in group.itertuples(index=False):

        i = match_index[record.match_id]

        j = stratum_index[
            (record.innings, record.match_phase)
        ]

        p_balls[i, j] = record.pressure_balls
        p_runs[i, j] = record.pressure_runs

        r_balls[i, j] = record.reference_balls
        r_runs[i, j] = record.reference_runs

    # Reconstruct the original point estimate
    original_p_balls = p_balls.sum(axis=0)
    original_r_balls = r_balls.sum(axis=0)

    original_r_rates = np.divide(
        r_runs.sum(axis=0),
        original_r_balls,
        out=np.zeros(s, dtype=float),
        where=original_r_balls > 0
    )

    original_expected = (
        original_p_balls * original_r_rates
    ).sum()

    original_actual = p_runs.sum()

    original_total_balls = original_p_balls.sum()

    reconstructed_uplift = (
        100 *
        (original_actual - original_expected) /
        original_total_balls
    )

    # Cluster bootstrap: resample whole matches
    weights = rng.multinomial(
        m,
        np.full(m, 1.0 / m),
        size=n_bootstrap
    )

    bp = weights @ p_balls
    rp = weights @ p_runs

    br = weights @ r_balls
    rr = weights @ r_runs

    # Recalculate reference scoring rates
    # independently for every resample.
    reference_rates = np.divide(
        rr,
        br,
        out=np.zeros_like(rr, dtype=float),
        where=br > 0
    )

    total_pressure_balls = bp.sum(axis=1)

    # A replicate cannot be evaluated when it contains
    # pressure balls in a stratum with no reference balls.
    missing_reference = (
        (bp > 0) & (br == 0)
    ).any(axis=1)

    valid_replicates = (
        (total_pressure_balls > 0) &
        ~missing_reference
    )

    uplift_samples = np.full(
        n_bootstrap,
        np.nan,
        dtype=float
    )

    expected_runs = (
        bp * reference_rates
    ).sum(axis=1)

    actual_runs = rp.sum(axis=1)

    uplift_samples[valid_replicates] = (
        100 *
        (
            actual_runs[valid_replicates] -
            expected_runs[valid_replicates]
        ) /
        total_pressure_balls[valid_replicates]
    )

    valid_values = uplift_samples[
        np.isfinite(uplift_samples)
    ]

    valid_count = len(valid_values)

    # Require at least 80% evaluable bootstrap draws
    enough_bootstrap_coverage = (
        valid_count >= 0.80 * n_bootstrap
    )

    if enough_bootstrap_coverage:

        lower, median, upper = np.quantile(
            valid_values,
            [0.025, 0.50, 0.975]
        )

        positive_share = (
            valid_values > 0
        ).mean()

    else:

        lower = np.nan
        median = np.nan
        upper = np.nan
        positive_share = np.nan

    bootstrap_results.append({
        "striker_id": player_id,
        "batting_pressure_type": pressure_type,
        "match_clusters": m,
        "bootstrap_valid_draws": valid_count,
        "bootstrap_valid_fraction": (
            valid_count / n_bootstrap
        ),
        "reconstructed_uplift": reconstructed_uplift,
        "bootstrap_lower_95": lower,
        "bootstrap_median": median,
        "bootstrap_upper_95": upper,
        "bootstrap_positive_share": positive_share
    })

bootstrap_estimates = pd.DataFrame(
    bootstrap_results
)

print("Bootstrap repetitions:", n_bootstrap)

print(
    "Profiles evaluated:",
    len(bootstrap_estimates)
)

print(
    "Profiles with sufficient bootstrap coverage:",
    int(
        bootstrap_estimates[
            "bootstrap_lower_95"
        ].notna().sum()
    )
)


# ============================================================
# 4. COMBINE BOOTSTRAP RESULTS WITH PLAYER PROFILES
# ============================================================

print("\n4. UNCERTAINTY-ADJUSTED PLAYER PROFILES")
print("=" * 70)

batter_uncertainty_profiles = (
    batter_type_profiles.merge(
        bootstrap_estimates,
        on=profile_keys,
        how="left",
        validate="one_to_one"
    )
)

batter_uncertainty_profiles["interval_width"] = (
    batter_uncertainty_profiles[
        "bootstrap_upper_95"
    ] -
    batter_uncertainty_profiles[
        "bootstrap_lower_95"
    ]
)

batter_uncertainty_profiles["interval_assessment"] = (
    np.select(
        [
            batter_uncertainty_profiles[
                "bootstrap_lower_95"
            ].gt(0),

            batter_uncertainty_profiles[
                "bootstrap_upper_95"
            ].lt(0),

            batter_uncertainty_profiles[
                "bootstrap_lower_95"
            ].notna()
        ],
        [
            "Interval Above Baseline",
            "Interval Below Baseline",
            "Interval Crosses Zero"
        ],
        default="Insufficient Bootstrap Coverage"
    )
)

display(
    batter_uncertainty_profiles[
        [
            "pressure_balls",
            "distinct_pressure_matches",
            "matched_sr_uplift",
            "bootstrap_lower_95",
            "bootstrap_upper_95",
            "interval_width",
            "bootstrap_valid_fraction"
        ]
    ].describe().T
)


# ============================================================
# 5. UNCERTAINTY BY PRESSURE TYPE
# ============================================================

print("\n5. BOOTSTRAP INTERVAL SUMMARY")
print("=" * 70)

display(
    pd.crosstab(
        batter_uncertainty_profiles[
            "batting_pressure_type"
        ],
        batter_uncertainty_profiles[
            "interval_assessment"
        ]
    )
)

display(
    batter_uncertainty_profiles.groupby(
        "batting_pressure_type"
    )
    .agg(
        players=("striker_id", "nunique"),
        median_pressure_balls=("pressure_balls", "median"),
        median_match_clusters=("match_clusters", "median"),
        median_interval_width=("interval_width", "median"),
        median_valid_fraction=(
            "bootstrap_valid_fraction", "median"
        )
    )
    .reset_index()
)


# ============================================================
# 6. HIGH-EXPOSURE PLAYER INTERVALS — NOT RANKINGS
# ============================================================

print("\n6. HIGH-EXPOSURE PLAYER INTERVALS")
print("=" * 70)

columns_to_show = [
    "player_name",
    "pressure_balls",
    "distinct_pressure_matches",
    "matched_sr_uplift",
    "bootstrap_lower_95",
    "bootstrap_upper_95",
    "bootstrap_valid_fraction",
    "interval_assessment"
]

for pressure_type in [
    "Wicket Pressure Only",
    "High Chase RRR Only",
    "Both"
]:

    print("\nPressure type:", pressure_type)

    display(
        batter_uncertainty_profiles.loc[
            batter_uncertainty_profiles[
                "batting_pressure_type"
            ].eq(pressure_type)
        ]
        .sort_values(
            "pressure_balls",
            ascending=False
        )[columns_to_show]
        .head(8)
    )


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

expected_pressure_balls = int(
    qualified_type_strata["pressure_balls"].sum()
)

expected_reference_balls = int(
    qualified_type_strata["reference_balls"].sum()
)

point_check = batter_uncertainty_profiles[
    [
        "matched_sr_uplift",
        "reconstructed_uplift"
    ]
].dropna()

checks = {
    "pressure_ball_reconciliation_error": abs(
        len(qualified_pressure_events_boot) -
        expected_pressure_balls
    ),

    "reference_ball_reconciliation_error": abs(
        len(qualified_reference_events_boot) -
        expected_reference_balls
    ),

    "missing_bootstrap_profiles": int(
        batter_uncertainty_profiles[
            "reconstructed_uplift"
        ].isna().sum()
    ),

    "duplicate_player_profiles": int(
        batter_uncertainty_profiles.duplicated(
            profile_keys
        ).sum()
    ),

    "point_estimate_mismatches": int(
        (
            ~np.isclose(
                point_check["matched_sr_uplift"],
                point_check["reconstructed_uplift"],
                rtol=1e-8,
                atol=1e-7
            )
        ).sum()
    ),

    "invalid_bootstrap_counts": int(
        (
            ~batter_uncertainty_profiles[
                "bootstrap_valid_draws"
            ].between(0, n_bootstrap)
        ).sum()
    ),

    "reversed_bootstrap_intervals": int(
        (
            batter_uncertainty_profiles[
                "bootstrap_lower_95"
            ] >
            batter_uncertainty_profiles[
                "bootstrap_upper_95"
            ]
        ).sum()
    ),

    "missing_pressure_type": int(
        batter_uncertainty_profiles[
            "batting_pressure_type"
        ].isna().sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 56 — Bowling Pressure and Personal Baseline Feasibility

print("=" * 70)
print("BOWLING PRESSURE AND PERSONAL BASELINE FEASIBILITY")
print("=" * 70)

# ============================================================
# 1. ESTABLISH BOWLING ANALYSIS SAMPLE
# ============================================================

print("\n1. BOWLING SAMPLE COVERAGE")
print("=" * 70)

bowling_eval = player_events.copy()

delivery_keys = [
    "match_id",
    "innings",
    "innings_ball_num"
]

chase_bowling_context = chase_quality[
    delivery_keys + [
        "rrr_observed",
        "rrr_one_run",
        "any_target_crossing_conflict"
    ]
].copy()

# Ensure compatible join-key types
for key in delivery_keys:

    bowling_eval[key] = pd.to_numeric(
        bowling_eval[key], errors="raise"
    ).astype("int64")

    chase_bowling_context[key] = pd.to_numeric(
        chase_bowling_context[key], errors="raise"
    ).astype("int64")

bowling_eval = bowling_eval.merge(
    chase_bowling_context,
    on=delivery_keys,
    how="left",
    validate="one_to_one",
    sort=False
)

print("Bowling delivery events:", len(bowling_eval))

print(
    "Unique bowlers:",
    bowling_eval["bowler_id"].nunique()
)

print(
    "Unique matches:",
    bowling_eval["match_id"].nunique()
)

print(
    "Second-innings deliveries:",
    int(bowling_eval["innings"].eq(2).sum())
)


# ============================================================
# 2. DEFINE CONSERVATIVE BOWLING OUTCOME PROXIES
# ============================================================

print("\n2. BOWLING OUTCOME DEFINITIONS")
print("=" * 70)

bowling_eval["clean_dot_proxy"] = (
    bowling_eval["bowler_legal_ball"] &
    bowling_eval["extras_type"].isna() &
    bowling_eval["batter_runs"].eq(0) &
    bowling_eval["extras_2_runs"].fillna(0).eq(0)
)

bowling_eval["batter_boundary_proxy"] = (
    bowling_eval["is_four"].eq(1) |
    bowling_eval["is_six"].eq(1)
)

bowling_eval["credited_wicket_proxy"] = (
    bowling_eval["bowler_credited_wicket_proxy"]
)

display(
    pd.DataFrame(
        [
            [
                "All bowling delivery events",
                len(bowling_eval)
            ],
            [
                "Legal bowling deliveries",
                int(
                    bowling_eval[
                        "bowler_legal_ball"
                    ].sum()
                )
            ],
            [
                "Conservative clean-dot events",
                int(
                    bowling_eval[
                        "clean_dot_proxy"
                    ].sum()
                )
            ],
            [
                "Batter boundary events",
                int(
                    bowling_eval[
                        "batter_boundary_proxy"
                    ].sum()
                )
            ],
            [
                "Bowler-credited wicket events",
                int(
                    bowling_eval[
                        "credited_wicket_proxy"
                    ].sum()
                )
            ]
        ],
        columns=["metric", "count"]
    )
)


# ============================================================
# 3. DEFINE EXPLORATORY BOWLING PRESSURE
# ============================================================

print("\n3. BOWLING PRESSURE CONTEXTS")
print("=" * 70)

# Scenario 1:
# First-innings death overs against a batting team
# with at least six wickets still available.

first_innings_attack = (
    bowling_eval["innings"].eq(1) &
    bowling_eval["match_phase"].eq("Death Overs") &
    bowling_eval["wickets_remaining_before"].ge(6)
)

# Scenario 2:
# Second-innings tight defence, with all three
# reconstructed scoring scenarios in agreement.

rrr_columns = [
    "rrr_observed",
    "rrr_one_run",
    "rrr_parity"
]

consensus_tight_rrr = (
    bowling_eval[rrr_columns].gt(0).all(axis=1) &
    bowling_eval[rrr_columns].le(8).all(axis=1)
)

no_target_conflict = (
    ~bowling_eval[
        "any_target_crossing_conflict"
    ].fillna(False).astype(bool)
)

second_innings_tight_defence = (
    bowling_eval["innings"].eq(2) &
    bowling_eval[
        "balls_remaining_before"
    ].between(1, 36) &
    bowling_eval[
        "wickets_remaining_before"
    ].ge(3) &
    consensus_tight_rrr &
    no_target_conflict
)

bowling_pressure_labels = [
    "First-Innings Death: Resources Intact",
    "Second-Innings: Tight Defence"
]

bowling_eval["bowling_pressure_type"] = np.select(
    [
        first_innings_attack,
        second_innings_tight_defence
    ],
    bowling_pressure_labels,
    default="Reference Context"
)

bowling_eval["is_bowling_pressure"] = (
    bowling_eval["bowling_pressure_type"] !=
    "Reference Context"
)

display(
    bowling_eval.groupby("bowling_pressure_type")
    .agg(
        delivery_events=("bowler_id", "size"),
        legal_balls=("bowler_legal_ball", "sum"),
        matches=("match_id", "nunique"),
        bowlers=("bowler_id", "nunique"),
        clean_dots=("clean_dot_proxy", "sum"),
        boundaries=("batter_boundary_proxy", "sum"),
        credited_wickets=("credited_wicket_proxy", "sum")
    )
    .reset_index()
)


# ============================================================
# 4. OPPORTUNITY BY INNINGS AND MATCH PHASE
# ============================================================

print("\n4. BOWLING PRESSURE BY INNINGS AND PHASE")
print("=" * 70)

display(
    bowling_eval.groupby(
        ["innings", "match_phase", "bowling_pressure_type"]
    )
    .agg(
        delivery_events=("bowler_id", "size"),
        legal_balls=("bowler_legal_ball", "sum"),
        unique_bowlers=("bowler_id", "nunique"),
        unique_matches=("match_id", "nunique"),
        clean_dots=("clean_dot_proxy", "sum"),
        credited_wickets=("credited_wicket_proxy", "sum")
    )
    .reset_index()
)


# ============================================================
# 5. PLAYER-MATCHED BOWLING STRATA
# ============================================================

print("\n5. BOWLER PERSONAL BASELINE FEASIBILITY")
print("=" * 70)

bowler_stratum_keys = [
    "bowler_id",
    "innings",
    "match_phase"
]

bowling_strata = (
    bowling_eval.groupby(
        bowler_stratum_keys + ["bowling_pressure_type"]
    )
    .agg(
        player_name=("bowler_name", "first"),
        delivery_events=("bowler_id", "size"),
        legal_balls=("bowler_legal_ball", "sum"),
        matches=("match_id", "nunique"),
        clean_dots=("clean_dot_proxy", "sum"),
        boundaries=("batter_boundary_proxy", "sum"),
        wickets=("credited_wicket_proxy", "sum")
    )
    .reset_index()
)

reference_bowling = (
    bowling_strata.loc[
        bowling_strata[
            "bowling_pressure_type"
        ].eq("Reference Context")
    ]
    .drop(
        columns=["bowling_pressure_type", "player_name"]
    )
    .rename(
        columns={
            "delivery_events": "reference_events",
            "legal_balls": "reference_legal_balls",
            "matches": "reference_matches",
            "clean_dots": "reference_clean_dots",
            "boundaries": "reference_boundaries",
            "wickets": "reference_wickets"
        }
    )
)

pressure_bowling = (
    bowling_strata.loc[
        bowling_strata["bowling_pressure_type"].isin(
            bowling_pressure_labels
        )
    ]
    .rename(
        columns={
            "delivery_events": "pressure_events",
            "legal_balls": "pressure_legal_balls",
            "matches": "pressure_matches",
            "clean_dots": "pressure_clean_dots",
            "boundaries": "pressure_boundaries",
            "wickets": "pressure_wickets"
        }
    )
)

bowler_comparisons = pressure_bowling.merge(
    reference_bowling,
    on=bowler_stratum_keys,
    how="left",
    validate="many_to_one"
)

reference_columns = [
    "reference_events",
    "reference_legal_balls",
    "reference_matches",
    "reference_clean_dots",
    "reference_boundaries",
    "reference_wickets"
]

bowler_comparisons[reference_columns] = (
    bowler_comparisons[reference_columns].fillna(0)
)

print("Pressure bowling strata:", len(bowler_comparisons))

print(
    "Strata with personal references:",
    int(
        bowler_comparisons[
            "reference_legal_balls"
        ].gt(0).sum()
    )
)

print(
    "Strata without personal references:",
    int(
        bowler_comparisons[
            "reference_legal_balls"
        ].eq(0).sum()
    )
)


# ============================================================
# 6. SAMPLE THRESHOLDS AND PLAYER EXPOSURE
# ============================================================

print("\n6. SAMPLE SUFFICIENCY BY BOWLING CONTEXT")
print("=" * 70)

thresholds = [
    (5, 20),
    (10, 30),
    (15, 50),
    (20, 75)
]

bowling_threshold_results = []

for context, df in bowler_comparisons.groupby(
    "bowling_pressure_type"
):

    for min_pressure, min_reference in thresholds:

        qualified = (
            df["pressure_legal_balls"].ge(min_pressure) &
            df["reference_legal_balls"].ge(min_reference) &
            df["pressure_matches"].ge(2) &
            df["reference_matches"].ge(2)
        )

        bowling_threshold_results.append({
            "pressure_type": context,
            "min_pressure_balls": min_pressure,
            "min_reference_balls": min_reference,
            "min_matches_each": 2,
            "qualified_strata": int(qualified.sum()),
            "qualified_bowlers": int(
                df.loc[qualified, "bowler_id"].nunique()
            ),
            "qualified_pressure_balls": int(
                df.loc[
                    qualified,
                    "pressure_legal_balls"
                ].sum()
            )
        })

display(pd.DataFrame(bowling_threshold_results))

print("\nHIGH-EXPOSURE BOWLER EXAMPLES — NOT RANKINGS")

for context in bowling_pressure_labels:

    print("\nContext:", context)

    display(
        bowler_comparisons.loc[
            bowler_comparisons[
                "bowling_pressure_type"
            ].eq(context)
        ]
        .sort_values(
            "pressure_legal_balls",
            ascending=False
        )[
            [
                "player_name",
                "innings",
                "match_phase",
                "pressure_events",
                "pressure_legal_balls",
                "pressure_matches",
                "reference_legal_balls",
                "reference_matches",
                "pressure_clean_dots",
                "pressure_boundaries",
                "pressure_wickets"
            ]
        ].head(10)
    )


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "row_count_difference": abs(
        len(bowling_eval) - len(player_events)
    ),

    "duplicate_delivery_keys": int(
        bowling_eval.duplicated(delivery_keys).sum()
    ),

    "missing_second_innings_rrr": int(
        bowling_eval.loc[
            bowling_eval["innings"].eq(2),
            rrr_columns
        ].isna().any(axis=1).sum()
    ),

    "unexpected_first_innings_rrr": int(
        bowling_eval.loc[
            bowling_eval["innings"].eq(1),
            ["rrr_observed", "rrr_one_run"]
        ].notna().any(axis=1).sum()
    ),

    "illegal_clean_dots": int(
        (
            bowling_eval["clean_dot_proxy"] &
            ~bowling_eval["bowler_legal_ball"]
        ).sum()
    ),

    "bowler_wickets_without_wicket_flag": int(
        (
            bowling_eval["credited_wicket_proxy"] &
            ~bowling_eval["is_wicket"].eq(True)
        ).sum()
    ),

    "pressure_event_reconciliation_error": abs(
        int(
            bowler_comparisons["pressure_events"].sum()
        ) -
        int(
            bowling_eval["is_bowling_pressure"].sum()
        )
    ),

    "duplicate_pressure_strata": int(
        bowler_comparisons.duplicated(
            bowler_stratum_keys +
            ["bowling_pressure_type"]
        ).sum()
    ),

    "bowling_context_in_wrong_innings": int(
        (
            (
                bowling_eval[
                    "bowling_pressure_type"
                ].eq(bowling_pressure_labels[0]) &
                bowling_eval["innings"].ne(1)
            ) |
            (
                bowling_eval[
                    "bowling_pressure_type"
                ].eq(bowling_pressure_labels[1]) &
                bowling_eval["innings"].ne(2)
            )
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 57 — Bowler Personal Baselines and Pressure Performance

print("=" * 70)
print("BOWLER PERSONAL BASELINES AND PRESSURE PERFORMANCE")
print("=" * 70)

bowler_phase_keys = [
    "bowler_id",
    "innings",
    "match_phase"
]

bowler_context_keys = bowler_phase_keys + [
    "bowling_pressure_type"
]

bowler_profile_keys = [
    "bowler_id",
    "bowling_pressure_type"
]

# ============================================================
# 1. USE A CONSISTENT LEGAL-BALL DENOMINATOR
# ============================================================

print("\n1. LEGAL-BALL ANALYSIS COVERAGE")
print("=" * 70)

bowling_legal_scope = bowling_eval.loc[
    bowling_eval["bowler_legal_ball"].eq(True)
].copy()

bowling_legal_scope["clean_dot_event"] = (
    bowling_legal_scope["clean_dot_proxy"].astype(int)
)

bowling_legal_scope["boundary_event"] = (
    bowling_legal_scope[
        "batter_boundary_proxy"
    ].astype(int)
)

bowling_legal_scope["credited_wicket_event"] = (
    bowling_legal_scope[
        "credited_wicket_proxy"
    ].astype(int)
)

print("Legal bowling balls:", len(bowling_legal_scope))

print(
    "Legal pressure balls:",
    int(
        bowling_legal_scope[
            "is_bowling_pressure"
        ].sum()
    )
)

print(
    "Unique bowlers:",
    bowling_legal_scope["bowler_id"].nunique()
)

# Informational: wickets on nonlegal events are
# outside this particular rate calculation.
print(
    "Credited-wicket events on nonlegal deliveries:",
    int(
        (
            bowling_eval["credited_wicket_proxy"] &
            ~bowling_eval["bowler_legal_ball"]
        ).sum()
    )
)


# ============================================================
# 2. BUILD MATCHED BOWLING STRATA
# ============================================================

print("\n2. BOWLER-PHASE-INNINGS STRATA")
print("=" * 70)

bowling_context_strata = (
    bowling_legal_scope.groupby(bowler_context_keys)
    .agg(
        player_name=("bowler_name", "first"),
        legal_balls=("bowler_id", "size"),
        matches=("match_id", "nunique"),
        clean_dots=("clean_dot_event", "sum"),
        boundaries=("boundary_event", "sum"),
        wickets=("credited_wicket_event", "sum")
    )
    .reset_index()
)

bowling_reference_strata = (
    bowling_context_strata.loc[
        bowling_context_strata[
            "bowling_pressure_type"
        ].eq("Reference Context")
    ]
    .drop(
        columns=["bowling_pressure_type", "player_name"]
    )
    .rename(
        columns={
            "legal_balls": "reference_legal_balls",
            "matches": "reference_matches",
            "clean_dots": "reference_clean_dots",
            "boundaries": "reference_boundaries",
            "wickets": "reference_wickets"
        }
    )
)

bowling_pressure_strata = (
    bowling_context_strata.loc[
        bowling_context_strata[
            "bowling_pressure_type"
        ].isin(bowling_pressure_labels)
    ]
    .rename(
        columns={
            "legal_balls": "pressure_legal_balls",
            "matches": "pressure_matches",
            "clean_dots": "pressure_clean_dots",
            "boundaries": "pressure_boundaries",
            "wickets": "pressure_wickets"
        }
    )
)

bowling_type_comparisons = (
    bowling_pressure_strata.merge(
        bowling_reference_strata,
        on=bowler_phase_keys,
        how="left",
        validate="many_to_one"
    )
)

reference_cols = [
    "reference_legal_balls",
    "reference_matches",
    "reference_clean_dots",
    "reference_boundaries",
    "reference_wickets"
]

bowling_type_comparisons[reference_cols] = (
    bowling_type_comparisons[
        reference_cols
    ].fillna(0)
)

print(
    "Pressure strata:",
    len(bowling_type_comparisons)
)

print(
    "Pressure strata with references:",
    int(
        bowling_type_comparisons[
            "reference_legal_balls"
        ].gt(0).sum()
    )
)

print(
    "Pressure strata without references:",
    int(
        bowling_type_comparisons[
            "reference_legal_balls"
        ].eq(0).sum()
    )
)


# ============================================================
# 3. COVERAGE BY PRESSURE TYPE
# ============================================================

print("\n3. MATCHED REFERENCE COVERAGE")
print("=" * 70)

bowling_type_comparisons["matched_pressure_balls"] = (
    np.where(
        bowling_type_comparisons[
            "reference_legal_balls"
        ].gt(0),
        bowling_type_comparisons[
            "pressure_legal_balls"
        ],
        0
    )
)

display(
    bowling_type_comparisons.groupby(
        "bowling_pressure_type"
    )
    .agg(
        strata=("bowler_id", "size"),
        bowlers=("bowler_id", "nunique"),
        pressure_balls=("pressure_legal_balls", "sum"),
        matched_pressure_balls=(
            "matched_pressure_balls", "sum"
        ),
        strata_with_reference=(
            "reference_legal_balls",
            lambda x: int(x.gt(0).sum())
        )
    )
    .reset_index()
)


# ============================================================
# 4. SAMPLE SIZE THRESHOLDS
# ============================================================

print("\n4. SAMPLE SUFFICIENCY")
print("=" * 70)

sample_thresholds = [
    (5, 20),
    (10, 30),
    (15, 50),
    (20, 75)
]

bowling_type_thresholds = []

for context, group in bowling_type_comparisons.groupby(
    "bowling_pressure_type"
):

    for min_pressure, min_reference in sample_thresholds:

        sufficient = (
            group["pressure_legal_balls"].ge(min_pressure) &
            group["reference_legal_balls"].ge(min_reference) &
            group["pressure_matches"].ge(2) &
            group["reference_matches"].ge(2)
        )

        bowling_type_thresholds.append({
            "pressure_type": context,
            "min_pressure_balls": min_pressure,
            "min_reference_balls": min_reference,
            "minimum_matches_each": 2,
            "eligible_strata": int(sufficient.sum()),
            "eligible_bowlers": int(
                group.loc[
                    sufficient, "bowler_id"
                ].nunique()
            ),
            "covered_pressure_balls": int(
                group.loc[
                    sufficient,
                    "pressure_legal_balls"
                ].sum()
            )
        })

display(pd.DataFrame(bowling_type_thresholds))


# ============================================================
# 5. CALCULATE EXPECTED BOWLING OUTCOMES
# ============================================================

print("\n5. PERSONAL EXPECTED BOWLING OUTCOMES")
print("=" * 70)

# Exploratory qualification, matching Cell 54
bowling_type_comparisons["qualified_stratum"] = (
    bowling_type_comparisons[
        "pressure_legal_balls"
    ].ge(10) &
    bowling_type_comparisons[
        "reference_legal_balls"
    ].ge(30) &
    bowling_type_comparisons[
        "pressure_matches"
    ].ge(2) &
    bowling_type_comparisons[
        "reference_matches"
    ].ge(2)
)

qualified_bowling_strata = (
    bowling_type_comparisons.loc[
        bowling_type_comparisons["qualified_stratum"]
    ].copy()
)

outcome_names = [
    "clean_dots",
    "boundaries",
    "wickets"
]

for outcome in outcome_names:

    qualified_bowling_strata[
        f"reference_{outcome}_rate"
    ] = (
        qualified_bowling_strata[
            f"reference_{outcome}"
        ] /
        qualified_bowling_strata[
            "reference_legal_balls"
        ]
    )

    qualified_bowling_strata[
        f"expected_{outcome}"
    ] = (
        qualified_bowling_strata[
            "pressure_legal_balls"
        ] *
        qualified_bowling_strata[
            f"reference_{outcome}_rate"
        ]
    )

    qualified_bowling_strata[
        f"{outcome}_difference"
    ] = (
        qualified_bowling_strata[
            f"pressure_{outcome}"
        ] -
        qualified_bowling_strata[
            f"expected_{outcome}"
        ]
    )

print(
    "Qualified bowling strata:",
    len(qualified_bowling_strata)
)

print(
    "Unique qualified bowlers:",
    qualified_bowling_strata[
        "bowler_id"
    ].nunique()
)

display(
    qualified_bowling_strata[
        [
            "pressure_legal_balls",
            "reference_legal_balls",
            "expected_clean_dots",
            "expected_boundaries",
            "expected_wickets",
            "clean_dots_difference",
            "boundaries_difference",
            "wickets_difference"
        ]
    ].describe().T
)


# ============================================================
# 6. PLAYER-LEVEL PRESSURE PROFILES
# ============================================================

print("\n6. BOWLER PERSONAL PRESSURE PROFILES")
print("=" * 70)

bowler_type_profiles = (
    qualified_bowling_strata.groupby(
        bowler_profile_keys
    )
    .agg(
        player_name=("player_name", "first"),
        qualified_strata=("match_phase", "size"),
        pressure_legal_balls=(
            "pressure_legal_balls", "sum"
        ),
        reference_legal_balls=(
            "reference_legal_balls", "sum"
        ),

        actual_clean_dots=("pressure_clean_dots", "sum"),
        expected_clean_dots=("expected_clean_dots", "sum"),

        actual_boundaries=("pressure_boundaries", "sum"),
        expected_boundaries=("expected_boundaries", "sum"),

        actual_wickets=("pressure_wickets", "sum"),
        expected_wickets=("expected_wickets", "sum")
    )
    .reset_index()
)

# Count unique pressure matches without double-counting
# matches represented in multiple qualified phases.
qualified_bowling_keys = (
    qualified_bowling_strata[
        bowler_context_keys
    ].drop_duplicates()
)

qualified_bowling_events = (
    bowling_legal_scope.loc[
        bowling_legal_scope["is_bowling_pressure"]
    ]
    .merge(
        qualified_bowling_keys,
        on=bowler_context_keys,
        how="inner",
        validate="many_to_one"
    )
)

distinct_bowling_matches = (
    qualified_bowling_events.groupby(
        bowler_profile_keys
    )
    .agg(
        distinct_pressure_matches=("match_id", "nunique")
    )
    .reset_index()
)

bowler_type_profiles = bowler_type_profiles.merge(
    distinct_bowling_matches,
    on=bowler_profile_keys,
    how="left",
    validate="one_to_one"
)

# All three measures are percentage-point differences
# per 100 legal deliveries.
bowler_type_profiles["clean_dot_uplift_pp"] = (
    100 *
    (
        bowler_type_profiles["actual_clean_dots"] -
        bowler_type_profiles["expected_clean_dots"]
    ) /
    bowler_type_profiles["pressure_legal_balls"]
)

bowler_type_profiles["boundary_suppression_pp"] = (
    100 *
    (
        bowler_type_profiles["expected_boundaries"] -
        bowler_type_profiles["actual_boundaries"]
    ) /
    bowler_type_profiles["pressure_legal_balls"]
)

bowler_type_profiles["wicket_rate_uplift_pp"] = (
    100 *
    (
        bowler_type_profiles["actual_wickets"] -
        bowler_type_profiles["expected_wickets"]
    ) /
    bowler_type_profiles["pressure_legal_balls"]
)

bowler_type_profiles["diagnostic_volume_pass"] = (
    bowler_type_profiles["pressure_legal_balls"].ge(30) &
    bowler_type_profiles["distinct_pressure_matches"].ge(3)
)

display(
    bowler_type_profiles.groupby(
        "bowling_pressure_type"
    )
    .agg(
        players=("bowler_id", "nunique"),
        qualified_pressure_balls=(
            "pressure_legal_balls", "sum"
        ),
        players_passing_volume=(
            "diagnostic_volume_pass", "sum"
        ),
        median_clean_dot_uplift=(
            "clean_dot_uplift_pp", "median"
        ),
        median_boundary_suppression=(
            "boundary_suppression_pp", "median"
        ),
        median_wicket_uplift=(
            "wicket_rate_uplift_pp", "median"
        )
    )
    .reset_index()
)

print("\nHIGH-EXPOSURE BOWLER EXAMPLES — NOT RANKINGS")

display_columns = [
    "player_name",
    "pressure_legal_balls",
    "distinct_pressure_matches",
    "actual_clean_dots",
    "expected_clean_dots",
    "clean_dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_rate_uplift_pp"
]

for context in bowling_pressure_labels:

    print("\nContext:", context)

    display(
        bowler_type_profiles.loc[
            bowler_type_profiles[
                "bowling_pressure_type"
            ].eq(context)
        ]
        .sort_values(
            "pressure_legal_balls",
            ascending=False
        )[display_columns]
        .head(10)
    )


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

expected_pressure_legal = int(
    bowling_legal_scope[
        "is_bowling_pressure"
    ].sum()
)

expected_reference_legal = int(
    bowling_legal_scope[
        "bowling_pressure_type"
    ].eq("Reference Context").sum()
)

checks = {
    "pressure_ball_accounting_error": abs(
        int(
            bowling_type_comparisons[
                "pressure_legal_balls"
            ].sum()
        ) - expected_pressure_legal
    ),

    "reference_ball_accounting_error": abs(
        int(
            bowling_reference_strata[
                "reference_legal_balls"
            ].sum()
        ) - expected_reference_legal
    ),

    "qualified_ball_reconciliation_error": abs(
        len(qualified_bowling_events) -
        int(
            qualified_bowling_strata[
                "pressure_legal_balls"
            ].sum()
        )
    ),

    "duplicate_pressure_strata": int(
        bowling_type_comparisons.duplicated(
            bowler_context_keys
        ).sum()
    ),

    "duplicate_bowler_profiles": int(
        bowler_type_profiles.duplicated(
            bowler_profile_keys
        ).sum()
    ),

    "missing_expected_outcomes": int(
        qualified_bowling_strata[
            [
                "expected_clean_dots",
                "expected_boundaries",
                "expected_wickets"
            ]
        ].isna().sum().sum()
    ),

    "negative_expected_outcomes": int(
        (
            qualified_bowling_strata[
                [
                    "expected_clean_dots",
                    "expected_boundaries",
                    "expected_wickets"
                ]
            ] < 0
        ).sum().sum()
    ),

    "missing_distinct_match_counts": int(
        bowler_type_profiles[
            "distinct_pressure_matches"
        ].isna().sum()
    ),

    "illegal_deliveries_in_qualified_sample": int(
        (
            ~qualified_bowling_events[
                "bowler_legal_ball"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 58 — Match-Cluster Bootstrap for Bowling Performance

print("=" * 70)
print("MATCH-CLUSTER BOOTSTRAP — BOWLING PRESSURE")
print("=" * 70)

rng = np.random.default_rng(2027)
n_bootstrap = 400

phase_keys = [
    "bowler_id",
    "innings",
    "match_phase"
]

context_keys = phase_keys + [
    "bowling_pressure_type"
]

profile_keys = [
    "bowler_id",
    "bowling_pressure_type"
]

match_keys = profile_keys + [
    "match_id",
    "innings",
    "match_phase"
]

outcome_config = {
    "clean_dot": {
        "event": "clean_dot_event",
        "profile_metric": "clean_dot_uplift_pp",
        "direction": 1
    },
    "boundary": {
        "event": "boundary_event",
        "profile_metric": "boundary_suppression_pp",
        "direction": -1
    },
    "wicket": {
        "event": "credited_wicket_event",
        "profile_metric": "wicket_rate_uplift_pp",
        "direction": 1
    }
}


# ============================================================
# 1. IDENTIFY QUALIFIED PRESSURE AND REFERENCE EVENTS
# ============================================================

print("\n1. QUALIFIED EVENT COVERAGE")
print("=" * 70)

qualified_keys = (
    qualified_bowling_strata[context_keys]
    .drop_duplicates()
    .copy()
)

bootstrap_pressure_events = (
    bowling_legal_scope.loc[
        bowling_legal_scope["is_bowling_pressure"]
    ]
    .merge(
        qualified_keys,
        on=context_keys,
        how="inner",
        validate="many_to_one"
    )
)

reference_source = bowling_legal_scope.loc[
    bowling_legal_scope[
        "bowling_pressure_type"
    ].eq("Reference Context"),
    phase_keys[:1] + [
        "match_id",
        "innings",
        "match_phase",
        "clean_dot_event",
        "boundary_event",
        "credited_wicket_event"
    ]
].copy()

bootstrap_reference_events = (
    reference_source.merge(
        qualified_keys,
        on=phase_keys,
        how="inner",
        validate="many_to_many"
    )
)

print(
    "Qualified pressure legal balls:",
    len(bootstrap_pressure_events)
)

print(
    "Matched reference legal balls:",
    len(bootstrap_reference_events)
)

print(
    "Unique qualified bowlers:",
    qualified_keys["bowler_id"].nunique()
)

print(
    "Qualified bowler-context profiles:",
    qualified_keys[profile_keys]
    .drop_duplicates().shape[0]
)


# ============================================================
# 2. AGGREGATE PRESSURE AND REFERENCE TO MATCH LEVEL
# ============================================================

print("\n2. MATCH-LEVEL AGGREGATION")
print("=" * 70)

pressure_by_match = (
    bootstrap_pressure_events.groupby(match_keys)
    .agg(
        pressure_balls=("bowler_id", "size"),
        pressure_clean_dot=("clean_dot_event", "sum"),
        pressure_boundary=("boundary_event", "sum"),
        pressure_wicket=("credited_wicket_event", "sum")
    )
    .reset_index()
)

reference_by_match = (
    bootstrap_reference_events.groupby(match_keys)
    .agg(
        reference_balls=("bowler_id", "size"),
        reference_clean_dot=("clean_dot_event", "sum"),
        reference_boundary=("boundary_event", "sum"),
        reference_wicket=("credited_wicket_event", "sum")
    )
    .reset_index()
)

bootstrap_bowling_matches = pressure_by_match.merge(
    reference_by_match,
    on=match_keys,
    how="outer",
    validate="one_to_one"
)

measure_columns = [
    "pressure_balls",
    "reference_balls",
    "pressure_clean_dot",
    "reference_clean_dot",
    "pressure_boundary",
    "reference_boundary",
    "pressure_wicket",
    "reference_wicket"
]

bootstrap_bowling_matches[measure_columns] = (
    bootstrap_bowling_matches[
        measure_columns
    ].fillna(0)
)

print(
    "Match-stratum records:",
    len(bootstrap_bowling_matches)
)

print(
    "Player-context profiles:",
    bootstrap_bowling_matches[profile_keys]
    .drop_duplicates().shape[0]
)


# ============================================================
# 3. MATCH-CLUSTER RESAMPLING
# ============================================================

print("\n3. BOOTSTRAP EXECUTION")
print("=" * 70)

strata_lookup = {
    key: list(
        group[
            ["innings", "match_phase"]
        ].itertuples(index=False, name=None)
    )
    for key, group in qualified_bowling_strata.groupby(
        profile_keys
    )
}

bootstrap_bowling_results = []

for key, group in bootstrap_bowling_matches.groupby(
    profile_keys, sort=False
):

    bowler_id, pressure_type = key

    strata = strata_lookup[key]

    match_ids = group["match_id"].unique().tolist()

    match_index = {
        match_id: i
        for i, match_id in enumerate(match_ids)
    }

    stratum_index = {
        stratum: j
        for j, stratum in enumerate(strata)
    }

    m = len(match_ids)
    s = len(strata)

    arrays = {
        column: np.zeros((m, s), dtype=float)
        for column in measure_columns
    }

    for record in group.itertuples(index=False):

        i = match_index[record.match_id]

        j = stratum_index[
            (record.innings, record.match_phase)
        ]

        for column in measure_columns:
            arrays[column][i, j] = getattr(
                record, column
            )

    # Resample entire matches, including their reference
    # and pressure events, with the same set of weights.
    weights = rng.multinomial(
        m,
        np.full(m, 1.0 / m),
        size=n_bootstrap
    )

    boot_pressure_balls = (
        weights @ arrays["pressure_balls"]
    )

    boot_reference_balls = (
        weights @ arrays["reference_balls"]
    )

    boot_total_pressure_balls = (
        boot_pressure_balls.sum(axis=1)
    )

    # A draw is invalid if a stratum has pressure
    # observations but no available personal reference.
    missing_reference = (
        (boot_pressure_balls > 0) &
        (boot_reference_balls == 0)
    ).any(axis=1)

    valid_draws = (
        (boot_total_pressure_balls > 0) &
        ~missing_reference
    )

    valid_count = int(valid_draws.sum())

    original_pressure_balls = (
        arrays["pressure_balls"].sum(axis=0)
    )

    original_reference_balls = (
        arrays["reference_balls"].sum(axis=0)
    )

    original_total_pressure = (
        original_pressure_balls.sum()
    )

    result = {
        "bowler_id": bowler_id,
        "bowling_pressure_type": pressure_type,
        "match_clusters": m,
        "bootstrap_valid_draws": valid_count,
        "bootstrap_valid_fraction": (
            valid_count / n_bootstrap
        )
    }

    # Calculate each bowling outcome separately.
    for metric, config in outcome_config.items():

        direction = config["direction"]

        p_column = f"pressure_{metric}"
        r_column = f"reference_{metric}"

        original_p = arrays[p_column].sum(axis=0)
        original_r = arrays[r_column].sum(axis=0)

        reference_rates = np.divide(
            original_r,
            original_reference_balls,
            out=np.zeros(s, dtype=float),
            where=original_reference_balls > 0
        )

        original_expected = (
            original_pressure_balls *
            reference_rates
        ).sum()

        original_actual = original_p.sum()

        result[f"{metric}_point_estimate"] = (
            direction * 100 *
            (original_actual - original_expected) /
            original_total_pressure
        )

        # Bootstrap pressure and reference outcomes
        boot_p = weights @ arrays[p_column]
        boot_r = weights @ arrays[r_column]

        boot_reference_rates = np.divide(
            boot_r,
            boot_reference_balls,
            out=np.zeros_like(
                boot_r, dtype=float
            ),
            where=boot_reference_balls > 0
        )

        boot_expected = (
            boot_pressure_balls *
            boot_reference_rates
        ).sum(axis=1)

        boot_actual = boot_p.sum(axis=1)

        samples = np.full(
            n_bootstrap,
            np.nan,
            dtype=float
        )

        samples[valid_draws] = (
            direction * 100 *
            (
                boot_actual[valid_draws] -
                boot_expected[valid_draws]
            ) /
            boot_total_pressure_balls[valid_draws]
        )

        valid_values = samples[
            np.isfinite(samples)
        ]

        if len(valid_values) >= 0.80 * n_bootstrap:

            lower, median, upper = np.quantile(
                valid_values,
                [0.025, 0.50, 0.975]
            )

        else:

            lower = np.nan
            median = np.nan
            upper = np.nan

        result[f"{metric}_lower_95"] = lower
        result[f"{metric}_median"] = median
        result[f"{metric}_upper_95"] = upper

    bootstrap_bowling_results.append(result)

bowling_bootstrap_estimates = pd.DataFrame(
    bootstrap_bowling_results
)

print("Bootstrap repetitions:", n_bootstrap)

print(
    "Profiles evaluated:",
    len(bowling_bootstrap_estimates)
)

print(
    "Profiles with sufficient coverage:",
    int(
        bowling_bootstrap_estimates[
            "clean_dot_lower_95"
        ].notna().sum()
    )
)


# ============================================================
# 4. COMBINE BOOTSTRAP RESULTS WITH BOWLER PROFILES
# ============================================================

print("\n4. UNCERTAINTY-ADJUSTED BOWLER PROFILES")
print("=" * 70)

bowler_uncertainty_profiles = (
    bowler_type_profiles.merge(
        bowling_bootstrap_estimates,
        on=profile_keys,
        how="left",
        validate="one_to_one"
    )
)

for metric in outcome_config:

    lower_col = f"{metric}_lower_95"
    upper_col = f"{metric}_upper_95"

    bowler_uncertainty_profiles[
        f"{metric}_interval_width"
    ] = (
        bowler_uncertainty_profiles[upper_col] -
        bowler_uncertainty_profiles[lower_col]
    )

    bowler_uncertainty_profiles[
        f"{metric}_assessment"
    ] = np.select(
        [
            bowler_uncertainty_profiles[
                lower_col
            ].gt(0),

            bowler_uncertainty_profiles[
                upper_col
            ].lt(0),

            bowler_uncertainty_profiles[
                lower_col
            ].notna()
        ],
        [
            "Above Baseline",
            "Below Baseline",
            "Interval Crosses Zero"
        ],
        default="Insufficient Bootstrap Coverage"
    )

display(
    bowler_uncertainty_profiles[
        [
            "pressure_legal_balls",
            "distinct_pressure_matches",
            "clean_dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_rate_uplift_pp",
            "clean_dot_interval_width",
            "boundary_interval_width",
            "wicket_interval_width",
            "bootstrap_valid_fraction"
        ]
    ].describe().T
)


# ============================================================
# 5. RELIABILITY SUMMARY BY BOWLING CONTEXT
# ============================================================

print("\n5. BOOTSTRAP RELIABILITY SUMMARY")
print("=" * 70)

for metric in outcome_config:

    print("\nOutcome:", metric)

    display(
        pd.crosstab(
            bowler_uncertainty_profiles[
                "bowling_pressure_type"
            ],
            bowler_uncertainty_profiles[
                f"{metric}_assessment"
            ]
        )
    )

display(
    bowler_uncertainty_profiles.groupby(
        "bowling_pressure_type"
    )
    .agg(
        profiles=("bowler_id", "size"),
        median_pressure_balls=(
            "pressure_legal_balls", "median"
        ),
        median_match_clusters=(
            "match_clusters", "median"
        ),
        median_valid_fraction=(
            "bootstrap_valid_fraction", "median"
        ),
        median_clean_dot_interval_width=(
            "clean_dot_interval_width", "median"
        ),
        median_boundary_interval_width=(
            "boundary_interval_width", "median"
        ),
        median_wicket_interval_width=(
            "wicket_interval_width", "median"
        )
    )
    .reset_index()
)


# ============================================================
# 6. HIGH-EXPOSURE INTERVALS — NOT RANKINGS
# ============================================================

print("\n6. HIGH-EXPOSURE BOWLER INTERVALS")
print("=" * 70)

sample_columns = [
    "player_name",
    "pressure_legal_balls",
    "distinct_pressure_matches",
    "clean_dot_uplift_pp",
    "clean_dot_lower_95",
    "clean_dot_upper_95",
    "boundary_suppression_pp",
    "boundary_lower_95",
    "boundary_upper_95",
    "wicket_rate_uplift_pp",
    "wicket_lower_95",
    "wicket_upper_95"
]

for context in bowling_pressure_labels:

    print("\nContext:", context)

    display(
        bowler_uncertainty_profiles.loc[
            bowler_uncertainty_profiles[
                "bowling_pressure_type"
            ].eq(context)
        ]
        .sort_values(
            "pressure_legal_balls",
            ascending=False
        )[sample_columns]
        .head(8)
    )


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

point_mismatches = {}

for metric, config in outcome_config.items():

    compared = bowler_uncertainty_profiles[
        [
            config["profile_metric"],
            f"{metric}_point_estimate"
        ]
    ].dropna()

    point_mismatches[metric] = int(
        (
            ~np.isclose(
                compared[config["profile_metric"]],
                compared[f"{metric}_point_estimate"],
                rtol=1e-8,
                atol=1e-7
            )
        ).sum()
    )

reversed_intervals = sum(
    int(
        (
            bowler_uncertainty_profiles[
                f"{metric}_lower_95"
            ] >
            bowler_uncertainty_profiles[
                f"{metric}_upper_95"
            ]
        ).sum()
    )
    for metric in outcome_config
)

checks = {
    "pressure_ball_reconciliation_error": abs(
        len(bootstrap_pressure_events) -
        int(
            qualified_bowling_strata[
                "pressure_legal_balls"
            ].sum()
        )
    ),

    "reference_ball_reconciliation_error": abs(
        len(bootstrap_reference_events) -
        int(
            qualified_bowling_strata[
                "reference_legal_balls"
            ].sum()
        )
    ),

    "missing_bootstrap_profiles": int(
        bowler_uncertainty_profiles[
            "bootstrap_valid_draws"
        ].isna().sum()
    ),

    "duplicate_profiles": int(
        bowler_uncertainty_profiles.duplicated(
            profile_keys
        ).sum()
    ),

    "clean_dot_point_mismatches": (
        point_mismatches["clean_dot"]
    ),

    "boundary_point_mismatches": (
        point_mismatches["boundary"]
    ),

    "wicket_point_mismatches": (
        point_mismatches["wicket"]
    ),

    "invalid_bootstrap_draw_counts": int(
        (
            ~bowler_uncertainty_profiles[
                "bootstrap_valid_draws"
            ].between(0, n_bootstrap)
        ).sum()
    ),

    "reversed_bootstrap_intervals": (
        reversed_intervals
    ),

    "illegal_deliveries_in_pressure_sample": int(
        (
            ~bootstrap_pressure_events[
                "bowler_legal_ball"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 59 — Player Eligibility Metadata and Identity Audit

print("=" * 70)
print("PLAYER ELIGIBILITY METADATA AND IDENTITY AUDIT")
print("=" * 70)

metadata_tables = [
    "competition_team_player_mapping",
    "playing_eleven",
    "competitions",
    "teams"
]

# ============================================================
# 1. DATASET CATALOG
# ============================================================

print("\n1. METADATA DATASET CATALOG")
print("=" * 70)

catalog = []

for table_name in metadata_tables:

    df = datasets[table_name]

    catalog.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(map(str, df.columns))
    })

display(pd.DataFrame(catalog))


# ============================================================
# 2. COLUMN SCHEMAS
# ============================================================

print("\n2. METADATA COLUMN SCHEMAS")
print("=" * 70)

for table_name in metadata_tables:

    df = datasets[table_name]

    print("\nTABLE:", table_name)

    schema = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "missing_pct": (
            df.isna().mean().values * 100
        ).round(2)
    })

    display(schema)


# ============================================================
# 3. INSPECT METADATA CONTENT
# ============================================================

print("\n3. METADATA SAMPLE RECORDS")
print("=" * 70)

for table_name in metadata_tables:

    print("\nTABLE:", table_name)

    display(
        datasets[table_name].head(5)
    )


# ============================================================
# 4. IDENTIFY ELIGIBILITY-RELATED FIELDS
# ============================================================

print("\n4. POTENTIAL ELIGIBILITY FIELDS")
print("=" * 70)

eligibility_keywords = [
    "birth",
    "dob",
    "age",
    "cap",
    "uncapped",
    "international",
    "nationality",
    "country",
    "ipl"
]

eligibility_fields = []

for table_name in metadata_tables:

    df = datasets[table_name]

    for column in df.columns:

        column_text = str(column).lower()

        matched_keywords = [
            keyword
            for keyword in eligibility_keywords
            if keyword in column_text
        ]

        if matched_keywords:

            examples = (
                df[column]
                .dropna()
                .astype(str)
                .unique()[:5]
                .tolist()
            )

            eligibility_fields.append({
                "table": table_name,
                "column": column,
                "matched_keywords": ", ".join(
                    matched_keywords
                ),
                "non_missing_rows": int(
                    df[column].notna().sum()
                ),
                "sample_values": examples
            })

if eligibility_fields:

    display(
        pd.DataFrame(eligibility_fields)
    )

else:

    print(
        "No obvious age, birth, cap, or "
        "nationality-related columns found."
    )


# ============================================================
# 5. LINK QUALIFIED PLAYERS TO PLAYER MAPPING
# ============================================================

print("\n5. QUALIFIED PLAYER ID LINKAGE")
print("=" * 70)

player_mapping = datasets[
    "competition_team_player_mapping"
].copy()

playing_eleven_data = datasets[
    "playing_eleven"
].copy()

# Find identifier fields without depending on
# capitalization or underscore conventions.

def normalized_column_name(column):
    return "".join(
        character
        for character in str(column).lower()
        if character.isalnum()
    )


def find_player_id_column(df):

    exact_candidates = [
        "playerid",
        "cricketerid"
    ]

    for column in df.columns:

        if normalized_column_name(column) in exact_candidates:
            return column

    possible = [
        column
        for column in df.columns
        if (
            "player" in normalized_column_name(column)
            and "id" in normalized_column_name(column)
        )
    ]

    return possible[0] if possible else None


mapping_player_col = find_player_id_column(
    player_mapping
)

eleven_player_col = find_player_id_column(
    playing_eleven_data
)

print("Mapping player ID column:", mapping_player_col)
print("Playing XI player ID column:", eleven_player_col)


def numeric_id_set(series):

    values = pd.to_numeric(
        series,
        errors="coerce"
    ).dropna()

    return set(
        values.astype("int64").tolist()
    )


qualified_batter_ids = numeric_id_set(
    batter_uncertainty_profiles["striker_id"]
)

qualified_bowler_ids = numeric_id_set(
    bowler_uncertainty_profiles["bowler_id"]
)

qualified_all_ids = (
    qualified_batter_ids |
    qualified_bowler_ids
)

linkage_results = []

source_id_sets = {}

for source_name, source_df, id_column in [
    (
        "Player mapping",
        player_mapping,
        mapping_player_col
    ),
    (
        "Playing XI",
        playing_eleven_data,
        eleven_player_col
    )
]:

    if id_column is None:

        print(
            "ID linkage unavailable for:",
            source_name
        )

        continue

    source_ids = numeric_id_set(
        source_df[id_column]
    )

    source_id_sets[source_name] = source_ids

    for player_group, target_ids in [
        ("Qualified batters", qualified_batter_ids),
        ("Qualified bowlers", qualified_bowler_ids),
        ("All unique qualified players", qualified_all_ids)
    ]:

        linked_ids = target_ids & source_ids

        linkage_results.append({
            "source": source_name,
            "player_group": player_group,
            "total_players": len(target_ids),
            "linked_players": len(linked_ids),
            "unmatched_players": len(
                target_ids - source_ids
            ),
            "linkage_pct": round(
                100 * len(linked_ids) /
                max(len(target_ids), 1),
                2
            )
        })

if linkage_results:

    display(
        pd.DataFrame(linkage_results)
    )

else:

    print(
        "No compatible player ID field found "
        "for automatic linkage."
    )


# ============================================================
# 6. COMPETITION COVERAGE
# ============================================================

print("\n6. COMPETITION COVERAGE")
print("=" * 70)

competition_counts = (
    matches.groupby("comp_id")
    .agg(
        matches=("match_id", "nunique"),
        first_match=("match_date", "min"),
        last_match=("match_date", "max")
    )
    .reset_index()
    .sort_values(
        "matches",
        ascending=False
    )
)

print(
    "Competitions represented in matches:",
    competition_counts["comp_id"].nunique()
)

print("\nTop competitions by match count:")

display(competition_counts.head(20))

# Attempt to identify a matching competition key
competition_metadata = datasets["competitions"]

competition_id_col = None

for column in competition_metadata.columns:

    if normalized_column_name(column) in [
        "compid",
        "competitionid"
    ]:
        competition_id_col = column
        break

print(
    "\nCompetition metadata join key:",
    competition_id_col
)

if competition_id_col is not None:

    competition_detail = competition_counts.merge(
        competition_metadata,
        left_on="comp_id",
        right_on=competition_id_col,
        how="left",
        validate="many_to_one",
        suffixes=("", "_metadata")
    )

    print("\nCompetitions with metadata:")

    display(
        competition_detail.head(20)
    )

else:

    print(
        "Competition metadata key requires "
        "manual identification."
    )


# ============================================================
# 7. FINAL ELIGIBILITY READINESS AUDIT
# ============================================================

print("\n7. ELIGIBILITY READINESS CHECKS")
print("=" * 70)

all_columns_normalized = [
    normalized_column_name(column)
    for table_name in metadata_tables
    for column in datasets[table_name].columns
]

has_age_field = any(
    (
        column == "age" or
        "dateofbirth" in column or
        "birthdate" in column or
        "dob" in column
    )
    for column in all_columns_normalized
)

has_cap_field = any(
    (
        "uncapped" in column or
        "internationalcap" in column or
        column == "iscapped" or
        column == "cappedstatus"
    )
    for column in all_columns_normalized
)

checks = {
    "qualified_unique_batters": len(
        qualified_batter_ids
    ),

    "qualified_unique_bowlers": len(
        qualified_bowler_ids
    ),

    "qualified_unique_players_combined": len(
        qualified_all_ids
    ),

    "player_mapping_key_available": (
        mapping_player_col is not None
    ),

    "playing_eleven_key_available": (
        eleven_player_col is not None
    ),

    "competition_join_key_available": (
        competition_id_col is not None
    ),

    "possible_age_or_birth_field_found": (
        has_age_field
    ),

    "possible_explicit_cap_status_field_found": (
        has_cap_field
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "value"]
    )
)

print("\nIMPORTANT:")
print(
    "A column's presence does not establish "
    "that its values are complete or accurate."
)
print(
    "International or IPL appearances must not "
    "be treated as definitive uncapped-status "
    "evidence without checking the competition rules."
)
print(
    "No player is being declared eligible "
    "or ineligible in this cell."
)

In [ ]:
# CELL 60 — Player Identity and Eligibility Evidence Register

print("=" * 70)
print("PLAYER IDENTITY AND ELIGIBILITY EVIDENCE REGISTER")
print("=" * 70)

mapping_source = datasets[
    "competition_team_player_mapping"
].copy()

xi_source = datasets["playing_eleven"].copy()
ball_source = datasets["ball_by_ball"].copy()

# ============================================================
# 1. CREATE MATCH AND COMPETITION METADATA
# ============================================================

print("\n1. MATCH AND COMPETITION EVIDENCE")
print("=" * 70)

match_metadata = datasets["matches"][
    ["match_id", "comp_id", "match_date"]
].copy()

match_metadata = match_metadata.merge(
    datasets["competitions"][
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

match_metadata["match_date"] = pd.to_datetime(
    match_metadata["match_date"],
    errors="coerce"
)

match_metadata["is_t20i"] = (
    match_metadata["match_format"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("T20I")
)

match_metadata["is_ipl"] = (
    match_metadata["comp_name"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"\bIPL\s*20\d{2}\b|Indian Premier League",
        case=False,
        regex=True
    )
)

match_metadata["is_other_competition"] = (
    ~match_metadata["is_t20i"] &
    ~match_metadata["is_ipl"]
)

display(
    pd.DataFrame([
        {
            "category": "T20I-labeled matches",
            "matches": int(match_metadata["is_t20i"].sum())
        },
        {
            "category": "IPL-labeled matches",
            "matches": int(match_metadata["is_ipl"].sum())
        },
        {
            "category": "Other competition matches",
            "matches": int(
                match_metadata["is_other_competition"].sum()
            )
        }
    ])
)

display(
    match_metadata.groupby(
        ["match_format"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
)


# ============================================================
# 2. CREATE MATCH-LEVEL PARTICIPATION EVIDENCE
# ============================================================

print("\n2. PLAYER PARTICIPATION COVERAGE")
print("=" * 70)

# Playing XI records explicitly flagged as played
xi_evidence = (
    xi_source.loc[
        xi_source["is_played"].eq(True),
        ["match_id", "player_id"]
    ]
    .drop_duplicates()
    .assign(
        xi_evidence=True,
        ball_evidence=False
    )
)

# Direct ball-by-ball participation:
# striker, non-striker or bowler
ball_participants = []

for role_column in [
    "striker_id",
    "non_striker_id",
    "bowler_id"
]:

    part = ball_source[
        ["match_id", role_column]
    ].rename(
        columns={role_column: "player_id"}
    )

    ball_participants.append(part)

ball_evidence = (
    pd.concat(
        ball_participants,
        ignore_index=True
    )
    .dropna(subset=["player_id"])
    .drop_duplicates()
    .assign(
        xi_evidence=False,
        ball_evidence=True
    )
)

participation_evidence = pd.concat(
    [xi_evidence, ball_evidence],
    ignore_index=True
)

participation_evidence["player_id"] = (
    pd.to_numeric(
        participation_evidence["player_id"],
        errors="raise"
    ).astype("int64")
)

participation_evidence["match_id"] = (
    pd.to_numeric(
        participation_evidence["match_id"],
        errors="raise"
    ).astype("int64")
)

# One match-player record regardless of number
# of deliveries or participation sources.
participation_evidence = (
    participation_evidence.groupby(
        ["match_id", "player_id"],
        as_index=False
    )
    .agg(
        xi_evidence=("xi_evidence", "max"),
        ball_evidence=("ball_evidence", "max")
    )
)

participation_evidence = (
    participation_evidence.merge(
        match_metadata,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

print(
    "Unique player-match participation records:",
    len(participation_evidence)
)

print(
    "Unique participants:",
    participation_evidence["player_id"].nunique()
)

print(
    "Matches represented:",
    participation_evidence["match_id"].nunique()
)

print(
    "Playing XI evidenced participations:",
    int(
        participation_evidence["xi_evidence"].sum()
    )
)

print(
    "Ball-recorded participations:",
    int(
        participation_evidence["ball_evidence"].sum()
    )
)


# ============================================================
# 3. SUMMARIZE EACH PLAYER'S RECORDED HISTORY
# ============================================================

print("\n3. PLAYER PARTICIPATION HISTORY")
print("=" * 70)

player_participation = (
    participation_evidence.groupby("player_id")
    .agg(
        observed_matches=("match_id", "nunique"),
        recorded_xi_matches=("xi_evidence", "sum"),
        recorded_ball_matches=("ball_evidence", "sum"),

        observed_t20i_matches=("is_t20i", "sum"),
        observed_ipl_matches=("is_ipl", "sum"),
        observed_other_matches=(
            "is_other_competition", "sum"
        ),

        observed_competitions=("comp_id", "nunique"),
        first_recorded_match=("match_date", "min"),
        last_recorded_match=("match_date", "max")
    )
    .reset_index()
)

display(
    player_participation[
        [
            "observed_matches",
            "recorded_xi_matches",
            "recorded_ball_matches",
            "observed_t20i_matches",
            "observed_ipl_matches",
            "observed_other_matches"
        ]
    ].describe().T
)


# ============================================================
# 4. RECONCILE PLAYER NAMES AND MODEL PROFILES
# ============================================================

print("\n4. PLAYER IDENTITY AND MODEL LINKAGE")
print("=" * 70)

# Select the most frequently recorded name for an ID.
# Different IDs are NEVER merged merely because names match.
name_frequency = (
    mapping_source.groupby(
        ["player_id", "player_name"]
    )
    .size()
    .reset_index(name="records")
    .sort_values(
        ["player_id", "records", "player_name"],
        ascending=[True, False, True]
    )
)

canonical_names = (
    name_frequency.drop_duplicates("player_id")[
        ["player_id", "player_name"]
    ]
    .copy()
)

# Fallback names for participants absent from mapping
ball_names = pd.concat(
    [
        ball_source[
            ["striker_id", "striker_name"]
        ].rename(
            columns={
                "striker_id": "player_id",
                "striker_name": "fallback_name"
            }
        ),

        ball_source[
            ["bowler_id", "bowler_name"]
        ].rename(
            columns={
                "bowler_id": "player_id",
                "bowler_name": "fallback_name"
            }
        )
    ],
    ignore_index=True
).dropna()

fallback_names = (
    ball_names.groupby(
        ["player_id", "fallback_name"]
    )
    .size()
    .reset_index(name="records")
    .sort_values(
        ["player_id", "records", "fallback_name"],
        ascending=[True, False, True]
    )
    .drop_duplicates("player_id")[
        ["player_id", "fallback_name"]
    ]
)

batting_roles = (
    batting_exposure[
        ["striker_id", "provisional_balls_faced"]
    ]
    .rename(
        columns={"striker_id": "player_id"}
    )
)

bowling_roles = (
    bowling_exposure[
        ["bowler_id", "legal_balls_bowled"]
    ]
    .rename(
        columns={"bowler_id": "player_id"}
    )
)

player_registry = (
    player_participation
    .merge(
        canonical_names,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        fallback_names,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        batting_roles,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        bowling_roles,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
)

player_registry["player_name"] = (
    player_registry["player_name"].fillna(
        player_registry["fallback_name"]
    )
)

player_registry[
    [
        "provisional_balls_faced",
        "legal_balls_bowled"
    ]
] = player_registry[
    [
        "provisional_balls_faced",
        "legal_balls_bowled"
    ]
].fillna(0)

qualified_batter_ids = set(
    batter_uncertainty_profiles[
        "striker_id"
    ].dropna().astype("int64")
)

qualified_bowler_ids = set(
    bowler_uncertainty_profiles[
        "bowler_id"
    ].dropna().astype("int64")
)

qualified_all_ids = (
    qualified_batter_ids | qualified_bowler_ids
)

player_registry["qualified_batting_profile"] = (
    player_registry["player_id"].isin(
        qualified_batter_ids
    )
)

player_registry["qualified_bowling_profile"] = (
    player_registry["player_id"].isin(
        qualified_bowler_ids
    )
)

player_registry["qualified_any_profile"] = (
    player_registry["qualified_batting_profile"] |
    player_registry["qualified_bowling_profile"]
)

player_registry["model_role"] = np.select(
    [
        player_registry["qualified_batting_profile"] &
        player_registry["qualified_bowling_profile"],

        player_registry["qualified_batting_profile"],

        player_registry["qualified_bowling_profile"]
    ],
    [
        "Batting and Bowling",
        "Batting Model",
        "Bowling Model"
    ],
    default="No Qualified Model Profile"
)

player_registry["eligibility_status"] = (
    "Unverified: DOB and official cap status required"
)

display(
    player_registry.groupby("model_role")
    .agg(
        players=("player_id", "nunique"),
        with_t20i_evidence=(
            "observed_t20i_matches",
            lambda x: int(x.gt(0).sum())
        ),
        with_ipl_evidence=(
            "observed_ipl_matches",
            lambda x: int(x.gt(0).sum())
        ),
        without_t20i_evidence=(
            "observed_t20i_matches",
            lambda x: int(x.eq(0).sum())
        )
    )
    .reset_index()
)


# ============================================================
# 5. ELIGIBILITY VERIFICATION QUEUE
# ============================================================

print("\n5. ELIGIBILITY VERIFICATION QUEUE")
print("=" * 70)

qualified_registry = player_registry.loc[
    player_registry["qualified_any_profile"]
].copy()

# This is NOT a list of confirmed uncapped players.
# Absence of T20I observations is only a dataset fact.
verification_queue = qualified_registry.loc[
    qualified_registry[
        "observed_t20i_matches"
    ].eq(0)
].copy()

verification_queue = verification_queue.sort_values(
    "player_id"
)

print(
    "Qualified players with no T20I appearance "
    "in supplied matches:",
    len(verification_queue)
)

print(
    "Qualified players with T20I evidence:",
    int(
        qualified_registry[
            "observed_t20i_matches"
        ].gt(0).sum()
    )
)

print(
    "\nSAMPLE REQUIRING EXTERNAL ELIGIBILITY CHECKS "
    "— NOT RANKINGS"
)

display(
    verification_queue[
        [
            "player_id",
            "player_name",
            "model_role",
            "observed_matches",
            "observed_ipl_matches",
            "observed_t20i_matches",
            "provisional_balls_faced",
            "legal_balls_bowled",
            "eligibility_status"
        ]
    ].head(25)
)

print("\nSAMPLE WITH RECORDED T20I PARTICIPATION")

display(
    qualified_registry.loc[
        qualified_registry[
            "observed_t20i_matches"
        ].gt(0)
    ]
    .sort_values("player_id")[
        [
            "player_id",
            "player_name",
            "model_role",
            "observed_t20i_matches",
            "observed_ipl_matches"
        ]
    ].head(20)
)


# ============================================================
# 6. IDENTITY AMBIGUITY AUDIT
# ============================================================

print("\n6. IDENTITY AMBIGUITY AUDIT")
print("=" * 70)

name_variants_by_id = (
    mapping_source.groupby("player_id")[
        "player_name"
    ].nunique()
)

print(
    "Player IDs with multiple mapped names:",
    int(name_variants_by_id.gt(1).sum())
)

normalized_names = mapping_source[
    ["player_id", "player_name"]
].drop_duplicates().copy()

normalized_names["normalized_name"] = (
    normalized_names["player_name"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

ids_per_name = (
    normalized_names.groupby("normalized_name")
    ["player_id"]
    .nunique()
    .reset_index(name="distinct_player_ids")
)

print(
    "Normalized names shared by multiple IDs:",
    int(ids_per_name["distinct_player_ids"].gt(1).sum())
)

display(
    ids_per_name.loc[
        ids_per_name[
            "distinct_player_ids"
        ].gt(1)
    ]
    .sort_values(
        "distinct_player_ids",
        ascending=False
    )
    .head(15)
)

print(
    "Participants without mapped/fallback names:",
    int(player_registry["player_name"].isna().sum())
)


# ============================================================
# 7. DATA INTEGRITY AND READINESS CHECKS
# ============================================================

print("\n7. DATA INTEGRITY AND READINESS CHECKS")
print("=" * 70)

participation_total = (
    player_participation[
        [
            "observed_t20i_matches",
            "observed_ipl_matches",
            "observed_other_matches"
        ]
    ].sum(axis=1)
)

checks = {
    "duplicate_player_registry_ids": int(
        player_registry["player_id"].duplicated().sum()
    ),

    "duplicate_player_match_records": int(
        participation_evidence.duplicated(
            ["match_id", "player_id"]
        ).sum()
    ),

    "participation_category_mismatches": int(
        participation_total.ne(
            player_participation["observed_matches"]
        ).sum()
    ),

    "participation_without_match_metadata": int(
        participation_evidence["comp_id"].isna().sum()
    ),

    "participation_without_competition_metadata": int(
        participation_evidence["match_format"].isna().sum()
    ),

    "qualified_players_missing_from_registry": int(
        len(
            qualified_all_ids -
            set(player_registry["player_id"])
        )
    ),

    "qualified_batting_profiles_missing": int(
        len(
            qualified_batter_ids -
            set(
                player_registry.loc[
                    player_registry["qualified_batting_profile"],
                    "player_id"
                ]
            )
        )
    ),

    "qualified_bowling_profiles_missing": int(
        len(
            qualified_bowler_ids -
            set(
                player_registry.loc[
                    player_registry["qualified_bowling_profile"],
                    "player_id"
                ]
            )
        )
    ),

    "qualified_players_without_names": int(
        qualified_registry["player_name"].isna().sum()
    ),

    "overlapping_ipl_and_t20i_match_flags": int(
        (
            match_metadata["is_ipl"] &
            match_metadata["is_t20i"]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

print("\nELIGIBILITY INTERPRETATION")
print("=" * 70)

print(
    "No recorded T20I appearance does NOT prove "
    "a player is uncapped."
)

print(
    "Recorded IPL participation does NOT, by itself, "
    "prove a player is internationally capped."
)

print(
    "All ages and final eligibility statuses remain "
    "unverified pending authoritative sources."
)

print(
    "Players without qualified statistical profiles "
    "remain in player_registry for future scouting."
)

In [ ]:
# CELL 61 — Verify Exact Round 1 Problem Statement

from io import BytesIO
import re
import zipfile

print("=" * 70)
print("OFFICIAL ROUND 1 PROBLEM STATEMENT VERIFICATION")
print("=" * 70)

# ============================================================
# 1. LOCATE THE PDF INSIDE THE ORIGINAL ZIP
# ============================================================

print("\n1. PROBLEM STATEMENT FILE")
print("=" * 70)

assert ZIP_PATH.exists(), (
    f"Original ZIP not found: {ZIP_PATH.resolve()}"
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    pdf_files = [
        name
        for name in z.namelist()
        if name.lower().endswith(".pdf")
    ]

    preferred_files = [
        name
        for name in pdf_files
        if "problem statement" in name.lower()
    ]

    selected_pdf = (
        preferred_files[0]
        if preferred_files
        else pdf_files[0] if pdf_files else None
    )

    assert selected_pdf is not None, (
        "No PDF found in the original ZIP."
    )

    pdf_bytes = z.read(selected_pdf)

print("Selected PDF:", selected_pdf)
print("PDF size:", len(pdf_bytes), "bytes")


# ============================================================
# 2. EXTRACT PDF TEXT WITH AVAILABLE READER
# ============================================================

print("\n2. PDF TEXT EXTRACTION")
print("=" * 70)

rubric_pages = []
reader_used = None
reader_errors = []

try:
    import fitz

    document = fitz.open(
        stream=pdf_bytes,
        filetype="pdf"
    )

    rubric_pages = [
        page.get_text("text", sort=True)
        for page in document
    ]

    reader_used = "PyMuPDF"

    document.close()

except Exception as exc:
    reader_errors.append(
        f"PyMuPDF: {type(exc).__name__}: {exc}"
    )

if reader_used is None:

    try:
        from pypdf import PdfReader

        reader = PdfReader(BytesIO(pdf_bytes))

        rubric_pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]

        reader_used = "pypdf"

    except Exception as exc:
        reader_errors.append(
            f"pypdf: {type(exc).__name__}: {exc}"
        )

if reader_used is None:

    try:
        from PyPDF2 import PdfReader

        reader = PdfReader(BytesIO(pdf_bytes))

        rubric_pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]

        reader_used = "PyPDF2"

    except Exception as exc:
        reader_errors.append(
            f"PyPDF2: {type(exc).__name__}: {exc}"
        )

if reader_used is None:

    print("No available PDF reader succeeded.")

    for error in reader_errors:
        print(error)

else:

    print("PDF reader:", reader_used)
    print("Total pages:", len(rubric_pages))

    rubric_text = "\n\n".join(rubric_pages)

    print(
        "Extracted characters:",
        len(rubric_text.strip())
    )


# ============================================================
# 3. DISPLAY THE ORIGINAL PROBLEM STATEMENT
# ============================================================

print("\n3. PROBLEM STATEMENT TEXT")
print("=" * 70)

if reader_used is not None:

    for page_number, page_text in enumerate(
        rubric_pages,
        start=1
    ):

        print(f"\n{'-' * 25}")
        print(f"PAGE {page_number}")
        print(f"{'-' * 25}")

        print(
            page_text[:14000]
            if page_text.strip()
            else "[No extractable text on this page]"
        )

        if len(page_text) > 14000:
            print(
                "\n[Page truncated in display; "
                "full text remains in rubric_pages]"
            )


# ============================================================
# 4. CHECK CONTEST REQUIREMENT WORDING
# ============================================================

print("\n4. REQUIREMENT KEYWORD AUDIT")
print("=" * 70)

requirements = {
    "Pressure": r"\bpressure\b",
    "Top 10": r"\btop\s*10\b|\bten\s+players\b",
    "Uncapped": r"\buncapped\b",
    "Youngest / Age": (
        r"\byoung(?:est)?\b|\bage\b|"
        r"\bdate of birth\b"
    ),
    "Batting": r"\bbatt(?:er|ers|ing)\b",
    "Bowling": r"\bbowl(?:er|ers|ing)\b",
    "Personal Baseline": (
        r"\bbaseline\b|\bnormal\b|"
        r"\busual\b|\brelative\b"
    ),
    "Minimum Sample": (
        r"\bminimum\b|\bsample size\b|"
        r"\bthreshold\b"
    ),
    "Ranking": r"\brank(?:ing|ings|ed)?\b",
    "Submission": (
        r"\bsubmission\b|\bsubmit\b|"
        r"\bpresentation\b"
    )
}

requirement_results = []

if reader_used is not None:

    for label, pattern in requirements.items():

        matches_found = list(
            re.finditer(
                pattern,
                rubric_text,
                flags=re.IGNORECASE
            )
        )

        requirement_results.append({
            "requirement": label,
            "text_matches": len(matches_found),
            "mentioned_in_extracted_text": (
                len(matches_found) > 0
            )
        })

    display(pd.DataFrame(requirement_results))


# ============================================================
# 5. EXTRACT IMPORTANT CONTEXT PASSAGES
# ============================================================

print("\n5. IMPORTANT REQUIREMENT PASSAGES")
print("=" * 70)

priority_pattern = (
    r"uncapped|youngest|top\s*10|"
    r"pressure|identify|task|objective|"
    r"submit|deliverable|evaluation|"
    r"criteria|batters|bowlers"
)

if reader_used is not None:

    passages = []

    for page_number, page_text in enumerate(
        rubric_pages,
        start=1
    ):

        lines = page_text.splitlines()

        for line_number, line in enumerate(lines):

            if re.search(
                priority_pattern,
                line,
                flags=re.IGNORECASE
            ):

                start = max(0, line_number - 1)
                end = min(
                    len(lines),
                    line_number + 3
                )

                excerpt = " ".join(
                    lines[start:end]
                ).strip()
            
                passages.append({
                    "page": page_number,
                    "passage": excerpt
                })

    if passages:

        display(
            pd.DataFrame(passages).head(45)
        )

    else:

        print(
            "No matching passages found in "
            "extracted PDF text."
        )


# ============================================================
# 6. DOCUMENT READINESS CHECK
# ============================================================

print("\n6. DOCUMENT READINESS CHECK")
print("=" * 70)

document_checks = {
    "pdf_found_in_zip": selected_pdf is not None,
    "reader_available": reader_used is not None,
    "page_count": len(rubric_pages),
    "extractable_text_available": (
        reader_used is not None
        and bool(rubric_text.strip())
    )
}

display(
    pd.DataFrame(
        list(document_checks.items()),
        columns=["check", "value"]
    )
)

print("\nINTERPRETATION:")
print(
    "Requirement wording must be reviewed "
    "in context, not inferred from keyword counts."
)
print(
    "Absence of a keyword in extracted text "
    "does not prove that the requirement is absent "
    "from images or scanned pages."
)

In [ ]:
# CELL 61 — Official Round 1 Problem Statement Verification
# CORRECTED VERSION — Automatic PDF Reader Installation

from pathlib import Path
from io import BytesIO
import zipfile
import subprocess
import sys
import re
import pandas as pd
from IPython.display import display

print("=" * 70)
print("OFFICIAL ROUND 1 PROBLEM STATEMENT VERIFICATION")
print("=" * 70)


# ============================================================
# 1. INSTALL PDF READER IF MISSING
# ============================================================

print("\n1. PDF READER ENVIRONMENT")
print("=" * 70)

try:
    from pypdf import PdfReader
    print("pypdf is already installed.")

except ModuleNotFoundError:

    print("Installing pypdf into the current Jupyter environment...")

    installation = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "pypdf"
        ],
        capture_output=True,
        text=True
    )

    if installation.returncode != 0:
        print(installation.stderr[-4000:])
        raise RuntimeError(
            "pypdf installation failed. "
            "Review the installation error above."
        )

    from pypdf import PdfReader

    print("pypdf installed successfully.")

print("Python environment:", sys.executable)


# ============================================================
# 2. LOCATE THE OFFICIAL PDF
# ============================================================

print("\n2. PROBLEM STATEMENT FILE")
print("=" * 70)

assert ZIP_PATH.exists(), (
    f"ZIP not found: {ZIP_PATH.resolve()}"
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    pdf_files = [
        name for name in z.namelist()
        if name.lower().endswith(".pdf")
    ]

    statement_files = [
        name for name in pdf_files
        if "problem statement" in name.lower()
    ]

    assert statement_files, (
        "Round 1 problem statement PDF not found."
    )

    selected_pdf = statement_files[0]

    pdf_bytes = z.read(selected_pdf)

print("Selected PDF:", selected_pdf)
print("File size:", len(pdf_bytes), "bytes")


# ============================================================
# 3. EXTRACT THE COMPLETE PDF TEXT
# ============================================================

print("\n3. PDF EXTRACTION")
print("=" * 70)

reader = PdfReader(BytesIO(pdf_bytes))

rubric_pages = []

for page in reader.pages:

    page_text = page.extract_text() or ""

    rubric_pages.append(page_text)

rubric_text = "\n\n".join(rubric_pages)

print("PDF pages:", len(rubric_pages))
print("Extracted characters:", len(rubric_text.strip()))

for page_number, page_text in enumerate(
    rubric_pages, start=1
):

    print("\n" + "=" * 60)
    print("PAGE", page_number)
    print("=" * 60)

    if page_text.strip():
        print(page_text[:18000])
    else:
        print("[No extractable text on this page]")

    if len(page_text) > 18000:
        print("[Remaining page text retained in rubric_pages]")


# ============================================================
# 4. CHECK OFFICIAL REQUIREMENT WORDING
# ============================================================

print("\n4. OFFICIAL REQUIREMENT KEYWORDS")
print("=" * 70)

requirement_patterns = {
    "Top 10": r"\btop\s*10\b",
    "Uncapped": r"\buncapped\b",
    "Youngest": r"\byoungest\b",
    "Young Players": r"\byoung\b",
    "Age": r"\bage\b",
    "Pressure": r"\bpressure\b",
    "Batters": r"\bbatters?\b",
    "Bowlers": r"\bbowlers?\b",
    "Baseline": r"\bbaseline\b",
    "Relative Performance": r"\brelative\b",
    "Ranking": r"\brank(?:ing|ings|ed)?\b",
    "Submission": r"\bsubmission\b|\bsubmit\b",
    "Evaluation": r"\bevaluation\b|\bcriteria\b"
}

keyword_results = []

for requirement, pattern in requirement_patterns.items():

    matches_found = re.findall(
        pattern,
        rubric_text,
        flags=re.IGNORECASE
    )

    keyword_results.append({
        "requirement": requirement,
        "mentions": len(matches_found),
        "found": bool(matches_found)
    })

display(pd.DataFrame(keyword_results))


# ============================================================
# 5. EXTRACT IMPORTANT PASSAGES
# ============================================================

print("\n5. IMPORTANT TASK AND ELIGIBILITY PASSAGES")
print("=" * 70)

priority_pattern = (
    r"uncapped|youngest|young players|"
    r"top\s*10|pressure|baseline|"
    r"batter|bowler|task|objective|"
    r"submission|evaluation|criteria|"
    r"deliverable"
)

passages = []

for page_number, page_text in enumerate(
    rubric_pages,
    start=1
):

    lines = page_text.splitlines()

    for index, line in enumerate(lines):

        if re.search(
            priority_pattern,
            line,
            flags=re.IGNORECASE
        ):

            start = max(0, index - 1)
            end = min(len(lines), index + 3)

            excerpt = " ".join(
                lines[start:end]
            ).strip()

            passages.append({
                "page": page_number,
                "passage": excerpt
            })

if passages:

    display(
        pd.DataFrame(passages).head(50)
    )

else:

    print("No matching passages found.")


# ============================================================
# 6. DOCUMENT INTEGRITY CHECKS
# ============================================================

print("\n6. DOCUMENT INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "pdf_found": bool(selected_pdf),
    "pdf_reader_available": True,
    "pages_detected": len(rubric_pages),
    "text_extraction_successful": bool(
        rubric_text.strip()
    ),
    "requirement_passages_found": len(passages)
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "value"]
    )
)

if not rubric_text.strip():
    print(
        "\nWARNING: The PDF may be image-only. "
        "Text extraction cannot verify its requirements."
    )

print("\nIMPORTANT:")
print(
    "Final eligibility rules must be determined "
    "from the complete wording of the PDF, "
    "not keyword counts alone."
)

In [ ]:
# CELL 61 — Competition Coverage and Scouting Evidence Audit

print("=" * 70)
print("COMPETITION COVERAGE AND SCOUTING EVIDENCE AUDIT")
print("=" * 70)

source_classes = ["IPL", "T20I", "Other T20"]

# ============================================================
# 1. CLASSIFY MATCHES BY COMPETITION SOURCE
# ============================================================

print("\n1. COMPETITION SOURCE COVERAGE")
print("=" * 70)

source_matches = match_metadata[
    [
        "match_id",
        "comp_id",
        "comp_name",
        "match_format",
        "is_ipl",
        "is_t20i"
    ]
].copy()

source_matches["source_class"] = np.select(
    [
        source_matches["is_ipl"],
        source_matches["is_t20i"]
    ],
    [
        "IPL",
        "T20I"
    ],
    default="Other T20"
)

display(
    source_matches.groupby("source_class")
    .agg(
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
)

print(
    "Total classified matches:",
    source_matches["match_id"].nunique()
)


# ============================================================
# 2. CHECK SOURCE COVERAGE OF THE VALIDATED SAMPLE
# ============================================================

print("\n2. VALIDATED ANALYSIS SAMPLE BY SOURCE")
print("=" * 70)

source_lookup = source_matches[
    ["match_id", "source_class"]
].copy()

scouting_source_events = player_events.merge(
    source_lookup,
    on="match_id",
    how="left",
    validate="many_to_one"
)

display(
    scouting_source_events.groupby("source_class")
    .agg(
        matches=("match_id", "nunique"),
        deliveries=("match_id", "size"),
        unique_batters=("striker_id", "nunique"),
        unique_bowlers=("bowler_id", "nunique"),
        legal_balls=("bowler_legal_ball", "sum")
    )
    .reset_index()
)


# ============================================================
# 3. SOURCE COVERAGE OF QUALIFIED PRESSURE OBSERVATIONS
# ============================================================

print("\n3. QUALIFIED PRESSURE OBSERVATIONS BY SOURCE")
print("=" * 70)

batting_source_events = (
    qualified_pressure_events_boot.merge(
        source_lookup,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

bowling_source_events = (
    bootstrap_pressure_events.merge(
        source_lookup,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

print("\nQualified batting pressure observations")

display(
    batting_source_events.groupby(
        ["batting_pressure_type", "source_class"]
    )
    .agg(
        pressure_balls=("match_id", "size"),
        unique_players=("striker_id", "nunique"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\nQualified bowling pressure observations")

display(
    bowling_source_events.groupby(
        ["bowling_pressure_type", "source_class"]
    )
    .agg(
        pressure_legal_balls=("match_id", "size"),
        unique_players=("bowler_id", "nunique"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)


# ============================================================
# 4. BUILD BATTING PROFILE SOURCE COVERAGE
# ============================================================

print("\n4. BATTING PROFILE SOURCE COVERAGE")
print("=" * 70)

batting_source_counts = (
    batting_source_events.groupby(
        [
            "striker_id",
            "batting_pressure_type",
            "source_class"
        ]
    )
    .size()
    .rename("balls")
    .reset_index()
)

batting_source_wide = (
    batting_source_counts.pivot_table(
        index=[
            "striker_id",
            "batting_pressure_type"
        ],
        columns="source_class",
        values="balls",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(columns=source_classes, fill_value=0)
    .add_prefix("source_balls_")
    .reset_index()
)

batting_source_profiles = (
    batter_uncertainty_profiles.merge(
        batting_source_wide,
        on=[
            "striker_id",
            "batting_pressure_type"
        ],
        how="left",
        validate="one_to_one"
    )
)

batting_source_columns = [
    f"source_balls_{source}"
    for source in source_classes
]

batting_source_profiles[batting_source_columns] = (
    batting_source_profiles[
        batting_source_columns
    ].fillna(0).astype(int)
)

batting_source_profiles["source_count"] = (
    batting_source_profiles[
        batting_source_columns
    ].gt(0).sum(axis=1)
)

display(
    batting_source_profiles.groupby(
        ["batting_pressure_type", "source_count"]
    )
    .agg(
        profiles=("striker_id", "size"),
        pressure_balls=("pressure_balls", "sum")
    )
    .reset_index()
)


# ============================================================
# 5. BUILD BOWLING PROFILE SOURCE COVERAGE
# ============================================================

print("\n5. BOWLING PROFILE SOURCE COVERAGE")
print("=" * 70)

bowling_source_counts = (
    bowling_source_events.groupby(
        [
            "bowler_id",
            "bowling_pressure_type",
            "source_class"
        ]
    )
    .size()
    .rename("balls")
    .reset_index()
)

bowling_source_wide = (
    bowling_source_counts.pivot_table(
        index=[
            "bowler_id",
            "bowling_pressure_type"
        ],
        columns="source_class",
        values="balls",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(columns=source_classes, fill_value=0)
    .add_prefix("source_balls_")
    .reset_index()
)

bowling_source_profiles = (
    bowler_uncertainty_profiles.merge(
        bowling_source_wide,
        on=[
            "bowler_id",
            "bowling_pressure_type"
        ],
        how="left",
        validate="one_to_one"
    )
)

bowling_source_columns = [
    f"source_balls_{source}"
    for source in source_classes
]

bowling_source_profiles[bowling_source_columns] = (
    bowling_source_profiles[
        bowling_source_columns
    ].fillna(0).astype(int)
)

bowling_source_profiles["source_count"] = (
    bowling_source_profiles[
        bowling_source_columns
    ].gt(0).sum(axis=1)
)

display(
    bowling_source_profiles.groupby(
        ["bowling_pressure_type", "source_count"]
    )
    .agg(
        profiles=("bowler_id", "size"),
        pressure_balls=("pressure_legal_balls", "sum")
    )
    .reset_index()
)


# ============================================================
# 6. EXPOSURE EXAMPLES — NOT PLAYER RANKINGS
# ============================================================

print("\n6. EXAMPLE PLAYER SOURCE PROFILES")
print("=" * 70)

batting_example_columns = [
    "player_name",
    "batting_pressure_type",
    "pressure_balls",
    "source_balls_IPL",
    "source_balls_T20I",
    "source_balls_Other T20",
    "matched_sr_uplift",
    "bootstrap_lower_95",
    "bootstrap_upper_95"
]

print("\nBatting — highest pressure exposure")

display(
    batting_source_profiles.sort_values(
        "pressure_balls",
        ascending=False
    )[batting_example_columns].head(12)
)

bowling_example_columns = [
    "player_name",
    "bowling_pressure_type",
    "pressure_legal_balls",
    "source_balls_IPL",
    "source_balls_T20I",
    "source_balls_Other T20",
    "clean_dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_rate_uplift_pp"
]

print("\nBowling — highest pressure exposure")

display(
    bowling_source_profiles.sort_values(
        "pressure_legal_balls",
        ascending=False
    )[bowling_example_columns].head(12)
)


# ============================================================
# 7. FINAL INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "duplicate_match_sources": int(
        source_matches["match_id"].duplicated().sum()
    ),

    "unclassified_analysis_deliveries": int(
        scouting_source_events["source_class"].isna().sum()
    ),

    "unclassified_batting_pressure_balls": int(
        batting_source_events["source_class"].isna().sum()
    ),

    "unclassified_bowling_pressure_balls": int(
        bowling_source_events["source_class"].isna().sum()
    ),

    "batting_pressure_ball_reconciliation": int(
        (
            batting_source_profiles[
                batting_source_columns
            ].sum(axis=1)
            != batting_source_profiles["pressure_balls"]
        ).sum()
    ),

    "bowling_pressure_ball_reconciliation": int(
        (
            bowling_source_profiles[
                bowling_source_columns
            ].sum(axis=1)
            != bowling_source_profiles[
                "pressure_legal_balls"
            ]
        ).sum()
    ),

    "missing_batting_source_profiles": abs(
        len(batting_source_profiles) -
        len(batter_uncertainty_profiles)
    ),

    "missing_bowling_source_profiles": abs(
        len(bowling_source_profiles) -
        len(bowler_uncertainty_profiles)
    ),

    "duplicate_batting_source_profiles": int(
        batting_source_profiles.duplicated(
            ["striker_id", "batting_pressure_type"]
        ).sum()
    ),

    "duplicate_bowling_source_profiles": int(
        bowling_source_profiles.duplicated(
            ["bowler_id", "bowling_pressure_type"]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 62 — Same-Source Personal Baseline Audit

print("=" * 70)
print("SAME-SOURCE PERSONAL BASELINE AUDIT")
print("=" * 70)

source_labels = ["IPL", "T20I", "Other T20"]

batting_source_keys = [
    "striker_id",
    "batting_pressure_type",
    "innings",
    "match_phase",
    "source_class"
]

bowling_source_keys = [
    "bowler_id",
    "bowling_pressure_type",
    "innings",
    "match_phase",
    "source_class"
]


# ============================================================
# 1. PREPARE SAME-SOURCE REFERENCE OBSERVATIONS
# ============================================================

print("\n1. REFERENCE SOURCE COVERAGE")
print("=" * 70)

batting_source_references = (
    qualified_reference_events_boot.merge(
        source_lookup,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

bowling_source_references = (
    bootstrap_reference_events.merge(
        source_lookup,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

display(
    pd.DataFrame([
        {
            "role": "Batting",
            "pressure_observations": len(
                batting_source_events
            ),
            "reference_observations": len(
                batting_source_references
            ),
            "pressure_profiles": (
                batting_source_events[
                    ["striker_id", "batting_pressure_type"]
                ].drop_duplicates().shape[0]
            )
        },
        {
            "role": "Bowling",
            "pressure_observations": len(
                bowling_source_events
            ),
            "reference_observations": len(
                bowling_source_references
            ),
            "pressure_profiles": (
                bowling_source_events[
                    ["bowler_id", "bowling_pressure_type"]
                ].drop_duplicates().shape[0]
            )
        }
    ])
)


# ============================================================
# 2. BATTER SAME-SOURCE MATCHED STRATA
# ============================================================

print("\n2. BATTING SAME-SOURCE BASELINE COVERAGE")
print("=" * 70)

batting_pressure_by_source = (
    batting_source_events.groupby(batting_source_keys)
    .agg(
        pressure_balls=("match_id", "size"),
        pressure_runs=("batter_runs", "sum"),
        pressure_matches=("match_id", "nunique")
    )
    .reset_index()
)

batting_reference_by_source = (
    batting_source_references.groupby(
        batting_source_keys
    )
    .agg(
        reference_balls=("match_id", "size"),
        reference_runs=("batter_runs", "sum"),
        reference_matches=("match_id", "nunique")
    )
    .reset_index()
)

batting_same_source = (
    batting_pressure_by_source.merge(
        batting_reference_by_source,
        on=batting_source_keys,
        how="left",
        validate="one_to_one"
    )
)

batting_reference_cols = [
    "reference_balls",
    "reference_runs",
    "reference_matches"
]

batting_same_source[batting_reference_cols] = (
    batting_same_source[
        batting_reference_cols
    ].fillna(0)
)

batting_same_source["matched_pressure_balls"] = (
    np.where(
        batting_same_source["reference_balls"].gt(0),
        batting_same_source["pressure_balls"],
        0
    )
)

batting_same_source["reference_sr"] = (
    100 * batting_same_source["reference_runs"].div(
        batting_same_source[
            "reference_balls"
        ].replace(0, np.nan)
    )
)

batting_coverage = (
    batting_same_source.groupby("source_class")
    .agg(
        pressure_strata=("striker_id", "size"),
        players=("striker_id", "nunique"),
        pressure_balls=("pressure_balls", "sum"),
        matched_pressure_balls=(
            "matched_pressure_balls", "sum"
        )
    )
    .reset_index()
)

batting_coverage["same_source_coverage_pct"] = (
    100 *
    batting_coverage["matched_pressure_balls"] /
    batting_coverage["pressure_balls"]
)

display(batting_coverage)


# ============================================================
# 3. BOWLER SAME-SOURCE MATCHED STRATA
# ============================================================

print("\n3. BOWLING SAME-SOURCE BASELINE COVERAGE")
print("=" * 70)

bowling_pressure_by_source = (
    bowling_source_events.groupby(bowling_source_keys)
    .agg(
        pressure_balls=("match_id", "size"),
        pressure_matches=("match_id", "nunique"),
        pressure_clean_dots=("clean_dot_event", "sum"),
        pressure_boundaries=("boundary_event", "sum"),
        pressure_wickets=("credited_wicket_event", "sum")
    )
    .reset_index()
)

bowling_reference_by_source = (
    bowling_source_references.groupby(
        bowling_source_keys
    )
    .agg(
        reference_balls=("match_id", "size"),
        reference_matches=("match_id", "nunique"),
        reference_clean_dots=("clean_dot_event", "sum"),
        reference_boundaries=("boundary_event", "sum"),
        reference_wickets=("credited_wicket_event", "sum")
    )
    .reset_index()
)

bowling_same_source = (
    bowling_pressure_by_source.merge(
        bowling_reference_by_source,
        on=bowling_source_keys,
        how="left",
        validate="one_to_one"
    )
)

bowling_reference_cols = [
    "reference_balls",
    "reference_matches",
    "reference_clean_dots",
    "reference_boundaries",
    "reference_wickets"
]

bowling_same_source[bowling_reference_cols] = (
    bowling_same_source[
        bowling_reference_cols
    ].fillna(0)
)

bowling_same_source["matched_pressure_balls"] = (
    np.where(
        bowling_same_source["reference_balls"].gt(0),
        bowling_same_source["pressure_balls"],
        0
    )
)

bowling_same_source["reference_clean_dot_rate"] = (
    100 * bowling_same_source[
        "reference_clean_dots"
    ].div(
        bowling_same_source[
            "reference_balls"
        ].replace(0, np.nan)
    )
)

bowling_same_source["reference_boundary_rate"] = (
    100 * bowling_same_source[
        "reference_boundaries"
    ].div(
        bowling_same_source[
            "reference_balls"
        ].replace(0, np.nan)
    )
)

bowling_coverage = (
    bowling_same_source.groupby("source_class")
    .agg(
        pressure_strata=("bowler_id", "size"),
        bowlers=("bowler_id", "nunique"),
        pressure_balls=("pressure_balls", "sum"),
        matched_pressure_balls=(
            "matched_pressure_balls", "sum"
        )
    )
    .reset_index()
)

bowling_coverage["same_source_coverage_pct"] = (
    100 *
    bowling_coverage["matched_pressure_balls"] /
    bowling_coverage["pressure_balls"]
)

display(bowling_coverage)


# ============================================================
# 4. SAME-SOURCE SAMPLE SUFFICIENCY
# ============================================================

print("\n4. SAMPLE SUFFICIENCY BY SOURCE AND ROLE")
print("=" * 70)

thresholds = [
    (5, 20),
    (10, 30),
    (15, 50)
]

source_threshold_results = []

for role, source_df, player_col in [
    ("Batting", batting_same_source, "striker_id"),
    ("Bowling", bowling_same_source, "bowler_id")
]:

    for source in source_labels:

        subset = source_df.loc[
            source_df["source_class"].eq(source)
        ]

        for min_pressure, min_reference in thresholds:

            sufficient = (
                subset["pressure_balls"].ge(min_pressure) &
                subset["reference_balls"].ge(min_reference) &
                subset["pressure_matches"].ge(2) &
                subset["reference_matches"].ge(2)
            )

            source_threshold_results.append({
                "role": role,
                "source": source,
                "minimum_pressure_balls": min_pressure,
                "minimum_reference_balls": min_reference,
                "minimum_matches_each": 2,
                "qualified_strata": int(sufficient.sum()),
                "qualified_players": int(
                    subset.loc[
                        sufficient, player_col
                    ].nunique()
                ),
                "covered_pressure_balls": int(
                    subset.loc[
                        sufficient, "pressure_balls"
                    ].sum()
                )
            })

display(pd.DataFrame(source_threshold_results))


# ============================================================
# 5. EXAMINE BATTING REFERENCE RATE DIFFERENCES
# ============================================================

print("\n5. IPL VERSUS T20I BATTING BASELINES")
print("=" * 70)

# Require a minimum amount of evidence in BOTH sources.
batting_rate_sample = batting_same_source.loc[
    batting_same_source["source_class"].isin(
        ["IPL", "T20I"]
    ) &
    batting_same_source["pressure_balls"].ge(5) &
    batting_same_source["reference_balls"].ge(20)
].copy()

batting_rate_pivot = (
    batting_rate_sample.pivot(
        index=[
            "striker_id",
            "batting_pressure_type",
            "innings",
            "match_phase"
        ],
        columns="source_class",
        values="reference_sr"
    )
    .reindex(columns=["IPL", "T20I"])
    .dropna()
    .reset_index()
)

batting_rate_pivot["ipl_minus_t20i_sr"] = (
    batting_rate_pivot["IPL"] -
    batting_rate_pivot["T20I"]
)

batting_rate_pivot["absolute_sr_gap"] = (
    batting_rate_pivot["ipl_minus_t20i_sr"].abs()
)

print(
    "Same-player, same-phase strata with both sources:",
    len(batting_rate_pivot)
)

display(
    batting_rate_pivot[
        "absolute_sr_gap"
    ].describe()
)

display(
    batting_rate_pivot.sort_values(
        "absolute_sr_gap",
        ascending=False
    ).head(10)
)


# ============================================================
# 6. EXAMINE BOWLING REFERENCE RATE DIFFERENCES
# ============================================================

print("\n6. IPL VERSUS T20I BOWLING BASELINES")
print("=" * 70)

bowling_rate_sample = bowling_same_source.loc[
    bowling_same_source["source_class"].isin(
        ["IPL", "T20I"]
    ) &
    bowling_same_source["pressure_balls"].ge(5) &
    bowling_same_source["reference_balls"].ge(20)
].copy()

bowling_rate_index = [
    "bowler_id",
    "bowling_pressure_type",
    "innings",
    "match_phase"
]

bowling_dot_pivot = (
    bowling_rate_sample.pivot(
        index=bowling_rate_index,
        columns="source_class",
        values="reference_clean_dot_rate"
    )
    .reindex(columns=["IPL", "T20I"])
    .dropna()
)

bowling_boundary_pivot = (
    bowling_rate_sample.pivot(
        index=bowling_rate_index,
        columns="source_class",
        values="reference_boundary_rate"
    )
    .reindex(columns=["IPL", "T20I"])
    .dropna()
)

common_bowling_index = (
    bowling_dot_pivot.index.intersection(
        bowling_boundary_pivot.index
    )
)

bowling_rate_comparison = pd.DataFrame({
    "dot_rate_gap_pp": (
        bowling_dot_pivot.loc[
            common_bowling_index, "IPL"
        ] -
        bowling_dot_pivot.loc[
            common_bowling_index, "T20I"
        ]
    ),

    "boundary_rate_gap_pp": (
        bowling_boundary_pivot.loc[
            common_bowling_index, "IPL"
        ] -
        bowling_boundary_pivot.loc[
            common_bowling_index, "T20I"
        ]
    )
}).reset_index()

print(
    "Same-bowler, same-phase strata with both sources:",
    len(bowling_rate_comparison)
)

display(
    bowling_rate_comparison[
        ["dot_rate_gap_pp", "boundary_rate_gap_pp"]
    ].describe().T
)

display(
    bowling_rate_comparison.assign(
        absolute_dot_gap=lambda df: (
            df["dot_rate_gap_pp"].abs()
        )
    )
    .sort_values(
        "absolute_dot_gap",
        ascending=False
    )
    .head(10)
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "batting_pressure_ball_reconciliation": abs(
        int(
            batting_same_source["pressure_balls"].sum()
        ) - len(batting_source_events)
    ),

    "batting_reference_ball_reconciliation": abs(
        int(
            batting_reference_by_source[
                "reference_balls"
            ].sum()
        ) - len(batting_source_references)
    ),

    "bowling_pressure_ball_reconciliation": abs(
        int(
            bowling_same_source["pressure_balls"].sum()
        ) - len(bowling_source_events)
    ),

    "bowling_reference_ball_reconciliation": abs(
        int(
            bowling_reference_by_source[
                "reference_balls"
            ].sum()
        ) - len(bowling_source_references)
    ),

    "missing_batting_reference_source": int(
        batting_source_references[
            "source_class"
        ].isna().sum()
    ),

    "missing_bowling_reference_source": int(
        bowling_source_references[
            "source_class"
        ].isna().sum()
    ),

    "duplicate_batting_source_strata": int(
        batting_same_source.duplicated(
            batting_source_keys
        ).sum()
    ),

    "duplicate_bowling_source_strata": int(
        bowling_same_source.duplicated(
            bowling_source_keys
        ).sum()
    ),

    "negative_batting_ball_counts": int(
        (
            (batting_same_source["pressure_balls"] < 0) |
            (batting_same_source["reference_balls"] < 0)
        ).sum()
    ),

    "negative_bowling_ball_counts": int(
        (
            (bowling_same_source["pressure_balls"] < 0) |
            (bowling_same_source["reference_balls"] < 0)
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

In [ ]:
# CELL 63 — Source-Adjusted Personal Pressure Performance

print("=" * 70)
print("SOURCE-ADJUSTED PERSONAL PRESSURE PERFORMANCE")
print("=" * 70)

bat_profile_keys = ["striker_id", "batting_pressure_type"]
bowl_profile_keys = ["bowler_id", "bowling_pressure_type"]

bat_stratum_keys = bat_profile_keys + [
    "innings", "match_phase"
]

bowl_stratum_keys = bowl_profile_keys + [
    "innings", "match_phase"
]


# ============================================================
# 1. REBUILD POOLED REFERENCES FOR FAIR COMPARISON
# ============================================================

print("\n1. POOLED REFERENCE STRATA")
print("=" * 70)

bat_pooled_reference = (
    batting_source_references.groupby(bat_stratum_keys)
    .agg(
        pooled_ref_balls=("match_id", "size"),
        pooled_ref_runs=("batter_runs", "sum")
    )
    .reset_index()
)

bowl_pooled_reference = (
    bowling_source_references.groupby(bowl_stratum_keys)
    .agg(
        pooled_ref_balls=("match_id", "size"),
        pooled_ref_clean_dots=("clean_dot_event", "sum"),
        pooled_ref_boundaries=("boundary_event", "sum"),
        pooled_ref_wickets=("credited_wicket_event", "sum")
    )
    .reset_index()
)

print("Batting pooled reference strata:", len(bat_pooled_reference))
print("Bowling pooled reference strata:", len(bowl_pooled_reference))


# ============================================================
# 2. SOURCE-ADJUSTED BATTING
# ============================================================

print("\n2. SOURCE-ADJUSTED BATTING PERFORMANCE")
print("=" * 70)

batting_matched_source = (
    batting_same_source.loc[
        batting_same_source["reference_balls"].gt(0)
    ]
    .merge(
        bat_pooled_reference,
        on=bat_stratum_keys,
        how="left",
        validate="many_to_one"
    )
    .copy()
)

assert batting_matched_source["pooled_ref_balls"].gt(0).all()

batting_matched_source["source_expected_runs"] = (
    batting_matched_source["pressure_balls"] *
    batting_matched_source["reference_runs"] /
    batting_matched_source["reference_balls"]
)

batting_matched_source["pooled_expected_runs"] = (
    batting_matched_source["pressure_balls"] *
    batting_matched_source["pooled_ref_runs"] /
    batting_matched_source["pooled_ref_balls"]
)

batting_source_adjustment = (
    batting_matched_source.groupby(bat_profile_keys)
    .agg(
        source_matched_balls=("pressure_balls", "sum"),
        source_actual_runs=("pressure_runs", "sum"),
        source_expected_runs=("source_expected_runs", "sum"),
        pooled_expected_runs=("pooled_expected_runs", "sum"),
        matched_sources=("source_class", "nunique")
    )
    .reset_index()
)

batting_source_adjustment["source_adjusted_sr_uplift"] = (
    100 * (
        batting_source_adjustment["source_actual_runs"] -
        batting_source_adjustment["source_expected_runs"]
    ) /
    batting_source_adjustment["source_matched_balls"]
)

batting_source_adjustment["pooled_matched_sr_uplift"] = (
    100 * (
        batting_source_adjustment["source_actual_runs"] -
        batting_source_adjustment["pooled_expected_runs"]
    ) /
    batting_source_adjustment["source_matched_balls"]
)

batting_source_adjustment["source_adjustment_sr"] = (
    batting_source_adjustment["source_adjusted_sr_uplift"] -
    batting_source_adjustment["pooled_matched_sr_uplift"]
)

batting_adjusted_profiles = batting_source_profiles.merge(
    batting_source_adjustment,
    on=bat_profile_keys,
    how="left",
    validate="one_to_one"
)

batting_adjusted_profiles["source_coverage_pct"] = (
    100 *
    batting_adjusted_profiles["source_matched_balls"] /
    batting_adjusted_profiles["pressure_balls"]
)

print(
    "Batting profiles retained:",
    len(batting_adjusted_profiles)
)

print(
    "Matched pressure balls:",
    int(
        batting_adjusted_profiles[
            "source_matched_balls"
        ].sum()
    )
)

display(
    batting_adjusted_profiles[
        [
            "source_coverage_pct",
            "matched_sr_uplift",
            "pooled_matched_sr_uplift",
            "source_adjusted_sr_uplift",
            "source_adjustment_sr"
        ]
    ].describe().T
)


# ============================================================
# 3. SOURCE-ADJUSTED BOWLING
# ============================================================

print("\n3. SOURCE-ADJUSTED BOWLING PERFORMANCE")
print("=" * 70)

bowling_matched_source = (
    bowling_same_source.loc[
        bowling_same_source["reference_balls"].gt(0)
    ]
    .merge(
        bowl_pooled_reference,
        on=bowl_stratum_keys,
        how="left",
        validate="many_to_one"
    )
    .copy()
)

assert bowling_matched_source["pooled_ref_balls"].gt(0).all()

outcomes = [
    "clean_dots",
    "boundaries",
    "wickets"
]

for outcome in outcomes:

    bowling_matched_source[
        f"source_expected_{outcome}"
    ] = (
        bowling_matched_source["pressure_balls"] *
        bowling_matched_source[f"reference_{outcome}"] /
        bowling_matched_source["reference_balls"]
    )

    bowling_matched_source[
        f"pooled_expected_{outcome}"
    ] = (
        bowling_matched_source["pressure_balls"] *
        bowling_matched_source[f"pooled_ref_{outcome}"] /
        bowling_matched_source["pooled_ref_balls"]
    )

aggregation = {
    "source_matched_balls": ("pressure_balls", "sum"),
    "matched_sources": ("source_class", "nunique")
}

for outcome in outcomes:

    aggregation[f"actual_{outcome}"] = (
        f"pressure_{outcome}", "sum"
    )

    aggregation[f"source_expected_{outcome}"] = (
        f"source_expected_{outcome}", "sum"
    )

    aggregation[f"pooled_expected_{outcome}"] = (
        f"pooled_expected_{outcome}", "sum"
    )

bowling_source_adjustment = (
    bowling_matched_source.groupby(bowl_profile_keys)
    .agg(**aggregation)
    .reset_index()
)

bowling_metric_specs = {
    "clean_dots": ("clean_dot_uplift_pp", 1),
    "boundaries": ("boundary_suppression_pp", -1),
    "wickets": ("wicket_rate_uplift_pp", 1)
}

for outcome, (metric_name, direction) in (
    bowling_metric_specs.items()
):

    bowling_source_adjustment[
        f"source_adjusted_{metric_name}"
    ] = (
        direction * 100 *
        (
            bowling_source_adjustment[f"actual_{outcome}"] -
            bowling_source_adjustment[
                f"source_expected_{outcome}"
            ]
        ) /
        bowling_source_adjustment["source_matched_balls"]
    )

    bowling_source_adjustment[
        f"pooled_matched_{metric_name}"
    ] = (
        direction * 100 *
        (
            bowling_source_adjustment[f"actual_{outcome}"] -
            bowling_source_adjustment[
                f"pooled_expected_{outcome}"
            ]
        ) /
        bowling_source_adjustment["source_matched_balls"]
    )

    bowling_source_adjustment[
        f"source_adjustment_{metric_name}"
    ] = (
        bowling_source_adjustment[
            f"source_adjusted_{metric_name}"
        ] -
        bowling_source_adjustment[
            f"pooled_matched_{metric_name}"
        ]
    )

bowling_adjusted_profiles = bowling_source_profiles.merge(
    bowling_source_adjustment,
    on=bowl_profile_keys,
    how="left",
    validate="one_to_one"
)

bowling_adjusted_profiles["source_coverage_pct"] = (
    100 *
    bowling_adjusted_profiles["source_matched_balls"] /
    bowling_adjusted_profiles["pressure_legal_balls"]
)

print(
    "Bowling profiles retained:",
    len(bowling_adjusted_profiles)
)

print(
    "Matched pressure legal balls:",
    int(
        bowling_adjusted_profiles[
            "source_matched_balls"
        ].sum()
    )
)

display(
    bowling_adjusted_profiles[
        [
            "source_coverage_pct",
            "source_adjustment_clean_dot_uplift_pp",
            "source_adjustment_boundary_suppression_pp",
            "source_adjustment_wicket_rate_uplift_pp"
        ]
    ].describe().T
)


# ============================================================
# 4. PROFILE DIRECTION CHANGES
# ============================================================

print("\n4. EFFECT DIRECTION SENSITIVITY")
print("=" * 70)

direction_results = []

bat_old = batting_adjusted_profiles[
    "pooled_matched_sr_uplift"
]

bat_new = batting_adjusted_profiles[
    "source_adjusted_sr_uplift"
]

direction_results.append({
    "role": "Batting",
    "metric": "SR uplift",
    "profiles": int(
        bat_old.notna().sum()
    ),
    "direction_changes": int(
        (
            (bat_old.gt(0) & bat_new.lt(0)) |
            (bat_old.lt(0) & bat_new.gt(0))
        ).sum()
    ),
    "median_absolute_adjustment": (
        (bat_new - bat_old).abs().median()
    )
})

for outcome, (metric_name, _) in (
    bowling_metric_specs.items()
):

    old = bowling_adjusted_profiles[
        f"pooled_matched_{metric_name}"
    ]

    new = bowling_adjusted_profiles[
        f"source_adjusted_{metric_name}"
    ]

    direction_results.append({
        "role": "Bowling",
        "metric": metric_name,
        "profiles": int(old.notna().sum()),
        "direction_changes": int(
            (
                (old.gt(0) & new.lt(0)) |
                (old.lt(0) & new.gt(0))
            ).sum()
        ),
        "median_absolute_adjustment": (
            (new - old).abs().median()
        )
    })

display(pd.DataFrame(direction_results))


# ============================================================
# 5. SAME-SOURCE SAMPLE SUFFICIENCY
# ============================================================

print("\n5. SAME-SOURCE SAMPLE SUFFICIENCY")
print("=" * 70)

sufficiency_results = []

for role, df, player_column in [
    (
        "Batting",
        batting_matched_source,
        "striker_id"
    ),
    (
        "Bowling",
        bowling_matched_source,
        "bowler_id"
    )
]:

    df = df.copy()

    df["source_sample_qualified"] = (
        df["pressure_balls"].ge(10) &
        df["reference_balls"].ge(30) &
        df["pressure_matches"].ge(2) &
        df["reference_matches"].ge(2)
    )

    for source in ["IPL", "T20I", "Other T20"]:

        source_df = df.loc[
            df["source_class"].eq(source)
        ]

        qualified_df = source_df.loc[
            source_df["source_sample_qualified"]
        ]

        sufficiency_results.append({
            "role": role,
            "source": source,
            "matched_pressure_balls": int(
                source_df["pressure_balls"].sum()
            ),
            "qualified_strata": len(qualified_df),
            "qualified_players": (
                qualified_df[player_column].nunique()
            ),
            "qualified_pressure_balls": int(
                qualified_df["pressure_balls"].sum()
            )
        })

display(pd.DataFrame(sufficiency_results))


# ============================================================
# 6. LARGEST SOURCE ADJUSTMENTS — NOT RANKINGS
# ============================================================

print("\n6. SENSITIVITY EXAMPLES — NOT PLAYER RANKINGS")
print("=" * 70)

batting_examples = batting_adjusted_profiles.copy()

batting_examples["absolute_source_adjustment"] = (
    batting_examples["source_adjustment_sr"].abs()
)

print("\nBatting: largest changes after source matching")

display(
    batting_examples.sort_values(
        "absolute_source_adjustment",
        ascending=False
    )[
        [
            "player_name",
            "batting_pressure_type",
            "pressure_balls",
            "source_matched_balls",
            "pooled_matched_sr_uplift",
            "source_adjusted_sr_uplift",
            "source_adjustment_sr"
        ]
    ].head(12)
)

bowling_examples = bowling_adjusted_profiles.copy()

bowling_examples["absolute_source_adjustment"] = (
    bowling_examples[
        "source_adjustment_boundary_suppression_pp"
    ].abs()
)

print(
    "\nBowling: largest boundary-suppression changes "
    "after source matching"
)

display(
    bowling_examples.sort_values(
        "absolute_source_adjustment",
        ascending=False
    )[
        [
            "player_name",
            "bowling_pressure_type",
            "pressure_legal_balls",
            "source_matched_balls",
            "pooled_matched_boundary_suppression_pp",
            "source_adjusted_boundary_suppression_pp",
            "source_adjustment_boundary_suppression_pp"
        ]
    ].head(12)
)


# ============================================================
# 7. INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

batting_matched_total = int(
    batting_matched_source["pressure_balls"].sum()
)

bowling_matched_total = int(
    bowling_matched_source["pressure_balls"].sum()
)

checks = {
    "batting_matched_ball_reconciliation": abs(
        batting_matched_total -
        int(
            batting_adjusted_profiles[
                "source_matched_balls"
            ].sum()
        )
    ),

    "bowling_matched_ball_reconciliation": abs(
        bowling_matched_total -
        int(
            bowling_adjusted_profiles[
                "source_matched_balls"
            ].sum()
        )
    ),

    "batting_source_coverage_exceeds_total": int(
        (
            batting_adjusted_profiles[
                "source_matched_balls"
            ] >
            batting_adjusted_profiles["pressure_balls"]
        ).sum()
    ),

    "bowling_source_coverage_exceeds_total": int(
        (
            bowling_adjusted_profiles[
                "source_matched_balls"
            ] >
            bowling_adjusted_profiles[
                "pressure_legal_balls"
            ]
        ).sum()
    ),

    "missing_batting_adjusted_profiles": int(
        batting_adjusted_profiles[
            "source_adjusted_sr_uplift"
        ].isna().sum()
    ),

    "missing_bowling_adjusted_profiles": int(
        bowling_adjusted_profiles[
            "source_adjusted_clean_dot_uplift_pp"
        ].isna().sum()
    ),

    "duplicate_batting_adjusted_profiles": int(
        batting_adjusted_profiles.duplicated(
            bat_profile_keys
        ).sum()
    ),

    "duplicate_bowling_adjusted_profiles": int(
        bowling_adjusted_profiles.duplicated(
            bowl_profile_keys
        ).sum()
    ),

    "negative_batting_expected_runs": int(
        (
            batting_matched_source[
                ["source_expected_runs", "pooled_expected_runs"]
            ] < 0
        ).any(axis=1).sum()
    ),

    "negative_bowling_expected_events": int(
        (
            bowling_matched_source[
                [
                    f"{prefix}_expected_{outcome}"
                    for prefix in ["source", "pooled"]
                    for outcome in outcomes
                ]
            ] < 0
        ).any(axis=1).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

print("\nINTERPRETATION")
print("=" * 70)

print(
    "Same-source and pooled effects are compared "
    "on the SAME matched pressure deliveries."
)

print(
    "The existing bootstrap intervals apply to "
    "the earlier pooled models, NOT these "
    "new source-adjusted estimates."
)

print(
    "These profiles remain exploratory. "
    "No final player eligibility or rankings "
    "have been established."
)

In [ ]:
# ============================================================
# CELL 64 — SOURCE-ADJUSTED BOOTSTRAP RELIABILITY
# COMPLETE REPLACEMENT
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("CELL 64 — SOURCE-ADJUSTED BOOTSTRAP RELIABILITY")
print("=" * 70)

N_BOOT = 400


# ============================================================
# 1. VERIFY INPUTS AND DEFINE BOOTSTRAP FUNCTION
# ============================================================

print("\n1. INPUT VERIFICATION AND BOOTSTRAP CONFIGURATION")
print("=" * 70)

required_inputs = [
    "batting_source_events",
    "batting_source_references",
    "batting_matched_source",
    "batting_adjusted_profiles",
    "bowling_source_events",
    "bowling_source_references",
    "bowling_matched_source",
    "bowling_adjusted_profiles"
]

missing_inputs = [
    name for name in required_inputs
    if name not in globals()
]

assert not missing_inputs, (
    f"Missing previous-cell variables: {missing_inputs}"
)

print("Required previous-cell variables: AVAILABLE")
print("Bootstrap repetitions:", N_BOOT)
print("Random seeds: Batting 2064 | Bowling 2065")


def source_cluster_bootstrap(
    pressure_events,
    reference_events,
    matched_strata,
    profile_table,
    profile_keys,
    event_columns,
    metric_labels,
    directions,
    seed
):

    strata_keys = profile_keys + [
        "innings",
        "match_phase",
        "source_class"
    ]

    qualified_strata = (
        matched_strata.loc[
            matched_strata["reference_balls"].gt(0),
            strata_keys
        ]
        .drop_duplicates()
    )

    pressure = pressure_events.merge(
        qualified_strata,
        on=strata_keys,
        how="inner",
        validate="many_to_one"
    )

    reference = reference_events.merge(
        qualified_strata,
        on=strata_keys,
        how="inner",
        validate="many_to_one"
    )

    aggregation = {
        "balls": ("match_id", "size")
    }

    event_names = []

    for i, column in enumerate(event_columns):

        event_name = f"event_{i}"
        event_names.append(event_name)

        aggregation[event_name] = (column, "sum")

    match_keys = strata_keys + ["match_id"]

    pressure_matches = (
        pressure.groupby(match_keys)
        .agg(**aggregation)
        .reset_index()
    )

    reference_matches = (
        reference.groupby(match_keys)
        .agg(**aggregation)
        .reset_index()
    )

    reference_lookup = {
        key: group
        for key, group in reference_matches.groupby(
            strata_keys,
            sort=False
        )
    }

    profile_order = (
        profile_table[profile_keys]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    profile_positions = {
        tuple(row): i
        for i, row in enumerate(
            profile_order.itertuples(
                index=False,
                name=None
            )
        )
    }

    n_profiles = len(profile_order)
    n_metrics = len(event_columns)

    rng = np.random.default_rng(seed)

    original_balls = np.zeros(n_profiles)

    original_differences = np.zeros(
        (n_profiles, n_metrics)
    )

    bootstrap_balls = np.zeros(
        (n_profiles, N_BOOT)
    )

    bootstrap_differences = np.zeros(
        (n_profiles, N_BOOT, n_metrics)
    )

    strata_counts = np.zeros(
        n_profiles,
        dtype=int
    )

    weak_strata_counts = np.zeros(
        n_profiles,
        dtype=int
    )

    matrix_columns = ["balls"] + event_names

    for key, p_group in pressure_matches.groupby(
        strata_keys,
        sort=False
    ):

        if key not in reference_lookup:
            raise ValueError(
                f"Missing source reference: {key}"
            )

        r_group = reference_lookup[key]

        profile_key = tuple(key[:len(profile_keys)])
        position = profile_positions[profile_key]

        p_array = p_group[
            matrix_columns
        ].to_numpy(dtype=float)

        r_array = r_group[
            matrix_columns
        ].to_numpy(dtype=float)

        n_p = len(p_array)
        n_r = len(r_array)

        strata_counts[position] += 1

        if n_p < 2 or n_r < 2:
            weak_strata_counts[position] += 1

        p_total = p_array.sum(axis=0)
        r_total = r_array.sum(axis=0)

        assert p_total[0] > 0
        assert r_total[0] > 0

        original_balls[position] += p_total[0]

        original_differences[position] += (
            p_total[1:] -
            p_total[0] *
            (r_total[1:] / r_total[0])
        )

        # Whole-match resampling within each
        # player, source and phase stratum.

        p_weights = rng.multinomial(
            n_p,
            np.full(n_p, 1.0 / n_p),
            size=N_BOOT
        )

        r_weights = rng.multinomial(
            n_r,
            np.full(n_r, 1.0 / n_r),
            size=N_BOOT
        )

        p_boot = p_weights @ p_array
        r_boot = r_weights @ r_array

        assert np.all(p_boot[:, 0] > 0)
        assert np.all(r_boot[:, 0] > 0)

        reference_rates = (
            r_boot[:, 1:] /
            r_boot[:, 0, None]
        )

        expected_events = (
            p_boot[:, 0, None] *
            reference_rates
        )

        bootstrap_balls[position] += (
            p_boot[:, 0]
        )

        bootstrap_differences[position] += (
            p_boot[:, 1:] - expected_events
        )

    assert np.all(original_balls > 0)
    assert np.all(bootstrap_balls > 0)

    direction_array = np.asarray(
        directions,
        dtype=float
    )

    point_estimates = (
        100 *
        original_differences *
        direction_array /
        original_balls[:, None]
    )

    bootstrap_estimates = (
        100 *
        bootstrap_differences *
        direction_array[None, None, :] /
        bootstrap_balls[:, :, None]
    )

    quantiles = np.quantile(
        bootstrap_estimates,
        [0.025, 0.50, 0.975],
        axis=1
    )

    result = profile_order.copy()

    result["source_bootstrap_strata"] = (
        strata_counts
    )

    result["source_bootstrap_weak_strata"] = (
        weak_strata_counts
    )

    result["source_bootstrap_original_balls"] = (
        original_balls
    )

    result["source_bootstrap_valid_draws"] = (
        np.isfinite(bootstrap_estimates)
        .all(axis=2)
        .sum(axis=1)
    )

    for i, label in enumerate(metric_labels):

        result[f"{label}_source_point"] = (
            point_estimates[:, i]
        )

        result[f"{label}_source_lower_95"] = (
            quantiles[0, :, i]
        )

        result[f"{label}_source_median"] = (
            quantiles[1, :, i]
        )

        result[f"{label}_source_upper_95"] = (
            quantiles[2, :, i]
        )

    pressure_match_count = (
        pressure_matches.groupby(profile_keys)[
            "match_id"
        ]
        .nunique()
        .rename("source_pressure_matches")
        .reset_index()
    )

    reference_match_count = (
        reference_matches.groupby(profile_keys)[
            "match_id"
        ]
        .nunique()
        .rename("source_reference_matches")
        .reset_index()
    )

    result = (
        result.merge(
            pressure_match_count,
            on=profile_keys,
            how="left",
            validate="one_to_one"
        )
        .merge(
            reference_match_count,
            on=profile_keys,
            how="left",
            validate="one_to_one"
        )
    )

    return result, pressure, reference


print("Bootstrap function defined successfully.")


# ============================================================
# 2. BATTING BOOTSTRAP
# ============================================================

print("\n2. SOURCE-ADJUSTED BATTING BOOTSTRAP")
print("=" * 70)

(
    batting_source_bootstrap,
    batting_boot_pressure_events,
    batting_boot_reference_events
) = source_cluster_bootstrap(
    pressure_events=batting_source_events,
    reference_events=batting_source_references,
    matched_strata=batting_matched_source,
    profile_table=batting_adjusted_profiles,
    profile_keys=[
        "striker_id",
        "batting_pressure_type"
    ],
    event_columns=["batter_runs"],
    metric_labels=["sr"],
    directions=[1],
    seed=2064
)

batting_reliability_profiles = (
    batting_adjusted_profiles.merge(
        batting_source_bootstrap,
        on=[
            "striker_id",
            "batting_pressure_type"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    "Batting profiles:",
    len(batting_reliability_profiles)
)

print(
    "Profiles with 400 valid draws:",
    int(
        batting_reliability_profiles[
            "source_bootstrap_valid_draws"
        ].eq(N_BOOT).sum()
    )
)


# ============================================================
# 3. BOWLING BOOTSTRAP
# ============================================================

print("\n3. SOURCE-ADJUSTED BOWLING BOOTSTRAP")
print("=" * 70)

(
    bowling_source_bootstrap,
    bowling_boot_pressure_events,
    bowling_boot_reference_events
) = source_cluster_bootstrap(
    pressure_events=bowling_source_events,
    reference_events=bowling_source_references,
    matched_strata=bowling_matched_source,
    profile_table=bowling_adjusted_profiles,
    profile_keys=[
        "bowler_id",
        "bowling_pressure_type"
    ],
    event_columns=[
        "clean_dot_event",
        "boundary_event",
        "credited_wicket_event"
    ],
    metric_labels=[
        "clean_dot",
        "boundary",
        "wicket"
    ],
    directions=[1, -1, 1],
    seed=2065
)

bowling_reliability_profiles = (
    bowling_adjusted_profiles.merge(
        bowling_source_bootstrap,
        on=[
            "bowler_id",
            "bowling_pressure_type"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    "Bowling profiles:",
    len(bowling_reliability_profiles)
)

print(
    "Profiles with 400 valid draws:",
    int(
        bowling_reliability_profiles[
            "source_bootstrap_valid_draws"
        ].eq(N_BOOT).sum()
    )
)


# ============================================================
# 4. RELIABILITY SCREENING
# ============================================================

print("\n4. RELIABILITY SCREENING")
print("=" * 70)

for df in [
    batting_reliability_profiles,
    bowling_reliability_profiles
]:

    df["source_sample_reliable"] = (
        df["source_coverage_pct"].ge(95) &
        df["source_matched_balls"].ge(30) &
        df["source_pressure_matches"].ge(4) &
        df["source_reference_matches"].ge(4) &
        df["source_bootstrap_weak_strata"].eq(0) &
        df["source_bootstrap_valid_draws"].eq(N_BOOT)
    )

    df["source_reliability_reason"] = np.select(
        [
            df["source_coverage_pct"].lt(95),
            df["source_matched_balls"].lt(30),
            df["source_pressure_matches"].lt(4),
            df["source_reference_matches"].lt(4),
            df["source_bootstrap_weak_strata"].gt(0),
            df["source_bootstrap_valid_draws"].lt(N_BOOT)
        ],
        [
            "Incomplete source coverage",
            "Limited pressure exposure",
            "Too few pressure matches",
            "Too few reference matches",
            "Weak source-phase stratum",
            "Incomplete bootstrap draws"
        ],
        default="Meets reliability criteria"
    )

reliability_summary = pd.DataFrame([
    {
        "role": "Batting",
        "profiles": len(batting_reliability_profiles),
        "reliable": int(
            batting_reliability_profiles[
                "source_sample_reliable"
            ].sum()
        ),
        "requires_caution": int(
            (
                ~batting_reliability_profiles[
                    "source_sample_reliable"
                ]
            ).sum()
        )
    },
    {
        "role": "Bowling",
        "profiles": len(bowling_reliability_profiles),
        "reliable": int(
            bowling_reliability_profiles[
                "source_sample_reliable"
            ].sum()
        ),
        "requires_caution": int(
            (
                ~bowling_reliability_profiles[
                    "source_sample_reliable"
                ]
            ).sum()
        )
    }
])

display(reliability_summary)

print("\nBatting reliability reasons:")

display(
    batting_reliability_profiles[
        "source_reliability_reason"
    ].value_counts()
)

print("\nBowling reliability reasons:")

display(
    bowling_reliability_profiles[
        "source_reliability_reason"
    ].value_counts()
)


# ============================================================
# 5. SOURCE-ADJUSTED INTERVAL CLASSIFICATION
# ============================================================

print("\n5. CONFIDENCE INTERVAL ASSESSMENT")
print("=" * 70)


def classify_source_interval(df, metric):

    lower = df[f"{metric}_source_lower_95"]
    upper = df[f"{metric}_source_upper_95"]

    return np.select(
        [
            lower.gt(0),
            upper.lt(0),
            lower.notna() & upper.notna()
        ],
        [
            "Above Baseline",
            "Below Baseline",
            "Interval Crosses Zero"
        ],
        default="Unavailable"
    )


batting_reliability_profiles[
    "sr_source_assessment"
] = classify_source_interval(
    batting_reliability_profiles,
    "sr"
)

print("\nBatting interval results:")

display(
    pd.crosstab(
        batting_reliability_profiles[
            "source_sample_reliable"
        ],
        batting_reliability_profiles[
            "sr_source_assessment"
        ]
    )
)

for metric in [
    "clean_dot",
    "boundary",
    "wicket"
]:

    result_column = (
        f"{metric}_source_assessment"
    )

    bowling_reliability_profiles[
        result_column
    ] = classify_source_interval(
        bowling_reliability_profiles,
        metric
    )

    print("\nBowling interval results:", metric)

    display(
        pd.crosstab(
            bowling_reliability_profiles[
                "source_sample_reliable"
            ],
            bowling_reliability_profiles[
                result_column
            ]
        )
    )


# ============================================================
# 6. EXAMPLE RESULTS — NOT RANKINGS
# ============================================================

print("\n6. HIGH-EXPOSURE PERFORMANCE EXAMPLES")
print("=" * 70)

print("\nBatting examples:")

display(
    batting_reliability_profiles.sort_values(
        "source_matched_balls",
        ascending=False
    )[
        [
            "player_name",
            "batting_pressure_type",
            "source_matched_balls",
            "source_adjusted_sr_uplift",
            "sr_source_lower_95",
            "sr_source_upper_95",
            "source_sample_reliable",
            "source_reliability_reason"
        ]
    ].head(12)
)

print("\nBowling examples:")

display(
    bowling_reliability_profiles.sort_values(
        "source_matched_balls",
        ascending=False
    )[
        [
            "player_name",
            "bowling_pressure_type",
            "source_matched_balls",
            "source_adjusted_clean_dot_uplift_pp",
            "clean_dot_source_lower_95",
            "clean_dot_source_upper_95",
            "source_adjusted_boundary_suppression_pp",
            "boundary_source_lower_95",
            "boundary_source_upper_95",
            "source_sample_reliable"
        ]
    ].head(12)
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)


def point_mismatch_count(
    df,
    original_column,
    bootstrap_column
):

    return int(
        (
            ~np.isclose(
                df[original_column],
                df[bootstrap_column],
                rtol=1e-8,
                atol=1e-7,
                equal_nan=False
            )
        ).sum()
    )


checks = {
    "batting_profile_count_mismatch": abs(
        len(batting_reliability_profiles) - 351
    ),

    "bowling_profile_count_mismatch": abs(
        len(bowling_reliability_profiles) - 215
    ),

    "batting_point_estimate_mismatches": (
        point_mismatch_count(
            batting_reliability_profiles,
            "source_adjusted_sr_uplift",
            "sr_source_point"
        )
    ),

    "bowling_dot_point_mismatches": (
        point_mismatch_count(
            bowling_reliability_profiles,
            "source_adjusted_clean_dot_uplift_pp",
            "clean_dot_source_point"
        )
    ),

    "bowling_boundary_point_mismatches": (
        point_mismatch_count(
            bowling_reliability_profiles,
            "source_adjusted_boundary_suppression_pp",
            "boundary_source_point"
        )
    ),

    "bowling_wicket_point_mismatches": (
        point_mismatch_count(
            bowling_reliability_profiles,
            "source_adjusted_wicket_rate_uplift_pp",
            "wicket_source_point"
        )
    ),

    "invalid_batting_bootstrap_draws": int(
        batting_reliability_profiles[
            "source_bootstrap_valid_draws"
        ].ne(N_BOOT).sum()
    ),

    "invalid_bowling_bootstrap_draws": int(
        bowling_reliability_profiles[
            "source_bootstrap_valid_draws"
        ].ne(N_BOOT).sum()
    ),

    "duplicate_batting_profiles": int(
        batting_reliability_profiles.duplicated(
            ["striker_id", "batting_pressure_type"]
        ).sum()
    ),

    "duplicate_bowling_profiles": int(
        bowling_reliability_profiles.duplicated(
            ["bowler_id", "bowling_pressure_type"]
        ).sum()
    ),

    "missing_batting_reliability_flags": int(
        batting_reliability_profiles[
            "source_sample_reliable"
        ].isna().sum()
    ),

    "missing_bowling_reliability_flags": int(
        bowling_reliability_profiles[
            "source_sample_reliable"
        ].isna().sum()
    )
}

integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(integrity_report)

if integrity_report["count"].eq(0).all():

    print("\nALL 12 INTEGRITY CHECKS PASSED")

else:

    print("\nWARNING: NON-ZERO INTEGRITY CHECKS FOUND")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "Source-adjusted confidence intervals use "
    "whole-match resampling within source-phase strata."
)

print(
    "Pressure and reference matches are resampled "
    "separately, so shared-match covariance is not "
    "fully preserved."
)

print(
    "These intervals are exploratory and are not "
    "adjusted for testing multiple players."
)

print(
    "Reliability flags describe statistical evidence, "
    "not final player eligibility or selection."
)

In [ ]:
# CELL 65 — PLAYER-LEVEL SCOUTING EVIDENCE CONSOLIDATION

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("PLAYER-LEVEL SCOUTING EVIDENCE CONSOLIDATION")
print("=" * 70)


# ============================================================
# 1. VERIFY SOURCE TABLES
# ============================================================

print("\n1. INPUT VALIDATION")
print("=" * 70)

bat_required = [
    "striker_id", "batting_pressure_type", "pressure_balls",
    "source_matched_balls", "source_sample_reliable",
    "source_adjusted_sr_uplift", "sr_source_lower_95",
    "sr_source_upper_95"
]

bowl_required = [
    "bowler_id", "bowling_pressure_type", "pressure_legal_balls",
    "source_matched_balls", "source_sample_reliable",
    "clean_dot_source_lower_95", "boundary_source_lower_95",
    "wicket_source_lower_95"
]

registry_required = [
    "player_id", "player_name", "observed_matches",
    "observed_t20i_matches", "observed_ipl_matches",
    "qualified_batting_profile", "qualified_bowling_profile",
    "eligibility_status"
]

for label, df, required in [
    ("Batting", batting_reliability_profiles, bat_required),
    ("Bowling", bowling_reliability_profiles, bowl_required),
    ("Registry", player_registry, registry_required)
]:

    missing = sorted(set(required) - set(df.columns))

    assert not missing, (
        f"{label}: missing columns {missing}"
    )

    print(
        f"{label}: {len(df):,} records; required fields available"
    )

assert not batting_reliability_profiles.duplicated(
    ["striker_id", "batting_pressure_type"]
).any()

assert not bowling_reliability_profiles.duplicated(
    ["bowler_id", "bowling_pressure_type"]
).any()

assert not player_registry["player_id"].duplicated().any()


# ============================================================
# 2. AGGREGATE BATTING PROFILES
# ============================================================

print("\n2. BATTING EVIDENCE BY PLAYER")
print("=" * 70)

b = batting_reliability_profiles.copy()

b["reliable_positive_sr"] = (
    b["source_sample_reliable"] &
    b["sr_source_lower_95"].gt(0)
).astype(int)

b["reliable_negative_sr"] = (
    b["source_sample_reliable"] &
    b["sr_source_upper_95"].lt(0)
).astype(int)

batting_player_evidence = (
    b.groupby("striker_id", as_index=False)
    .agg(
        batting_profiles=("batting_pressure_type", "size"),
        batting_pressure_balls=("pressure_balls", "sum"),
        batting_source_matched_balls=(
            "source_matched_balls", "sum"
        ),
        reliable_batting_profiles=(
            "source_sample_reliable", "sum"
        ),
        reliable_positive_sr_profiles=(
            "reliable_positive_sr", "sum"
        ),
        reliable_negative_sr_profiles=(
            "reliable_negative_sr", "sum"
        )
    )
    .rename(columns={"striker_id": "player_id"})
)

print(
    "Unique qualified batters:",
    len(batting_player_evidence)
)

print(
    "Total qualified batting profiles:",
    int(batting_player_evidence["batting_profiles"].sum())
)

print(
    "Reliable batting profiles:",
    int(
        batting_player_evidence[
            "reliable_batting_profiles"
        ].sum()
    )
)

print(
    "Reliable batting profiles above baseline:",
    int(
        batting_player_evidence[
            "reliable_positive_sr_profiles"
        ].sum()
    )
)


# ============================================================
# 3. AGGREGATE BOWLING PROFILES
# ============================================================

print("\n3. BOWLING EVIDENCE BY PLAYER")
print("=" * 70)

w = bowling_reliability_profiles.copy()

for metric in ["clean_dot", "boundary", "wicket"]:

    w[f"reliable_positive_{metric}"] = (
        w["source_sample_reliable"] &
        w[f"{metric}_source_lower_95"].gt(0)
    ).astype(int)

bowling_player_evidence = (
    w.groupby("bowler_id", as_index=False)
    .agg(
        bowling_profiles=("bowling_pressure_type", "size"),
        bowling_pressure_balls=(
            "pressure_legal_balls", "sum"
        ),
        bowling_source_matched_balls=(
            "source_matched_balls", "sum"
        ),
        reliable_bowling_profiles=(
            "source_sample_reliable", "sum"
        ),
        reliable_positive_dot_profiles=(
            "reliable_positive_clean_dot", "sum"
        ),
        reliable_positive_boundary_profiles=(
            "reliable_positive_boundary", "sum"
        ),
        reliable_positive_wicket_profiles=(
            "reliable_positive_wicket", "sum"
        )
    )
    .rename(columns={"bowler_id": "player_id"})
)

print(
    "Unique qualified bowlers:",
    len(bowling_player_evidence)
)

print(
    "Total qualified bowling profiles:",
    int(bowling_player_evidence["bowling_profiles"].sum())
)

print(
    "Reliable bowling profiles:",
    int(
        bowling_player_evidence[
            "reliable_bowling_profiles"
        ].sum()
    )
)


# ============================================================
# 4. CONSOLIDATE WITHOUT FILTERING PLAYERS
# ============================================================

print("\n4. ALL-PLAYER EVIDENCE MATRIX")
print("=" * 70)

scouting_evidence_matrix = (
    player_registry.copy()
    .merge(
        batting_player_evidence,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        bowling_player_evidence,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "batting_profiles",
    "batting_pressure_balls",
    "batting_source_matched_balls",
    "reliable_batting_profiles",
    "reliable_positive_sr_profiles",
    "reliable_negative_sr_profiles",
    "bowling_profiles",
    "bowling_pressure_balls",
    "bowling_source_matched_balls",
    "reliable_bowling_profiles",
    "reliable_positive_dot_profiles",
    "reliable_positive_boundary_profiles",
    "reliable_positive_wicket_profiles"
]

scouting_evidence_matrix[count_columns] = (
    scouting_evidence_matrix[count_columns]
    .fillna(0)
    .astype("int64")
)

scouting_evidence_matrix["any_model_profile"] = (
    scouting_evidence_matrix["batting_profiles"].gt(0) |
    scouting_evidence_matrix["bowling_profiles"].gt(0)
)

scouting_evidence_matrix["any_reliable_profile"] = (
    scouting_evidence_matrix[
        "reliable_batting_profiles"
    ].gt(0) |
    scouting_evidence_matrix[
        "reliable_bowling_profiles"
    ].gt(0)
)

scouting_evidence_matrix["t20i_sample_evidence"] = (
    np.where(
        scouting_evidence_matrix[
            "observed_t20i_matches"
        ].gt(0),
        "Recorded T20I participation",
        "No T20I recorded in supplied data"
    )
)

scouting_evidence_matrix["statistical_evidence_level"] = (
    np.select(
        [
            scouting_evidence_matrix[
                "any_reliable_profile"
            ],
            scouting_evidence_matrix[
                "any_model_profile"
            ]
        ],
        [
            "At least one reliable model profile",
            "Model profiles; additional caution"
        ],
        default="No qualified model profile"
    )
)

print(
    "Registry players retained:",
    len(scouting_evidence_matrix)
)

print(
    "Players with qualified model evidence:",
    int(
        scouting_evidence_matrix[
            "any_model_profile"
        ].sum()
    )
)

print(
    "Players with reliable model evidence:",
    int(
        scouting_evidence_matrix[
            "any_reliable_profile"
        ].sum()
    )
)

print(
    "Players without qualified model profiles:",
    int(
        (
            ~scouting_evidence_matrix[
                "any_model_profile"
            ]
        ).sum()
    )
)


# ============================================================
# 5. EVIDENCE COVERAGE AND VERIFICATION GROUPS
# ============================================================

print("\n5. EVIDENCE COVERAGE AND VERIFICATION GROUPS")
print("=" * 70)

display(
    scouting_evidence_matrix.groupby(
        [
            "statistical_evidence_level",
            "t20i_sample_evidence"
        ],
        dropna=False
    )
    .agg(
        players=("player_id", "nunique"),
        batting_profiles=("batting_profiles", "sum"),
        bowling_profiles=("bowling_profiles", "sum")
    )
    .reset_index()
)

no_t20i_qualified = scouting_evidence_matrix.loc[
    scouting_evidence_matrix[
        "observed_t20i_matches"
    ].eq(0) &
    scouting_evidence_matrix[
        "any_model_profile"
    ]
]

print(
    "\nQualified players without T20I evidence "
    "in supplied data:",
    len(no_t20i_qualified)
)

print(
    "IMPORTANT: This is NOT a confirmed "
    "uncapped or age-eligible player count."
)


# ============================================================
# 6. REVIEW EXAMPLES — NOT RANKINGS
# ============================================================

print("\n6. PLAYER REVIEW EXAMPLES — NOT RANKINGS")
print("=" * 70)

batting_examples = (
    b.loc[
        b["reliable_positive_sr"].eq(1)
    ]
    .merge(
        player_registry[
            ["player_id", "observed_t20i_matches"]
        ],
        left_on="striker_id",
        right_on="player_id",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        "source_matched_balls",
        ascending=False
    )
)

print(
    "\nReliable positive batting profiles "
    "(sorted by exposure, not performance):"
)

display(
    batting_examples[
        [
            "player_name",
            "batting_pressure_type",
            "source_matched_balls",
            "source_adjusted_sr_uplift",
            "sr_source_lower_95",
            "sr_source_upper_95",
            "observed_t20i_matches"
        ]
    ].head(15)
)

bowling_examples = (
    w.loc[
        w["reliable_positive_clean_dot"].eq(1) |
        w["reliable_positive_boundary"].eq(1) |
        w["reliable_positive_wicket"].eq(1)
    ]
    .sort_values(
        "source_matched_balls",
        ascending=False
    )
)

print(
    "\nReliable positive bowling-metric profiles "
    "(sorted by exposure, not performance):"
)

display(
    bowling_examples[
        [
            "player_name",
            "bowling_pressure_type",
            "source_matched_balls",
            "reliable_positive_clean_dot",
            "reliable_positive_boundary",
            "reliable_positive_wicket"
        ]
    ].head(15)
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. PLAYER EVIDENCE INTEGRITY CHECKS")
print("=" * 70)

actual_batting_ids = set(
    b["striker_id"].astype("int64")
)

actual_bowling_ids = set(
    w["bowler_id"].astype("int64")
)

registry_ids = set(
    scouting_evidence_matrix[
        "player_id"
    ].astype("int64")
)

checks = {
    "duplicate_registry_ids": int(
        scouting_evidence_matrix[
            "player_id"
        ].duplicated().sum()
    ),

    "missing_player_names": int(
        scouting_evidence_matrix[
            "player_name"
        ].isna().sum()
    ),

    "registry_player_count_difference": abs(
        len(scouting_evidence_matrix) -
        len(player_registry)
    ),

    "batting_profile_count_difference": abs(
        int(
            scouting_evidence_matrix[
                "batting_profiles"
            ].sum()
        ) - len(b)
    ),

    "bowling_profile_count_difference": abs(
        int(
            scouting_evidence_matrix[
                "bowling_profiles"
            ].sum()
        ) - len(w)
    ),

    "reliable_batting_count_difference": abs(
        int(
            scouting_evidence_matrix[
                "reliable_batting_profiles"
            ].sum()
        ) -
        int(b["source_sample_reliable"].sum())
    ),

    "reliable_bowling_count_difference": abs(
        int(
            scouting_evidence_matrix[
                "reliable_bowling_profiles"
            ].sum()
        ) -
        int(w["source_sample_reliable"].sum())
    ),

    "qualified_players_missing_from_registry": len(
        (
            actual_batting_ids |
            actual_bowling_ids
        ) - registry_ids
    ),

    "batting_flag_disagreements": int(
        (
            scouting_evidence_matrix[
                "batting_profiles"
            ].gt(0) !=
            scouting_evidence_matrix[
                "qualified_batting_profile"
            ]
        ).sum()
    ),

    "bowling_flag_disagreements": int(
        (
            scouting_evidence_matrix[
                "bowling_profiles"
            ].gt(0) !=
            scouting_evidence_matrix[
                "qualified_bowling_profile"
            ]
        ).sum()
    ),

    "missing_eligibility_status": int(
        scouting_evidence_matrix[
            "eligibility_status"
        ].isna().sum()
    )
}

evidence_integrity = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(evidence_integrity)

if evidence_integrity["count"].eq(0).all():
    print("\nALL 11 INTEGRITY CHECKS PASSED")
else:
    print("\nREVIEW NONZERO CHECKS BEFORE NEXT CELL")

print("\nINTERPRETATION")
print("=" * 70)

print(
    "No player has been declared uncapped, "
    "age-eligible or selected."
)

print(
    "Intervals excluding zero remain exploratory "
    "and are not adjusted for multiple testing."
)

print(
    "All players, including those without qualified "
    "model profiles, remain available for scouting."
)

In [ ]:
# ============================================================
# CELL 66 — SCOUTING REVIEW AND ELIGIBILITY VERIFICATION QUEUE
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("SCOUTING REVIEW AND ELIGIBILITY VERIFICATION QUEUE")
print("=" * 70)


# ============================================================
# 1. VERIFY INPUTS
# ============================================================

print("\n1. INPUT VALIDATION")
print("=" * 70)

required_variables = [
    "scouting_evidence_matrix",
    "batting_reliability_profiles",
    "bowling_reliability_profiles",
    "verification_queue"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

assert len(scouting_evidence_matrix) == 969

assert (
    scouting_evidence_matrix["player_id"]
    .is_unique
)

assert len(batting_reliability_profiles) == 351
assert len(bowling_reliability_profiles) == 215

print("Registered players:", len(scouting_evidence_matrix))
print("Batting profiles:", len(batting_reliability_profiles))
print("Bowling profiles:", len(bowling_reliability_profiles))
print("Previous variables successfully verified.")


# ============================================================
# 2. CREATE RELIABLE POSITIVE PERFORMANCE SIGNALS
# ============================================================

print("\n2. RELIABLE POSITIVE PERFORMANCE SIGNALS")
print("=" * 70)

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()

positive_signals = []

batting_positive = bat.loc[
    bat["source_sample_reliable"].eq(True) &
    bat["sr_source_lower_95"].gt(0)
].copy()

for row in batting_positive.itertuples(index=False):

    positive_signals.append({
        "player_id": row.striker_id,
        "role": "Batting",
        "pressure_context": row.batting_pressure_type,
        "metric": "Strike-rate uplift",
        "effect_per_100_balls": row.source_adjusted_sr_uplift,
        "lower_95": row.sr_source_lower_95,
        "upper_95": row.sr_source_upper_95,
        "matched_pressure_balls": row.source_matched_balls
    })

bowling_metric_configuration = {
    "clean_dot": (
        "Clean-dot uplift",
        "source_adjusted_clean_dot_uplift_pp"
    ),
    "boundary": (
        "Boundary suppression",
        "source_adjusted_boundary_suppression_pp"
    ),
    "wicket": (
        "Wicket-rate uplift",
        "source_adjusted_wicket_rate_uplift_pp"
    )
}

for metric, (metric_label, effect_column) in (
    bowling_metric_configuration.items()
):

    positive_bowling = bowl.loc[
        bowl["source_sample_reliable"].eq(True) &
        bowl[f"{metric}_source_lower_95"].gt(0)
    ].copy()

    for _, row in positive_bowling.iterrows():

        positive_signals.append({
            "player_id": row["bowler_id"],
            "role": "Bowling",
            "pressure_context": row[
                "bowling_pressure_type"
            ],
            "metric": metric_label,
            "effect_per_100_balls": row[effect_column],
            "lower_95": row[
                f"{metric}_source_lower_95"
            ],
            "upper_95": row[
                f"{metric}_source_upper_95"
            ],
            "matched_pressure_balls": row[
                "source_matched_balls"
            ]
        })

positive_signal_columns = [
    "player_id",
    "role",
    "pressure_context",
    "metric",
    "effect_per_100_balls",
    "lower_95",
    "upper_95",
    "matched_pressure_balls"
]

positive_signal_details = pd.DataFrame(
    positive_signals,
    columns=positive_signal_columns
)

print(
    "Positive batting profile signals:",
    len(batting_positive)
)

print(
    "Positive bowling metric signals:",
    len(positive_signal_details) - len(batting_positive)
)

print(
    "Unique players with positive reliable signals:",
    positive_signal_details["player_id"].nunique()
)


# ============================================================
# 3. BUILD THE COMPLETE PLAYER REVIEW QUEUE
# ============================================================

print("\n3. COMPLETE PLAYER REVIEW QUEUE")
print("=" * 70)

scouting_review_queue = scouting_evidence_matrix.copy()

positive_counts = (
    positive_signal_details.groupby("player_id")
    .size()
    .rename("reliable_positive_signal_count")
    .reset_index()
)

scouting_review_queue = scouting_review_queue.merge(
    positive_counts,
    on="player_id",
    how="left",
    validate="one_to_one"
)

scouting_review_queue[
    "reliable_positive_signal_count"
] = (
    scouting_review_queue[
        "reliable_positive_signal_count"
    ]
    .fillna(0)
    .astype(int)
)

scouting_review_queue["has_reliable_positive_signal"] = (
    scouting_review_queue[
        "reliable_positive_signal_count"
    ].gt(0)
)

scouting_review_queue["review_track"] = np.select(
    [
        scouting_review_queue[
            "observed_t20i_matches"
        ].gt(0),

        scouting_review_queue[
            "has_reliable_positive_signal"
        ],

        scouting_review_queue[
            "any_reliable_profile"
        ],

        scouting_review_queue[
            "any_model_profile"
        ]
    ],
    [
        "T20I recorded — verify eligibility",
        "No recorded T20I — positive reliable signal",
        "No recorded T20I — reliable evidence",
        "No recorded T20I — exploratory evidence"
    ],
    default="No recorded T20I — limited model evidence"
)

# These are research fields, NOT verified facts.
# Do not fill them using guesses or name similarity.

scouting_review_queue["verified_date_of_birth"] = pd.NaT

scouting_review_queue["date_of_birth_source"] = pd.NA

scouting_review_queue["verified_cap_status"] = (
    "Not verified"
)

scouting_review_queue["cap_status_source"] = pd.NA

scouting_review_queue["age_eligibility_status"] = (
    "Not verified"
)

scouting_review_queue["final_selection_status"] = (
    "Not evaluated"
)

print(
    "Players retained:",
    len(scouting_review_queue)
)

print(
    "Players with positive reliable signals:",
    int(
        scouting_review_queue[
            "has_reliable_positive_signal"
        ].sum()
    )
)


# ============================================================
# 4. IDENTITY COLLISION AUDIT
# ============================================================

print("\n4. PLAYER IDENTITY COLLISION AUDIT")
print("=" * 70)

scouting_review_queue["normalized_player_name"] = (
    scouting_review_queue["player_name"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

name_id_counts = (
    scouting_review_queue.groupby(
        "normalized_player_name"
    )["player_id"]
    .nunique()
    .rename("ids_sharing_name")
    .reset_index()
)

scouting_review_queue = scouting_review_queue.merge(
    name_id_counts,
    on="normalized_player_name",
    how="left",
    validate="many_to_one"
)

scouting_review_queue["identity_collision_flag"] = (
    scouting_review_queue[
        "ids_sharing_name"
    ].gt(1)
)

print(
    "Names shared by multiple player IDs:",
    int(
        name_id_counts[
            "ids_sharing_name"
        ].gt(1).sum()
    )
)

print(
    "Player records requiring name-collision review:",
    int(
        scouting_review_queue[
            "identity_collision_flag"
        ].sum()
    )
)

display(
    scouting_review_queue.loc[
        scouting_review_queue[
            "identity_collision_flag"
        ],
        [
            "player_id",
            "player_name",
            "ids_sharing_name",
            "observed_matches",
            "observed_t20i_matches",
            "observed_ipl_matches"
        ]
    ].sort_values(
        ["player_name", "player_id"]
    )
)

print(
    "\nPlayer IDs remain separate even when "
    "the displayed names are identical."
)


# ============================================================
# 5. EVIDENCE REVIEW TRACKS
# ============================================================

print("\n5. SCOUTING REVIEW GROUPS")
print("=" * 70)

review_summary = (
    scouting_review_queue.groupby("review_track")
    .agg(
        players=("player_id", "nunique"),
        with_reliable_profiles=(
            "any_reliable_profile", "sum"
        ),
        with_positive_signals=(
            "has_reliable_positive_signal", "sum"
        ),
        identity_collision_records=(
            "identity_collision_flag", "sum"
        )
    )
    .reset_index()
)

display(review_summary)

print(
    "\nTotal no-recorded-T20I players:",
    int(
        scouting_review_queue[
            "observed_t20i_matches"
        ].eq(0).sum()
    )
)

print(
    "Qualified no-recorded-T20I players:",
    int(
        (
            scouting_review_queue[
                "observed_t20i_matches"
            ].eq(0) &
            scouting_review_queue[
                "any_model_profile"
            ]
        ).sum()
    )
)


# ============================================================
# 6. REVIEW THE ACTUAL POSITIVE EVIDENCE
# ============================================================

print("\n6. PLAYER SIGNALS REQUIRING REVIEW")
print("=" * 70)

signal_details_with_identity = (
    positive_signal_details.merge(
        scouting_review_queue[
            [
                "player_id",
                "player_name",
                "observed_t20i_matches",
                "observed_ipl_matches",
                "identity_collision_flag"
            ]
        ],
        on="player_id",
        how="left",
        validate="many_to_one"
    )
)

print("\nAll reliability-screened positive signals")

display(
    signal_details_with_identity[
        [
            "player_id",
            "player_name",
            "role",
            "pressure_context",
            "metric",
            "matched_pressure_balls",
            "effect_per_100_balls",
            "lower_95",
            "upper_95",
            "observed_t20i_matches"
        ]
    ].sort_values(
        ["role", "matched_pressure_balls"],
        ascending=[True, False]
    )
)

print(
    "\nQualified profiles without recorded T20I "
    "participation — NOT CONFIRMED UNCAPPED"
)

no_recorded_t20i_review = (
    scouting_review_queue.loc[
        scouting_review_queue[
            "observed_t20i_matches"
        ].eq(0) &
        scouting_review_queue[
            "any_model_profile"
        ]
    ]
    .sort_values("player_id")
    .copy()
)

display(
    no_recorded_t20i_review[
        [
            "player_id",
            "player_name",
            "model_role",
            "observed_ipl_matches",
            "batting_pressure_balls",
            "bowling_pressure_balls",
            "reliable_batting_profiles",
            "reliable_bowling_profiles",
            "reliable_positive_signal_count",
            "identity_collision_flag",
            "verified_cap_status"
        ]
    ].head(30)
)


# ============================================================
# 7. INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

registry_ids = set(
    scouting_evidence_matrix["player_id"]
)

queue_ids = set(
    scouting_review_queue["player_id"]
)

original_positive_total = (
    scouting_evidence_matrix[
        [
            "reliable_positive_sr_profiles",
            "reliable_positive_dot_profiles",
            "reliable_positive_boundary_profiles",
            "reliable_positive_wicket_profiles"
        ]
    ].sum().sum()
)

checks = {
    "registry_player_count_difference": abs(
        len(scouting_review_queue) -
        len(scouting_evidence_matrix)
    ),

    "missing_registry_players": len(
        registry_ids - queue_ids
    ),

    "unexpected_queue_players": len(
        queue_ids - registry_ids
    ),

    "duplicate_queue_player_ids": int(
        scouting_review_queue[
            "player_id"
        ].duplicated().sum()
    ),

    "missing_player_names": int(
        scouting_review_queue[
            "player_name"
        ].isna().sum()
    ),

    "positive_signal_count_mismatch": abs(
        int(original_positive_total) -
        int(
            scouting_review_queue[
                "reliable_positive_signal_count"
            ].sum()
        )
    ),

    "qualified_no_t20i_queue_mismatch": abs(
        len(no_recorded_t20i_review) -
        len(verification_queue)
    ),

    "missing_review_tracks": int(
        scouting_review_queue[
            "review_track"
        ].isna().sum()
    ),

    "unverified_cap_status_mismatch": int(
        scouting_review_queue[
            "verified_cap_status"
        ].ne("Not verified").sum()
    ),

    "missing_identity_flags": int(
        scouting_review_queue[
            "identity_collision_flag"
        ].isna().sum()
    ),

    "positive_signals_without_registry_id": int(
        (
            ~positive_signal_details[
                "player_id"
            ].isin(queue_ids)
        ).sum()
    )
}

review_integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(review_integrity_report)

if review_integrity_report["count"].eq(0).all():
    print("\nALL 11 INTEGRITY CHECKS PASSED")
else:
    print("\nWARNING — REVIEW NONZERO CHECKS")

print("\nIMPORTANT INTERPRETATION")
print("=" * 70)

print(
    "No T20I recorded is NOT equivalent "
    "to internationally uncapped."
)

print(
    "A positive exploratory interval is NOT "
    "proof of future performance."
)

print(
    "Every player is retained for scouting."
)

print(
    "Birth dates, international cap status and "
    "final eligibility remain unverified."
)

print(
    "This is an evidence review queue, "
    "NOT the final Top 10 ranking."
)

In [ ]:
# ============================================================
# CELL 67 — PLAYER-LEVEL PERFORMANCE AND SCOUTING RESEARCH TIERS
# COMPLETE CORRECTED VERSION
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("PLAYER-LEVEL PERFORMANCE AND SCOUTING RESEARCH TIERS")
print("=" * 70)


# ============================================================
# 1. VERIFY REQUIRED INPUTS
# ============================================================

print("\n1. INPUT VERIFICATION")
print("=" * 70)

required_variables = [
    "scouting_review_queue",
    "batting_reliability_profiles",
    "bowling_reliability_profiles",
    "batting_matched_source",
    "bowling_matched_source",
    "positive_signal_details"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing previous-cell variables: {missing_variables}"
)

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()

required_columns = {
    "Batting profiles": (
        bat,
        [
            "striker_id",
            "batting_pressure_type",
            "source_matched_balls",
            "source_sample_reliable",
            "sr_source_lower_95"
        ]
    ),
    "Bowling profiles": (
        bowl,
        [
            "bowler_id",
            "bowling_pressure_type",
            "source_matched_balls",
            "source_sample_reliable",
            "clean_dot_source_lower_95",
            "boundary_source_lower_95",
            "wicket_source_lower_95"
        ]
    ),
    "Batting source strata": (
        batting_matched_source,
        [
            "striker_id",
            "batting_pressure_type",
            "pressure_balls",
            "pressure_runs",
            "source_expected_runs"
        ]
    ),
    "Bowling source strata": (
        bowling_matched_source,
        [
            "bowler_id",
            "bowling_pressure_type",
            "pressure_balls",
            "pressure_clean_dots",
            "pressure_boundaries",
            "pressure_wickets",
            "source_expected_clean_dots",
            "source_expected_boundaries",
            "source_expected_wickets"
        ]
    )
}

for label, (df, columns) in required_columns.items():

    missing = sorted(set(columns) - set(df.columns))

    assert not missing, (
        f"{label}: missing columns {missing}"
    )

    print(f"{label}: {len(df):,} rows — VERIFIED")

assert scouting_review_queue["player_id"].is_unique

assert not bat.duplicated(
    ["striker_id", "batting_pressure_type"]
).any()

assert not bowl.duplicated(
    ["bowler_id", "bowling_pressure_type"]
).any()

print(
    "Registered players:",
    len(scouting_review_queue)
)

print("Required source-adjusted data: AVAILABLE")


# ============================================================
# 2. CONSOLIDATE SOURCE-ADJUSTED BATTING PERFORMANCE
# ============================================================

print("\n2. PLAYER-LEVEL BATTING PERFORMANCE")
print("=" * 70)

bat["positive_reliable_context"] = (
    bat["source_sample_reliable"].eq(True) &
    bat["sr_source_lower_95"].gt(0)
)

# Use original verified source-matched strata.
# Do not depend on merge-suffixed profile columns.

batting_source_totals = (
    batting_matched_source.groupby(
        "striker_id",
        as_index=False
    )
    .agg(
        batting_matched_balls=("pressure_balls", "sum"),
        batting_actual_runs=("pressure_runs", "sum"),
        batting_expected_runs=(
            "source_expected_runs", "sum"
        )
    )
)

batting_context_counts = (
    bat.groupby("striker_id", as_index=False)
    .agg(
        batting_contexts=(
            "batting_pressure_type", "nunique"
        ),
        batting_reliable_contexts=(
            "source_sample_reliable", "sum"
        ),
        batting_positive_contexts=(
            "positive_reliable_context", "sum"
        )
    )
)

batting_player_performance = (
    batting_source_totals.merge(
        batting_context_counts,
        on="striker_id",
        how="outer",
        validate="one_to_one"
    )
    .rename(columns={"striker_id": "player_id"})
)

batting_player_performance[
    "overall_source_adjusted_sr_uplift"
] = (
    100 *
    (
        batting_player_performance["batting_actual_runs"] -
        batting_player_performance["batting_expected_runs"]
    ).div(
        batting_player_performance[
            "batting_matched_balls"
        ].replace(0, np.nan)
    )
)

print(
    "Qualified batters:",
    len(batting_player_performance)
)

print(
    "Source-matched batting pressure balls:",
    int(
        batting_player_performance[
            "batting_matched_balls"
        ].sum()
    )
)

display(
    batting_player_performance[
        [
            "batting_matched_balls",
            "batting_contexts",
            "batting_reliable_contexts",
            "overall_source_adjusted_sr_uplift"
        ]
    ].describe().T
)


# ============================================================
# 3. CONSOLIDATE SOURCE-ADJUSTED BOWLING PERFORMANCE
# ============================================================

print("\n3. PLAYER-LEVEL BOWLING PERFORMANCE")
print("=" * 70)

for metric in [
    "clean_dot",
    "boundary",
    "wicket"
]:

    bowl[f"positive_reliable_{metric}"] = (
        bowl["source_sample_reliable"].eq(True) &
        bowl[f"{metric}_source_lower_95"].gt(0)
    )

bowl["positive_reliable_metric_count"] = (
    bowl[
        [
            "positive_reliable_clean_dot",
            "positive_reliable_boundary",
            "positive_reliable_wicket"
        ]
    ].sum(axis=1)
)

# The source-matched strata have unambiguous
# pressure and expected-outcome columns.

bowling_source_totals = (
    bowling_matched_source.groupby(
        "bowler_id",
        as_index=False
    )
    .agg(
        bowling_matched_balls=("pressure_balls", "sum"),

        bowling_actual_dots=(
            "pressure_clean_dots", "sum"
        ),
        bowling_expected_dots=(
            "source_expected_clean_dots", "sum"
        ),

        bowling_actual_boundaries=(
            "pressure_boundaries", "sum"
        ),
        bowling_expected_boundaries=(
            "source_expected_boundaries", "sum"
        ),

        bowling_actual_wickets=(
            "pressure_wickets", "sum"
        ),
        bowling_expected_wickets=(
            "source_expected_wickets", "sum"
        )
    )
)

bowling_context_counts = (
    bowl.groupby("bowler_id", as_index=False)
    .agg(
        bowling_contexts=(
            "bowling_pressure_type", "nunique"
        ),
        bowling_reliable_contexts=(
            "source_sample_reliable", "sum"
        ),
        bowling_positive_metrics=(
            "positive_reliable_metric_count", "sum"
        )
    )
)

bowling_player_performance = (
    bowling_source_totals.merge(
        bowling_context_counts,
        on="bowler_id",
        how="outer",
        validate="one_to_one"
    )
    .rename(columns={"bowler_id": "player_id"})
)

bowling_denominator = (
    bowling_player_performance[
        "bowling_matched_balls"
    ].replace(0, np.nan)
)

bowling_player_performance[
    "overall_clean_dot_uplift_pp"
] = (
    100 *
    (
        bowling_player_performance["bowling_actual_dots"] -
        bowling_player_performance["bowling_expected_dots"]
    ).div(bowling_denominator)
)

bowling_player_performance[
    "overall_boundary_suppression_pp"
] = (
    100 *
    (
        bowling_player_performance[
            "bowling_expected_boundaries"
        ] -
        bowling_player_performance[
            "bowling_actual_boundaries"
        ]
    ).div(bowling_denominator)
)

bowling_player_performance[
    "overall_wicket_uplift_pp"
] = (
    100 *
    (
        bowling_player_performance["bowling_actual_wickets"] -
        bowling_player_performance["bowling_expected_wickets"]
    ).div(bowling_denominator)
)

print(
    "Qualified bowlers:",
    len(bowling_player_performance)
)

print(
    "Source-matched bowling pressure balls:",
    int(
        bowling_player_performance[
            "bowling_matched_balls"
        ].sum()
    )
)

display(
    bowling_player_performance[
        [
            "bowling_matched_balls",
            "bowling_contexts",
            "bowling_reliable_contexts",
            "overall_clean_dot_uplift_pp",
            "overall_boundary_suppression_pp",
            "overall_wicket_uplift_pp"
        ]
    ].describe().T
)


# ============================================================
# 4. BUILD COMPLETE PLAYER PERFORMANCE MATRIX
# ============================================================

print("\n4. ALL-PLAYER PERFORMANCE MATRIX")
print("=" * 70)

player_performance_matrix = (
    scouting_review_queue.copy()
    .merge(
        batting_player_performance,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        bowling_player_performance,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "batting_contexts",
    "batting_matched_balls",
    "batting_reliable_contexts",
    "batting_positive_contexts",
    "bowling_contexts",
    "bowling_matched_balls",
    "bowling_reliable_contexts",
    "bowling_positive_metrics"
]

for column in count_columns:

    player_performance_matrix[column] = (
        player_performance_matrix[column]
        .fillna(0)
        .astype("int64")
    )

# Missing performance estimates stay NaN.
# They must not be interpreted as zero performance.

print(
    "Players retained:",
    len(player_performance_matrix)
)

print(
    "Players with batting model estimates:",
    int(
        player_performance_matrix[
            "overall_source_adjusted_sr_uplift"
        ].notna().sum()
    )
)

print(
    "Players with bowling model estimates:",
    int(
        player_performance_matrix[
            "overall_clean_dot_uplift_pp"
        ].notna().sum()
    )
)

print(
    "Players without qualified model profiles:",
    int(
        (
            ~player_performance_matrix["any_model_profile"]
        ).sum()
    )
)


# ============================================================
# 5. CREATE SEPARATE BATTING AND BOWLING EVIDENCE TIERS
# ============================================================

print("\n5. BATTING AND BOWLING EVIDENCE TIERS")
print("=" * 70)


def create_evidence_tier(
    df,
    profile_column,
    reliable_column,
    positive_column,
    exposure_column
):

    conditions = [
        df[positive_column].gt(0),
        df[reliable_column].gt(0),
        df[profile_column].gt(0),
        df[exposure_column].gt(0)
    ]

    labels = [
        "A | Positive reliability-screened signal",
        "B | Reliable profile without positive signal",
        "C | Qualified profile with sample caution",
        "D | Recorded exposure, no qualified profile"
    ]

    return np.select(
        conditions,
        labels,
        default="E | No recorded role-delivery exposure"
    )


player_performance_matrix["batting_evidence_tier"] = (
    create_evidence_tier(
        player_performance_matrix,
        "batting_contexts",
        "batting_reliable_contexts",
        "batting_positive_contexts",
        "provisional_balls_faced"
    )
)

player_performance_matrix["bowling_evidence_tier"] = (
    create_evidence_tier(
        player_performance_matrix,
        "bowling_contexts",
        "bowling_reliable_contexts",
        "bowling_positive_metrics",
        "legal_balls_bowled"
    )
)

print("\nBatting evidence tiers:")

display(
    player_performance_matrix[
        "batting_evidence_tier"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("tier")
    .reset_index(name="players")
)

print("\nBowling evidence tiers:")

display(
    player_performance_matrix[
        "bowling_evidence_tier"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("tier")
    .reset_index(name="players")
)


# ============================================================
# 6. BUILD SCOUTING RESEARCH POOLS
# ============================================================

print("\n6. SCOUTING RESEARCH POOLS")
print("=" * 70)

player_performance_matrix[
    "t20i_evidence_group"
] = np.where(
    player_performance_matrix[
        "observed_t20i_matches"
    ].gt(0),
    "T20I participation recorded",
    "No T20I in supplied sample"
)

print("\nBatting evidence coverage:")

display(
    pd.crosstab(
        player_performance_matrix[
            "batting_evidence_tier"
        ],
        player_performance_matrix[
            "t20i_evidence_group"
        ]
    )
)

print("\nBowling evidence coverage:")

display(
    pd.crosstab(
        player_performance_matrix[
            "bowling_evidence_tier"
        ],
        player_performance_matrix[
            "t20i_evidence_group"
        ]
    )
)

no_t20i_research_pool = (
    player_performance_matrix.loc[
        player_performance_matrix[
            "observed_t20i_matches"
        ].eq(0)
    ]
    .copy()
)

no_t20i_research_pool["research_status"] = (
    "Age, identity and official cap status unverified"
)

print(
    "\nPlayers with no recorded T20I:",
    len(no_t20i_research_pool)
)

print(
    "IMPORTANT: This is NOT a confirmed "
    "uncapped-player count."
)

print("\nBatting research examples — NOT RANKINGS")

display(
    no_t20i_research_pool.loc[
        no_t20i_research_pool[
            "batting_contexts"
        ].gt(0)
    ]
    .sort_values(
        "batting_matched_balls",
        ascending=False
    )[
        [
            "player_id",
            "player_name",
            "batting_evidence_tier",
            "batting_matched_balls",
            "overall_source_adjusted_sr_uplift",
            "observed_ipl_matches",
            "research_status"
        ]
    ].head(15)
)

print("\nBowling research examples — NOT RANKINGS")

display(
    no_t20i_research_pool.loc[
        no_t20i_research_pool[
            "bowling_contexts"
        ].gt(0)
    ]
    .sort_values(
        "bowling_matched_balls",
        ascending=False
    )[
        [
            "player_id",
            "player_name",
            "bowling_evidence_tier",
            "bowling_matched_balls",
            "overall_clean_dot_uplift_pp",
            "overall_boundary_suppression_pp",
            "overall_wicket_uplift_pp",
            "research_status"
        ]
    ].head(15)
)


# ============================================================
# 7. DATA INTEGRITY AND RECONCILIATION
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

# Reconcile each pressure profile against the
# independently source-aggregated pressure balls.

batting_profile_reconciliation = (
    batting_matched_source.groupby(
        ["striker_id", "batting_pressure_type"],
        as_index=False
    )
    .agg(calculated_balls=("pressure_balls", "sum"))
    .merge(
        bat[
            [
                "striker_id",
                "batting_pressure_type",
                "source_matched_balls"
            ]
        ],
        on=[
            "striker_id",
            "batting_pressure_type"
        ],
        how="outer",
        validate="one_to_one"
    )
)

bowling_profile_reconciliation = (
    bowling_matched_source.groupby(
        ["bowler_id", "bowling_pressure_type"],
        as_index=False
    )
    .agg(calculated_balls=("pressure_balls", "sum"))
    .merge(
        bowl[
            [
                "bowler_id",
                "bowling_pressure_type",
                "source_matched_balls"
            ]
        ],
        on=[
            "bowler_id",
            "bowling_pressure_type"
        ],
        how="outer",
        validate="one_to_one"
    )
)

batting_profile_mismatches = int(
    (
        ~np.isclose(
            batting_profile_reconciliation["calculated_balls"],
            batting_profile_reconciliation["source_matched_balls"],
            equal_nan=False
        )
    ).sum()
)

bowling_profile_mismatches = int(
    (
        ~np.isclose(
            bowling_profile_reconciliation["calculated_balls"],
            bowling_profile_reconciliation["source_matched_balls"],
            equal_nan=False
        )
    ).sum()
)

checks = {
    "registry_player_count_mismatch": abs(
        len(player_performance_matrix) -
        len(scouting_review_queue)
    ),

    "duplicate_player_ids": int(
        player_performance_matrix[
            "player_id"
        ].duplicated().sum()
    ),

    "batting_model_player_count_mismatch": abs(
        int(
            player_performance_matrix[
                "batting_contexts"
            ].gt(0).sum()
        ) -
        bat["striker_id"].nunique()
    ),

    "bowling_model_player_count_mismatch": abs(
        int(
            player_performance_matrix[
                "bowling_contexts"
            ].gt(0).sum()
        ) -
        bowl["bowler_id"].nunique()
    ),

    "batting_pressure_ball_mismatch": abs(
        int(
            player_performance_matrix[
                "batting_matched_balls"
            ].sum()
        ) -
        int(bat["source_matched_balls"].sum())
    ),

    "bowling_pressure_ball_mismatch": abs(
        int(
            player_performance_matrix[
                "bowling_matched_balls"
            ].sum()
        ) -
        int(bowl["source_matched_balls"].sum())
    ),

    "batting_profile_source_mismatches": (
        batting_profile_mismatches
    ),

    "bowling_profile_source_mismatches": (
        bowling_profile_mismatches
    ),

    "batting_reliable_context_mismatch": abs(
        int(
            player_performance_matrix[
                "batting_reliable_contexts"
            ].sum()
        ) -
        int(bat["source_sample_reliable"].sum())
    ),

    "bowling_reliable_context_mismatch": abs(
        int(
            player_performance_matrix[
                "bowling_reliable_contexts"
            ].sum()
        ) -
        int(bowl["source_sample_reliable"].sum())
    ),

    "batting_positive_signal_mismatch": abs(
        int(
            player_performance_matrix[
                "batting_positive_contexts"
            ].sum()
        ) -
        int(
            positive_signal_details[
                "role"
            ].eq("Batting").sum()
        )
    ),

    "bowling_positive_signal_mismatch": abs(
        int(
            player_performance_matrix[
                "bowling_positive_metrics"
            ].sum()
        ) -
        int(
            positive_signal_details[
                "role"
            ].eq("Bowling").sum()
        )
    ),

    "missing_evidence_tiers": int(
        player_performance_matrix[
            [
                "batting_evidence_tier",
                "bowling_evidence_tier"
            ]
        ].isna().any(axis=1).sum()
    ),

    "missing_t20i_research_groups": int(
        player_performance_matrix[
            "t20i_evidence_group"
        ].isna().sum()
    ),

    "missing_player_names": int(
        player_performance_matrix[
            "player_name"
        ].isna().sum()
    )
}

performance_integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(performance_integrity_report)

if performance_integrity_report["count"].eq(0).all():

    print("\nALL 15 INTEGRITY CHECKS PASSED")

else:

    print("\nWARNING: REVIEW NONZERO CHECKS")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "Player-level performance is calculated from "
    "source-matched pressure and expected outcomes."
)

print(
    "Aggregated player-level metrics are descriptive. "
    "Profile confidence intervals cannot be "
    "transferred to the aggregated metrics."
)

print(
    "Statistical evidence tiers are not rankings "
    "or official player eligibility decisions."
)

print(
    "No recorded T20I does not establish "
    "uncapped status."
)

print(
    "All 969 players remain available "
    "for the scouting process."
)

In [ ]:
# ============================================================
# CELL 68 — RECOVER LOW-SAMPLE SCOUTING EVIDENCE
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("LOW-SAMPLE AND UNQUALIFIED SCOUTING EVIDENCE")
print("=" * 70)


# ============================================================
# 1. VERIFY THE VALIDATED INPUT DATA
# ============================================================

print("\n1. INPUT VERIFICATION")
print("=" * 70)

required_variables = [
    "player_events",
    "bowling_eval",
    "source_lookup",
    "player_performance_matrix"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing previous-cell variables: {missing_variables}"
)

required_columns = {
    "Player events": (
        player_events,
        [
            "match_id", "striker_id", "batter_runs",
            "batter_ball_proxy", "is_four", "is_six"
        ]
    ),
    "Bowling events": (
        bowling_eval,
        [
            "match_id", "bowler_id", "bowler_legal_ball",
            "clean_dot_proxy", "batter_boundary_proxy",
            "credited_wicket_proxy"
        ]
    ),
    "Competition lookup": (
        source_lookup,
        ["match_id", "source_class"]
    ),
    "Player matrix": (
        player_performance_matrix,
        [
            "player_id", "player_name",
            "batting_evidence_tier",
            "bowling_evidence_tier",
            "provisional_balls_faced",
            "legal_balls_bowled"
        ]
    )
}

for label, (df, columns) in required_columns.items():

    missing = sorted(set(columns) - set(df.columns))

    assert not missing, (
        f"{label}: missing columns {missing}"
    )

    print(f"{label}: {len(df):,} rows — VERIFIED")

assert source_lookup["match_id"].is_unique
assert player_performance_matrix["player_id"].is_unique

print("Previous analysis remains unchanged.")


# ============================================================
# 2. CREATE DESCRIPTIVE BATTING EVIDENCE
# ============================================================

print("\n2. ALL-PLAYER BATTING EXPOSURE")
print("=" * 70)

raw_batting_events = (
    player_events.loc[
        player_events["batter_ball_proxy"].eq(True)
    ]
    .merge(
        source_lookup,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
    .copy()
)

raw_batting_events["raw_boundary_event"] = (
    raw_batting_events["is_four"].eq(1) |
    raw_batting_events["is_six"].eq(1)
).astype("int8")

raw_batting_player = (
    raw_batting_events.groupby(
        "striker_id",
        as_index=False
    )
    .agg(
        raw_batting_balls=("match_id", "size"),
        raw_batting_runs=("batter_runs", "sum"),
        raw_batting_matches=("match_id", "nunique"),
        raw_batting_boundaries=(
            "raw_boundary_event", "sum"
        ),
        raw_batting_ipl_balls=(
            "source_class",
            lambda x: int(x.eq("IPL").sum())
        ),
        raw_batting_t20i_balls=(
            "source_class",
            lambda x: int(x.eq("T20I").sum())
        ),
        raw_batting_other_balls=(
            "source_class",
            lambda x: int(x.eq("Other T20").sum())
        )
    )
    .rename(columns={"striker_id": "player_id"})
)

print(
    "Players with batting delivery exposure:",
    len(raw_batting_player)
)

print(
    "Provisional batting balls:",
    len(raw_batting_events)
)

display(
    raw_batting_player[
        [
            "raw_batting_balls",
            "raw_batting_runs",
            "raw_batting_matches",
            "raw_batting_boundaries"
        ]
    ].describe().T
)


# ============================================================
# 3. CREATE DESCRIPTIVE BOWLING EVIDENCE
# ============================================================

print("\n3. ALL-PLAYER BOWLING EXPOSURE")
print("=" * 70)

raw_bowling_events = (
    bowling_eval.loc[
        bowling_eval["bowler_legal_ball"].eq(True)
    ]
    .merge(
        source_lookup,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
    .copy()
)

raw_bowling_player = (
    raw_bowling_events.groupby(
        "bowler_id",
        as_index=False
    )
    .agg(
        raw_bowling_balls=("match_id", "size"),
        raw_bowling_matches=("match_id", "nunique"),
        raw_bowling_clean_dots=(
            "clean_dot_proxy", "sum"
        ),
        raw_bowling_boundaries=(
            "batter_boundary_proxy", "sum"
        ),
        raw_bowling_wickets=(
            "credited_wicket_proxy", "sum"
        ),
        raw_bowling_ipl_balls=(
            "source_class",
            lambda x: int(x.eq("IPL").sum())
        ),
        raw_bowling_t20i_balls=(
            "source_class",
            lambda x: int(x.eq("T20I").sum())
        ),
        raw_bowling_other_balls=(
            "source_class",
            lambda x: int(x.eq("Other T20").sum())
        )
    )
    .rename(columns={"bowler_id": "player_id"})
)

print(
    "Players with bowling delivery exposure:",
    len(raw_bowling_player)
)

print(
    "Legal bowling balls:",
    len(raw_bowling_events)
)

display(
    raw_bowling_player[
        [
            "raw_bowling_balls",
            "raw_bowling_matches",
            "raw_bowling_clean_dots",
            "raw_bowling_boundaries",
            "raw_bowling_wickets"
        ]
    ].describe().T
)


# ============================================================
# 4. MERGE INTO THE EXISTING PLAYER MATRIX
# ============================================================

print("\n4. COMPREHENSIVE ROLE EXPOSURE MATRIX")
print("=" * 70)

scouting_opportunity_matrix = (
    player_performance_matrix.copy()
    .merge(
        raw_batting_player,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        raw_bowling_player,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
)

raw_count_columns = [
    "raw_batting_balls",
    "raw_batting_runs",
    "raw_batting_matches",
    "raw_batting_boundaries",
    "raw_batting_ipl_balls",
    "raw_batting_t20i_balls",
    "raw_batting_other_balls",
    "raw_bowling_balls",
    "raw_bowling_matches",
    "raw_bowling_clean_dots",
    "raw_bowling_boundaries",
    "raw_bowling_wickets",
    "raw_bowling_ipl_balls",
    "raw_bowling_t20i_balls",
    "raw_bowling_other_balls"
]

scouting_opportunity_matrix[raw_count_columns] = (
    scouting_opportunity_matrix[raw_count_columns]
    .fillna(0)
    .astype("int64")
)

batting_denominator = (
    scouting_opportunity_matrix[
        "raw_batting_balls"
    ].replace(0, np.nan)
)

bowling_denominator = (
    scouting_opportunity_matrix[
        "raw_bowling_balls"
    ].replace(0, np.nan)
)

scouting_opportunity_matrix["raw_batting_sr_proxy"] = (
    100 *
    scouting_opportunity_matrix["raw_batting_runs"] /
    batting_denominator
)

scouting_opportunity_matrix[
    "raw_batting_boundary_pct"
] = (
    100 *
    scouting_opportunity_matrix[
        "raw_batting_boundaries"
    ] /
    batting_denominator
)

scouting_opportunity_matrix[
    "raw_bowling_clean_dot_pct"
] = (
    100 *
    scouting_opportunity_matrix[
        "raw_bowling_clean_dots"
    ] /
    bowling_denominator
)

scouting_opportunity_matrix[
    "raw_bowling_boundary_pct"
] = (
    100 *
    scouting_opportunity_matrix[
        "raw_bowling_boundaries"
    ] /
    bowling_denominator
)

scouting_opportunity_matrix[
    "raw_bowling_wickets_per_100_balls"
] = (
    100 *
    scouting_opportunity_matrix[
        "raw_bowling_wickets"
    ] /
    bowling_denominator
)

print(
    "Players retained:",
    len(scouting_opportunity_matrix)
)

print(
    "Players with raw batting statistics:",
    int(
        scouting_opportunity_matrix[
            "raw_batting_balls"
        ].gt(0).sum()
    )
)

print(
    "Players with raw bowling statistics:",
    int(
        scouting_opportunity_matrix[
            "raw_bowling_balls"
        ].gt(0).sum()
    )
)


# ============================================================
# 5. ASSESS SAMPLE SIZE AND MODEL COVERAGE
# ============================================================

print("\n5. SAMPLE SIZE AND QUALIFICATION COVERAGE")
print("=" * 70)

scouting_opportunity_matrix["batting_sample_band"] = (
    pd.cut(
        scouting_opportunity_matrix["raw_batting_balls"],
        bins=[-1, 0, 29, 59, 119, np.inf],
        labels=[
            "None",
            "1–29",
            "30–59",
            "60–119",
            "120+"
        ]
    )
)

scouting_opportunity_matrix["bowling_sample_band"] = (
    pd.cut(
        scouting_opportunity_matrix["raw_bowling_balls"],
        bins=[-1, 0, 29, 119, 239, np.inf],
        labels=[
            "None",
            "1–29",
            "30–119",
            "120–239",
            "240+"
        ]
    )
)

print("\nBatting sample sizes by evidence tier")

display(
    pd.crosstab(
        scouting_opportunity_matrix[
            "batting_evidence_tier"
        ],
        scouting_opportunity_matrix[
            "batting_sample_band"
        ]
    )
)

print("\nBowling sample sizes by evidence tier")

display(
    pd.crosstab(
        scouting_opportunity_matrix[
            "bowling_evidence_tier"
        ],
        scouting_opportunity_matrix[
            "bowling_sample_band"
        ]
    )
)

print("\nPlayers without qualified pressure profiles")

display(
    pd.DataFrame([
        {
            "role": "Batting",
            "unqualified_with_exposure": int(
                (
                    scouting_opportunity_matrix[
                        "batting_evidence_tier"
                    ].str.startswith("D")
                ).sum()
            ),
            "total_raw_balls": int(
                scouting_opportunity_matrix.loc[
                    scouting_opportunity_matrix[
                        "batting_evidence_tier"
                    ].str.startswith("D"),
                    "raw_batting_balls"
                ].sum()
            )
        },
        {
            "role": "Bowling",
            "unqualified_with_exposure": int(
                (
                    scouting_opportunity_matrix[
                        "bowling_evidence_tier"
                    ].str.startswith("D")
                ).sum()
            ),
            "total_raw_balls": int(
                scouting_opportunity_matrix.loc[
                    scouting_opportunity_matrix[
                        "bowling_evidence_tier"
                    ].str.startswith("D"),
                    "raw_bowling_balls"
                ].sum()
            )
        }
    ])
)


# ============================================================
# 6. REVIEW UNQUALIFIED PLAYERS AND CONTEXT DIFFERENCES
# ============================================================

print("\n6. SCOUTING OPPORTUNITY REVIEW — NOT RANKINGS")
print("=" * 70)

unqualified_batters = (
    scouting_opportunity_matrix.loc[
        scouting_opportunity_matrix[
            "batting_evidence_tier"
        ].str.startswith("D")
    ]
    .sort_values(
        "raw_batting_balls",
        ascending=False
    )
)

unqualified_bowlers = (
    scouting_opportunity_matrix.loc[
        scouting_opportunity_matrix[
            "bowling_evidence_tier"
        ].str.startswith("D")
    ]
    .sort_values(
        "raw_bowling_balls",
        ascending=False
    )
)

print("\nBatters with exposure but no qualified model profile")

display(
    unqualified_batters[
        [
            "player_id",
            "player_name",
            "raw_batting_balls",
            "raw_batting_matches",
            "raw_batting_sr_proxy",
            "raw_batting_ipl_balls",
            "raw_batting_t20i_balls",
            "observed_t20i_matches"
        ]
    ].head(15)
)

print("\nBowlers with exposure but no qualified model profile")

display(
    unqualified_bowlers[
        [
            "player_id",
            "player_name",
            "raw_bowling_balls",
            "raw_bowling_matches",
            "raw_bowling_clean_dot_pct",
            "raw_bowling_boundary_pct",
            "raw_bowling_wickets_per_100_balls",
            "observed_t20i_matches"
        ]
    ].head(15)
)

# A positive result in one pressure context does not
# guarantee a positive aggregate across all contexts.

batting_context_disagreements = (
    scouting_opportunity_matrix.loc[
        scouting_opportunity_matrix[
            "batting_positive_contexts"
        ].gt(0) &
        scouting_opportunity_matrix[
            "overall_source_adjusted_sr_uplift"
        ].lt(0)
    ]
    .copy()
)

print(
    "\nPlayers with a positive reliable batting "
    "context but negative overall batting uplift:",
    len(batting_context_disagreements)
)

display(
    batting_context_disagreements[
        [
            "player_id",
            "player_name",
            "batting_positive_contexts",
            "overall_source_adjusted_sr_uplift",
            "batting_matched_balls"
        ]
    ]
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

matrix = scouting_opportunity_matrix

registry_ids = set(matrix["player_id"])

checks = {
    "player_count_mismatch": abs(
        len(matrix) - len(player_performance_matrix)
    ),

    "duplicate_player_ids": int(
        matrix["player_id"].duplicated().sum()
    ),

    "raw_batting_players_missing": len(
        set(raw_batting_player["player_id"]) -
        registry_ids
    ),

    "raw_bowling_players_missing": len(
        set(raw_bowling_player["player_id"]) -
        registry_ids
    ),

    "raw_batting_ball_reconciliation": abs(
        int(matrix["raw_batting_balls"].sum()) -
        len(raw_batting_events)
    ),

    "raw_bowling_ball_reconciliation": abs(
        int(matrix["raw_bowling_balls"].sum()) -
        len(raw_bowling_events)
    ),

    "batting_source_ball_mismatch": int(
        (
            matrix[
                [
                    "raw_batting_ipl_balls",
                    "raw_batting_t20i_balls",
                    "raw_batting_other_balls"
                ]
            ].sum(axis=1) != matrix["raw_batting_balls"]
        ).sum()
    ),

    "bowling_source_ball_mismatch": int(
        (
            matrix[
                [
                    "raw_bowling_ipl_balls",
                    "raw_bowling_t20i_balls",
                    "raw_bowling_other_balls"
                ]
            ].sum(axis=1) != matrix["raw_bowling_balls"]
        ).sum()
    ),

    "batting_exposure_mismatch": int(
        (
            matrix["raw_batting_balls"] !=
            matrix["provisional_balls_faced"]
        ).sum()
    ),

    "bowling_exposure_mismatch": int(
        (
            matrix["raw_bowling_balls"] !=
            matrix["legal_balls_bowled"]
        ).sum()
    ),

    "batting_boundaries_exceed_balls": int(
        (
            matrix["raw_batting_boundaries"] >
            matrix["raw_batting_balls"]
        ).sum()
    ),

    "bowling_clean_dots_exceed_balls": int(
        (
            matrix["raw_bowling_clean_dots"] >
            matrix["raw_bowling_balls"]
        ).sum()
    ),

    "bowling_boundaries_exceed_balls": int(
        (
            matrix["raw_bowling_boundaries"] >
            matrix["raw_bowling_balls"]
        ).sum()
    ),

    "bowling_wickets_exceed_balls": int(
        (
            matrix["raw_bowling_wickets"] >
            matrix["raw_bowling_balls"]
        ).sum()
    ),

    "batting_matched_exceeds_raw": int(
        (
            matrix["batting_matched_balls"] >
            matrix["raw_batting_balls"]
        ).sum()
    ),

    "bowling_matched_exceeds_raw": int(
        (
            matrix["bowling_matched_balls"] >
            matrix["raw_bowling_balls"]
        ).sum()
    ),

    "negative_raw_counts": int(
        (matrix[raw_count_columns] < 0)
        .any(axis=1).sum()
    ),

    "missing_sample_bands": int(
        matrix[
            [
                "batting_sample_band",
                "bowling_sample_band"
            ]
        ].isna().any(axis=1).sum()
    )
}

opportunity_integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(opportunity_integrity_report)

if opportunity_integrity_report["count"].eq(0).all():
    print("\nALL 18 INTEGRITY CHECKS PASSED")
else:
    print("\nWARNING: INVESTIGATE NONZERO CHECKS")

print("\nINTERPRETATION")
print("=" * 70)

print(
    "Raw statistics are descriptive and use "
    "provisional delivery definitions."
)

print(
    "Raw strike rates and bowling event rates "
    "are not pressure-adjusted or opposition-adjusted."
)

print(
    "Failing a pressure-model qualification threshold "
    "does not make a player unsuitable for scouting."
)

print(
    "A context-specific positive result can coexist "
    "with a negative overall pressure-performance effect."
)

print(
    "All 969 registered players remain available."
)

In [ ]:
# ============================================================
# CELL 69 — CONTEXT-ADJUSTED EMERGING TALENT RESEARCH
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("CONTEXT-ADJUSTED EMERGING TALENT RESEARCH")
print("=" * 70)


# ============================================================
# 1. VERIFY INPUTS
# ============================================================

print("\n1. INPUT VERIFICATION")
print("=" * 70)

required_variables = [
    "raw_batting_events",
    "raw_bowling_events",
    "scouting_opportunity_matrix"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing Cell 68 variables: {missing_variables}"
)

context_keys = [
    "source_class",
    "innings",
    "match_phase"
]

batting_required = [
    "match_id", "striker_id", "batter_runs"
] + context_keys

bowling_required = [
    "match_id", "bowler_id",
    "clean_dot_proxy",
    "batter_boundary_proxy",
    "credited_wicket_proxy"
] + context_keys

assert set(batting_required).issubset(
    raw_batting_events.columns
), sorted(
    set(batting_required) -
    set(raw_batting_events.columns)
)

assert set(bowling_required).issubset(
    raw_bowling_events.columns
), sorted(
    set(bowling_required) -
    set(raw_bowling_events.columns)
)

assert scouting_opportunity_matrix["player_id"].is_unique

assert raw_batting_events[context_keys].notna().all().all()
assert raw_bowling_events[context_keys].notna().all().all()

print("Batting deliveries:", len(raw_batting_events))
print("Bowling legal deliveries:", len(raw_bowling_events))
print("Players:", len(scouting_opportunity_matrix))
print("Context fields: competition, innings, match phase")


# ============================================================
# 2. DEFINE LEAVE-ONE-PLAYER-OUT CONTEXT COMPARISON
# ============================================================

print("\n2. PEER-COMPARISON METHOD")
print("=" * 70)


def calculate_peer_context(
    events,
    player_column,
    outcome_mapping,
    prefix
):

    necessary = (
        ["match_id", player_column] +
        context_keys +
        list(outcome_mapping.values())
    )

    data = events[necessary].copy()

    aggregation = {
        "stratum_balls": ("match_id", "size")
    }

    for outcome_name, source_column in outcome_mapping.items():

        new_column = f"outcome_{outcome_name}"

        data[new_column] = pd.to_numeric(
            data[source_column],
            errors="raise"
        )

        assert data[new_column].notna().all()

        aggregation[f"observed_{outcome_name}"] = (
            new_column, "sum"
        )

    player_context = (
        data.groupby(
            [player_column] + context_keys,
            as_index=False
        )
        .agg(**aggregation)
    )

    overall_aggregation = {
        "total_balls": ("stratum_balls", "sum")
    }

    for outcome_name in outcome_mapping:

        overall_aggregation[f"total_{outcome_name}"] = (
            f"observed_{outcome_name}", "sum"
        )

    context_totals = (
        player_context.groupby(
            context_keys,
            as_index=False
        )
        .agg(**overall_aggregation)
    )

    player_context = player_context.merge(
        context_totals,
        on=context_keys,
        how="left",
        validate="many_to_one"
    )

    # Exclude the evaluated player's deliveries.
    player_context["peer_reference_balls"] = (
        player_context["total_balls"] -
        player_context["stratum_balls"]
    )

    for outcome_name in outcome_mapping:

        player_context[
            f"peer_reference_{outcome_name}"
        ] = (
            player_context[f"total_{outcome_name}"] -
            player_context[f"observed_{outcome_name}"]
        )

        player_context[
            f"expected_{outcome_name}"
        ] = (
            player_context["stratum_balls"] *
            player_context[
                f"peer_reference_{outcome_name}"
            ] /
            player_context[
                "peer_reference_balls"
            ].replace(0, np.nan)
        )

    # An observation is comparable only when at least
    # one other player contributes reference deliveries.
    matched_context = player_context.loc[
        player_context["peer_reference_balls"].gt(0)
    ].copy()

    summary_aggregation = {
        "matched_balls": ("stratum_balls", "sum"),
        "matched_strata": ("stratum_balls", "size")
    }

    for outcome_name in outcome_mapping:

        summary_aggregation[f"observed_{outcome_name}"] = (
            f"observed_{outcome_name}", "sum"
        )

        summary_aggregation[f"expected_{outcome_name}"] = (
            f"expected_{outcome_name}", "sum"
        )

    player_summary = (
        matched_context.groupby(
            player_column,
            as_index=False
        )
        .agg(**summary_aggregation)
    )

    rename_columns = {
        player_column: "player_id",
        "matched_balls": f"{prefix}_peer_balls",
        "matched_strata": f"{prefix}_peer_strata"
    }

    for outcome_name in outcome_mapping:

        rename_columns[
            f"observed_{outcome_name}"
        ] = (
            f"{prefix}_peer_observed_{outcome_name}"
        )

        rename_columns[
            f"expected_{outcome_name}"
        ] = (
            f"{prefix}_peer_expected_{outcome_name}"
        )

    player_summary = player_summary.rename(
        columns=rename_columns
    )

    return (
        player_summary,
        player_context,
        matched_context
    )


print(
    "Reference: other players in the same "
    "competition category, innings and match phase."
)

print(
    "A player's own deliveries are excluded "
    "from their peer reference."
)


# ============================================================
# 3. CALCULATE BATTING PEER COMPARISONS
# ============================================================

print("\n3. BATTING PEER COMPARISON")
print("=" * 70)

(
    batting_peer_summary,
    batting_peer_strata,
    batting_peer_matched
) = calculate_peer_context(
    events=raw_batting_events,
    player_column="striker_id",
    outcome_mapping={
        "runs": "batter_runs"
    },
    prefix="bat"
)

print(
    "Batters with peer comparisons:",
    len(batting_peer_summary)
)

print(
    "Matched batting balls:",
    int(
        batting_peer_summary[
            "bat_peer_balls"
        ].sum()
    )
)

print(
    "Unmatched batting balls:",
    int(
        batting_peer_strata.loc[
            batting_peer_strata[
                "peer_reference_balls"
            ].eq(0),
            "stratum_balls"
        ].sum()
    )
)


# ============================================================
# 4. CALCULATE BOWLING PEER COMPARISONS
# ============================================================

print("\n4. BOWLING PEER COMPARISON")
print("=" * 70)

(
    bowling_peer_summary,
    bowling_peer_strata,
    bowling_peer_matched
) = calculate_peer_context(
    events=raw_bowling_events,
    player_column="bowler_id",
    outcome_mapping={
        "dots": "clean_dot_proxy",
        "boundaries": "batter_boundary_proxy",
        "wickets": "credited_wicket_proxy"
    },
    prefix="bowl"
)

print(
    "Bowlers with peer comparisons:",
    len(bowling_peer_summary)
)

print(
    "Matched bowling legal balls:",
    int(
        bowling_peer_summary[
            "bowl_peer_balls"
        ].sum()
    )
)

print(
    "Unmatched bowling legal balls:",
    int(
        bowling_peer_strata.loc[
            bowling_peer_strata[
                "peer_reference_balls"
            ].eq(0),
            "stratum_balls"
        ].sum()
    )
)


# ============================================================
# 5. BUILD CONTEXT-ADJUSTED PLAYER RESEARCH MATRIX
# ============================================================

print("\n5. CONTEXT-ADJUSTED PLAYER RESEARCH MATRIX")
print("=" * 70)

context_research_matrix = (
    scouting_opportunity_matrix.copy()
    .merge(
        batting_peer_summary,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        bowling_peer_summary,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
)

count_fields = [
    "bat_peer_balls",
    "bat_peer_strata",
    "bowl_peer_balls",
    "bowl_peer_strata"
]

context_research_matrix[count_fields] = (
    context_research_matrix[count_fields]
    .fillna(0)
    .astype(int)
)

context_research_matrix["bat_peer_coverage_pct"] = (
    100 *
    context_research_matrix["bat_peer_balls"] /
    context_research_matrix[
        "raw_batting_balls"
    ].replace(0, np.nan)
)

context_research_matrix["bowl_peer_coverage_pct"] = (
    100 *
    context_research_matrix["bowl_peer_balls"] /
    context_research_matrix[
        "raw_bowling_balls"
    ].replace(0, np.nan)
)

bat_denominator = (
    context_research_matrix[
        "bat_peer_balls"
    ].replace(0, np.nan)
)

bowl_denominator = (
    context_research_matrix[
        "bowl_peer_balls"
    ].replace(0, np.nan)
)

context_research_matrix["bat_peer_sr_gap"] = (
    100 *
    (
        context_research_matrix[
            "bat_peer_observed_runs"
        ] -
        context_research_matrix[
            "bat_peer_expected_runs"
        ]
    ) /
    bat_denominator
)

context_research_matrix["bowl_peer_dot_gap_pp"] = (
    100 *
    (
        context_research_matrix[
            "bowl_peer_observed_dots"
        ] -
        context_research_matrix[
            "bowl_peer_expected_dots"
        ]
    ) /
    bowl_denominator
)

context_research_matrix[
    "bowl_peer_boundary_suppression_pp"
] = (
    100 *
    (
        context_research_matrix[
            "bowl_peer_expected_boundaries"
        ] -
        context_research_matrix[
            "bowl_peer_observed_boundaries"
        ]
    ) /
    bowl_denominator
)

context_research_matrix["bowl_peer_wicket_gap_pp"] = (
    100 *
    (
        context_research_matrix[
            "bowl_peer_observed_wickets"
        ] -
        context_research_matrix[
            "bowl_peer_expected_wickets"
        ]
    ) /
    bowl_denominator
)

# Explicit heuristic shrinkage toward zero.
# Not an estimated empirical-Bayes model.
#
# Batting: 60 pseudo-balls.
# Bowling: 120 pseudo-balls.

bat_shrinkage_factor = (
    context_research_matrix["bat_peer_balls"] /
    (
        context_research_matrix["bat_peer_balls"] + 60
    )
)

bowl_shrinkage_factor = (
    context_research_matrix["bowl_peer_balls"] /
    (
        context_research_matrix["bowl_peer_balls"] + 120
    )
)

context_research_matrix["bat_peer_sr_gap_shrunk"] = (
    context_research_matrix["bat_peer_sr_gap"] *
    bat_shrinkage_factor
)

for metric in [
    "bowl_peer_dot_gap_pp",
    "bowl_peer_boundary_suppression_pp",
    "bowl_peer_wicket_gap_pp"
]:

    context_research_matrix[
        f"{metric}_shrunk"
    ] = (
        context_research_matrix[metric] *
        bowl_shrinkage_factor
    )

print(
    "Players retained:",
    len(context_research_matrix)
)

print(
    "Batting peer estimates:",
    int(
        context_research_matrix[
            "bat_peer_sr_gap"
        ].notna().sum()
    )
)

print(
    "Bowling peer estimates:",
    int(
        context_research_matrix[
            "bowl_peer_dot_gap_pp"
        ].notna().sum()
    )
)

print("\nContext-adjusted descriptive distributions")

display(
    context_research_matrix[
        [
            "bat_peer_sr_gap_shrunk",
            "bowl_peer_dot_gap_pp_shrunk",
            "bowl_peer_boundary_suppression_pp_shrunk",
            "bowl_peer_wicket_gap_pp_shrunk"
        ]
    ].describe().T
)


# ============================================================
# 6. BUILD UNVERIFIED EMERGING-TALENT RESEARCH SCREENS
# ============================================================

print("\n6. EMERGING-TALENT RESEARCH SCREENS")
print("=" * 70)

# Research workload, NOT confirmed uncapped status.
no_recorded_t20i = context_research_matrix.loc[
    context_research_matrix[
        "observed_t20i_matches"
    ].eq(0)
].copy()

batting_research_candidates = (
    no_recorded_t20i.loc[
        no_recorded_t20i[
            "raw_batting_balls"
        ].ge(60) &
        no_recorded_t20i[
            "raw_batting_matches"
        ].ge(3) &
        no_recorded_t20i[
            "bat_peer_coverage_pct"
        ].ge(95)
    ]
    .sort_values(
        "bat_peer_sr_gap_shrunk",
        ascending=False
    )
    .copy()
)

bowling_research_candidates = (
    no_recorded_t20i.loc[
        no_recorded_t20i[
            "raw_bowling_balls"
        ].ge(120) &
        no_recorded_t20i[
            "raw_bowling_matches"
        ].ge(3) &
        no_recorded_t20i[
            "bowl_peer_coverage_pct"
        ].ge(95)
    ]
    .sort_values(
        "bowl_peer_boundary_suppression_pp_shrunk",
        ascending=False
    )
    .copy()
)

print(
    "Players with no T20I in supplied sample:",
    len(no_recorded_t20i)
)

print(
    "Batting research candidates meeting "
    "minimum descriptive exposure:",
    len(batting_research_candidates)
)

print(
    "Bowling research candidates meeting "
    "minimum descriptive exposure:",
    len(bowling_research_candidates)
)

print(
    "\nBATTING — strongest source/phase-referenced "
    "descriptive gaps, NOT final rankings"
)

display(
    batting_research_candidates[
        [
            "player_id",
            "player_name",
            "raw_batting_balls",
            "raw_batting_matches",
            "raw_batting_sr_proxy",
            "bat_peer_sr_gap",
            "bat_peer_sr_gap_shrunk",
            "batting_evidence_tier",
            "observed_ipl_matches",
            "verified_cap_status"
        ]
    ].head(20)
)

print(
    "\nBOWLING — source/phase-referenced "
    "boundary suppression, NOT final rankings"
)

display(
    bowling_research_candidates[
        [
            "player_id",
            "player_name",
            "raw_bowling_balls",
            "raw_bowling_matches",
            "bowl_peer_dot_gap_pp_shrunk",
            "bowl_peer_boundary_suppression_pp_shrunk",
            "bowl_peer_wicket_gap_pp_shrunk",
            "bowling_evidence_tier",
            "observed_ipl_matches",
            "verified_cap_status"
        ]
    ].head(20)
)

print(
    "\nIMPORTANT: Age, identity and official "
    "cap status remain unverified."
)

print(
    "Players below minimum sample thresholds "
    "are retained in context_research_matrix."
)


# ============================================================
# 7. INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

bat_unmatched = int(
    batting_peer_strata.loc[
        batting_peer_strata[
            "peer_reference_balls"
        ].eq(0),
        "stratum_balls"
    ].sum()
)

bowl_unmatched = int(
    bowling_peer_strata.loc[
        bowling_peer_strata[
            "peer_reference_balls"
        ].eq(0),
        "stratum_balls"
    ].sum()
)

checks = {
    "registry_player_count_difference": abs(
        len(context_research_matrix) -
        len(scouting_opportunity_matrix)
    ),

    "duplicate_player_ids": int(
        context_research_matrix[
            "player_id"
        ].duplicated().sum()
    ),

    "unrepresented_batting_ids": len(
        set(batting_peer_summary["player_id"]) -
        set(context_research_matrix["player_id"])
    ),

    "unrepresented_bowling_ids": len(
        set(bowling_peer_summary["player_id"]) -
        set(context_research_matrix["player_id"])
    ),

    "batting_context_ball_reconciliation": abs(
        int(
            batting_peer_strata["stratum_balls"].sum()
        ) - len(raw_batting_events)
    ),

    "bowling_context_ball_reconciliation": abs(
        int(
            bowling_peer_strata["stratum_balls"].sum()
        ) - len(raw_bowling_events)
    ),

    "batting_matched_unmatched_reconciliation": abs(
        int(
            context_research_matrix[
                "bat_peer_balls"
            ].sum()
        ) +
        bat_unmatched -
        len(raw_batting_events)
    ),

    "bowling_matched_unmatched_reconciliation": abs(
        int(
            context_research_matrix[
                "bowl_peer_balls"
            ].sum()
        ) +
        bowl_unmatched -
        len(raw_bowling_events)
    ),

    "negative_batting_peer_reference": int(
        batting_peer_strata[
            "peer_reference_balls"
        ].lt(0).sum()
    ),

    "negative_bowling_peer_reference": int(
        bowling_peer_strata[
            "peer_reference_balls"
        ].lt(0).sum()
    ),

    "batting_peer_exposure_exceeds_raw": int(
        (
            context_research_matrix[
                "bat_peer_balls"
            ] >
            context_research_matrix[
                "raw_batting_balls"
            ]
        ).sum()
    ),

    "bowling_peer_exposure_exceeds_raw": int(
        (
            context_research_matrix[
                "bowl_peer_balls"
            ] >
            context_research_matrix[
                "raw_bowling_balls"
            ]
        ).sum()
    ),

    "negative_batting_expected_runs": int(
        batting_peer_matched[
            "expected_runs"
        ].lt(0).sum()
    ),

    "negative_bowling_expected_events": int(
        (
            bowling_peer_matched[
                [
                    "expected_dots",
                    "expected_boundaries",
                    "expected_wickets"
                ]
            ] < 0
        ).any(axis=1).sum()
    ),

    "duplicate_batting_peer_strata": int(
        batting_peer_strata.duplicated(
            ["striker_id"] + context_keys
        ).sum()
    ),

    "duplicate_bowling_peer_strata": int(
        bowling_peer_strata.duplicated(
            ["bowler_id"] + context_keys
        ).sum()
    )
}

peer_integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(peer_integrity_report)

if peer_integrity_report["count"].eq(0).all():

    print("\nALL 16 INTEGRITY CHECKS PASSED")

else:

    print("\nWARNING: INVESTIGATE NONZERO CHECKS")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "All peer reference rates exclude "
    "the evaluated player's own observations."
)

print(
    "The comparison controls for broad "
    "competition category, innings and match phase."
)

print(
    "It does not fully control for opposition "
    "quality, pitch, venue or individual role."
)

print(
    "Shrinkage uses fixed heuristic pseudo-ball "
    "counts, not fitted statistical priors."
)

print(
    "This is a research screen only. "
    "No age or cap eligibility has been established."
)

In [ ]:
# ============================================================
# CELL 70 — RECENT IPL FORM AND SEASON CONSISTENCY
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("RECENT IPL FORM AND SEASON CONSISTENCY")
print("=" * 70)


# ============================================================
# 1. IDENTIFY THE TWO MOST RECENT IPL SEASONS
# ============================================================

print("\n1. IPL SEASON IDENTIFICATION")
print("=" * 70)

required_metadata = [
    "match_id", "is_ipl", "season", "match_date"
]

assert set(required_metadata).issubset(match_metadata.columns)

ipl_season_lookup = (
    match_metadata.loc[
        match_metadata["is_ipl"].eq(True),
        required_metadata
    ]
    .copy()
)

assert ipl_season_lookup["match_id"].is_unique

year_from_season = pd.to_numeric(
    ipl_season_lookup["season"]
    .astype("string")
    .str.extract(r"(20\d{2})", expand=False),
    errors="coerce"
)

year_from_date = pd.to_datetime(
    ipl_season_lookup["match_date"],
    errors="coerce"
).dt.year

ipl_season_lookup["season_year"] = (
    year_from_season
    .fillna(year_from_date)
    .astype("Int64")
)

assert ipl_season_lookup["season_year"].notna().all(), (
    "Some IPL matches have no recoverable season year."
)

available_years = sorted(
    ipl_season_lookup["season_year"]
    .astype(int)
    .unique()
    .tolist()
)

assert len(available_years) >= 2, (
    "At least two IPL seasons are required."
)

previous_year, latest_year = available_years[-2:]

print("Available IPL seasons:", available_years)
print("Previous season:", previous_year)
print("Latest season:", latest_year)

display(
    ipl_season_lookup.groupby(
        "season_year"
    )["match_id"]
    .nunique()
    .rename("matches")
    .reset_index()
)


# ============================================================
# 2. PREPARE RECENT IPL DELIVERY DATA
# ============================================================

print("\n2. RECENT IPL DELIVERY COVERAGE")
print("=" * 70)

season_map = ipl_season_lookup[
    ["match_id", "season_year"]
].copy()

recent_batting_events = (
    raw_batting_events.loc[
        raw_batting_events["source_class"].eq("IPL")
    ]
    .merge(
        season_map,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

recent_bowling_events = (
    raw_bowling_events.loc[
        raw_bowling_events["source_class"].eq("IPL")
    ]
    .merge(
        season_map,
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

assert recent_batting_events["season_year"].notna().all()
assert recent_bowling_events["season_year"].notna().all()

recent_batting_events = recent_batting_events.loc[
    recent_batting_events[
        "season_year"
    ].isin([previous_year, latest_year])
].copy()

recent_bowling_events = recent_bowling_events.loc[
    recent_bowling_events[
        "season_year"
    ].isin([previous_year, latest_year])
].copy()

assert len(recent_batting_events) > 0
assert len(recent_bowling_events) > 0

print("\nBatting exposure by IPL season")

display(
    recent_batting_events.groupby("season_year")
    .agg(
        matches=("match_id", "nunique"),
        batters=("striker_id", "nunique"),
        batting_balls=("match_id", "size")
    )
    .reset_index()
)

print("\nBowling exposure by IPL season")

display(
    recent_bowling_events.groupby("season_year")
    .agg(
        matches=("match_id", "nunique"),
        bowlers=("bowler_id", "nunique"),
        legal_balls=("match_id", "size")
    )
    .reset_index()
)


# ============================================================
# 3. DEFINE SEASON-SPECIFIC PEER COMPARISON
# ============================================================

print("\n3. SEASON-SPECIFIC PEER COMPARISON METHOD")
print("=" * 70)


def calculate_season_peer_profiles(
    events,
    player_column,
    outcomes,
    directions,
    pseudo_balls
):

    context = [
        "season_year",
        "innings",
        "match_phase"
    ]

    player_context = [player_column] + context
    player_season = [player_column, "season_year"]

    required = (
        player_season +
        ["match_id", "innings", "match_phase"] +
        list(outcomes.values())
    )

    data = events[required].copy()

    aggregation = {
        "stratum_balls": ("match_id", "size")
    }

    for label, source_column in outcomes.items():

        aggregation[f"observed_{label}"] = (
            source_column, "sum"
        )

    player_strata = (
        data.groupby(
            player_context,
            as_index=False
        )
        .agg(**aggregation)
    )

    total_aggregation = {
        "total_balls": ("stratum_balls", "sum")
    }

    for label in outcomes:

        total_aggregation[f"total_{label}"] = (
            f"observed_{label}", "sum"
        )

    context_totals = (
        player_strata.groupby(
            context,
            as_index=False
        )
        .agg(**total_aggregation)
    )

    player_strata = player_strata.merge(
        context_totals,
        on=context,
        how="left",
        validate="many_to_one"
    )

    player_strata["peer_balls"] = (
        player_strata["total_balls"] -
        player_strata["stratum_balls"]
    )

    for label in outcomes:

        peer_events = (
            player_strata[f"total_{label}"] -
            player_strata[f"observed_{label}"]
        )

        player_strata[f"expected_{label}"] = (
            player_strata["stratum_balls"] *
            peer_events /
            player_strata["peer_balls"].replace(
                0, np.nan
            )
        )

    matched = player_strata.loc[
        player_strata["peer_balls"].gt(0)
    ].copy()

    exposure = (
        data.groupby(
            player_season,
            as_index=False
        )
        .agg(
            raw_balls=("match_id", "size"),
            raw_matches=("match_id", "nunique")
        )
    )

    matched_aggregation = {
        "matched_balls": ("stratum_balls", "sum")
    }

    for label in outcomes:

        matched_aggregation[f"observed_{label}"] = (
            f"observed_{label}", "sum"
        )

        matched_aggregation[f"expected_{label}"] = (
            f"expected_{label}", "sum"
        )

    summary = (
        matched.groupby(
            player_season,
            as_index=False
        )
        .agg(**matched_aggregation)
    )

    summary = exposure.merge(
        summary,
        on=player_season,
        how="left",
        validate="one_to_one"
    )

    summary["matched_balls"] = (
        summary["matched_balls"]
        .fillna(0)
        .astype(int)
    )

    summary["peer_coverage_pct"] = (
        100 *
        summary["matched_balls"] /
        summary["raw_balls"]
    )

    denominator = summary[
        "matched_balls"
    ].replace(0, np.nan)

    shrinkage = (
        summary["matched_balls"] /
        (summary["matched_balls"] + pseudo_balls)
    )

    for label, direction in directions.items():

        summary[f"{label}_gap"] = (
            100 * direction *
            (
                summary[f"observed_{label}"] -
                summary[f"expected_{label}"]
            ) /
            denominator
        )

        summary[f"{label}_gap_shrunk"] = (
            summary[f"{label}_gap"] *
            shrinkage
        )

    summary = summary.rename(
        columns={player_column: "player_id"}
    )

    return summary, player_strata


print(
    "Peer reference excludes the evaluated "
    "player's own IPL deliveries."
)

print(
    "Matching: IPL season, innings and match phase."
)

print(
    "Shrinkage: 60 pseudo-balls for batting; "
    "120 pseudo-balls for bowling."
)


# ============================================================
# 4. CALCULATE BATTING AND BOWLING SEASON PROFILES
# ============================================================

print("\n4. RECENT IPL SEASON PERFORMANCE")
print("=" * 70)

(
    batting_season_profiles,
    batting_season_strata
) = calculate_season_peer_profiles(
    events=recent_batting_events,
    player_column="striker_id",
    outcomes={"runs": "batter_runs"},
    directions={"runs": 1},
    pseudo_balls=60
)

(
    bowling_season_profiles,
    bowling_season_strata
) = calculate_season_peer_profiles(
    events=recent_bowling_events,
    player_column="bowler_id",
    outcomes={
        "dots": "clean_dot_proxy",
        "boundaries": "batter_boundary_proxy",
        "wickets": "credited_wicket_proxy"
    },
    directions={
        "dots": 1,
        "boundaries": -1,
        "wickets": 1
    },
    pseudo_balls=120
)

print("\nSeason-specific batting profile coverage")

display(
    batting_season_profiles.groupby("season_year")
    .agg(
        players=("player_id", "nunique"),
        raw_balls=("raw_balls", "sum"),
        matched_balls=("matched_balls", "sum")
    )
    .reset_index()
)

print("\nSeason-specific bowling profile coverage")

display(
    bowling_season_profiles.groupby("season_year")
    .agg(
        players=("player_id", "nunique"),
        raw_balls=("raw_balls", "sum"),
        matched_balls=("matched_balls", "sum")
    )
    .reset_index()
)


# ============================================================
# 5. COMBINE PREVIOUS AND LATEST SEASON
# ============================================================

print("\n5. TWO-SEASON PLAYER RESEARCH MATRIX")
print("=" * 70)


def attach_season_profile(
    base,
    profiles,
    year,
    prefix
):

    selection = (
        profiles.loc[
            profiles["season_year"].eq(year)
        ]
        .drop(columns=["season_year"])
        .copy()
    )

    selection = selection.rename(
        columns={
            column: f"{prefix}_{column}"
            for column in selection.columns
            if column != "player_id"
        }
    )

    return base.merge(
        selection,
        on="player_id",
        how="left",
        validate="one_to_one"
    )


recent_ipl_research_matrix = (
    context_research_matrix.copy()
)

for profiles, role in [
    (batting_season_profiles, "bat"),
    (bowling_season_profiles, "bowl")
]:

    recent_ipl_research_matrix = attach_season_profile(
        recent_ipl_research_matrix,
        profiles,
        previous_year,
        f"previous_{role}"
    )

    recent_ipl_research_matrix = attach_season_profile(
        recent_ipl_research_matrix,
        profiles,
        latest_year,
        f"latest_{role}"
    )

for prefix in [
    "previous_bat",
    "latest_bat",
    "previous_bowl",
    "latest_bowl"
]:

    for field in [
        "raw_balls",
        "raw_matches",
        "matched_balls"
    ]:

        column = f"{prefix}_{field}"

        recent_ipl_research_matrix[column] = (
            recent_ipl_research_matrix[column]
            .fillna(0)
            .astype(int)
        )

print(
    "Players retained:",
    len(recent_ipl_research_matrix)
)

print(
    "Players with latest-season batting deliveries:",
    int(
        recent_ipl_research_matrix[
            "latest_bat_raw_balls"
        ].gt(0).sum()
    )
)

print(
    "Players with latest-season bowling deliveries:",
    int(
        recent_ipl_research_matrix[
            "latest_bowl_raw_balls"
        ].gt(0).sum()
    )
)


# ============================================================
# 6. RECENT IPL RESEARCH LEADS AND CONSISTENCY
# ============================================================

print("\n6. RECENT IPL RESEARCH LEADS")
print("=" * 70)

# No recorded T20I is a research grouping,
# not verified international uncapped status.

unverified_ipl_pool = (
    recent_ipl_research_matrix.loc[
        recent_ipl_research_matrix[
            "observed_t20i_matches"
        ].eq(0)
    ]
    .copy()
)

recent_batting_leads = (
    unverified_ipl_pool.loc[
        unverified_ipl_pool[
            "latest_bat_matched_balls"
        ].ge(60) &
        unverified_ipl_pool[
            "latest_bat_raw_matches"
        ].ge(3) &
        unverified_ipl_pool[
            "latest_bat_peer_coverage_pct"
        ].ge(95)
    ]
    .copy()
)

recent_bowling_leads = (
    unverified_ipl_pool.loc[
        unverified_ipl_pool[
            "latest_bowl_matched_balls"
        ].ge(120) &
        unverified_ipl_pool[
            "latest_bowl_raw_matches"
        ].ge(5) &
        unverified_ipl_pool[
            "latest_bowl_peer_coverage_pct"
        ].ge(95)
    ]
    .copy()
)

bat_prior_sufficient = (
    recent_batting_leads[
        "previous_bat_matched_balls"
    ].ge(60) &
    recent_batting_leads[
        "previous_bat_raw_matches"
    ].ge(3)
)

recent_batting_leads["season_consistency"] = (
    np.select(
        [
            bat_prior_sufficient &
            recent_batting_leads[
                "latest_bat_runs_gap_shrunk"
            ].gt(0) &
            recent_batting_leads[
                "previous_bat_runs_gap_shrunk"
            ].gt(0),

            bat_prior_sufficient
        ],
        [
            "Positive peer gap in both seasons",
            "Mixed or nonpositive peer gaps"
        ],
        default="Insufficient previous-season sample"
    )
)

bowl_prior_sufficient = (
    recent_bowling_leads[
        "previous_bowl_matched_balls"
    ].ge(120) &
    recent_bowling_leads[
        "previous_bowl_raw_matches"
    ].ge(5)
)

recent_bowling_leads["season_consistency"] = (
    np.select(
        [
            bowl_prior_sufficient &
            recent_bowling_leads[
                "latest_bowl_boundaries_gap_shrunk"
            ].gt(0) &
            recent_bowling_leads[
                "previous_bowl_boundaries_gap_shrunk"
            ].gt(0),

            bowl_prior_sufficient
        ],
        [
            "Positive boundary suppression in both seasons",
            "Mixed or nonpositive suppression"
        ],
        default="Insufficient previous-season sample"
    )
)

recent_batting_leads = (
    recent_batting_leads.sort_values(
        "latest_bat_runs_gap_shrunk",
        ascending=False
    )
)

recent_bowling_leads = (
    recent_bowling_leads.sort_values(
        "latest_bowl_boundaries_gap_shrunk",
        ascending=False
    )
)

print(
    f"Latest season: {latest_year}"
)

print(
    "Batting leads meeting recent IPL sample thresholds:",
    len(recent_batting_leads)
)

print(
    "Bowling leads meeting recent IPL sample thresholds:",
    len(recent_bowling_leads)
)

print("\nBatting — recent IPL peer comparisons")

display(
    recent_batting_leads[
        [
            "player_id",
            "player_name",
            "latest_bat_raw_balls",
            "latest_bat_raw_matches",
            "latest_bat_runs_gap_shrunk",
            "previous_bat_raw_balls",
            "previous_bat_runs_gap_shrunk",
            "season_consistency",
            "batting_evidence_tier"
        ]
    ].head(20)
)

print("\nBowling — recent IPL peer comparisons")

display(
    recent_bowling_leads[
        [
            "player_id",
            "player_name",
            "latest_bowl_raw_balls",
            "latest_bowl_raw_matches",
            "latest_bowl_dots_gap_shrunk",
            "latest_bowl_boundaries_gap_shrunk",
            "latest_bowl_wickets_gap_shrunk",
            "previous_bowl_boundaries_gap_shrunk",
            "season_consistency",
            "bowling_evidence_tier"
        ]
    ].head(20)
)

print("\nBatting season-consistency groups")

display(
    recent_batting_leads[
        "season_consistency"
    ].value_counts()
)

print("\nBowling season-consistency groups")

display(
    recent_bowling_leads[
        "season_consistency"
    ].value_counts()
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

batting_matched_total = int(
    batting_season_profiles[
        "matched_balls"
    ].sum()
)

bowling_matched_total = int(
    bowling_season_profiles[
        "matched_balls"
    ].sum()
)

checks = {
    "duplicate_ipl_match_ids": int(
        ipl_season_lookup["match_id"]
        .duplicated().sum()
    ),

    "missing_ipl_season_years": int(
        ipl_season_lookup[
            "season_year"
        ].isna().sum()
    ),

    "missing_batting_season_labels": int(
        recent_batting_events[
            "season_year"
        ].isna().sum()
    ),

    "missing_bowling_season_labels": int(
        recent_bowling_events[
            "season_year"
        ].isna().sum()
    ),

    "batting_season_strata_reconciliation": abs(
        int(
            batting_season_strata[
                "stratum_balls"
            ].sum()
        ) -
        len(recent_batting_events)
    ),

    "bowling_season_strata_reconciliation": abs(
        int(
            bowling_season_strata[
                "stratum_balls"
            ].sum()
        ) -
        len(recent_bowling_events)
    ),

    "batting_season_profile_reconciliation": abs(
        int(
            batting_season_profiles[
                "raw_balls"
            ].sum()
        ) -
        len(recent_batting_events)
    ),

    "bowling_season_profile_reconciliation": abs(
        int(
            bowling_season_profiles[
                "raw_balls"
            ].sum()
        ) -
        len(recent_bowling_events)
    ),

    "batting_matched_exceeds_raw": int(
        (
            batting_season_profiles[
                "matched_balls"
            ] >
            batting_season_profiles[
                "raw_balls"
            ]
        ).sum()
    ),

    "bowling_matched_exceeds_raw": int(
        (
            bowling_season_profiles[
                "matched_balls"
            ] >
            bowling_season_profiles[
                "raw_balls"
            ]
        ).sum()
    ),

    "batting_matched_total_exceeds_events": int(
        batting_matched_total >
        len(recent_batting_events)
    ),

    "bowling_matched_total_exceeds_events": int(
        bowling_matched_total >
        len(recent_bowling_events)
    ),

    "missing_registry_players": abs(
        len(recent_ipl_research_matrix) -
        len(context_research_matrix)
    ),

    "duplicate_research_player_ids": int(
        recent_ipl_research_matrix[
            "player_id"
        ].duplicated().sum()
    ),

    "batting_leads_with_recorded_t20i": int(
        recent_batting_leads[
            "observed_t20i_matches"
        ].gt(0).sum()
    ),

    "bowling_leads_with_recorded_t20i": int(
        recent_bowling_leads[
            "observed_t20i_matches"
        ].gt(0).sum()
    )
}

recent_ipl_integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(recent_ipl_integrity_report)

if recent_ipl_integrity_report["count"].eq(0).all():
    print("\nALL 16 INTEGRITY CHECKS PASSED")
else:
    print("\nWARNING: REVIEW NONZERO INTEGRITY CHECKS")

print("\nINTERPRETATION")
print("=" * 70)

print(
    "Season comparisons use only the two latest "
    "IPL seasons available in the data."
)

print(
    "Peer benchmarks match season, innings "
    "and match phase, excluding each player's "
    "own deliveries."
)

print(
    "Season-to-season agreement is descriptive, "
    "not a formal predictive validation."
)

print(
    "No player has been confirmed young, "
    "internationally uncapped or auction-eligible."
)

print(
    "The full scouting database remains intact."
)

In [ ]:
# ============================================================
# CELL 71 — 2027 AUCTION SCOUTING VERIFICATION QUEUE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

print("=" * 70)
print("2027 AUCTION SCOUTING VERIFICATION QUEUE")
print("=" * 70)


# ============================================================
# 1. VERIFY INPUTS
# ============================================================

print("\n1. INPUT VERIFICATION")
print("=" * 70)

required_variables = [
    "recent_ipl_research_matrix",
    "recent_batting_leads",
    "recent_bowling_leads",
    "previous_year",
    "latest_year"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing previous-cell variables: {missing_variables}"
)

bat_leads = recent_batting_leads.reset_index(
    drop=True
).copy()

bowl_leads = recent_bowling_leads.reset_index(
    drop=True
).copy()

metadata_columns = [
    "player_id",
    "player_name",
    "observed_t20i_matches",
    "observed_ipl_matches",
    "identity_collision_flag",
    "verified_date_of_birth",
    "verified_cap_status",
    "age_eligibility_status",
    "eligibility_status"
]

assert set(metadata_columns).issubset(
    recent_ipl_research_matrix.columns
)

assert recent_ipl_research_matrix["player_id"].is_unique

print("Registered players:", len(recent_ipl_research_matrix))
print("Batting research leads:", len(bat_leads))
print("Bowling research leads:", len(bowl_leads))
print("Seasons:", previous_year, "and", latest_year)


# ============================================================
# 2. CREATE STANDARDIZED BATTING DOSSIERS
# ============================================================

print("\n2. BATTING RESEARCH DOSSIERS")
print("=" * 70)

batting_role_dossiers = pd.DataFrame({
    "player_id": bat_leads["player_id"],
    "research_role": "Batting",
    "latest_ipl_balls": (
        bat_leads["latest_bat_matched_balls"]
    ),
    "latest_ipl_matches": (
        bat_leads["latest_bat_raw_matches"]
    ),
    "previous_ipl_balls": (
        bat_leads["previous_bat_matched_balls"]
    ),
    "latest_primary_peer_gap": (
        bat_leads["latest_bat_runs_gap_shrunk"]
    ),
    "previous_primary_peer_gap": (
        bat_leads["previous_bat_runs_gap_shrunk"]
    ),
    "latest_dot_gap_pp": np.nan,
    "latest_wicket_gap_pp": np.nan,
    "season_consistency": (
        bat_leads["season_consistency"]
    ),
    "pressure_evidence_tier": (
        bat_leads["batting_evidence_tier"]
    ),
    "historical_positive_pressure_signal": (
        bat_leads["batting_positive_contexts"].gt(0)
    ),
    "primary_metric": (
        "Batting SR advantage per 100 balls"
    )
})

print("Batting dossiers created:", len(batting_role_dossiers))


# ============================================================
# 3. CREATE STANDARDIZED BOWLING DOSSIERS
# ============================================================

print("\n3. BOWLING RESEARCH DOSSIERS")
print("=" * 70)

bowling_role_dossiers = pd.DataFrame({
    "player_id": bowl_leads["player_id"],
    "research_role": "Bowling",
    "latest_ipl_balls": (
        bowl_leads["latest_bowl_matched_balls"]
    ),
    "latest_ipl_matches": (
        bowl_leads["latest_bowl_raw_matches"]
    ),
    "previous_ipl_balls": (
        bowl_leads["previous_bowl_matched_balls"]
    ),
    "latest_primary_peer_gap": (
        bowl_leads["latest_bowl_boundaries_gap_shrunk"]
    ),
    "previous_primary_peer_gap": (
        bowl_leads[
            "previous_bowl_boundaries_gap_shrunk"
        ]
    ),
    "latest_dot_gap_pp": (
        bowl_leads["latest_bowl_dots_gap_shrunk"]
    ),
    "latest_wicket_gap_pp": (
        bowl_leads["latest_bowl_wickets_gap_shrunk"]
    ),
    "season_consistency": (
        bowl_leads["season_consistency"]
    ),
    "pressure_evidence_tier": (
        bowl_leads["bowling_evidence_tier"]
    ),
    "historical_positive_pressure_signal": (
        bowl_leads["bowling_positive_metrics"].gt(0)
    ),
    "primary_metric": (
        "Boundary suppression, percentage points"
    )
})

print("Bowling dossiers created:", len(bowling_role_dossiers))


# ============================================================
# 4. COMBINE DOSSIERS AND DEFINE RESEARCH PRIORITIES
# ============================================================

print("\n4. EVIDENCE-BASED RESEARCH PRIORITIES")
print("=" * 70)

role_dossiers = pd.concat(
    [
        batting_role_dossiers,
        bowling_role_dossiers
    ],
    ignore_index=True
)

player_identity = recent_ipl_research_matrix[
    metadata_columns
].copy()

role_dossiers = role_dossiers.merge(
    player_identity,
    on="player_id",
    how="left",
    validate="many_to_one"
)

role_dossiers["latest_primary_positive"] = (
    role_dossiers[
        "latest_primary_peer_gap"
    ].gt(0)
)

role_dossiers["latest_secondary_positive"] = (
    role_dossiers[
        "latest_dot_gap_pp"
    ].gt(0) |
    role_dossiers[
        "latest_wicket_gap_pp"
    ].gt(0)
)

role_dossiers["two_season_primary_positive"] = (
    role_dossiers[
        "season_consistency"
    ].str.startswith("Positive")
)

role_dossiers["previous_sample_limited"] = (
    role_dossiers[
        "season_consistency"
    ].eq("Insufficient previous-season sample")
)

# PRIORITIES ARE FOR VERIFICATION WORKLOAD ONLY.
# Different batting and bowling metrics are NOT
# placed on one numerical performance scale.

priority_one = (
    role_dossiers["latest_primary_positive"] &
    role_dossiers["two_season_primary_positive"]
)

priority_two = (
    role_dossiers["latest_primary_positive"] |
    role_dossiers["latest_secondary_positive"] |
    role_dossiers[
        "historical_positive_pressure_signal"
    ]
)

role_dossiers["research_priority_number"] = (
    np.select(
        [
            priority_one,
            priority_two
        ],
        [
            1,
            2
        ],
        default=3
    ).astype(int)
)

priority_labels = {
    1: "P1 | Repeat positive signal",
    2: "P2 | Recent or context-specific signal",
    3: "P3 | Additional scouting review"
}

role_dossiers["research_priority"] = (
    role_dossiers[
        "research_priority_number"
    ].map(priority_labels)
)

role_dossiers["official_eligibility_review"] = (
    "Pending DOB, cap-status and rule verification"
)

role_dossiers["data_scope_note"] = (
    "Latest IPL sample; descriptive peer comparison"
)

print("\nResearch priorities by role")

display(
    pd.crosstab(
        role_dossiers["research_role"],
        role_dossiers["research_priority"]
    )
)

print("\nRepeated positive batting signals")

display(
    role_dossiers.loc[
        (role_dossiers["research_role"] == "Batting") &
        role_dossiers["two_season_primary_positive"],
        [
            "player_id",
            "player_name",
            "latest_ipl_balls",
            "latest_primary_peer_gap",
            "previous_primary_peer_gap",
            "pressure_evidence_tier",
            "research_priority"
        ]
    ].sort_values(
        "latest_primary_peer_gap",
        ascending=False
    )
)

print("\nRepeated positive bowling boundary signals")

display(
    role_dossiers.loc[
        (role_dossiers["research_role"] == "Bowling") &
        role_dossiers["two_season_primary_positive"],
        [
            "player_id",
            "player_name",
            "latest_ipl_balls",
            "latest_primary_peer_gap",
            "previous_primary_peer_gap",
            "latest_dot_gap_pp",
            "latest_wicket_gap_pp",
            "research_priority"
        ]
    ].sort_values(
        "latest_primary_peer_gap",
        ascending=False
    )
)


# ============================================================
# 5. CREATE UNIQUE-PLAYER ELIGIBILITY VERIFICATION QUEUE
# ============================================================

print("\n5. OFFICIAL ELIGIBILITY VERIFICATION QUEUE")
print("=" * 70)

verification_batch = (
    role_dossiers.groupby(
        "player_id",
        as_index=False
    )
    .agg(
        research_roles=(
            "research_role",
            lambda values: " / ".join(
                sorted(set(values))
            )
        ),
        role_dossier_count=("research_role", "size"),
        research_priority_number=(
            "research_priority_number", "min"
        ),
        latest_primary_positive_roles=(
            "latest_primary_positive", "sum"
        ),
        historical_pressure_signals=(
            "historical_positive_pressure_signal",
            "sum"
        )
    )
    .merge(
        player_identity,
        on="player_id",
        how="left",
        validate="one_to_one"
    )
)

verification_batch["research_priority"] = (
    verification_batch[
        "research_priority_number"
    ].map(priority_labels)
)

verification_batch["dob_source_url"] = ""
verification_batch["cap_status_source_url"] = ""
verification_batch["eligibility_rules_source"] = ""

verification_batch["official_eligibility_result"] = (
    "Pending verification"
)

verification_batch["scouting_notes"] = ""

verification_batch = (
    verification_batch.sort_values(
        [
            "research_priority_number",
            "player_name",
            "player_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Role-specific dossiers:",
    len(role_dossiers)
)

print(
    "Unique players needing verification:",
    len(verification_batch)
)

print(
    "Players with identity-collision warnings:",
    int(
        verification_batch[
            "identity_collision_flag"
        ].sum()
    )
)

print("\nUnique-player verification priorities")

display(
    verification_batch[
        "research_priority"
    ].value_counts()
    .rename_axis("priority")
    .reset_index(name="players")
)

print("\nFirst verification batch — NOT A FINAL SHORTLIST")

display(
    verification_batch[
        [
            "player_id",
            "player_name",
            "research_roles",
            "research_priority",
            "observed_t20i_matches",
            "verified_cap_status",
            "age_eligibility_status",
            "identity_collision_flag"
        ]
    ].head(30)
)


# ============================================================
# 6. EXPORT THE TWO WORKING CSV FILES
# ============================================================

print("\n6. EXPORT SCOUTING WORKING FILES")
print("=" * 70)

role_dossier_file = Path(
    "rr_round1_role_research_dossiers.csv"
)

verification_file = Path(
    "rr_round1_eligibility_verification_queue.csv"
)

role_dossiers.to_csv(
    role_dossier_file,
    index=False
)

verification_batch.to_csv(
    verification_file,
    index=False
)

print(
    "Role dossiers:",
    role_dossier_file.resolve()
)

print(
    "Eligibility verification queue:",
    verification_file.resolve()
)

print(
    "\nThese CSV files are working research files. "
    "They do not establish official eligibility."
)


# ============================================================
# 7. DATA INTEGRITY CHECKS
# ============================================================

print("\n7. DATA INTEGRITY CHECKS")
print("=" * 70)

registry_ids = set(
    recent_ipl_research_matrix["player_id"]
)

dossier_ids = set(
    role_dossiers["player_id"]
)

verification_ids = set(
    verification_batch["player_id"]
)

checks = {
    "registered_player_count_difference": abs(
        len(recent_ipl_research_matrix) - 969
    ),

    "batting_dossier_count_difference": abs(
        int(
            role_dossiers[
                "research_role"
            ].eq("Batting").sum()
        ) - len(bat_leads)
    ),

    "bowling_dossier_count_difference": abs(
        int(
            role_dossiers[
                "research_role"
            ].eq("Bowling").sum()
        ) - len(bowl_leads)
    ),

    "duplicate_player_role_dossiers": int(
        role_dossiers.duplicated(
            ["player_id", "research_role"]
        ).sum()
    ),

    "dossier_ids_missing_from_registry": len(
        dossier_ids - registry_ids
    ),

    "verification_ids_missing_from_dossiers": len(
        verification_ids - dossier_ids
    ),

    "dossier_ids_missing_from_verification": len(
        dossier_ids - verification_ids
    ),

    "duplicate_verification_player_ids": int(
        verification_batch[
            "player_id"
        ].duplicated().sum()
    ),

    "unique_player_count_mismatch": abs(
        len(verification_batch) -
        len(dossier_ids)
    ),

    "missing_dossier_player_names": int(
        role_dossiers[
            "player_name"
        ].isna().sum()
    ),

    "missing_research_priorities": int(
        role_dossiers[
            "research_priority"
        ].isna().sum()
    ),

    "missing_recent_primary_metrics": int(
        role_dossiers[
            "latest_primary_peer_gap"
        ].isna().sum()
    ),

    "recorded_t20i_in_initial_batch": int(
        verification_batch[
            "observed_t20i_matches"
        ].gt(0).sum()
    ),

    "unverified_dob_incorrectly_filled": int(
        role_dossiers[
            "verified_date_of_birth"
        ].notna().sum()
    ),

    "unexpected_verified_cap_status": int(
        role_dossiers[
            "verified_cap_status"
        ].ne("Not verified").sum()
    ),

    "negative_latest_exposure": int(
        role_dossiers[
            "latest_ipl_balls"
        ].lt(0).sum()
    ),

    "missing_exported_files": int(
        not role_dossier_file.exists()
    ) + int(
        not verification_file.exists()
    )
}

verification_integrity_report = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(verification_integrity_report)

if verification_integrity_report["count"].eq(0).all():

    print("\nALL 17 INTEGRITY CHECKS PASSED")

else:

    print("\nWARNING: REVIEW NONZERO CHECKS")

print("\nINTERPRETATION")
print("=" * 70)

print(
    "P1, P2 and P3 are research-verification "
    "priorities, not talent rankings."
)

print(
    "The initial batch is limited to players "
    "meeting recent IPL exposure thresholds."
)

print(
    "Players outside this batch remain in the "
    "full 969-player scouting database."
)

print(
    "No recorded T20I appearance does NOT "
    "prove that a player is uncapped."
)

print(
    "Official date of birth, international "
    "cap status and Round 1 eligibility "
    "must be independently verified."
)

print(
    "No final Top 10 has been selected."
)

In [ ]:
# CELL 72 — Pressure Model Ranking Readiness

import pandas as pd
from IPython.display import display

print("=" * 70)
print("ROUND 1 — PRESSURE MODEL RANKING READINESS")
print("=" * 70)

# ============================================================
# 1. IDENTIFY AVAILABLE PLAYER-LEVEL MODEL TABLES
# ============================================================

model_tables = [
    "player_registry",
    "batting_reliability_profiles",
    "bowling_reliability_profiles",
    "player_performance_matrix",
    "scouting_review_queue"
]

table_status = []

for name in model_tables:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):

        table_status.append({
            "table": name,
            "available": True,
            "rows": len(obj),
            "columns": len(obj.columns)
        })

    else:

        table_status.append({
            "table": name,
            "available": False,
            "rows": None,
            "columns": None
        })

print("\n1. MODEL TABLE AVAILABILITY")
print("=" * 70)

display(pd.DataFrame(table_status))


# ============================================================
# 2. INSPECT BATTING MODEL COLUMNS
# ============================================================

print("\n2. BATTING PRESSURE MODEL COLUMNS")
print("=" * 70)

bat = globals().get("batting_reliability_profiles")

if isinstance(bat, pd.DataFrame):

    print("Profiles:", len(bat))

    for i, column in enumerate(bat.columns, 1):
        print(f"{i:02d}. {column}")

else:
    print("Batting reliability table not found.")


# ============================================================
# 3. INSPECT BOWLING MODEL COLUMNS
# ============================================================

print("\n3. BOWLING PRESSURE MODEL COLUMNS")
print("=" * 70)

bowl = globals().get("bowling_reliability_profiles")

if isinstance(bowl, pd.DataFrame):

    print("Profiles:", len(bowl))

    for i, column in enumerate(bowl.columns, 1):
        print(f"{i:02d}. {column}")

else:
    print("Bowling reliability table not found.")


# ============================================================
# 4. CHECK PLAYER ID COVERAGE
# ============================================================

print("\n4. PLAYER IDENTIFIER COVERAGE")
print("=" * 70)

id_checks = []

for name in [
    "batting_reliability_profiles",
    "bowling_reliability_profiles"
]:

    df = globals().get(name)

    if not isinstance(df, pd.DataFrame):
        continue

    id_columns = [
        col for col in df.columns
        if col in [
            "player_id",
            "striker_id",
            "batter_id",
            "bowler_id"
        ]
    ]

    id_checks.append({
        "table": name,
        "player_id_columns": ", ".join(id_columns)
            if id_columns else "Not identified",
        "profiles": len(df)
    })

display(pd.DataFrame(id_checks))


# ============================================================
# 5. READINESS SUMMARY
# ============================================================

print("\n5. NEXT STAGE")
print("=" * 70)

print("Next objective:")
print("Build evidence-based pressure rankings for batters and bowlers.")

print("\nRanking requirements:")
print("- Performance relative to personal baseline")
print("- Pressure context")
print("- Exposure and reliability")
print("- Uncertainty")
print("- Independent batting and bowling evaluation")

print("\nNo players have been selected yet.")
print("No existing source data has been modified.")

In [ ]:
# CELL 73 — Verify Source-Adjusted Pressure Metrics

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("PRESSURE PERFORMANCE — MATHEMATICAL VALIDATION")
print("=" * 70)

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()


# ============================================================
# 1. FUNCTION TO VERIFY PERFORMANCE METRICS
# ============================================================

def verify_pressure_metric(
    df,
    metric_name,
    actual_column,
    expected_column,
    denominator_column,
    reported_column,
    direction="higher"
):

    actual = pd.to_numeric(
        df[actual_column], errors="coerce"
    )

    expected = pd.to_numeric(
        df[expected_column], errors="coerce"
    )

    denominator = pd.to_numeric(
        df[denominator_column], errors="coerce"
    )

    reported = pd.to_numeric(
        df[reported_column], errors="coerce"
    )

    valid = (
        actual.notna() &
        expected.notna() &
        denominator.gt(0) &
        reported.notna()
    )

    if direction == "higher":

        calculated = (
            100 * (actual - expected) / denominator
        )

    else:

        calculated = (
            100 * (expected - actual) / denominator
        )

    difference = (
        calculated.loc[valid] -
        reported.loc[valid]
    ).abs()

    return {
        "metric": metric_name,
        "total_profiles": len(df),
        "comparable_profiles": int(valid.sum()),
        "unverifiable_profiles": int((~valid).sum()),
        "disagreements_above_0.001": int(
            (difference > 0.001).sum()
        ),
        "maximum_difference": (
            round(float(difference.max()), 6)
            if len(difference) else np.nan
        )
    }


# ============================================================
# 2. VERIFY BATTING STRIKE-RATE UPLIFT
# ============================================================

print("\n1. BATTING PERFORMANCE VALIDATION")
print("=" * 70)

batting_check = verify_pressure_metric(
    df=bat,
    metric_name="Source-adjusted SR uplift",
    actual_column="source_actual_runs",
    expected_column="source_expected_runs",
    denominator_column="source_matched_balls",
    reported_column="source_adjusted_sr_uplift",
    direction="higher"
)

display(pd.DataFrame([batting_check]))


# ============================================================
# 3. VERIFY BOWLING PERFORMANCE METRICS
# ============================================================

print("\n2. BOWLING PERFORMANCE VALIDATION")
print("=" * 70)

bowling_checks = []

bowling_checks.append(
    verify_pressure_metric(
        df=bowl,
        metric_name="Clean-dot uplift (pp)",
        actual_column="actual_clean_dots_y",
        expected_column="source_expected_clean_dots",
        denominator_column="source_matched_balls",
        reported_column="source_adjusted_clean_dot_uplift_pp",
        direction="higher"
    )
)

bowling_checks.append(
    verify_pressure_metric(
        df=bowl,
        metric_name="Boundary suppression (pp)",
        actual_column="actual_boundaries_y",
        expected_column="source_expected_boundaries",
        denominator_column="source_matched_balls",
        reported_column="source_adjusted_boundary_suppression_pp",
        direction="lower"
    )
)

bowling_checks.append(
    verify_pressure_metric(
        df=bowl,
        metric_name="Wicket-rate uplift (pp)",
        actual_column="actual_wickets_y",
        expected_column="source_expected_wickets",
        denominator_column="source_matched_balls",
        reported_column="source_adjusted_wicket_rate_uplift_pp",
        direction="higher"
    )
)

bowling_validation = pd.DataFrame(bowling_checks)

display(bowling_validation)


# ============================================================
# 4. CHECK RELIABILITY COVERAGE
# ============================================================

print("\n3. RELIABILITY COVERAGE")
print("=" * 70)

reliability_results = []

for role, df in [
    ("Batting", bat),
    ("Bowling", bowl)
]:

    reliable = (
        df["source_sample_reliable"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    player_id = (
        "striker_id"
        if role == "Batting"
        else "bowler_id"
    )

    reliability_results.append({
        "role": role,
        "profiles": len(df),
        "unique_players": df[player_id].nunique(),
        "reliable_profiles": int(reliable.sum()),
        "players_with_reliable_profile": (
            df.loc[reliable, player_id].nunique()
        ),
        "total_source_matched_balls": (
            pd.to_numeric(
                df["source_matched_balls"],
                errors="coerce"
            ).sum()
        )
    })

display(pd.DataFrame(reliability_results))


# ============================================================
# 5. VALIDATION DECISION
# ============================================================

print("\n4. VALIDATION DECISION")
print("=" * 70)

validation_results = pd.DataFrame(
    [batting_check] + bowling_checks
)

total_disagreements = int(
    validation_results[
        "disagreements_above_0.001"
    ].sum()
)

total_unverifiable = int(
    validation_results[
        "unverifiable_profiles"
    ].sum()
)

print("Metric disagreements:", total_disagreements)
print("Unverifiable profiles:", total_unverifiable)

if total_disagreements == 0 and total_unverifiable == 0:

    print("\nPASS: All source-adjusted metrics reconcile.")
    print("Ready for player-level evidence consolidation.")

else:

    print("\nREVIEW REQUIRED:")
    print("Investigate differences before combining")
    print("pressure profiles into player rankings.")

print("\nOriginal model tables remain unchanged.")
print("No final player rankings have been assigned.")

In [ ]:
# CELL 74 — Player-Level Pressure Evidence Consolidation

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("PLAYER-LEVEL PRESSURE EVIDENCE CONSOLIDATION")
print("=" * 70)


# ============================================================
# 1. PREPARE RELIABLE PRESSURE PROFILES
# ============================================================

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()

bat["_reliable"] = (
    bat["source_sample_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

bowl["_reliable"] = (
    bowl["source_sample_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

bat_reliable = bat.loc[bat["_reliable"]].copy()
bowl_reliable = bowl.loc[bowl["_reliable"]].copy()

print("\n1. RELIABLE EVIDENCE INPUT")
print("=" * 70)

print("Batting profiles:", len(bat_reliable))
print("Bowling profiles:", len(bowl_reliable))


# ============================================================
# 2. CONSOLIDATE BATTING
# ============================================================

print("\n2. BATTING PLAYER-LEVEL EVIDENCE")
print("=" * 70)

batting_player_evidence = (
    bat_reliable
    .groupby("striker_id", as_index=False)
    .agg(
        player_name=("player_name", "first"),
        reliable_profiles=("batting_pressure_type", "size"),
        pressure_contexts=("batting_pressure_type", "nunique"),
        matched_balls=("source_matched_balls", "sum"),
        observed_runs=("source_actual_runs", "sum"),
        expected_runs=("source_expected_runs", "sum")
    )
)

batting_player_evidence["runs_above_expected"] = (
    batting_player_evidence["observed_runs"] -
    batting_player_evidence["expected_runs"]
)

batting_player_evidence["pressure_sr_uplift"] = (
    100 *
    batting_player_evidence["runs_above_expected"] /
    batting_player_evidence["matched_balls"]
)

batting_player_evidence = (
    batting_player_evidence.sort_values(
        ["pressure_sr_uplift", "matched_balls"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

batting_player_evidence["exploratory_position"] = (
    np.arange(1, len(batting_player_evidence) + 1)
)

print(
    "Players with reliable batting evidence:",
    len(batting_player_evidence)
)

display(
    batting_player_evidence[
        [
            "exploratory_position",
            "player_name",
            "reliable_profiles",
            "pressure_contexts",
            "matched_balls",
            "observed_runs",
            "expected_runs",
            "pressure_sr_uplift"
        ]
    ].head(20).round(2)
)


# ============================================================
# 3. CONSOLIDATE BOWLING
# ============================================================

print("\n3. BOWLING PLAYER-LEVEL EVIDENCE")
print("=" * 70)

bowling_player_evidence = (
    bowl_reliable
    .groupby("bowler_id", as_index=False)
    .agg(
        player_name=("player_name", "first"),
        reliable_profiles=("bowling_pressure_type", "size"),
        pressure_contexts=("bowling_pressure_type", "nunique"),
        matched_balls=("source_matched_balls", "sum"),

        actual_dots=("actual_clean_dots_y", "sum"),
        expected_dots=("source_expected_clean_dots", "sum"),

        actual_boundaries=("actual_boundaries_y", "sum"),
        expected_boundaries=("source_expected_boundaries", "sum"),

        actual_wickets=("actual_wickets_y", "sum"),
        expected_wickets=("source_expected_wickets", "sum")
    )
)

n = bowling_player_evidence["matched_balls"]

bowling_player_evidence["dot_uplift_pp"] = (
    100 *
    (
        bowling_player_evidence["actual_dots"] -
        bowling_player_evidence["expected_dots"]
    ) / n
)

bowling_player_evidence["boundary_suppression_pp"] = (
    100 *
    (
        bowling_player_evidence["expected_boundaries"] -
        bowling_player_evidence["actual_boundaries"]
    ) / n
)

bowling_player_evidence["wicket_uplift_pp"] = (
    100 *
    (
        bowling_player_evidence["actual_wickets"] -
        bowling_player_evidence["expected_wickets"]
    ) / n
)

print(
    "Players with reliable bowling evidence:",
    len(bowling_player_evidence)
)


# ============================================================
# 4. BOWLING — DOT-BALL LEADERS
# ============================================================

print("\n4. BOWLING CONTROL — DOT-BALL IMPROVEMENT")
print("=" * 70)

display(
    bowling_player_evidence.sort_values(
        ["dot_uplift_pp", "matched_balls"],
        ascending=[False, False]
    )[
        [
            "player_name",
            "reliable_profiles",
            "matched_balls",
            "dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_uplift_pp"
        ]
    ].head(10).round(3)
)


# ============================================================
# 5. BOWLING — BOUNDARY PREVENTION LEADERS
# ============================================================

print("\n5. BOWLING CONTROL — BOUNDARY SUPPRESSION")
print("=" * 70)

display(
    bowling_player_evidence.sort_values(
        ["boundary_suppression_pp", "matched_balls"],
        ascending=[False, False]
    )[
        [
            "player_name",
            "reliable_profiles",
            "matched_balls",
            "dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_uplift_pp"
        ]
    ].head(10).round(3)
)


# ============================================================
# 6. BOWLING — WICKET-TAKING LEADERS
# ============================================================

print("\n6. BOWLING ATTACK — WICKET-RATE IMPROVEMENT")
print("=" * 70)

display(
    bowling_player_evidence.sort_values(
        ["wicket_uplift_pp", "matched_balls"],
        ascending=[False, False]
    )[
        [
            "player_name",
            "reliable_profiles",
            "matched_balls",
            "dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_uplift_pp"
        ]
    ].head(10).round(3)
)


# ============================================================
# 7. CONSOLIDATION INTEGRITY
# ============================================================

print("\n7. CONSOLIDATION INTEGRITY")
print("=" * 70)

checks = {
    "batting_player_count_difference": (
        len(batting_player_evidence) -
        bat_reliable["striker_id"].nunique()
    ),

    "bowling_player_count_difference": (
        len(bowling_player_evidence) -
        bowl_reliable["bowler_id"].nunique()
    ),

    "batting_ball_count_difference": (
        batting_player_evidence["matched_balls"].sum() -
        bat_reliable["source_matched_balls"].sum()
    ),

    "bowling_ball_count_difference": (
        bowling_player_evidence["matched_balls"].sum() -
        bowl_reliable["source_matched_balls"].sum()
    ),

    "duplicate_batting_players": (
        batting_player_evidence["striker_id"].duplicated().sum()
    ),

    "duplicate_bowling_players": (
        bowling_player_evidence["bowler_id"].duplicated().sum()
    ),

    "invalid_batting_denominators": (
        batting_player_evidence["matched_balls"].le(0).sum()
    ),

    "invalid_bowling_denominators": (
        bowling_player_evidence["matched_balls"].le(0).sum()
    )
}

check_df = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(check_df)

if np.allclose(check_df["count"], 0):

    print("\nALL CONSOLIDATION CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED — DO NOT FINALIZE RANKINGS")


print("\nIMPORTANT INTERPRETATION")
print("=" * 70)

print(
    "These are exploratory rankings based on observed "
    "source-adjusted performance."
)

print(
    "Only reliability-screened profiles were consolidated."
)

print(
    "Combined player-level uncertainty intervals "
    "have NOT yet been calculated."
)

print(
    "Pressure contexts must be checked for overlapping "
    "deliveries before treating combined exposure as independent."
)

print(
    "No final Top 10 selections have been made."
)

In [ ]:
# CELL 75 — Pressure Leader Uncertainty Investigation

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("PRESSURE LEADERS — UNCERTAINTY AND MATCH SUPPORT")
print("=" * 70)


# ============================================================
# 1. EXAMINE LEADING BATTERS
# ============================================================

print("\n1. BATTING LEADERS — PROFILE-LEVEL EVIDENCE")
print("=" * 70)

top_batter_ids = (
    batting_player_evidence
    .head(20)["striker_id"]
    .tolist()
)

batting_leader_profiles = (
    bat_reliable.loc[
        bat_reliable["striker_id"].isin(top_batter_ids)
    ]
    .copy()
)

batting_leader_profiles["interval_above_zero"] = (
    batting_leader_profiles["sr_source_lower_95"] > 0
)

batting_leader_profiles["interval_width"] = (
    batting_leader_profiles["sr_source_upper_95"] -
    batting_leader_profiles["sr_source_lower_95"]
)

batting_leader_profiles["exploratory_position"] = (
    batting_leader_profiles["striker_id"].map(
        batting_player_evidence.set_index(
            "striker_id"
        )["exploratory_position"]
    )
)

display(
    batting_leader_profiles.sort_values(
        [
            "exploratory_position",
            "source_matched_balls"
        ],
        ascending=[True, False]
    )[
        [
            "exploratory_position",
            "player_name",
            "batting_pressure_type",
            "source_matched_balls",
            "source_pressure_matches",
            "source_adjusted_sr_uplift",
            "sr_source_lower_95",
            "sr_source_upper_95",
            "interval_width",
            "interval_above_zero"
        ]
    ].round(2)
)


# ============================================================
# 2. BATTING UNCERTAINTY SUMMARY
# ============================================================

print("\n2. BATTING UNCERTAINTY SUMMARY")
print("=" * 70)

print(
    "Leading batters examined:",
    batting_leader_profiles["striker_id"].nunique()
)

print(
    "Reliable profiles examined:",
    len(batting_leader_profiles)
)

print(
    "Profiles with nominal 95% interval above zero:",
    int(
        batting_leader_profiles[
            "interval_above_zero"
        ].sum()
    )
)


# ============================================================
# 3. SELECT BOWLING LEADERS ACROSS ALL THREE METRICS
# ============================================================

print("\n3. BOWLING LEADER COVERAGE")
print("=" * 70)

bowling_leader_ids = set()

for metric in [
    "dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_uplift_pp"
]:

    leaders = bowling_player_evidence.nlargest(
        10, metric
    )

    bowling_leader_ids.update(
        leaders["bowler_id"].tolist()
    )

bowling_leader_profiles = (
    bowl_reliable.loc[
        bowl_reliable["bowler_id"].isin(
            bowling_leader_ids
        )
    ]
    .copy()
)

print(
    "Unique bowling leaders:",
    len(bowling_leader_ids)
)

print(
    "Reliable bowling profiles:",
    len(bowling_leader_profiles)
)


# ============================================================
# 4. IDENTIFY POSITIVE BOWLING INTERVALS
# ============================================================

print("\n4. BOWLING LEADERS — UNCERTAINTY")
print("=" * 70)

bowling_leader_profiles["dot_interval_positive"] = (
    bowling_leader_profiles[
        "clean_dot_source_lower_95"
    ] > 0
)

bowling_leader_profiles["boundary_interval_positive"] = (
    bowling_leader_profiles[
        "boundary_source_lower_95"
    ] > 0
)

bowling_leader_profiles["wicket_interval_positive"] = (
    bowling_leader_profiles[
        "wicket_source_lower_95"
    ] > 0
)

display(
    bowling_leader_profiles.sort_values(
        "source_matched_balls",
        ascending=False
    )[
        [
            "player_name",
            "bowling_pressure_type",
            "source_matched_balls",
            "source_pressure_matches",

            "source_adjusted_clean_dot_uplift_pp",
            "clean_dot_source_lower_95",

            "source_adjusted_boundary_suppression_pp",
            "boundary_source_lower_95",

            "source_adjusted_wicket_rate_uplift_pp",
            "wicket_source_lower_95",

            "dot_interval_positive",
            "boundary_interval_positive",
            "wicket_interval_positive"
        ]
    ].round(3)
)


# ============================================================
# 5. BOWLING UNCERTAINTY SUMMARY
# ============================================================

print("\n5. BOWLING UNCERTAINTY SUMMARY")
print("=" * 70)

summary = {
    "nominal_positive_dot_intervals": int(
        bowling_leader_profiles[
            "dot_interval_positive"
        ].sum()
    ),

    "nominal_positive_boundary_intervals": int(
        bowling_leader_profiles[
            "boundary_interval_positive"
        ].sum()
    ),

    "nominal_positive_wicket_intervals": int(
        bowling_leader_profiles[
            "wicket_interval_positive"
        ].sum()
    )
}

display(
    pd.DataFrame(
        list(summary.items()),
        columns=["measure", "profiles"]
    )
)


# ============================================================
# 6. PRESSURE CONTEXT BREADTH
# ============================================================

print("\n6. MULTIPLE-CONTEXT SUPPORT")
print("=" * 70)

context_summary = pd.DataFrame([
    {
        "role": "Batting",
        "players_with_reliable_evidence": len(
            batting_player_evidence
        ),
        "players_with_multiple_reliable_contexts": int(
            (
                batting_player_evidence[
                    "pressure_contexts"
                ] > 1
            ).sum()
        )
    },
    {
        "role": "Bowling",
        "players_with_reliable_evidence": len(
            bowling_player_evidence
        ),
        "players_with_multiple_reliable_contexts": int(
            (
                bowling_player_evidence[
                    "pressure_contexts"
                ] > 1
            ).sum()
        )
    }
])

display(context_summary)


# ============================================================
# 7. INTERPRETATION
# ============================================================

print("\n7. INTERPRETATION")
print("=" * 70)

print(
    "Positive interval flags are exploratory."
)

print(
    "They are not adjusted for multiple comparisons."
)

print(
    "Profile-level intervals cannot automatically "
    "be treated as combined player-level intervals."
)

print(
    "Players with higher observed uplift are not "
    "necessarily the most reliable."
)

print(
    "Final rankings require uncertainty-aware "
    "comparison and sensitivity analysis."
)

print(
    "\nNo original data or model outputs were modified."
)

In [ ]:
# CELL 76 — Pressure Model Sample Adequacy Audit

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("PRESSURE MODEL — SAMPLE ADEQUACY INVESTIGATION")
print("=" * 70)


# ============================================================
# 1. PREPARE MODEL TABLES
# ============================================================

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()

for df in [bat, bowl]:

    df["_reliable"] = (
        df["source_sample_reliable"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


# ============================================================
# 2. CONTEXT-LEVEL SAMPLE COVERAGE
# ============================================================

def context_coverage(
    df,
    context_column,
    player_column,
    pressure_ball_column
):

    results = []

    for context, group in df.groupby(
        context_column,
        dropna=False
    ):

        reliable = group.loc[group["_reliable"]]

        results.append({
            "pressure_context": context,

            "qualified_profiles": len(group),

            "qualified_players": (
                group[player_column].nunique()
            ),

            "reliable_profiles": len(reliable),

            "reliable_players": (
                reliable[player_column].nunique()
            ),

            "reliable_profile_pct": round(
                100 * len(reliable) / len(group),
                2
            ) if len(group) else np.nan,

            "qualified_pressure_balls": (
                group[pressure_ball_column].sum()
            ),

            "qualified_matched_balls": (
                group["source_matched_balls"].sum()
            ),

            "reliable_matched_balls": (
                reliable["source_matched_balls"].sum()
            ),

            "median_reliable_matches": (
                reliable["source_pressure_matches"].median()
            )
        })

    return pd.DataFrame(results)


print("\n1. BATTING — SAMPLE COVERAGE BY PRESSURE TYPE")
print("=" * 70)

batting_context_coverage = context_coverage(
    df=bat,
    context_column="batting_pressure_type",
    player_column="striker_id",
    pressure_ball_column="pressure_balls"
)

display(batting_context_coverage)


print("\n2. BOWLING — SAMPLE COVERAGE BY PRESSURE TYPE")
print("=" * 70)

bowling_context_coverage = context_coverage(
    df=bowl,
    context_column="bowling_pressure_type",
    player_column="bowler_id",
    pressure_ball_column="pressure_legal_balls"
)

display(bowling_context_coverage)


# ============================================================
# 3. SOURCE-MATCHED BALL DISTRIBUTION
# ============================================================

print("\n3. SOURCE-MATCHED BALL DISTRIBUTION")
print("=" * 70)

sample_summary = []

for role, df in [
    ("Batting", bat),
    ("Bowling", bowl)
]:

    reliable = df.loc[df["_reliable"]]

    sample_summary.append({
        "role": role,
        "all_profiles": len(df),
        "reliable_profiles": len(reliable),

        "median_balls_all": (
            df["source_matched_balls"].median()
        ),

        "median_balls_reliable": (
            reliable["source_matched_balls"].median()
        ),

        "minimum_balls_reliable": (
            reliable["source_matched_balls"].min()
        ),

        "maximum_balls_reliable": (
            reliable["source_matched_balls"].max()
        ),

        "median_pressure_matches": (
            reliable["source_pressure_matches"].median()
        ),

        "total_reliable_matched_balls": (
            reliable["source_matched_balls"].sum()
        )
    })

display(pd.DataFrame(sample_summary).round(2))


# ============================================================
# 4. BOWLING — UNCERTAINTY WIDTH
# ============================================================

print("\n4. BOWLING — UNCERTAINTY WIDTH")
print("=" * 70)

bowl_reliable = bowl.loc[bowl["_reliable"]].copy()

interval_pairs = {
    "Clean-dot uplift": (
        "clean_dot_source_lower_95",
        "clean_dot_source_upper_95"
    ),

    "Boundary suppression": (
        "boundary_source_lower_95",
        "boundary_source_upper_95"
    ),

    "Wicket-rate uplift": (
        "wicket_source_lower_95",
        "wicket_source_upper_95"
    )
}

width_results = []

for metric, (lower_col, upper_col) in interval_pairs.items():

    widths = (
        bowl_reliable[upper_col] -
        bowl_reliable[lower_col]
    )

    width_results.append({
        "metric": metric,
        "profiles": len(widths),
        "median_interval_width_pp": widths.median(),
        "mean_interval_width_pp": widths.mean(),
        "maximum_interval_width_pp": widths.max()
    })

display(pd.DataFrame(width_results).round(3))


# ============================================================
# 5. BOWLING — HIGHEST EXPOSURE PLAYERS
# ============================================================

print("\n5. BOWLERS WITH MOST RELIABLE PRESSURE EXPOSURE")
print("=" * 70)

bowling_exposure = (
    bowl_reliable.groupby(
        "bowler_id",
        as_index=False
    )
    .agg(
        player_name=("player_name", "first"),
        reliable_profiles=("bowling_pressure_type", "size"),
        pressure_contexts=("bowling_pressure_type", "nunique"),
        total_matched_balls=("source_matched_balls", "sum")
    )
    .sort_values(
        "total_matched_balls",
        ascending=False
    )
)

display(bowling_exposure.head(20))


# ============================================================
# 6. BATTING — HIGHEST EXPOSURE PLAYERS
# ============================================================

print("\n6. BATTERS WITH MOST RELIABLE PRESSURE EXPOSURE")
print("=" * 70)

bat_reliable = bat.loc[bat["_reliable"]].copy()

batting_exposure = (
    bat_reliable.groupby(
        "striker_id",
        as_index=False
    )
    .agg(
        player_name=("player_name", "first"),
        reliable_profiles=("batting_pressure_type", "size"),
        pressure_contexts=("batting_pressure_type", "nunique"),
        total_matched_balls=("source_matched_balls", "sum")
    )
    .sort_values(
        "total_matched_balls",
        ascending=False
    )
)

display(batting_exposure.head(20))


# ============================================================
# 7. DATA INTEGRITY
# ============================================================

print("\n7. INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "batting_profile_difference": (
        batting_context_coverage[
            "qualified_profiles"
        ].sum() - len(bat)
    ),

    "bowling_profile_difference": (
        bowling_context_coverage[
            "qualified_profiles"
        ].sum() - len(bowl)
    ),

    "batting_reliable_profile_difference": (
        batting_context_coverage[
            "reliable_profiles"
        ].sum() - len(bat_reliable)
    ),

    "bowling_reliable_profile_difference": (
        bowling_context_coverage[
            "reliable_profiles"
        ].sum() - len(bowl_reliable)
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "difference"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL SAMPLE COVERAGE CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "This audit measures available statistical evidence."
)

print(
    "It does not establish player superiority."
)

print(
    "A wider interval implies greater uncertainty "
    "about the estimated pressure advantage."
)

print(
    "The next decision is whether the existing "
    "pressure definitions provide sufficient "
    "evidence for credible player rankings."
)

In [ ]:
# CELL 77 — Pressure Framework Expansion Readiness

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("PRESSURE FRAMEWORK — EVENT-LEVEL READINESS")
print("=" * 70)


# ============================================================
# 1. LOCATE EXISTING EVENT-LEVEL TABLES
# ============================================================

candidate_tables = [
    "player_events",
    "chase_quality",
    "chase_state",
    "pressure_features",
    "standard_state",
    "match_state"
]

inventory = []

for name in candidate_tables:

    df = globals().get(name)

    if isinstance(df, pd.DataFrame):

        inventory.append({
            "table": name,
            "available": True,
            "rows": len(df),
            "columns": len(df.columns),
            "has_bowler_id": "bowler_id" in df.columns,
            "has_match_phase": "match_phase" in df.columns,
            "has_legal_delivery": (
                "is_legal_delivery" in df.columns
            )
        })

    else:

        inventory.append({
            "table": name,
            "available": False,
            "rows": None,
            "columns": None,
            "has_bowler_id": False,
            "has_match_phase": False,
            "has_legal_delivery": False
        })

print("\n1. EVENT TABLE INVENTORY")
print("=" * 70)

display(pd.DataFrame(inventory))


# ============================================================
# 2. IDENTIFY THE PRIMARY EVENT TABLE
# ============================================================

events = globals().get("player_events")

if not isinstance(events, pd.DataFrame):

    print("\nWARNING: player_events is unavailable.")

else:

    print("\n2. EVENT-LEVEL FEATURE COLUMNS")
    print("=" * 70)

    keywords = [
        "pressure",
        "phase",
        "wicket",
        "ball",
        "score",
        "target",
        "required",
        "rrr",
        "chase",
        "legal",
        "boundary",
        "dot",
        "innings",
        "match_id",
        "bowler",
        "comp_id",
        "format"
    ]

    relevant_columns = [
        column for column in events.columns
        if any(
            keyword in column.lower()
            for keyword in keywords
        )
    ]

    for number, column in enumerate(
        relevant_columns,
        start=1
    ):

        print(f"{number:02d}. {column}")


    # ========================================================
    # 3. LEGAL-BALL COVERAGE
    # ========================================================

    print("\n3. LEGAL-BALL COVERAGE")
    print("=" * 70)

    if "is_legal_delivery" in events.columns:

        legal_mask = (
            pd.to_numeric(
                events["is_legal_delivery"],
                errors="coerce"
            ).eq(1)
        )

        legal_events = events.loc[
            legal_mask
        ].copy()

        print(
            "Total event records:",
            len(events)
        )

        print(
            "Legal delivery records:",
            len(legal_events)
        )

    else:

        legal_events = pd.DataFrame()

        print(
            "Legal-delivery flag unavailable."
        )


    # ========================================================
    # 4. PHASE AND INNINGS COVERAGE
    # ========================================================

    print("\n4. LEGAL DELIVERIES BY INNINGS AND PHASE")
    print("=" * 70)

    required = {
        "innings",
        "match_phase"
    }

    if (
        not legal_events.empty and
        required.issubset(legal_events.columns)
    ):

        phase_coverage = (
            legal_events.groupby(
                ["innings", "match_phase"],
                dropna=False
            )
            .size()
            .reset_index(
                name="legal_deliveries"
            )
        )

        display(phase_coverage)

    else:

        print(
            "Required phase columns unavailable."
        )


    # ========================================================
    # 5. POSSIBLE CONTEXT FEATURES
    # ========================================================

    print("\n5. AVAILABLE MATCH-SITUATION FEATURES")
    print("=" * 70)

    situation_keywords = [
        "wicket",
        "remaining",
        "score",
        "required",
        "rrr",
        "run_rate",
        "shock",
        "depletion",
        "pressure"
    ]

    situation_columns = [
        column for column in events.columns
        if any(
            keyword in column.lower()
            for keyword in situation_keywords
        )
    ]

    feature_inventory = []

    for column in situation_columns:

        feature_inventory.append({
            "feature": column,
            "dtype": str(events[column].dtype),
            "missing_pct": round(
                100 * events[column].isna().mean(),
                2
            ),
            "unique_values": (
                events[column].nunique(
                    dropna=True
                )
            )
        })

    display(pd.DataFrame(feature_inventory))


# ============================================================
# 6. EXISTING PRESSURE SAMPLE COMPARISON
# ============================================================

print("\n6. EXISTING BOWLING PRESSURE SAMPLE")
print("=" * 70)

bowl = bowling_reliability_profiles.copy()

reliable = (
    bowl["source_sample_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

print(
    "Total source-matched bowling profile balls:",
    bowl["source_matched_balls"].sum()
)

print(
    "Reliable-profile source-matched balls:",
    bowl.loc[
        reliable,
        "source_matched_balls"
    ].sum()
)

print(
    "Reliable bowling profiles:",
    int(reliable.sum())
)

print(
    "Players with reliable profiles:",
    bowl.loc[
        reliable,
        "bowler_id"
    ].nunique()
)


# ============================================================
# 7. NEXT DECISION
# ============================================================

print("\n7. NEXT DECISION")
print("=" * 70)

print(
    "Determine which broader bowling contexts "
    "are supported by existing event-level data."
)

print(
    "Retain original pressure scenarios "
    "for sensitivity and robustness testing."
)

print(
    "Do not overwrite existing models."
)

print(
    "Do not select final players yet."
)

In [ ]:
# CELL 78 — Alternative Bowling Context Coverage

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("ALTERNATIVE BOWLING CONTEXT — COVERAGE ANALYSIS")
print("=" * 70)


# ============================================================
# 1. PREPARE LEGAL BOWLING DELIVERIES
# ============================================================

events = player_events.copy()

bowling_events = events.loc[
    pd.to_numeric(
        events["is_legal_delivery"],
        errors="coerce"
    ).eq(1)
    &
    events["innings"].isin([1, 2])
    &
    events["bowler_id"].notna()
].copy()

print("\n1. SOURCE COVERAGE")
print("=" * 70)

print("Legal bowling deliveries:", len(bowling_events))
print("Unique bowlers:", bowling_events["bowler_id"].nunique())
print("Unique matches:", bowling_events["match_id"].nunique())


# ============================================================
# 2. DEFINE CANDIDATE CHALLENGE CONTEXTS
# ============================================================

phase = bowling_events["match_phase"]
innings = bowling_events["innings"]

wickets_left = pd.to_numeric(
    bowling_events["wickets_remaining_before"],
    errors="coerce"
)

wickets_lost = pd.to_numeric(
    bowling_events["wickets_before"],
    errors="coerce"
)

candidate_contexts = {
    "All death overs": (
        phase.eq("Death Overs")
    ),

    "First-innings death": (
        innings.eq(1) &
        phase.eq("Death Overs")
    ),

    "First-innings death, 5+ wickets left": (
        innings.eq(1) &
        phase.eq("Death Overs") &
        wickets_left.ge(5)
    ),

    "Second-innings death": (
        innings.eq(2) &
        phase.eq("Death Overs")
    ),

    "Second-innings death, 5+ wickets left": (
        innings.eq(2) &
        phase.eq("Death Overs") &
        wickets_left.ge(5)
    ),

    "Death overs, 5+ wickets left": (
        phase.eq("Death Overs") &
        wickets_left.ge(5)
    ),

    "Powerplay, 0-1 wickets lost": (
        phase.eq("Powerplay") &
        wickets_lost.le(1)
    ),

    "Middle overs, 7+ wickets left": (
        phase.eq("Middle Overs") &
        wickets_left.ge(7)
    )
}


# ============================================================
# 3. EVALUATE EXPOSURE IN EACH CONTEXT
# ============================================================

print("\n2. CANDIDATE CONTEXT COVERAGE")
print("=" * 70)

coverage_results = []

for name, mask in candidate_contexts.items():

    subset = bowling_events.loc[mask]

    bowler_counts = (
        subset.groupby("bowler_id").size()
    )

    coverage_results.append({
        "context": name,
        "legal_balls": len(subset),
        "matches": subset["match_id"].nunique(),
        "bowlers": subset["bowler_id"].nunique(),

        "median_balls_per_bowler": (
            bowler_counts.median()
            if len(bowler_counts) else np.nan
        ),

        "bowlers_with_30_balls": int(
            (bowler_counts >= 30).sum()
        ),

        "bowlers_with_60_balls": int(
            (bowler_counts >= 60).sum()
        ),

        "bowlers_with_120_balls": int(
            (bowler_counts >= 120).sum()
        )
    })

context_coverage = pd.DataFrame(
    coverage_results
)

display(context_coverage)


# ============================================================
# 4. INVESTIGATE SECOND-INNINGS CHASE INFORMATION
# ============================================================

print("\n3. SECOND-INNINGS DEATH — CHASE VARIABLES")
print("=" * 70)

second_death = bowling_events.loc[
    innings.eq(2) &
    phase.eq("Death Overs")
].copy()

for column in [
    "rrr_parity",
    "rrr_scenario_spread"
]:

    if column in second_death.columns:

        print("\n", column)

        display(
            pd.to_numeric(
                second_death[column],
                errors="coerce"
            ).describe(
                percentiles=[
                    0.10,
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                    0.95
                ]
            )
        )


# ============================================================
# 5. CHASE RELIABILITY GROUPS
# ============================================================

print("\n4. CHASE-STATE RELIABILITY")
print("=" * 70)

if "pressure_reliability_group" in second_death.columns:

    display(
        second_death[
            "pressure_reliability_group"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis("reliability_group")
        .reset_index(name="deliveries")
    )


# ============================================================
# 6. VERIFY COVERAGE
# ============================================================

print("\n5. CONTEXT VALIDATION")
print("=" * 70)

checks = {
    "negative_ball_counts": int(
        (context_coverage["legal_balls"] < 0).sum()
    ),

    "contexts_exceeding_source": int(
        (
            context_coverage["legal_balls"] >
            len(bowling_events)
        ).sum()
    ),

    "missing_context_labels": int(
        context_coverage["context"].isna().sum()
    ),

    "duplicate_context_labels": int(
        context_coverage["context"].duplicated().sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL COVERAGE CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


print("\nIMPORTANT")
print("=" * 70)

print(
    "These contexts measure potential bowling challenges."
)

print(
    "They are not automatically equivalent "
    "to match pressure."
)

print(
    "Overlapping contexts are intentionally "
    "compared, not added together."
)

print(
    "The final framework must distinguish "
    "cricketing difficulty from match pressure."
)

In [ ]:
# CELL 79 — Candidate Bowling Pressure Context Construction

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("BOWLING PRESSURE — CANDIDATE CONTEXT ENGINE")
print("=" * 70)


# ============================================================
# 1. CREATE AN INDEPENDENT LEGAL-BALL WORKING TABLE
# ============================================================

source = player_events

required_columns = [
    "match_id",
    "innings",
    "bowler_id",
    "is_legal_delivery",
    "match_phase",
    "wickets_remaining_before",
    "rrr_parity",
    "pressure_reliability_group"
]

missing = [
    column for column in required_columns
    if column not in source.columns
]

assert not missing, (
    f"Required columns unavailable: {missing}"
)

legal_mask = (
    pd.to_numeric(
        source["is_legal_delivery"],
        errors="coerce"
    ).eq(1)
    &
    source["innings"].isin([1, 2])
    &
    source["bowler_id"].notna()
)

bowling_context_candidates = (
    source.loc[legal_mask].copy()
)

print("\n1. WORKING DATASET")
print("=" * 70)

print("Legal deliveries:", len(bowling_context_candidates))
print(
    "Unique bowlers:",
    bowling_context_candidates["bowler_id"].nunique()
)


# ============================================================
# 2. PREPARE MATCH-SITUATION VARIABLES
# ============================================================

df = bowling_context_candidates

death = df["match_phase"].eq("Death Overs")

second_innings = df["innings"].eq(2)

wickets_left = pd.to_numeric(
    df["wickets_remaining_before"],
    errors="coerce"
)

required_rate = pd.to_numeric(
    df["rrr_parity"],
    errors="coerce"
)

reliable_chase_state = (
    df["pressure_reliability_group"]
    .eq("Consistent Below Threshold")
)


# ============================================================
# 3. DEFINE CANDIDATE CHALLENGE CONDITIONS
# ============================================================

def death_resources(minimum_wickets):

    return (
        death &
        wickets_left.ge(minimum_wickets)
    )


def live_chase(maximum_rrr):

    return (
        second_innings &
        death &
        reliable_chase_state &
        required_rate.gt(0) &
        required_rate.le(maximum_rrr)
    )


candidate_masks = {
    "Death challenge: 4+ wickets": death_resources(4),
    "Death challenge: 5+ wickets": death_resources(5),
    "Death challenge: 6+ wickets": death_resources(6),

    "Live chase: RRR <= 8": live_chase(8),
    "Live chase: RRR <= 10": live_chase(10),
    "Live chase: RRR <= 12": live_chase(12)
}


# ============================================================
# 4. COMPARE SAMPLE SIZES
# ============================================================

print("\n2. SENSITIVITY OF CONTEXT DEFINITIONS")
print("=" * 70)

context_results = []

for context, mask in candidate_masks.items():

    subset = df.loc[mask]

    bowler_exposure = (
        subset.groupby("bowler_id").size()
    )

    context_results.append({
        "candidate_context": context,
        "legal_deliveries": len(subset),
        "matches": subset["match_id"].nunique(),
        "bowlers": subset["bowler_id"].nunique(),

        "bowlers_30_plus_balls": int(
            (bowler_exposure >= 30).sum()
        ),

        "bowlers_60_plus_balls": int(
            (bowler_exposure >= 60).sum()
        ),

        "bowlers_120_plus_balls": int(
            (bowler_exposure >= 120).sum()
        )
    })

display(pd.DataFrame(context_results))


# ============================================================
# 5. SELECT PRELIMINARY CONTEXTS
# ============================================================

print("\n3. PRELIMINARY TWO-CONTEXT FRAMEWORK")
print("=" * 70)

resource_challenge = death_resources(5)

attainable_chase = live_chase(12)

# The two conditions can overlap.
# Assign the live-chase condition first so each delivery
# receives exactly one primary classification.

df["candidate_bowling_context"] = np.select(
    [
        attainable_chase,
        resource_challenge & ~attainable_chase
    ],
    [
        "Live-Chase Defence",
        "Death vs Batting Resources"
    ],
    default="Reference / Other"
)

df["is_candidate_challenge"] = (
    df["candidate_bowling_context"]
    .ne("Reference / Other")
)

display(
    df.groupby(
        "candidate_bowling_context"
    )
    .agg(
        legal_deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)


# ============================================================
# 6. MEASURE CONTEXT OVERLAP
# ============================================================

print("\n4. RAW CONTEXT OVERLAP")
print("=" * 70)

overlap = pd.crosstab(
    resource_challenge,
    attainable_chase,
    rownames=["Death resource challenge"],
    colnames=["Live chase defence"],
    margins=True
)

display(overlap)


# ============================================================
# 7. ASSESS CHASE-SCORE UNCERTAINTY
# ============================================================

print("\n5. LIVE-CHASE SCORE UNCERTAINTY")
print("=" * 70)

live_events = df.loc[
    df["candidate_bowling_context"]
    .eq("Live-Chase Defence")
].copy()

if len(live_events):

    print(
        "Live-chase deliveries:",
        len(live_events)
    )

    display(
        pd.to_numeric(
            live_events["rrr_scenario_spread"],
            errors="coerce"
        ).describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95
            ]
        )
    )

else:

    print("No qualifying live-chase deliveries.")


# ============================================================
# 8. INTEGRITY VALIDATION
# ============================================================

print("\n6. INTEGRITY VALIDATION")
print("=" * 70)

checks = {
    "source_row_count_difference": (
        len(df) -
        int(legal_mask.sum())
    ),

    "missing_context_classifications": (
        df["candidate_bowling_context"].isna().sum()
    ),

    "classified_row_count_difference": (
        df["candidate_bowling_context"]
        .value_counts()
        .sum() - len(df)
    ),

    "live_chase_outside_second_innings": (
        (
            df["candidate_bowling_context"]
            .eq("Live-Chase Defence")
            &
            ~df["innings"].eq(2)
        ).sum()
    ),

    "live_chase_outside_death": (
        (
            df["candidate_bowling_context"]
            .eq("Live-Chase Defence")
            &
            ~death
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL CONTEXT INTEGRITY CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "These are preliminary classifications."
)

print(
    "Death vs Batting Resources measures role difficulty."
)

print(
    "Live-Chase Defence uses a provisional "
    "score-reconstruction-based match state."
)

print(
    "Sensitivity, player-level coverage and "
    "baseline matching must be evaluated "
    "before these classifications are accepted."
)

print(
    "\nExisting source data and pressure models "
    "remain unchanged."
)

In [ ]:
# CELL 80 — Bowling Outcome and Discipline Engine

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("BOWLING OUTCOME AND DISCIPLINE ENGINE")
print("=" * 70)


# ============================================================
# 1. SELECT REQUIRED EVENT FIELDS
# ============================================================

required_columns = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "bowler_id",
    "bowler_name",
    "match_phase",
    "wickets_before",
    "wickets_remaining_before",
    "rrr_parity",
    "pressure_reliability_group",
    "is_legal_delivery",
    "runs",
    "extras_type",
    "extras_2_runs",
    "is_four",
    "is_six",
    "bowler_credited_wicket_proxy"
]

missing_columns = [
    col for col in required_columns
    if col not in player_events.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

bowling_outcome_events = (
    player_events.loc[
        player_events["innings"].isin([1, 2]) &
        player_events["bowler_id"].notna(),
        required_columns
    ]
    .copy()
)


# ============================================================
# 2. DEFINE LEGAL AND ILLEGAL DELIVERIES
# ============================================================

bowling_outcome_events["legal_delivery"] = (
    pd.to_numeric(
        bowling_outcome_events["is_legal_delivery"],
        errors="coerce"
    ).eq(1)
)

bowling_outcome_events["illegal_delivery"] = (
    ~bowling_outcome_events["legal_delivery"]
)

extras = (
    bowling_outcome_events["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

bowling_outcome_events["wide_event"] = (
    extras.eq("wide")
)

bowling_outcome_events["overstep_event"] = (
    extras.eq("overstep")
)


# ============================================================
# 3. DEFINE BOWLING OUTCOMES
# ============================================================

runs = pd.to_numeric(
    bowling_outcome_events["runs"],
    errors="coerce"
)

extra_runs = pd.to_numeric(
    bowling_outcome_events["extras_2_runs"],
    errors="coerce"
)

legal = bowling_outcome_events["legal_delivery"]

bowling_outcome_events["clean_dot"] = (
    legal &
    runs.eq(0) &
    bowling_outcome_events["extras_type"].isna() &
    extra_runs.fillna(0).eq(0)
)

bowling_outcome_events["boundary_conceded"] = (
    legal &
    (
        bowling_outcome_events["is_four"].eq(1) |
        bowling_outcome_events["is_six"].eq(1)
    )
)

bowling_outcome_events["credited_wicket"] = (
    legal &
    bowling_outcome_events[
        "bowler_credited_wicket_proxy"
    ].eq(1)
)


# ============================================================
# 4. DEFINE TWO INDEPENDENT CHALLENGE INDICATORS
# ============================================================

phase = bowling_outcome_events["match_phase"]

wickets_left = pd.to_numeric(
    bowling_outcome_events[
        "wickets_remaining_before"
    ],
    errors="coerce"
)

rrr = pd.to_numeric(
    bowling_outcome_events["rrr_parity"],
    errors="coerce"
)

bowling_outcome_events["resource_death_challenge"] = (
    phase.eq("Death Overs") &
    wickets_left.ge(5)
)

bowling_outcome_events["live_chase_defence"] = (
    bowling_outcome_events["innings"].eq(2) &
    phase.eq("Death Overs") &
    bowling_outcome_events[
        "pressure_reliability_group"
    ].eq("Consistent Below Threshold") &
    rrr.gt(0) &
    rrr.le(12)
)


# ============================================================
# 5. OVERALL DELIVERY ACCOUNTING
# ============================================================

print("\n1. DELIVERY ACCOUNTING")
print("=" * 70)

print(
    "Total bowling events:",
    len(bowling_outcome_events)
)

print(
    "Legal deliveries:",
    int(
        bowling_outcome_events[
            "legal_delivery"
        ].sum()
    )
)

print(
    "Illegal deliveries:",
    int(
        bowling_outcome_events[
            "illegal_delivery"
        ].sum()
    )
)

print(
    "Wides:",
    int(
        bowling_outcome_events[
            "wide_event"
        ].sum()
    )
)

print(
    "Oversteps:",
    int(
        bowling_outcome_events[
            "overstep_event"
        ].sum()
    )
)


# ============================================================
# 6. INSPECT ILLEGAL EXTRAS CLASSIFICATIONS
# ============================================================

print("\n2. ILLEGAL DELIVERY CLASSIFICATIONS")
print("=" * 70)

display(
    bowling_outcome_events.loc[
        bowling_outcome_events["illegal_delivery"],
        "extras_type"
    ]
    .value_counts(dropna=False)
    .rename_axis("extras_type")
    .reset_index(name="events")
)


# ============================================================
# 7. COMPARE CHALLENGE CONTEXTS
# ============================================================

print("\n3. OUTCOME RATES BY CONTEXT")
print("=" * 70)

context_masks = {
    "Death: 5+ wickets available": (
        bowling_outcome_events[
            "resource_death_challenge"
        ]
    ),

    "Live-chase defence": (
        bowling_outcome_events[
            "live_chase_defence"
        ]
    ),

    "All death overs": (
        phase.eq("Death Overs")
    ),

    "Powerplay: 0-1 wickets lost": (
        phase.eq("Powerplay") &
        pd.to_numeric(
            bowling_outcome_events["wickets_before"],
            errors="coerce"
        ).le(1)
    )
}

context_results = []

for context, mask in context_masks.items():

    subset = bowling_outcome_events.loc[mask]

    legal_subset = subset.loc[
        subset["legal_delivery"]
    ]

    total_events = len(subset)
    legal_balls = len(legal_subset)

    context_results.append({
        "context": context,
        "events": total_events,
        "legal_balls": legal_balls,

        "illegal_delivery_pct": (
            100 * subset["illegal_delivery"].mean()
            if total_events else np.nan
        ),

        "clean_dot_pct": (
            100 * legal_subset["clean_dot"].mean()
            if legal_balls else np.nan
        ),

        "boundary_pct": (
            100 *
            legal_subset["boundary_conceded"].mean()
            if legal_balls else np.nan
        ),

        "credited_wicket_pct": (
            100 *
            legal_subset["credited_wicket"].mean()
            if legal_balls else np.nan
        )
    })

context_outcomes = pd.DataFrame(
    context_results
)

display(context_outcomes.round(3))


# ============================================================
# 8. IDENTIFY IRREGULAR WICKET RECORDS
# ============================================================

print("\n4. NONLEGAL BOWLER-CREDITED WICKET RECORDS")
print("=" * 70)

nonlegal_wicket_records = (
    bowling_outcome_events[
        "illegal_delivery"
    ]
    &
    bowling_outcome_events[
        "bowler_credited_wicket_proxy"
    ].eq(1)
)

print(
    "Nonlegal events with credited-wicket proxy:",
    int(nonlegal_wicket_records.sum())
)

print(
    "These events remain flagged for investigation."
)


# ============================================================
# 9. DATA INTEGRITY CHECKS
# ============================================================

print("\n5. INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "legal_illegal_count_difference": (
        int(
            bowling_outcome_events[
                "legal_delivery"
            ].sum()
        )
        +
        int(
            bowling_outcome_events[
                "illegal_delivery"
            ].sum()
        )
        -
        len(bowling_outcome_events)
    ),

    "clean_dots_on_illegal_deliveries": int(
        (
            bowling_outcome_events["clean_dot"] &
            bowling_outcome_events["illegal_delivery"]
        ).sum()
    ),

    "legal_boundaries_on_illegal_deliveries": int(
        (
            bowling_outcome_events[
                "boundary_conceded"
            ] &
            bowling_outcome_events[
                "illegal_delivery"
            ]
        ).sum()
    ),

    "credited_legal_wickets_on_illegal_deliveries": int(
        (
            bowling_outcome_events[
                "credited_wicket"
            ] &
            bowling_outcome_events[
                "illegal_delivery"
            ]
        ).sum()
    ),

    "resource_challenge_outside_death": int(
        (
            bowling_outcome_events[
                "resource_death_challenge"
            ] &
            ~phase.eq("Death Overs")
        ).sum()
    ),

    "live_chase_outside_second_innings": int(
        (
            bowling_outcome_events[
                "live_chase_defence"
            ] &
            ~bowling_outcome_events["innings"].eq(2)
        ).sum()
    ),

    "duplicate_event_keys": int(
        bowling_outcome_events.duplicated(
            [
                "match_id",
                "innings",
                "innings_ball_num"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL ENGINE INTEGRITY CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "Clean-dot, boundary and credited-wicket rates "
    "use legal deliveries as denominators."
)

print(
    "Illegal-delivery frequency uses all "
    "recorded bowling events."
)

print(
    "The two challenge indicators can overlap "
    "and must not be added as independent evidence."
)

print(
    "Context-level averages are descriptive, "
    "not adjusted individual player effects."
)

print(
    "\nOriginal datasets and earlier models "
    "have not been modified."
)

In [ ]:
# CELL 81 — Audit Nonlegal Wickets and Boundaries

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("NONLEGAL DELIVERIES — WICKET AND BOUNDARY AUDIT")
print("=" * 70)


# ============================================================
# 1. PREPARE SOURCE EVENTS
# ============================================================

required_columns = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "bowler_id",
    "bowler_name",
    "extras_type",
    "dismissal_type",
    "is_wicket",
    "bowler_credited_wicket_proxy",
    "is_legal_delivery",
    "is_four",
    "is_six",
    "runs",
    "extras_2_runs"
]

missing_columns = [
    col for col in required_columns
    if col not in player_events.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

audit_events = player_events.loc[
    player_events["innings"].isin([1, 2]) &
    player_events["bowler_id"].notna(),
    required_columns
].copy()

audit_events["legal_delivery"] = (
    pd.to_numeric(
        audit_events["is_legal_delivery"],
        errors="coerce"
    ).eq(1)
)

audit_events["illegal_delivery"] = (
    ~audit_events["legal_delivery"]
)

audit_events["credited_wicket_proxy"] = (
    audit_events["bowler_credited_wicket_proxy"]
    .eq(1)
)


# ============================================================
# 2. IDENTIFY NONLEGAL CREDITED WICKETS
# ============================================================

print("\n1. NONLEGAL CREDITED-WICKET EVENTS")
print("=" * 70)

nonlegal_credited = audit_events.loc[
    audit_events["illegal_delivery"] &
    audit_events["credited_wicket_proxy"]
].copy()

print("Events:", len(nonlegal_credited))

display(
    nonlegal_credited[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "bowler_name",
            "extras_type",
            "dismissal_type",
            "is_wicket"
        ]
    ]
)


# ============================================================
# 3. DISMISSAL TYPE DISTRIBUTION
# ============================================================

print("\n2. EXTRAS TYPE AND DISMISSAL COMBINATIONS")
print("=" * 70)

display(
    nonlegal_credited.groupby(
        ["extras_type", "dismissal_type"],
        dropna=False
    )
    .size()
    .reset_index(name="events")
    .sort_values(
        "events",
        ascending=False
    )
)


# ============================================================
# 4. CRICKET-LAW COMPATIBILITY SCREEN
# ============================================================

print("\n3. DISMISSAL COMPATIBILITY SCREEN")
print("=" * 70)

extras_normalized = (
    nonlegal_credited["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

dismissal_normalized = (
    nonlegal_credited["dismissal_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

wide_credit_candidate = (
    extras_normalized.eq("wide") &
    dismissal_normalized.isin(
        [
            "stump",
            "stumped",
            "hit wicket"
        ]
    )
)

nonlegal_credited["classification"] = np.where(
    wide_credit_candidate,
    "Potentially valid bowler-credit wicket",
    "Requires manual verification"
)

display(
    nonlegal_credited[
        "classification"
    ]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="events")
)


# ============================================================
# 5. PLAYERS AFFECTED BY NONLEGAL WICKETS
# ============================================================

print("\n4. AFFECTED BOWLERS")
print("=" * 70)

if len(nonlegal_credited):

    display(
        nonlegal_credited.groupby(
            ["bowler_id", "bowler_name"],
            as_index=False
        )
        .agg(
            nonlegal_credited_events=(
                "credited_wicket_proxy", "sum"
            ),
            affected_matches=("match_id", "nunique")
        )
        .sort_values(
            "nonlegal_credited_events",
            ascending=False
        )
    )

else:

    print("No affected bowlers.")


# ============================================================
# 6. IDENTIFY BOUNDARIES ON NONLEGAL EVENTS
# ============================================================

print("\n5. NONLEGAL BATTING BOUNDARIES")
print("=" * 70)

audit_events["boundary_flag"] = (
    audit_events["is_four"].eq(1) |
    audit_events["is_six"].eq(1)
)

nonlegal_boundaries = audit_events.loc[
    audit_events["illegal_delivery"] &
    audit_events["boundary_flag"]
].copy()

print(
    "Boundary events on nonlegal deliveries:",
    len(nonlegal_boundaries)
)

display(
    nonlegal_boundaries.groupby(
        "extras_type",
        dropna=False
    )
    .size()
    .reset_index(name="boundary_events")
)


# ============================================================
# 7. RECONCILE ORIGINAL WICKET COUNTS
# ============================================================

print("\n6. WICKET COUNT RECONCILIATION")
print("=" * 70)

wicket_accounting = {
    "all_credited_wicket_events": int(
        audit_events["credited_wicket_proxy"].sum()
    ),

    "legal_credited_wicket_events": int(
        (
            audit_events["credited_wicket_proxy"] &
            audit_events["legal_delivery"]
        ).sum()
    ),

    "nonlegal_credited_wicket_events": int(
        (
            audit_events["credited_wicket_proxy"] &
            audit_events["illegal_delivery"]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(wicket_accounting.items()),
        columns=["measure", "events"]
    )
)


# ============================================================
# 8. INTEGRITY CHECKS
# ============================================================

print("\n7. INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "credited_wicket_reconciliation": (
        wicket_accounting[
            "legal_credited_wicket_events"
        ]
        +
        wicket_accounting[
            "nonlegal_credited_wicket_events"
        ]
        -
        wicket_accounting[
            "all_credited_wicket_events"
        ]
    ),

    "nonlegal_wicket_count_difference": (
        len(nonlegal_credited)
        -
        wicket_accounting[
            "nonlegal_credited_wicket_events"
        ]
    ),

    "duplicate_delivery_keys": int(
        audit_events.duplicated(
            [
                "match_id",
                "innings",
                "innings_ball_num"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL RECONCILIATION CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "A wicket credited to a bowler may occur "
    "on a wide delivery."
)

print(
    "The event's dismissal mechanism must "
    "be checked before including it."
)

print(
    "Nonlegal batting boundaries may also "
    "affect bowling concession metrics."
)

print(
    "No wicket records have been reclassified."
)

print(
    "No earlier model results have been overwritten."
)

In [ ]:
# CELL 82 — Corrected Bowling Performance Engine

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("CORRECTED BOWLING PERFORMANCE ENGINE")
print("=" * 70)


# ============================================================
# 1. CREATE INDEPENDENT WORKING DATASET
# ============================================================

bowling_pressure_v2_events = (
    bowling_outcome_events.copy()
)

print("\n1. SOURCE DATA")
print("=" * 70)

print(
    "Total bowling events:",
    len(bowling_pressure_v2_events)
)


# ============================================================
# 2. DEFINE CORRECTED OUTCOME COUNTS
# ============================================================

df = bowling_pressure_v2_events

df["legal_ball_count"] = (
    df["legal_delivery"].astype(int)
)

df["illegal_event_count"] = (
    df["illegal_delivery"].astype(int)
)

df["clean_dot_count"] = (
    df["clean_dot"].astype(int)
)

# Include verified bowler-credited stumpings on wides.
df["credited_wicket_count"] = (
    df["bowler_credited_wicket_proxy"]
    .eq(1)
    .astype(int)
)

# Include batting boundaries on legal and illegal deliveries.
df["boundary_count"] = (
    (
        df["is_four"].eq(1) |
        df["is_six"].eq(1)
    )
    .astype(int)
)


# ============================================================
# 3. RECONCILE CORRECTED OUTCOMES
# ============================================================

print("\n2. CORRECTED OUTCOME ACCOUNTING")
print("=" * 70)

outcome_totals = {
    "legal_balls": int(
        df["legal_ball_count"].sum()
    ),

    "illegal_events": int(
        df["illegal_event_count"].sum()
    ),

    "clean_dots": int(
        df["clean_dot_count"].sum()
    ),

    "credited_wickets_all_events": int(
        df["credited_wicket_count"].sum()
    ),

    "batting_boundaries_all_events": int(
        df["boundary_count"].sum()
    )
}

display(
    pd.DataFrame(
        list(outcome_totals.items()),
        columns=["measure", "events"]
    )
)


# ============================================================
# 4. ATTACH MATCH AND COMPETITION IDENTIFIERS
# ============================================================

print("\n3. MATCH METADATA COVERAGE")
print("=" * 70)

match_lookup = (
    datasets["matches"][
        [
            "match_id",
            "comp_id",
            "match_date"
        ]
    ]
    .copy()
)

assert match_lookup["match_id"].is_unique, (
    "Duplicate match IDs found in match metadata."
)

original_rows = len(df)

bowling_pressure_v2_events = df.merge(
    match_lookup,
    on="match_id",
    how="left",
    validate="many_to_one"
)

print(
    "Rows after metadata merge:",
    len(bowling_pressure_v2_events)
)

print(
    "Missing competition IDs:",
    int(
        bowling_pressure_v2_events[
            "comp_id"
        ].isna().sum()
    )
)


# ============================================================
# 5. EXAMINE COMPETITION METADATA
# ============================================================

print("\n4. COMPETITION METADATA STRUCTURE")
print("=" * 70)

print(
    "Available competition columns:",
    datasets["competitions"].columns.tolist()
)

display(
    datasets["competitions"].head(8)
)


# ============================================================
# 6. COMPETITION COVERAGE
# ============================================================

print("\n5. COMPETITION COVERAGE")
print("=" * 70)

display(
    bowling_pressure_v2_events.groupby(
        "comp_id",
        dropna=False
    )
    .agg(
        bowling_events=("match_id", "size"),
        matches=("match_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "bowling_events",
        ascending=False
    )
    .head(15)
)


# ============================================================
# 7. CALCULATE CORRECTED CONTEXT OUTCOMES
# ============================================================

print("\n6. CORRECTED CONTEXT PERFORMANCE")
print("=" * 70)

df = bowling_pressure_v2_events

context_masks = {
    "Death vs Batting Resources": (
        df["resource_death_challenge"]
    ),

    "Live-Chase Defence": (
        df["live_chase_defence"]
    ),

    "All Death Overs": (
        df["match_phase"].eq("Death Overs")
    )
}

corrected_context_results = []

for context, mask in context_masks.items():

    subset = df.loc[mask]

    legal_balls = int(
        subset["legal_ball_count"].sum()
    )

    total_events = len(subset)

    credited_wickets = int(
        subset["credited_wicket_count"].sum()
    )

    boundaries = int(
        subset["boundary_count"].sum()
    )

    clean_dots = int(
        subset["clean_dot_count"].sum()
    )

    illegal_events = int(
        subset["illegal_event_count"].sum()
    )

    corrected_context_results.append({
        "context": context,

        "events": total_events,

        "legal_balls": legal_balls,

        "clean_dot_per_100_legal": (
            100 * clean_dots / legal_balls
            if legal_balls else np.nan
        ),

        "boundaries_per_100_legal": (
            100 * boundaries / legal_balls
            if legal_balls else np.nan
        ),

        "wickets_per_100_legal": (
            100 * credited_wickets / legal_balls
            if legal_balls else np.nan
        ),

        "illegal_events_per_100_legal": (
            100 * illegal_events / legal_balls
            if legal_balls else np.nan
        )
    })

corrected_context_summary = pd.DataFrame(
    corrected_context_results
)

display(
    corrected_context_summary.round(3)
)


# ============================================================
# 8. FINAL ENGINE INTEGRITY
# ============================================================

print("\n7. ENGINE INTEGRITY")
print("=" * 70)

checks = {
    "row_count_difference": (
        len(bowling_pressure_v2_events) -
        original_rows
    ),

    "duplicate_delivery_keys": int(
        bowling_pressure_v2_events.duplicated(
            [
                "match_id",
                "innings",
                "innings_ball_num"
            ]
        ).sum()
    ),

    "missing_competition_ids": int(
        bowling_pressure_v2_events[
            "comp_id"
        ].isna().sum()
    ),

    "wicket_accounting_difference": (
        int(
            bowling_pressure_v2_events[
                "credited_wicket_count"
            ].sum()
        )
        -
        (
            int(
                bowling_pressure_v2_events.loc[
                    bowling_pressure_v2_events[
                        "legal_delivery"
                    ],
                    "credited_wicket_count"
                ].sum()
            )
            +
            int(
                bowling_pressure_v2_events.loc[
                    bowling_pressure_v2_events[
                        "illegal_delivery"
                    ],
                    "credited_wicket_count"
                ].sum()
            )
        )
    ),

    "clean_dots_on_illegal_events": int(
        (
            bowling_pressure_v2_events[
                "clean_dot_count"
            ].gt(0)
            &
            bowling_pressure_v2_events[
                "illegal_delivery"
            ]
        ).sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL CORRECTED ENGINE CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


print("\nINTERPRETATION")
print("=" * 70)

print(
    "Verified wide-ball stumpings are included "
    "in bowler-credited wickets."
)

print(
    "Batting boundaries on nonlegal deliveries "
    "are included in boundary concessions."
)

print(
    "All bowling outcome rates are expressed "
    "per 100 legal balls."
)

print(
    "The resulting data is ready for "
    "competition-source matching."
)

print(
    "\nOriginal datasets and models remain unchanged."
)

In [ ]:
# CELL 83 — Round 1 Player Selection Candidate Board

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("ROUND 1 — PLAYER SELECTION CANDIDATE BOARD")
print("=" * 70)


# ============================================================
# 1. LOAD EXISTING VALIDATED PLAYER EVIDENCE
# ============================================================

print("\n1. INPUT VERIFICATION")
print("=" * 70)

bat = batting_player_evidence.copy()
bowl = bowling_player_evidence.copy()

bat_profiles = batting_reliability_profiles.copy()
bowl_profiles = bowling_reliability_profiles.copy()

print("Batters with reliable evidence:", len(bat))
print("Bowlers with reliable evidence:", len(bowl))

assert bat["striker_id"].is_unique
assert bowl["bowler_id"].is_unique

assert len(bat) >= 10
assert len(bowl) >= 10


# ============================================================
# 2. BATTING — STATISTICAL RELIABILITY
# ============================================================

print("\n2. BATTING RELIABILITY EVIDENCE")
print("=" * 70)

bat_reliable = bat_profiles.loc[
    bat_profiles["source_sample_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
].copy()

bat_reliable["positive_interval"] = (
    bat_reliable["sr_source_lower_95"] > 0
)

bat_support = (
    bat_reliable.groupby(
        "striker_id",
        as_index=False
    )
    .agg(
        pressure_matches=(
            "source_pressure_matches", "max"
        ),
        positive_interval_contexts=(
            "positive_interval", "sum"
        ),
        lowest_context_lower_95=(
            "sr_source_lower_95", "min"
        )
    )
)

bat = bat.merge(
    bat_support,
    on="striker_id",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 3. BATTING — GENERATE CANDIDATES
# ============================================================

print("\n3. BATTING CANDIDATE BOARD")
print("=" * 70)

# Include leading performers.
batting_candidate_ids = set(
    bat.nlargest(
        20,
        "pressure_sr_uplift"
    )["striker_id"]
)

# Preserve high-exposure players for comparison.
batting_candidate_ids.update(
    bat.nlargest(
        10,
        "matched_balls"
    )["striker_id"]
)

# Include players with at least one positive
# nominal bootstrap interval.
batting_candidate_ids.update(
    bat.loc[
        bat["positive_interval_contexts"] > 0,
        "striker_id"
    ]
)

batting_candidate_board = (
    bat.loc[
        bat["striker_id"].isin(
            batting_candidate_ids
        )
    ]
    .sort_values(
        "pressure_sr_uplift",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Batting candidates identified:",
    len(batting_candidate_board)
)

display(
    batting_candidate_board[
        [
            "player_name",
            "matched_balls",
            "reliable_profiles",
            "pressure_contexts",
            "pressure_matches",
            "pressure_sr_uplift",
            "positive_interval_contexts",
            "lowest_context_lower_95"
        ]
    ].round(2)
)


# ============================================================
# 4. BOWLING — STATISTICAL RELIABILITY
# ============================================================

print("\n4. BOWLING RELIABILITY EVIDENCE")
print("=" * 70)

bowl_reliable = bowl_profiles.loc[
    bowl_profiles["source_sample_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
].copy()

bowl_reliable["positive_dot_interval"] = (
    bowl_reliable[
        "clean_dot_source_lower_95"
    ] > 0
)

bowl_reliable["positive_boundary_interval"] = (
    bowl_reliable[
        "boundary_source_lower_95"
    ] > 0
)

bowl_reliable["positive_wicket_interval"] = (
    bowl_reliable[
        "wicket_source_lower_95"
    ] > 0
)

bowl_support = (
    bowl_reliable.groupby(
        "bowler_id",
        as_index=False
    )
    .agg(
        pressure_matches=(
            "source_pressure_matches", "max"
        ),
        positive_dot_intervals=(
            "positive_dot_interval", "sum"
        ),
        positive_boundary_intervals=(
            "positive_boundary_interval", "sum"
        ),
        positive_wicket_intervals=(
            "positive_wicket_interval", "sum"
        )
    )
)

bowl = bowl.merge(
    bowl_support,
    on="bowler_id",
    how="left",
    validate="one_to_one"
)

bowl["positive_point_dimensions"] = (
    bowl[
        [
            "dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_uplift_pp"
        ]
    ].gt(0).sum(axis=1)
)


# ============================================================
# 5. BOWLING — GENERATE CANDIDATES
# ============================================================

print("\n5. BOWLING CANDIDATE BOARD")
print("=" * 70)

bowling_candidate_ids = set()

# Screen across three separate performance dimensions
# and overall exposure, without inventing a composite score.
screening_metrics = [
    "dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_uplift_pp",
    "matched_balls"
]

for metric in screening_metrics:

    leaders = bowl.nlargest(
        12,
        metric
    )

    bowling_candidate_ids.update(
        leaders["bowler_id"].tolist()
    )

bowling_candidate_board = (
    bowl.loc[
        bowl["bowler_id"].isin(
            bowling_candidate_ids
        )
    ]
    .sort_values(
        "matched_balls",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Bowling candidates identified:",
    len(bowling_candidate_board)
)

display(
    bowling_candidate_board[
        [
            "player_name",
            "matched_balls",
            "reliable_profiles",
            "pressure_contexts",
            "pressure_matches",
            "dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_uplift_pp",
            "positive_point_dimensions",
            "positive_dot_intervals",
            "positive_boundary_intervals",
            "positive_wicket_intervals"
        ]
    ].round(3)
)


# ============================================================
# 6. CANDIDATE INTEGRITY
# ============================================================

print("\n6. CANDIDATE INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "duplicate_batting_candidates": int(
        batting_candidate_board[
            "striker_id"
        ].duplicated().sum()
    ),

    "duplicate_bowling_candidates": int(
        bowling_candidate_board[
            "bowler_id"
        ].duplicated().sum()
    ),

    "missing_batting_names": int(
        batting_candidate_board[
            "player_name"
        ].isna().sum()
    ),

    "missing_bowling_names": int(
        bowling_candidate_board[
            "player_name"
        ].isna().sum()
    ),

    "missing_batting_reliability": int(
        batting_candidate_board[
            "pressure_matches"
        ].isna().sum()
    ),

    "missing_bowling_reliability": int(
        bowling_candidate_board[
            "pressure_matches"
        ].isna().sum()
    )
}

display(
    pd.DataFrame(
        list(checks.items()),
        columns=["check", "count"]
    )
)

if all(value == 0 for value in checks.values()):

    print("\nALL CANDIDATE CHECKS PASSED")

else:

    print("\nREVIEW REQUIRED")


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n7. SCOUTING CANDIDATE SUMMARY")
print("=" * 70)

print(
    "Batting candidates:",
    len(batting_candidate_board)
)

print(
    "Bowling candidates:",
    len(bowling_candidate_board)
)

print("\nIMPORTANT:")
print(
    "These are exploratory candidate pools, "
    "not final ordered rankings."
)

print(
    "Positive confidence interval flags are nominal "
    "and not adjusted for multiple comparisons."
)

print(
    "Combined player-level uncertainty intervals "
    "have not yet been established."
)

print(
    "All existing source data and model outputs "
    "remain unchanged."
)

print("\nNEXT STAGE:")
print(
    "Evaluate candidates and develop the provisional "
    "Top 10 Batters and Top 10 Bowlers."
)

In [ ]:
# ============================================================
# CELL 84 — ROUND 1 PROVISIONAL PLAYER RANKINGS
# Sample-size adjustment + exposure + sensitivity
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("ROUND 1 — PROVISIONAL PRESSURE PERFORMANCE RANKINGS")
print("=" * 70)


# ============================================================
# 1. VERIFY INPUT TABLES
# ============================================================

print("\n1. INPUT VERIFICATION")
print("=" * 70)

assert "batting_candidate_board" in globals(), (
    "batting_candidate_board is missing. Run the previous cell."
)

assert "bowling_candidate_board" in globals(), (
    "bowling_candidate_board is missing. Run the previous cell."
)

bat_required = [
    "player_name",
    "matched_balls",
    "pressure_matches",
    "pressure_contexts",
    "pressure_sr_uplift",
    "positive_interval_contexts"
]

bowl_required = [
    "player_name",
    "matched_balls",
    "pressure_matches",
    "pressure_contexts",
    "dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_uplift_pp",
    "positive_dot_intervals",
    "positive_boundary_intervals",
    "positive_wicket_intervals"
]

assert set(bat_required).issubset(
    batting_candidate_board.columns
), "Missing batting columns"

assert set(bowl_required).issubset(
    bowling_candidate_board.columns
), "Missing bowling columns"

bat = batting_candidate_board.copy()
bowl = bowling_candidate_board.copy()

for col in [
    "matched_balls",
    "pressure_matches",
    "pressure_contexts",
    "pressure_sr_uplift",
    "positive_interval_contexts"
]:
    bat[col] = pd.to_numeric(bat[col], errors="raise")

for col in [
    "matched_balls",
    "pressure_matches",
    "pressure_contexts",
    "dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_uplift_pp",
    "positive_dot_intervals",
    "positive_boundary_intervals",
    "positive_wicket_intervals"
]:
    bowl[col] = pd.to_numeric(bowl[col], errors="raise")

assert bat[bat_required].notna().all().all()
assert bowl[bowl_required].notna().all().all()
assert bat["player_name"].is_unique
assert bowl["player_name"].is_unique
assert bat["matched_balls"].gt(0).all()
assert bowl["matched_balls"].gt(0).all()

print("Batting candidates:", len(bat))
print("Bowling candidates:", len(bowl))


# ============================================================
# 2. COMMON EXPOSURE ADJUSTMENT
# ============================================================

print("\n2. EXPOSURE ADJUSTMENT")
print("=" * 70)

# More distinct matches provide broader exposure.
# This is a transparent ranking preference, not a
# statistical confidence interval.

def match_exposure_factor(matches):
    return (
        0.80
        + 0.20 * np.minimum(matches / 15.0, 1.0)
    )


bat["match_exposure_factor"] = match_exposure_factor(
    bat["pressure_matches"]
)

bowl["match_exposure_factor"] = match_exposure_factor(
    bowl["pressure_matches"]
)


# ============================================================
# 3. BATTING RANKING MODEL
# ============================================================

print("\n3. BATTING RANKING MODEL")
print("=" * 70)

# Player's pressure SR uplift is measured against
# their source-adjusted personal baseline.
#
# Shrinkage reduces the impact of extreme estimates
# from small samples.
#
# The 120-ball constant is a decision assumption;
# sensitivity checks below vary it.

BATTING_SHRINKAGE_BALLS = 120.0

bat["batting_shrinkage_factor"] = (
    bat["matched_balls"]
    / (
        bat["matched_balls"]
        + BATTING_SHRINKAGE_BALLS
    )
)

bat["adjusted_batting_uplift"] = (
    bat["pressure_sr_uplift"]
    * bat["batting_shrinkage_factor"]
)

bat["batting_selection_score"] = (
    bat["adjusted_batting_uplift"]
    * bat["match_exposure_factor"]
)

bat["batting_evidence_status"] = np.select(
    [
        (bat["matched_balls"] >= 150)
        & (bat["pressure_matches"] >= 10),

        (bat["matched_balls"] >= 75)
        & (bat["pressure_matches"] >= 5)
    ],
    [
        "Broader exposure",
        "Moderate exposure"
    ],
    default="Limited exposure"
)

# Nominal positive intervals are shown as evidence.
# They are NOT used as a bonus in the ranking score
# because multiple-testing adjustment is unavailable.

bat["any_positive_interval"] = (
    bat["positive_interval_contexts"] > 0
)


# ============================================================
# 4. BATTING SENSITIVITY
# ============================================================

print("\n4. BATTING SENSITIVITY")
print("=" * 70)

bat_sensitivity_columns = []

for shrinkage in [60.0, 120.0, 200.0]:

    score_col = f"bat_score_k{int(shrinkage)}"
    rank_col = f"bat_rank_k{int(shrinkage)}"

    bat[score_col] = (
        bat["pressure_sr_uplift"]
        * (
            bat["matched_balls"]
            / (bat["matched_balls"] + shrinkage)
        )
        * bat["match_exposure_factor"]
    )

    bat[rank_col] = (
        bat[score_col]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    bat_sensitivity_columns.append(rank_col)

bat["best_sensitivity_rank"] = (
    bat[bat_sensitivity_columns].min(axis=1)
)

bat["worst_sensitivity_rank"] = (
    bat[bat_sensitivity_columns].max(axis=1)
)

bat["sensitivity_rank_spread"] = (
    bat["worst_sensitivity_rank"]
    - bat["best_sensitivity_rank"]
)


# ============================================================
# 5. BOWLING RANKING MODEL
# ============================================================

print("\n5. BOWLING RANKING MODEL")
print("=" * 70)

# All three measures are improvements relative to
# each bowler's matched personal baseline:
#
# 1. Clean-dot uplift: higher is better.
# 2. Boundary suppression: higher is better.
# 3. Wicket-rate uplift: higher is better.
#
# The divisors put the three percentage-point
# metrics onto broadly comparable decision scales.
#
# The weights express a scouting preference.
# They are NOT coefficients fitted from outcomes.

BOWLING_SHRINKAGE_BALLS = 100.0

BOWLING_WEIGHTS = {
    "dot": 0.25,
    "boundary": 0.40,
    "wicket": 0.35
}

BOWLING_SCALES = {
    "dot": 10.0,
    "boundary": 10.0,
    "wicket": 5.0
}

bowl["bowling_shrinkage_factor"] = (
    bowl["matched_balls"]
    / (
        bowl["matched_balls"]
        + BOWLING_SHRINKAGE_BALLS
    )
)

bowl["normalized_dot"] = (
    bowl["dot_uplift_pp"]
    / BOWLING_SCALES["dot"]
)

bowl["normalized_boundary"] = (
    bowl["boundary_suppression_pp"]
    / BOWLING_SCALES["boundary"]
)

bowl["normalized_wicket"] = (
    bowl["wicket_uplift_pp"]
    / BOWLING_SCALES["wicket"]
)

bowl["raw_bowling_utility"] = (
    BOWLING_WEIGHTS["dot"]
    * bowl["normalized_dot"]

    + BOWLING_WEIGHTS["boundary"]
    * bowl["normalized_boundary"]

    + BOWLING_WEIGHTS["wicket"]
    * bowl["normalized_wicket"]
)

bowl["bowling_selection_score"] = (
    bowl["raw_bowling_utility"]
    * bowl["bowling_shrinkage_factor"]
    * bowl["match_exposure_factor"]
)

bowl["positive_point_dimensions"] = (
    (bowl["dot_uplift_pp"] > 0).astype(int)
    + (bowl["boundary_suppression_pp"] > 0).astype(int)
    + (bowl["wicket_uplift_pp"] > 0).astype(int)
)

bowl["positive_interval_dimensions"] = (
    (bowl["positive_dot_intervals"] > 0).astype(int)
    + (bowl["positive_boundary_intervals"] > 0).astype(int)
    + (bowl["positive_wicket_intervals"] > 0).astype(int)
)

bowl["bowling_evidence_status"] = np.select(
    [
        (bowl["matched_balls"] >= 120)
        & (bowl["pressure_matches"] >= 10),

        (bowl["matched_balls"] >= 60)
        & (bowl["pressure_matches"] >= 5)
    ],
    [
        "Broader exposure",
        "Moderate exposure"
    ],
    default="Limited exposure"
)


# ============================================================
# 6. BOWLING SENSITIVITY
# ============================================================

print("\n6. BOWLING SENSITIVITY")
print("=" * 70)

# Three weighting philosophies:
#
# Balanced: initial preference.
# Wicket-focused: greater value on wicket uplift.
# Control-focused: greater value on dots/boundaries.
#
# Test each under three shrinkage strengths.

bowling_weight_scenarios = {
    "balanced": (0.25, 0.40, 0.35),
    "wicket_focus": (0.20, 0.30, 0.50),
    "control_focus": (0.35, 0.40, 0.25)
}

bowl_sensitivity_columns = []

for scenario, weights in bowling_weight_scenarios.items():

    dot_weight, boundary_weight, wicket_weight = weights

    utility = (
        dot_weight * bowl["normalized_dot"]
        + boundary_weight * bowl["normalized_boundary"]
        + wicket_weight * bowl["normalized_wicket"]
    )

    for shrinkage in [60.0, 100.0, 150.0]:

        rank_col = (
            f"bow_rank_{scenario}_k{int(shrinkage)}"
        )

        scenario_score = (
            utility
            * (
                bowl["matched_balls"]
                / (bowl["matched_balls"] + shrinkage)
            )
            * bowl["match_exposure_factor"]
        )

        bowl[rank_col] = (
            scenario_score
            .rank(ascending=False, method="min")
            .astype(int)
        )

        bowl_sensitivity_columns.append(rank_col)

bowl["best_sensitivity_rank"] = (
    bowl[bowl_sensitivity_columns].min(axis=1)
)

bowl["worst_sensitivity_rank"] = (
    bowl[bowl_sensitivity_columns].max(axis=1)
)

bowl["sensitivity_rank_spread"] = (
    bowl["worst_sensitivity_rank"]
    - bowl["best_sensitivity_rank"]
)


# ============================================================
# 7. CONSTRUCT PROVISIONAL RANKINGS
# ============================================================

print("\n7. PROVISIONAL TOP-15 BATTING BOARD")
print("=" * 70)

batting_provisional_rankings = (
    bat.sort_values(
        [
            "batting_selection_score",
            "matched_balls"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
    .copy()
)

batting_provisional_rankings.insert(
    0,
    "provisional_rank",
    np.arange(1, len(batting_provisional_rankings) + 1)
)

display(
    batting_provisional_rankings[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "pressure_matches",
            "pressure_contexts",
            "pressure_sr_uplift",
            "adjusted_batting_uplift",
            "batting_selection_score",
            "any_positive_interval",
            "batting_evidence_status",
            "best_sensitivity_rank",
            "worst_sensitivity_rank"
        ]
    ].head(15).round(3)
)


print("\n8. PROVISIONAL TOP-15 BOWLING BOARD")
print("=" * 70)

bowling_provisional_rankings = (
    bowl.sort_values(
        [
            "bowling_selection_score",
            "matched_balls"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
    .copy()
)

bowling_provisional_rankings.insert(
    0,
    "provisional_rank",
    np.arange(1, len(bowling_provisional_rankings) + 1)
)

display(
    bowling_provisional_rankings[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "pressure_matches",
            "dot_uplift_pp",
            "boundary_suppression_pp",
            "wicket_uplift_pp",
            "positive_point_dimensions",
            "positive_interval_dimensions",
            "bowling_selection_score",
            "bowling_evidence_status",
            "best_sensitivity_rank",
            "worst_sensitivity_rank"
        ]
    ].head(15).round(3)
)


# ============================================================
# 9. PROVISIONAL TOP-10 SUMMARY
# ============================================================

print("\n9. TOP-10 CANDIDATE SUMMARY")
print("=" * 70)

bat_top10 = batting_provisional_rankings.head(10)
bowl_top10 = bowling_provisional_rankings.head(10)

summary = pd.DataFrame([
    {
        "role": "Batting",
        "provisional_candidates": len(bat_top10),
        "broader_exposure": int(
            (bat_top10["batting_evidence_status"]
             == "Broader exposure").sum()
        ),
        "limited_exposure": int(
            (bat_top10["batting_evidence_status"]
             == "Limited exposure").sum()
        ),
        "positive_nominal_interval_players": int(
            bat_top10["any_positive_interval"].sum()
        )
    },
    {
        "role": "Bowling",
        "provisional_candidates": len(bowl_top10),
        "broader_exposure": int(
            (bowl_top10["bowling_evidence_status"]
             == "Broader exposure").sum()
        ),
        "limited_exposure": int(
            (bowl_top10["bowling_evidence_status"]
             == "Limited exposure").sum()
        ),
        "positive_nominal_interval_players": int(
            (bowl_top10["positive_interval_dimensions"] > 0).sum()
        )
    }
])

display(summary)


# ============================================================
# 10. RANKING INTEGRITY
# ============================================================

print("\n10. RANKING INTEGRITY")
print("=" * 70)

checks = {
    "batting_candidates_preserved": (
        len(batting_provisional_rankings) - len(bat)
    ),
    "bowling_candidates_preserved": (
        len(bowling_provisional_rankings) - len(bowl)
    ),
    "duplicate_batting_names": int(
        batting_provisional_rankings["player_name"]
        .duplicated().sum()
    ),
    "duplicate_bowling_names": int(
        bowling_provisional_rankings["player_name"]
        .duplicated().sum()
    ),
    "missing_batting_scores": int(
        batting_provisional_rankings[
            "batting_selection_score"
        ].isna().sum()
    ),
    "missing_bowling_scores": int(
        bowling_provisional_rankings[
            "bowling_selection_score"
        ].isna().sum()
    ),
    "nonfinite_batting_scores": int(
        (~np.isfinite(
            batting_provisional_rankings[
                "batting_selection_score"
            ]
        )).sum()
    ),
    "nonfinite_bowling_scores": int(
        (~np.isfinite(
            bowling_provisional_rankings[
                "bowling_selection_score"
            ]
        )).sum()
    )
}

checks_df = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(checks_df)

assert (checks_df["count"] == 0).all(), (
    "Ranking integrity checks failed."
)

print("\nALL RANKING INTEGRITY CHECKS PASSED")


# ============================================================
# FINAL INTERPRETATION
# ============================================================

print("\nINTERPRETATION")
print("=" * 70)

print(
    "These are provisional scouting priorities, "
    "not confirmed final selections."
)

print(
    "Sample-size adjustments and bowling weights "
    "are explicitly chosen decision assumptions."
)

print(
    "Rank ranges show sensitivity to those assumptions."
)

print(
    "Nominal bootstrap intervals are not corrected "
    "for multiple comparisons."
)

print(
    "Next: review the leading 15 players in each role "
    "for pressure-context quality, cricketing role, "
    "2027 RR squad fit, and auction feasibility."
)

print(
    "The source models remain unchanged."
)

In [ ]:
# ============================================================
# CELL 85 — BOWLING DEATH-OVER PEER PERFORMANCE
# Absolute quality anchor for pressure scouting
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 70)
print("BOWLING — ABSOLUTE DEATH-OVER PERFORMANCE")
print("=" * 70)


# ============================================================
# 1. INPUT VALIDATION
# ============================================================

print("\n1. INPUT VALIDATION")
print("=" * 70)

assert "player_events" in globals()
assert "bowling_provisional_rankings" in globals()

required = [
    "match_id",
    "bowler_id",
    "bowler_name",
    "match_phase",
    "is_legal_delivery",
    "runs",
    "is_four",
    "is_six",
    "extras_type",
    "bowler_credited_wicket_proxy"
]

missing = set(required) - set(player_events.columns)

assert not missing, f"Missing event columns: {missing}"

assert "bowler_id" in bowling_provisional_rankings.columns

events = player_events[required].copy()

events["bowler_id"] = pd.to_numeric(
    events["bowler_id"],
    errors="raise"
).astype("Int64")

for col in [
    "is_legal_delivery",
    "is_four",
    "is_six",
    "bowler_credited_wicket_proxy"
]:
    assert events[col].notna().all(), (
        f"Unexpected missing values in {col}"
    )

    events[col] = events[col].astype(bool)

events["runs"] = pd.to_numeric(
    events["runs"],
    errors="raise"
)

assert events["runs"].notna().all()
assert events["bowler_id"].notna().all()

print("Source bowling events:", len(events))
print("Unique bowlers:", events["bowler_id"].nunique())


# ============================================================
# 2. CONSTRUCT BOWLING OUTCOMES
# ============================================================

print("\n2. OUTCOME DEFINITIONS")
print("=" * 70)

events["legal_ball"] = events["is_legal_delivery"]

events["illegal_event"] = ~events["legal_ball"]

# Legal ball with no batting runs and no recorded extras.
# This is a strict clean-dot proxy.

events["clean_dot"] = (
    events["legal_ball"]
    & events["runs"].eq(0)
    & events["extras_type"].isna()
)

# Includes boundaries on no-balls and other illegal events.

events["batting_boundary"] = (
    events["is_four"] | events["is_six"]
)

# Includes verified wide-ball stumpings.

events["credited_wicket"] = (
    events["bowler_credited_wicket_proxy"]
)

assert not (
    events["clean_dot"] & events["illegal_event"]
).any()

print("Legal balls:", int(events["legal_ball"].sum()))
print("Clean dots:", int(events["clean_dot"].sum()))
print("Credited wickets:", int(events["credited_wicket"].sum()))
print("Batting boundaries:", int(events["batting_boundary"].sum()))


# ============================================================
# 3. ISOLATE DEATH OVERS
# ============================================================

print("\n3. DEATH-OVER COVERAGE")
print("=" * 70)

death = events.loc[
    events["match_phase"].eq("Death Overs")
].copy()

assert len(death) > 0

death_legal = int(death["legal_ball"].sum())

assert death_legal > 0

print("Death-over events:", len(death))
print("Death-over legal balls:", death_legal)
print("Death-over matches:", death["match_id"].nunique())
print("Death-over bowlers:", death["bowler_id"].nunique())


# ============================================================
# 4. DEATH-OVER PEER BASELINE
# ============================================================

print("\n4. DEATH-OVER PEER BASELINE")
print("=" * 70)

peer_rates = {
    "clean_dot": (
        100 * death["clean_dot"].sum() / death_legal
    ),
    "boundary": (
        100 * death["batting_boundary"].sum() / death_legal
    ),
    "wicket": (
        100 * death["credited_wicket"].sum() / death_legal
    ),
    "illegal": (
        100 * death["illegal_event"].sum() / death_legal
    )
}

peer_table = pd.DataFrame([
    {
        "measure": "Clean dots per 100 legal balls",
        "peer_rate": peer_rates["clean_dot"]
    },
    {
        "measure": "Batting boundaries per 100 legal balls",
        "peer_rate": peer_rates["boundary"]
    },
    {
        "measure": "Credited wickets per 100 legal balls",
        "peer_rate": peer_rates["wicket"]
    },
    {
        "measure": "Illegal events per 100 legal balls",
        "peer_rate": peer_rates["illegal"]
    }
])

display(peer_table.round(3))


# ============================================================
# 5. PLAYER-LEVEL DEATH-OVER OUTCOMES
# ============================================================

print("\n5. PLAYER-LEVEL DEATH PERFORMANCE")
print("=" * 70)

death_player_stats = (
    death.groupby("bowler_id", observed=True)
    .agg(
        bowler_name=("bowler_name", "first"),
        death_events=("match_id", "size"),
        death_matches=("match_id", "nunique"),
        death_legal_balls=("legal_ball", "sum"),
        death_clean_dots=("clean_dot", "sum"),
        death_boundaries=("batting_boundary", "sum"),
        death_wickets=("credited_wicket", "sum"),
        death_illegal_events=("illegal_event", "sum")
    )
    .reset_index()
)

death_player_stats = death_player_stats.loc[
    death_player_stats["death_legal_balls"] > 0
].copy()

# Explicit 60-ball smoothing strength.
# This reduces small-sample extremes toward the
# observed death-over peer rate.

PRIOR_BALLS = 60.0

outcomes = [
    ("clean_dots", "clean_dot"),
    ("boundaries", "boundary"),
    ("wickets", "wicket"),
    ("illegal_events", "illegal")
]

for count_suffix, peer_key in outcomes:

    count_col = f"death_{count_suffix}"

    raw_col = f"raw_{peer_key}_per_100"

    smooth_col = f"smoothed_{peer_key}_per_100"

    death_player_stats[raw_col] = (
        100
        * death_player_stats[count_col]
        / death_player_stats["death_legal_balls"]
    )

    death_player_stats[smooth_col] = (
        100
        * (
            death_player_stats[count_col]
            + PRIOR_BALLS * peer_rates[peer_key] / 100
        )
        / (
            death_player_stats["death_legal_balls"]
            + PRIOR_BALLS
        )
    )


# ============================================================
# 6. PERFORMANCE RELATIVE TO DEATH-OVER PEERS
# ============================================================

print("\n6. PEER-RELATIVE PERFORMANCE")
print("=" * 70)

# Positive values are better for all three measures.

death_player_stats["peer_dot_advantage_pp"] = (
    death_player_stats["smoothed_clean_dot_per_100"]
    - peer_rates["clean_dot"]
)

death_player_stats["peer_boundary_advantage_pp"] = (
    peer_rates["boundary"]
    - death_player_stats["smoothed_boundary_per_100"]
)

death_player_stats["peer_wicket_advantage_pp"] = (
    death_player_stats["smoothed_wicket_per_100"]
    - peer_rates["wicket"]
)

death_player_stats["positive_peer_dimensions"] = (
    (
        death_player_stats["peer_dot_advantage_pp"] > 0
    ).astype(int)

    + (
        death_player_stats["peer_boundary_advantage_pp"] > 0
    ).astype(int)

    + (
        death_player_stats["peer_wicket_advantage_pp"] > 0
    ).astype(int)
)

death_player_stats["death_peer_evidence"] = np.select(
    [
        (
            death_player_stats["death_legal_balls"] >= 120
        )
        & (
            death_player_stats["death_matches"] >= 10
        ),

        (
            death_player_stats["death_legal_balls"] >= 60
        )
        & (
            death_player_stats["death_matches"] >= 5
        )
    ],
    [
        "Broader death exposure",
        "Moderate death exposure"
    ],
    default="Limited death exposure"
)


# ============================================================
# 7. JOIN WITH EXISTING PROVISIONAL RANKINGS
# ============================================================

print("\n7. CANDIDATE CROSS-CHECK")
print("=" * 70)

candidate_comparison = (
    bowling_provisional_rankings.copy()
)

candidate_comparison["bowler_id"] = pd.to_numeric(
    candidate_comparison["bowler_id"],
    errors="raise"
).astype("Int64")

assert candidate_comparison["bowler_id"].is_unique
assert death_player_stats["bowler_id"].is_unique

# Remove duplicate name column from the right table.

comparison_stats = death_player_stats.drop(
    columns=["bowler_name"]
)

candidate_comparison = candidate_comparison.merge(
    comparison_stats,
    on="bowler_id",
    how="left",
    validate="one_to_one"
)

assert len(candidate_comparison) == len(
    bowling_provisional_rankings
)

print(
    "Candidates with death-over evidence:",
    int(
        candidate_comparison[
            "death_legal_balls"
        ].notna().sum()
    )
)

print(
    "Candidates with 60+ death-over balls:",
    int(
        candidate_comparison[
            "death_legal_balls"
        ].ge(60).sum()
    )
)

display(
    candidate_comparison[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "death_legal_balls",
            "death_matches",
            "peer_dot_advantage_pp",
            "peer_boundary_advantage_pp",
            "peer_wicket_advantage_pp",
            "positive_peer_dimensions",
            "death_peer_evidence"
        ]
    ]
    .head(20)
    .round(3)
)


# ============================================================
# 8. REVIEW ESTABLISHED HIGH-EXPOSURE BOWLERS
# ============================================================

print("\n8. HIGH-EXPOSURE BOWLER REVIEW")
print("=" * 70)

high_exposure_review = candidate_comparison.loc[
    candidate_comparison["death_legal_balls"].ge(100)
].copy()

high_exposure_review = high_exposure_review.sort_values(
    [
        "positive_peer_dimensions",
        "death_legal_balls"
    ],
    ascending=[False, False]
)

display(
    high_exposure_review[
        [
            "player_name",
            "provisional_rank",
            "death_legal_balls",
            "death_matches",
            "peer_dot_advantage_pp",
            "peer_boundary_advantage_pp",
            "peer_wicket_advantage_pp",
            "positive_peer_dimensions"
        ]
    ].round(3)
)


# ============================================================
# 9. FINAL CHECKS
# ============================================================

print("\n9. INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "candidate_row_difference": (
        len(candidate_comparison)
        - len(bowling_provisional_rankings)
    ),
    "duplicate_candidate_ids": int(
        candidate_comparison["bowler_id"].duplicated().sum()
    ),
    "duplicate_death_stat_ids": int(
        death_player_stats["bowler_id"].duplicated().sum()
    ),
    "negative_death_legal_balls": int(
        death_player_stats["death_legal_balls"].lt(0).sum()
    ),
    "death_legal_ball_reconciliation": int(
        death_player_stats["death_legal_balls"].sum()
        - death_legal
    )
}

checks_df = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(checks_df)

assert checks_df["count"].eq(0).all()

print("\nALL CHECKS PASSED")


# ============================================================
# 10. INTERPRETATION
# ============================================================

print("\nINTERPRETATION")
print("=" * 70)

print(
    "These are absolute death-over performance "
    "comparisons against the overall death-over pool."
)

print(
    "They complement, but do not replace, "
    "source-adjusted personal-baseline pressure effects."
)

print(
    "Death-over averages are not matched by opposition, "
    "venue, competition, bowling type, or match state."
)

print(
    "Smoothed rates use a stated 60-ball prior "
    "and are descriptive, not calibrated confidence intervals."
)

print(
    "No final player selections have been made. "
    "Original models remain unchanged."
)

In [ ]:
# ============================================================
# CELL 86 — FINAL SCOUTING DECISION BOARD
# Round 1: Most Reliable Players Under Pressure
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("ROUND 1 — FINAL SCOUTING DECISION BOARD")
print("=" * 70)


# ============================================================
# 1. VERIFY EXISTING EVIDENCE
# ============================================================

print("\n1. EVIDENCE AVAILABILITY")
print("=" * 70)

required_tables = [
    "batting_provisional_rankings",
    "bowling_provisional_rankings",
    "candidate_comparison"
]

missing_tables = [
    name for name in required_tables
    if name not in globals()
]

assert not missing_tables, (
    f"Missing tables: {missing_tables}. "
    "Run the corresponding earlier cells."
)

bat = batting_provisional_rankings.copy()
bowl = candidate_comparison.copy()

assert "provisional_rank" in bat.columns
assert "provisional_rank" in bowl.columns

assert bat["player_name"].is_unique
assert bowl["player_name"].is_unique

print("Batting candidates:", len(bat))
print("Bowling candidates:", len(bowl))


# ============================================================
# 2. VERIFIED PUBLIC FRANCHISE CONTEXT
# ============================================================

print("\n2. PUBLIC FRANCHISE CONTEXT")
print("=" * 70)

# These notes describe verified historical or announced
# franchise associations.
#
# A 2026 squad association does NOT establish a player's
# final 2027 retention or auction availability.
#
# Players not listed here are UNVERIFIED, not free agents.

public_player_context = {

    "Namboori Tilak Varma":
        "Mumbai Indians — IPL 2026 squad",

    "Sherfane Rutherford":
        "Mumbai Indians — IPL 2026 squad",

    "Jasprit Bumrah":
        "Mumbai Indians — IPL 2026 squad",

    "Dhruv Jurel":
        "Rajasthan Royals — IPL 2026 squad",

    "Shimron Hetmyer":
        "Rajasthan Royals — IPL 2026 squad",

    "Jofra Archer":
        "Rajasthan Royals — IPL 2026 squad",

    "Ravi Bishnoi":
        "Rajasthan Royals — IPL 2026 squad",

    "Sanju Samson":
        "Chennai Super Kings — IPL 2026 squad",

    "Ashutosh Sharma":
        "Delhi Capitals — IPL 2026 squad",

    "Arshdeep Singh":
        "Mumbai Indians — confirmed IPL 2027 trade",

    "Nathan Ellis":
        "Sunrisers Hyderabad — confirmed IPL 2027 trade",

    "Maheesh Theekshana":
        "Released by RR before IPL 2026; latest status unverified",

    "Haris Rauf":
        "Pakistan player; IPL 2027 registration unverified"
}


def franchise_context(player_name):

    return public_player_context.get(
        player_name,
        "2027 franchise/auction status not yet verified"
    )


bat["public_franchise_context"] = (
    bat["player_name"].map(franchise_context)
)

bowl["public_franchise_context"] = (
    bowl["player_name"].map(franchise_context)
)


# ============================================================
# 3. BATTING EVIDENCE CLASSIFICATION
# ============================================================

print("\n3. BATTING EVIDENCE CLASSIFICATION")
print("=" * 70)

# Evidence descriptions only.
# These are NOT new scores or eligibility restrictions.

bat["evidence_review"] = np.select(
    [
        (
            (bat["matched_balls"] >= 100)
            & (bat["positive_interval_contexts"] > 0)
            & (bat["pressure_sr_uplift"] > 0)
        ),

        (
            (bat["matched_balls"] >= 150)
            & (bat["pressure_sr_uplift"] > 0)
        ),

        (
            (bat["positive_interval_contexts"] > 0)
            & (bat["pressure_sr_uplift"] > 0)
        ),

        (
            (bat["pressure_sr_uplift"] > 0)
            & (bat["matched_balls"] < 75)
        )
    ],
    [
        "Positive interval; broader sample",
        "Broad sample; interval uncertainty",
        "Positive interval; smaller sample",
        "Exploratory uplift; limited sample"
    ],
    default="Additional cricketing review required"
)

bat["final_selection_status"] = "UNDER REVIEW"


# ============================================================
# 4. BATTING SCOUTING BOARD
# ============================================================

print("\n4. BATTING SCOUTING BOARD")
print("=" * 70)

batting_selection_review = bat.copy()

display(
    batting_selection_review[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "pressure_matches",
            "pressure_contexts",
            "pressure_sr_uplift",
            "positive_interval_contexts",
            "best_sensitivity_rank",
            "worst_sensitivity_rank",
            "evidence_review",
            "public_franchise_context"
        ]
    ].head(20).round(3)
)


# ============================================================
# 5. BATTING ALTERNATIVES OUTSIDE MODEL TOP 10
# ============================================================

print("\n5. BATTING ALTERNATIVES")
print("=" * 70)

batting_alternatives = (
    batting_selection_review.loc[
        (batting_selection_review["provisional_rank"] > 10)
        & (batting_selection_review["pressure_sr_uplift"] > 0)
    ]
    .sort_values("provisional_rank")
)

display(
    batting_alternatives[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "pressure_sr_uplift",
            "positive_interval_contexts",
            "evidence_review"
        ]
    ].round(3)
)


# ============================================================
# 6. BOWLING EVIDENCE CLASSIFICATION
# ============================================================

print("\n6. BOWLING EVIDENCE CLASSIFICATION")
print("=" * 70)

# The death-over comparison is an absolute peer benchmark.
# It is NOT equivalent to the personal-baseline model.

bowl["evidence_review"] = np.select(
    [
        (
            (bowl["death_legal_balls"] >= 150)
            & (bowl["positive_peer_dimensions"] == 3)
        ),

        (
            (bowl["death_legal_balls"] >= 200)
            & (bowl["positive_peer_dimensions"] >= 2)
        ),

        (
            (bowl["positive_peer_dimensions"] >= 2)
            & (bowl["matched_balls"] < 60)
        )
    ],
    [
        "Strong death-over peer evidence",
        "Broad death-over evidence",
        "Interesting signal; limited matched sample"
    ],
    default="Mixed evidence; cricketing review required"
)

bowl["final_selection_status"] = "UNDER REVIEW"


# ============================================================
# 7. BOWLING SCOUTING BOARD
# ============================================================

print("\n7. BOWLING SCOUTING BOARD")
print("=" * 70)

bowling_selection_review = bowl.copy()

display(
    bowling_selection_review[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "pressure_matches",
            "death_legal_balls",
            "death_matches",
            "positive_peer_dimensions",
            "peer_dot_advantage_pp",
            "peer_boundary_advantage_pp",
            "peer_wicket_advantage_pp",
            "evidence_review",
            "public_franchise_context"
        ]
    ].head(20).round(3)
)


# ============================================================
# 8. BOWLING CHALLENGERS OUTSIDE MODEL TOP 10
# ============================================================

print("\n8. BOWLING ALTERNATIVES")
print("=" * 70)

bowling_alternatives = (
    bowling_selection_review.loc[
        (bowling_selection_review["provisional_rank"] > 10)
        & (
            (bowling_selection_review["positive_peer_dimensions"] >= 2)
            | (bowling_selection_review["death_legal_balls"] >= 300)
        )
    ]
    .sort_values(
        [
            "positive_peer_dimensions",
            "death_legal_balls"
        ],
        ascending=[False, False]
    )
)

display(
    bowling_alternatives[
        [
            "provisional_rank",
            "player_name",
            "matched_balls",
            "death_legal_balls",
            "positive_peer_dimensions",
            "peer_dot_advantage_pp",
            "peer_boundary_advantage_pp",
            "peer_wicket_advantage_pp",
            "public_franchise_context"
        ]
    ].round(3)
)


# ============================================================
# 9. SCOUTING REVIEW SUMMARY
# ============================================================

print("\n9. SCOUTING REVIEW SUMMARY")
print("=" * 70)

summary = pd.DataFrame([
    {
        "role": "Batting",
        "model_candidates": len(batting_selection_review),
        "outside_top10_challengers": len(batting_alternatives),
        "final_selections": 0
    },
    {
        "role": "Bowling",
        "model_candidates": len(bowling_selection_review),
        "outside_top10_challengers": len(bowling_alternatives),
        "final_selections": 0
    }
])

display(summary)


# ============================================================
# 10. DECISION READINESS
# ============================================================

print("\n10. DECISION READINESS")
print("=" * 70)

checks = {
    "duplicate_batting_players": int(
        batting_selection_review["player_name"]
        .duplicated().sum()
    ),
    "duplicate_bowling_players": int(
        bowling_selection_review["player_name"]
        .duplicated().sum()
    ),
    "missing_batting_names": int(
        batting_selection_review["player_name"].isna().sum()
    ),
    "missing_bowling_names": int(
        bowling_selection_review["player_name"].isna().sum()
    ),
    "missing_batting_context": int(
        batting_selection_review[
            "public_franchise_context"
        ].isna().sum()
    ),
    "missing_bowling_context": int(
        bowling_selection_review[
            "public_franchise_context"
        ].isna().sum()
    )
}

checks_df = pd.DataFrame(
    list(checks.items()),
    columns=["check", "count"]
)

display(checks_df)

assert checks_df["count"].eq(0).all()

print("\nALL DECISION-BOARD CHECKS PASSED")


print("\nNEXT DECISION")
print("=" * 70)

print(
    "Review the top candidates alongside overlooked alternatives."
)

print(
    "Do not automatically exclude players because of "
    "nationality, international retirement or current IPL team."
)

print(
    "Historical squad information is not proof of "
    "2027 auction availability."
)

print(
    "Next: select and justify the final 10 batters "
    "and 10 bowlers."
)

print(
    "No original models or source tables were modified."
)

In [ ]:
# ============================================================
# CELL 87 — FINAL PRESSURE RELIABILITY EVIDENCE REVIEW
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("ROUND 1 — PRESSURE RELIABILITY EVIDENCE REVIEW")
print("=" * 70)

# ============================================================
# 1. VALIDATE EXISTING MODEL TABLES
# ============================================================

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()

bat_required = [
    "striker_id",
    "player_name",
    "batting_pressure_type",
    "source_matched_balls",
    "source_pressure_matches",
    "source_sample_reliable",
    "source_adjusted_sr_uplift",
    "sr_source_lower_95",
    "sr_source_upper_95"
]

bowl_required = [
    "bowler_id",
    "player_name",
    "bowling_pressure_type",
    "source_matched_balls",
    "source_pressure_matches",
    "source_sample_reliable",
    "source_adjusted_clean_dot_uplift_pp",
    "source_adjusted_boundary_suppression_pp",
    "source_adjusted_wicket_rate_uplift_pp",
    "clean_dot_source_lower_95",
    "boundary_source_lower_95",
    "wicket_source_lower_95"
]

assert set(bat_required).issubset(bat.columns), (
    "Missing batting columns: "
    + str(sorted(set(bat_required) - set(bat.columns)))
)

assert set(bowl_required).issubset(bowl.columns), (
    "Missing bowling columns: "
    + str(sorted(set(bowl_required) - set(bowl.columns)))
)

print("\n1. SOURCE COVERAGE")
print("=" * 70)

print("Batting profiles:", len(bat))
print("Bowling profiles:", len(bowl))

# Normalize identifier types to avoid mixed integer-factorizer errors.
for frame, id_col in [
    (bat, "striker_id"),
    (bowl, "bowler_id")
]:
    assert frame[id_col].notna().all()
    frame[id_col] = pd.to_numeric(
        frame[id_col], errors="raise"
    ).astype("int64")

def reliable_flag(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

for frame in [bat, bowl]:
    frame["is_reliable"] = reliable_flag(
        frame["source_sample_reliable"]
    )

    frame["source_matched_balls"] = pd.to_numeric(
        frame["source_matched_balls"],
        errors="coerce"
    )

    frame["source_pressure_matches"] = pd.to_numeric(
        frame["source_pressure_matches"],
        errors="coerce"
    )

# ============================================================
# 2. BATTING: CONTEXT-LEVEL EVIDENCE
# ============================================================

print("\n2. BATTING PRESSURE EVIDENCE")
print("=" * 70)

bat["sr_uplift"] = pd.to_numeric(
    bat["source_adjusted_sr_uplift"],
    errors="coerce"
)

bat["lower_95"] = pd.to_numeric(
    bat["sr_source_lower_95"],
    errors="coerce"
)

bat["upper_95"] = pd.to_numeric(
    bat["sr_source_upper_95"],
    errors="coerce"
)

bat["positive_interval"] = (
    bat["lower_95"] > 0
)

bat["negative_interval"] = (
    bat["upper_95"] < 0
)

bat["positive_point_estimate"] = (
    bat["sr_uplift"] > 0
)

bat_contexts = (
    bat.groupby("striker_id", as_index=False)
    .agg(
        batting_contexts=("batting_pressure_type", "nunique"),
        reliable_contexts=("is_reliable", "sum"),
        positive_interval_contexts=("positive_interval", "sum"),
        negative_interval_contexts=("negative_interval", "sum"),
        positive_point_contexts=("positive_point_estimate", "sum")
    )
)

# Choose the best-supported individual context per player.
# Do not add overlapping pressure-context ball counts.

bat_primary = (
    bat.sort_values(
        [
            "is_reliable",
            "source_matched_balls",
            "source_pressure_matches"
        ],
        ascending=[False, False, False],
        kind="stable"
    )
    .drop_duplicates("striker_id")
    .copy()
)

bat_primary = bat_primary.merge(
    bat_contexts,
    on="striker_id",
    how="left",
    validate="one_to_one"
)

bat_primary["mixed_direction"] = (
    (bat_primary["positive_point_contexts"] > 0)
    &
    (
        bat_primary["positive_point_contexts"]
        < bat_primary["batting_contexts"]
    )
)

bat_primary = bat_primary.sort_values(
    [
        "is_reliable",
        "positive_interval_contexts",
        "source_matched_balls",
        "source_pressure_matches"
    ],
    ascending=[False, False, False, False],
    kind="stable"
).reset_index(drop=True)

bat_primary["evidence_order"] = (
    np.arange(1, len(bat_primary) + 1)
)

bat_display = [
    "evidence_order",
    "player_name",
    "batting_pressure_type",
    "source_matched_balls",
    "source_pressure_matches",
    "sr_uplift",
    "lower_95",
    "upper_95",
    "is_reliable",
    "batting_contexts",
    "positive_interval_contexts",
    "mixed_direction"
]

print("\nTop 25 batters by evidence-review order:")
display(
    bat_primary[bat_display]
    .head(25)
    .round(3)
)

# ============================================================
# 3. BOWLING: THREE INDEPENDENT PERFORMANCE METRICS
# ============================================================

print("\n3. BOWLING PRESSURE EVIDENCE")
print("=" * 70)

bowl_metrics = {
    "clean_dot": (
        "source_adjusted_clean_dot_uplift_pp",
        "clean_dot_source_lower_95"
    ),
    "boundary": (
        "source_adjusted_boundary_suppression_pp",
        "boundary_source_lower_95"
    ),
    "wicket": (
        "source_adjusted_wicket_rate_uplift_pp",
        "wicket_source_lower_95"
    )
}

for label, (point_col, lower_col) in bowl_metrics.items():

    bowl[f"{label}_point"] = pd.to_numeric(
        bowl[point_col],
        errors="coerce"
    )

    bowl[f"{label}_lower"] = pd.to_numeric(
        bowl[lower_col],
        errors="coerce"
    )

    bowl[f"{label}_positive_point"] = (
        bowl[f"{label}_point"] > 0
    )

    bowl[f"{label}_positive_interval"] = (
        bowl[f"{label}_lower"] > 0
    )

point_flags = [
    "clean_dot_positive_point",
    "boundary_positive_point",
    "wicket_positive_point"
]

interval_flags = [
    "clean_dot_positive_interval",
    "boundary_positive_interval",
    "wicket_positive_interval"
]

bowl["positive_point_dimensions"] = (
    bowl[point_flags].sum(axis=1)
)

bowl["positive_interval_dimensions"] = (
    bowl[interval_flags].sum(axis=1)
)

bowl_contexts = (
    bowl.groupby("bowler_id", as_index=False)
    .agg(
        bowling_contexts=("bowling_pressure_type", "nunique"),
        reliable_contexts=("is_reliable", "sum"),
        contexts_with_positive_interval=(
            "positive_interval_dimensions",
            lambda x: int((x > 0).sum())
        )
    )
)

# Select a single best-supported pressure context.
# No averaging or double-counting overlapping contexts.

bowl_primary = (
    bowl.sort_values(
        [
            "is_reliable",
            "source_matched_balls",
            "source_pressure_matches"
        ],
        ascending=[False, False, False],
        kind="stable"
    )
    .drop_duplicates("bowler_id")
    .copy()
)

bowl_primary = bowl_primary.merge(
    bowl_contexts,
    on="bowler_id",
    how="left",
    validate="one_to_one"
)

# This is an evidence-review order, NOT a composite player score.
# Bowling dimensions remain independent.

bowl_primary = bowl_primary.sort_values(
    [
        "is_reliable",
        "positive_interval_dimensions",
        "positive_point_dimensions",
        "source_matched_balls",
        "source_pressure_matches"
    ],
    ascending=[False, False, False, False, False],
    kind="stable"
).reset_index(drop=True)

bowl_primary["evidence_order"] = (
    np.arange(1, len(bowl_primary) + 1)
)

bowl_display = [
    "evidence_order",
    "player_name",
    "bowling_pressure_type",
    "source_matched_balls",
    "source_pressure_matches",
    "clean_dot_point",
    "boundary_point",
    "wicket_point",
    "positive_point_dimensions",
    "positive_interval_dimensions",
    "is_reliable",
    "bowling_contexts"
]

print("\nTop 30 bowlers by evidence-review order:")
display(
    bowl_primary[bowl_display]
    .head(30)
    .round(3)
)

# ============================================================
# 4. EVIDENCE COVERAGE SUMMARY
# ============================================================

print("\n4. RELIABILITY SUMMARY")
print("=" * 70)

summary = pd.DataFrame([
    {
        "role": "Batting",
        "unique_players": len(bat_primary),
        "players_with_reliable_context": int(
            (bat_primary["reliable_contexts"] > 0).sum()
        ),
        "players_with_positive_interval": int(
            (bat_primary["positive_interval_contexts"] > 0).sum()
        )
    },
    {
        "role": "Bowling",
        "unique_players": len(bowl_primary),
        "players_with_reliable_context": int(
            (bowl_primary["reliable_contexts"] > 0).sum()
        ),
        "players_with_positive_interval": int(
            (bowl_primary["contexts_with_positive_interval"] > 0).sum()
        )
    }
])

display(summary)

# ============================================================
# 5. DATA INTEGRITY
# ============================================================

print("\n5. INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "batting_profile_count_difference":
        len(bat) - 351,

    "bowling_profile_count_difference":
        len(bowl) - 215,

    "duplicate_batting_primary_ids":
        int(bat_primary["striker_id"].duplicated().sum()),

    "duplicate_bowling_primary_ids":
        int(bowl_primary["bowler_id"].duplicated().sum()),

    "missing_batting_primary_names":
        int(bat_primary["player_name"].isna().sum()),

    "missing_bowling_primary_names":
        int(bowl_primary["player_name"].isna().sum())
}

display(
    pd.DataFrame(
        checks.items(),
        columns=["check", "count"]
    )
)

assert all(value == 0 for value in checks.values())

print("\nALL EVIDENCE-REVIEW CHECKS PASSED")

print("\nIMPORTANT:")
print(
    "Evidence order is not the final ranking. "
    "It prioritizes reliable profiles, positive intervals "
    "and exposure without inventing a composite score."
)

print(
    "Death-over peer evidence must be reviewed separately "
    "against these personal-pressure results."
)

print(
    "No player was excluded by nationality, "
    "age, franchise, retirement or auction availability."
)

print(
    "Original model tables remain unchanged. "
    "No final selections have been assigned."
)

In [ ]:
# ============================================================
# CELL 88 — FINAL SHORTLIST HEAD-TO-HEAD AUDIT
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 70)
print("ROUND 1 — FINAL SHORTLIST HEAD-TO-HEAD AUDIT")
print("=" * 70)

# ============================================================
# 1. PLAYER REVIEW POOLS
# ============================================================

current_batters = [
    "Namboori Tilak Varma",
    "Sherfane Rutherford",
    "Shai Hope",
    "Dhruv Jurel",
    "Rahul Tewatia",
    "Ashutosh Sharma",
    "Sanju Samson",
    "Shimron Hetmyer",
    "Heinrich Klaasen",
    "Towhid Hridoy"
]

batting_challengers = [
    "Nitish Rana",
    "Moeen Ali",
    "Rovman Powell",
    "David Miller",
    "Jason Holder",
    "Tim David",
    "Hardik Pandya"
]

current_bowlers = [
    "Rashid Khan",
    "Maheesh Theekshana",
    "Lungi Ngidi",
    "Tim Southee",
    "Haris Rauf",
    "Brad Evans",
    "Blessing Muzarabani",
    "Joshua Little",
    "Obed McCoy",
    "Mustafizur Rahman"
]

bowling_challengers = [
    "Ravi Bishnoi",
    "Trent Boult",
    "Mohammed Siraj",
    "Yash Dayal",
    "Mukesh Kumar",
    "Anrich Nortje",
    "Jasprit Bumrah",
    "Jofra Archer",
    "Prasidh Krishna",
    "Harshal Patel"
]

bat = batting_reliability_profiles.copy()
bowl = bowling_reliability_profiles.copy()

def normalize_name(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

def normalize_bool(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

for frame, player_id in [
    (bat, "striker_id"),
    (bowl, "bowler_id")
]:
    frame[player_id] = pd.to_numeric(
        frame[player_id],
        errors="raise"
    ).astype("int64")

    frame["player_key"] = normalize_name(
        frame["player_name"]
    )

    frame["reliable"] = normalize_bool(
        frame["source_sample_reliable"]
    )

    frame["source_matched_balls"] = pd.to_numeric(
        frame["source_matched_balls"],
        errors="coerce"
    )

    frame["source_pressure_matches"] = pd.to_numeric(
        frame["source_pressure_matches"],
        errors="coerce"
    )

print("\n1. REVIEW COVERAGE")
print("=" * 70)

print("Batting model profiles:", len(bat))
print("Bowling model profiles:", len(bowl))

# ============================================================
# 2. BATTING — ALL PRESSURE CONTEXTS
# ============================================================

print("\n2. BATTING HEAD-TO-HEAD")
print("=" * 70)

bat["uplift"] = pd.to_numeric(
    bat["source_adjusted_sr_uplift"],
    errors="coerce"
)

bat["lower"] = pd.to_numeric(
    bat["sr_source_lower_95"],
    errors="coerce"
)

bat["upper"] = pd.to_numeric(
    bat["sr_source_upper_95"],
    errors="coerce"
)

bat["reliable_positive_interval"] = (
    bat["reliable"] & (bat["lower"] > 0)
)

bat["reliable_negative_interval"] = (
    bat["reliable"] & (bat["upper"] < 0)
)

bat["reliable_positive_point"] = (
    bat["reliable"] & (bat["uplift"] > 0)
)

bat["reliable_negative_point"] = (
    bat["reliable"] & (bat["uplift"] < 0)
)

bat_summary = (
    bat.groupby("striker_id", as_index=False)
    .agg(
        player_name=("player_name", "first"),
        contexts=("batting_pressure_type", "nunique"),
        reliable_contexts=("reliable", "sum"),
        positive_intervals=("reliable_positive_interval", "sum"),
        negative_intervals=("reliable_negative_interval", "sum"),
        positive_points=("reliable_positive_point", "sum"),
        negative_points=("reliable_negative_point", "sum"),
        max_matched_balls=("source_matched_balls", "max"),
        max_pressure_matches=("source_pressure_matches", "max")
    )
)

bat_summary["player_key"] = normalize_name(
    bat_summary["player_name"]
)

bat_summary["review_group"] = np.select(
    [
        bat_summary["player_key"].isin(
            [x.lower() for x in current_batters]
        ),
        bat_summary["player_key"].isin(
            [x.lower() for x in batting_challengers]
        )
    ],
    ["Current Top 10", "Challenger"],
    default="Other"
)

bat_summary["mixed_reliable_direction"] = (
    (bat_summary["positive_points"] > 0)
    &
    (bat_summary["negative_points"] > 0)
)

bat_review = (
    bat_summary[
        bat_summary["review_group"] != "Other"
    ]
    .sort_values(
        [
            "positive_intervals",
            "reliable_contexts",
            "max_matched_balls"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(
    bat_review[
        [
            "review_group",
            "player_name",
            "contexts",
            "reliable_contexts",
            "positive_intervals",
            "negative_intervals",
            "positive_points",
            "negative_points",
            "max_matched_balls",
            "max_pressure_matches",
            "mixed_reliable_direction"
        ]
    ]
)

print("\nBatting context-level evidence:")

bat_detail_keys = set(bat_review["player_key"])

bat_details = bat[
    bat["player_key"].isin(bat_detail_keys)
].copy()

display(
    bat_details[
        [
            "player_name",
            "batting_pressure_type",
            "source_matched_balls",
            "source_pressure_matches",
            "uplift",
            "lower",
            "upper",
            "reliable",
            "reliable_positive_interval"
        ]
    ]
    .sort_values(
        ["player_name", "source_matched_balls"],
        ascending=[True, False]
    )
    .round(3)
)

# ============================================================
# 3. BOWLING — THREE PERSONAL PRESSURE METRICS
# ============================================================

print("\n3. BOWLING HEAD-TO-HEAD")
print("=" * 70)

bowling_metrics = {
    "dot": (
        "source_adjusted_clean_dot_uplift_pp",
        "clean_dot_source_lower_95"
    ),
    "boundary": (
        "source_adjusted_boundary_suppression_pp",
        "boundary_source_lower_95"
    ),
    "wicket": (
        "source_adjusted_wicket_rate_uplift_pp",
        "wicket_source_lower_95"
    )
}

for metric, (point_col, lower_col) in bowling_metrics.items():

    bowl[f"{metric}_point"] = pd.to_numeric(
        bowl[point_col],
        errors="coerce"
    )

    bowl[f"{metric}_lower"] = pd.to_numeric(
        bowl[lower_col],
        errors="coerce"
    )

    bowl[f"{metric}_positive"] = (
        bowl[f"{metric}_point"] > 0
    )

    bowl[f"{metric}_interval_positive"] = (
        bowl["reliable"]
        &
        (bowl[f"{metric}_lower"] > 0)
    )

bowl["positive_point_dimensions"] = (
    bowl[
        [
            "dot_positive",
            "boundary_positive",
            "wicket_positive"
        ]
    ].sum(axis=1)
)

bowl["positive_interval_dimensions"] = (
    bowl[
        [
            "dot_interval_positive",
            "boundary_interval_positive",
            "wicket_interval_positive"
        ]
    ].sum(axis=1)
)

# Choose the largest reliable pressure context for review.
# This does not aggregate overlapping contexts.

bowl_primary = (
    bowl.sort_values(
        [
            "reliable",
            "source_matched_balls",
            "source_pressure_matches"
        ],
        ascending=[False, False, False],
        kind="stable"
    )
    .drop_duplicates("bowler_id")
    .copy()
)

bowl_primary["review_group"] = np.select(
    [
        bowl_primary["player_key"].isin(
            [x.lower() for x in current_bowlers]
        ),
        bowl_primary["player_key"].isin(
            [x.lower() for x in bowling_challengers]
        )
    ],
    ["Current Top 10", "Challenger"],
    default="Other"
)

bowl_review = (
    bowl_primary[
        bowl_primary["review_group"] != "Other"
    ]
    .sort_values(
        [
            "positive_interval_dimensions",
            "positive_point_dimensions",
            "source_matched_balls"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(
    bowl_review[
        [
            "review_group",
            "player_name",
            "bowling_pressure_type",
            "source_matched_balls",
            "source_pressure_matches",
            "dot_point",
            "boundary_point",
            "wicket_point",
            "positive_point_dimensions",
            "positive_interval_dimensions",
            "reliable"
        ]
    ].round(3)
)

# ============================================================
# 4. BOWLING — OTHER PRESSURE CONTEXTS
# ============================================================

print("\n4. BOWLING CONTEXT DISAGREEMENTS")
print("=" * 70)

bowl_detail_keys = set(bowl_review["player_key"])

bowl_details = bowl[
    bowl["player_key"].isin(bowl_detail_keys)
].copy()

display(
    bowl_details[
        [
            "player_name",
            "bowling_pressure_type",
            "source_matched_balls",
            "source_pressure_matches",
            "dot_point",
            "boundary_point",
            "wicket_point",
            "positive_point_dimensions",
            "positive_interval_dimensions",
            "reliable"
        ]
    ]
    .sort_values(
        ["player_name", "source_matched_balls"],
        ascending=[True, False]
    )
    .round(3)
)

# ============================================================
# 5. RELIABLE-ONLY INTERVAL RECONCILIATION
# ============================================================

print("\n5. RELIABLE-ONLY CONFIDENCE CHECK")
print("=" * 70)

reconciliation = pd.DataFrame([
    {
        "role": "Batting",
        "reliable_profiles": int(bat["reliable"].sum()),
        "reliable_profiles_with_positive_interval": int(
            bat["reliable_positive_interval"].sum()
        )
    },
    {
        "role": "Bowling",
        "reliable_profiles": int(bowl["reliable"].sum()),
        "reliable_profiles_with_positive_interval": int(
            (
                bowl["positive_interval_dimensions"] > 0
            ).sum()
        )
    }
])

display(reconciliation)

# ============================================================
# 6. INTEGRITY
# ============================================================

print("\n6. INTEGRITY CHECKS")
print("=" * 70)

checks = {
    "batting_profile_difference": len(bat) - 351,
    "bowling_profile_difference": len(bowl) - 215,
    "duplicate_batting_summary_ids":
        int(bat_summary["striker_id"].duplicated().sum()),
    "duplicate_bowling_primary_ids":
        int(bowl_primary["bowler_id"].duplicated().sum()),
    "negative_source_ball_counts":
        int(
            (bat["source_matched_balls"] < 0).sum()
            +
            (bowl["source_matched_balls"] < 0).sum()
        )
}

display(
    pd.DataFrame(
        checks.items(),
        columns=["check", "count"]
    )
)

assert all(value == 0 for value in checks.values())

print("\nALL HEAD-TO-HEAD INTEGRITY CHECKS PASSED")

print("\nINTERPRETATION")
print("=" * 70)

print(
    "Positive point estimates, positive confidence intervals "
    "and consistency across contexts are separate evidence."
)

print(
    "Pressure contexts may overlap; exposure must not "
    "be added across contexts."
)

print(
    "These tables support human selection. "
    "They do not automatically define the final ranking."
)

print(
    "No filtering by IPL franchise, nationality, "
    "retirement, age or auction availability."
)

print(
    "Original models are unchanged. "
    "No final selections have been locked."
)

In [ ]:
# ============================================================
# STEP 2 — CELL 1
# PRESSURE FRAMEWORK: DATA SOURCE DISCOVERY
# Read-only inspection. Existing datasets remain unchanged.
# ============================================================

import pandas as pd
from IPython.display import display

required_columns = [
    "match_id",
    "innings",
    "bowler_id",
    "is_legal_delivery",
    "match_phase",
    "wickets_remaining_before",
    "balls_remaining_before",
    "wickets_last_6_balls",
    "wicket_shock_6",
    "rrr_parity",
    "rrr_scenario_spread",
    "pressure_reliability_group"
]

target_tables = [
    "player_events",
    "pressure_features",
    "chase_quality",
    "chase_state",
    "standard_state",
    "match_state"
]

inventory = []

# Search existing DataFrame variables
for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        if (
            name in target_tables
            or (
                "match_id" in obj.columns
                and "bowler_id" in obj.columns
            )
        ):

            inventory.append({
                "source": name,
                "type": "DataFrame",
                "rows": len(obj),
                "columns": len(obj.columns),
                "pressure_columns_available": sum(
                    col in obj.columns
                    for col in required_columns
                ),
                "has_phase": "match_phase" in obj.columns,
                "has_rrr": "rrr_parity" in obj.columns
            })

print("=" * 65)
print("STEP 2 — PRESSURE DATA SOURCE INVENTORY")
print("=" * 65)

if inventory:
    display(
        pd.DataFrame(inventory)
        .sort_values(
            "pressure_columns_available",
            ascending=False
        )
        .reset_index(drop=True)
    )
else:
    print("No matching DataFrame variables found.")

# Check common database connections without changing data
for connection_name in [
    "con", "conn", "duckdb_conn", "db"
]:

    connection = globals().get(connection_name)

    if connection is not None and hasattr(connection, "execute"):

        try:
            available_tables = connection.execute(
                "SHOW TABLES"
            ).fetchdf()

            print(
                f"\nDatabase tables via '{connection_name}':"
            )

            display(available_tables)

        except Exception as error:
            print(
                f"{connection_name}: "
                f"Could not inspect tables — {error}"
            )

print("\nDiscovery complete. No source data modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 2
# BOWLING PRESSURE DEFINITION RECONCILIATION
# Read-only validation. Existing models remain unchanged.
# ============================================================

import pandas as pd
from IPython.display import display

df = bowling_legal_scope

required = [
    "match_id",
    "innings",
    "is_legal_delivery",
    "match_phase",
    "wickets_remaining_before",
    "rrr_parity",
    "pressure_reliability_group"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 1. Normalize comparison values without changing source data
# ------------------------------------------------------------

legal = df["is_legal_delivery"].eq(True)

death = (
    df["match_phase"]
    .astype("string")
    .str.strip()
    .eq("Death Overs")
    .fillna(False)
)

second_innings = pd.to_numeric(
    df["innings"], errors="coerce"
).eq(2)

wickets_left = pd.to_numeric(
    df["wickets_remaining_before"],
    errors="coerce"
)

rrr = pd.to_numeric(
    df["rrr_parity"],
    errors="coerce"
)

reliability = (
    df["pressure_reliability_group"]
    .astype("string")
    .str.strip()
)

consistent_below = (
    reliability
    .eq("Consistent Below Threshold")
    .fillna(False)
)

# ------------------------------------------------------------
# 2. Reconstruct bowling challenge contexts
# ------------------------------------------------------------

death_4 = legal & death & wickets_left.ge(4)
death_5 = legal & death & wickets_left.ge(5)
death_6 = legal & death & wickets_left.ge(6)

# Live chase defence:
# Second innings + death overs + consistent below-threshold
# chase reconstruction + relevant required run rate.

live_8 = (
    legal & second_innings & death
    & consistent_below & rrr.le(8)
)

live_10 = (
    legal & second_innings & death
    & consistent_below & rrr.le(10)
)

live_12 = (
    legal & second_innings & death
    & consistent_below & rrr.le(12)
)

# ------------------------------------------------------------
# 3. Compare against earlier notebook checkpoints
# ------------------------------------------------------------

checks = [
    ("Death challenge: 4+ wickets", death_4, 22799),
    ("Death challenge: 5+ wickets", death_5, 17351),
    ("Death challenge: 6+ wickets", death_6, 10971),
    ("Live chase: RRR <= 8", live_8, 2590),
    ("Live chase: RRR <= 10", live_10, 3641),
    ("Live chase: RRR <= 12", live_12, 4244),
    ("Overlap: death 5+ and live 12",
     death_5 & live_12, 3278)
]

results = []

for name, mask, previous in checks:

    current = int(mask.sum())

    results.append({
        "context": name,
        "current_count": current,
        "previous_checkpoint": previous,
        "difference": current - previous,
        "status": "MATCH" if current == previous else "REVIEW"
    })

print("=" * 70)
print("STEP 2 — BOWLING PRESSURE DEFINITION VALIDATION")
print("=" * 70)

print(f"Source records: {len(df):,}")
print(f"Legal deliveries: {int(legal.sum()):,}")
print(f"Nonlegal records in scope: {int((~legal).sum()):,}")

display(pd.DataFrame(results))

# ------------------------------------------------------------
# 4. Inspect chase reconstruction reliability
# ------------------------------------------------------------

chase_death = legal & second_innings & death

reliability_summary = (
    reliability[chase_death]
    .value_counts(dropna=False)
    .rename_axis("reliability_group")
    .reset_index(name="legal_deliveries")
)

print("\nSECOND-INNINGS DEATH: RELIABILITY GROUPS")

display(reliability_summary)

# ------------------------------------------------------------
# 5. Check whether uncertain scenarios affect classification
# ------------------------------------------------------------

raw_live_12 = (
    legal & second_innings & death & rrr.le(12)
)

print("\nCHASE CLASSIFICATION UNCERTAINTY")

print(
    "RRR <= 12 before reliability filtering:",
    int(raw_live_12.sum())
)

print(
    "RRR <= 12 after reliability filtering:",
    int(live_12.sum())
)

print(
    "Excluded by reliability filter:",
    int((raw_live_12 & ~live_12).sum())
)

print("\nValidation complete. No source data modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 2
# BOWLING PRESSURE DEFINITION RECONCILIATION
# Read-only validation. Existing models remain unchanged.
# ============================================================

import pandas as pd
from IPython.display import display

df = bowling_legal_scope

required = [
    "match_id",
    "innings",
    "is_legal_delivery",
    "match_phase",
    "wickets_remaining_before",
    "rrr_parity",
    "pressure_reliability_group"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 1. Normalize comparison values without changing source data
# ------------------------------------------------------------

legal = df["is_legal_delivery"].eq(True)

death = (
    df["match_phase"]
    .astype("string")
    .str.strip()
    .eq("Death Overs")
    .fillna(False)
)

second_innings = pd.to_numeric(
    df["innings"], errors="coerce"
).eq(2)

wickets_left = pd.to_numeric(
    df["wickets_remaining_before"],
    errors="coerce"
)

rrr = pd.to_numeric(
    df["rrr_parity"],
    errors="coerce"
)

reliability = (
    df["pressure_reliability_group"]
    .astype("string")
    .str.strip()
)

consistent_below = (
    reliability
    .eq("Consistent Below Threshold")
    .fillna(False)
)

# ------------------------------------------------------------
# 2. Reconstruct bowling challenge contexts
# ------------------------------------------------------------

death_4 = legal & death & wickets_left.ge(4)
death_5 = legal & death & wickets_left.ge(5)
death_6 = legal & death & wickets_left.ge(6)

# Live chase defence:
# Second innings + death overs + consistent below-threshold
# chase reconstruction + relevant required run rate.

live_8 = (
    legal & second_innings & death
    & consistent_below & rrr.le(8)
)

live_10 = (
    legal & second_innings & death
    & consistent_below & rrr.le(10)
)

live_12 = (
    legal & second_innings & death
    & consistent_below & rrr.le(12)
)

# ------------------------------------------------------------
# 3. Compare against earlier notebook checkpoints
# ------------------------------------------------------------

checks = [
    ("Death challenge: 4+ wickets", death_4, 22799),
    ("Death challenge: 5+ wickets", death_5, 17351),
    ("Death challenge: 6+ wickets", death_6, 10971),
    ("Live chase: RRR <= 8", live_8, 2590),
    ("Live chase: RRR <= 10", live_10, 3641),
    ("Live chase: RRR <= 12", live_12, 4244),
    ("Overlap: death 5+ and live 12",
     death_5 & live_12, 3278)
]

results = []

for name, mask, previous in checks:

    current = int(mask.sum())

    results.append({
        "context": name,
        "current_count": current,
        "previous_checkpoint": previous,
        "difference": current - previous,
        "status": "MATCH" if current == previous else "REVIEW"
    })

print("=" * 70)
print("STEP 2 — BOWLING PRESSURE DEFINITION VALIDATION")
print("=" * 70)

print(f"Source records: {len(df):,}")
print(f"Legal deliveries: {int(legal.sum()):,}")
print(f"Nonlegal records in scope: {int((~legal).sum()):,}")

display(pd.DataFrame(results))

# ------------------------------------------------------------
# 4. Inspect chase reconstruction reliability
# ------------------------------------------------------------

chase_death = legal & second_innings & death

reliability_summary = (
    reliability[chase_death]
    .value_counts(dropna=False)
    .rename_axis("reliability_group")
    .reset_index(name="legal_deliveries")
)

print("\nSECOND-INNINGS DEATH: RELIABILITY GROUPS")

display(reliability_summary)

# ------------------------------------------------------------
# 5. Check whether uncertain scenarios affect classification
# ------------------------------------------------------------

raw_live_12 = (
    legal & second_innings & death & rrr.le(12)
)

print("\nCHASE CLASSIFICATION UNCERTAINTY")

print(
    "RRR <= 12 before reliability filtering:",
    int(raw_live_12.sum())
)

print(
    "RRR <= 12 after reliability filtering:",
    int(live_12.sum())
)

print(
    "Excluded by reliability filter:",
    int((raw_live_12 & ~live_12).sum())
)

print("\nValidation complete. No source data modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 3
# PRE-DELIVERY MATCH-STATE INTEGRITY
# Read-only validation. Existing datasets remain unchanged.
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

df = bowling_legal_scope

required = [
    "match_id",
    "innings",
    "is_legal_delivery",
    "legal_balls_before",
    "legal_balls_after",
    "balls_remaining_before",
    "balls_remaining_after",
    "wickets_before",
    "wickets_after",
    "is_confirmed_wicket",
    "wickets_remaining_before",
    "wickets_remaining_after",
    "match_phase"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Required columns missing: {missing}")

# Numeric comparisons without modifying original columns
num = {
    c: pd.to_numeric(df[c], errors="coerce")
    for c in required
    if c not in [
        "match_id",
        "match_phase",
        "is_legal_delivery"
    ]
}

confirmed_wicket = pd.to_numeric(
    df["is_confirmed_wicket"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. State consistency checks
# ------------------------------------------------------------

checks = {
    "legal_ball_progression": (
        num["legal_balls_after"]
        - num["legal_balls_before"]
    ).eq(1),

    "remaining_ball_progression": (
        num["balls_remaining_before"]
        - num["balls_remaining_after"]
    ).eq(1),

    "wicket_progression": (
        num["wickets_after"]
        - num["wickets_before"]
    ).eq(confirmed_wicket),

    "wickets_remaining_before": (
        num["wickets_before"]
        + num["wickets_remaining_before"]
    ).eq(10),

    "wickets_remaining_after": (
        num["wickets_after"]
        + num["wickets_remaining_after"]
    ).eq(10)
}

# ------------------------------------------------------------
# 2. Phase-label consistency for standard 120-ball innings
# ------------------------------------------------------------

if "scheduled_balls" in df.columns:

    scheduled = pd.to_numeric(
        df["scheduled_balls"],
        errors="coerce"
    )

    standard = scheduled.eq(120)

    before = num["legal_balls_before"]

    expected_phase = pd.Series(
        np.select(
            [
                before.lt(36),
                before.between(36, 95)
            ],
            [
                "Powerplay",
                "Middle Overs"
            ],
            default="Death Overs"
        ),
        index=df.index
    )

    phase_match = (
        df["match_phase"].astype("string")
        .str.strip()
        .eq(expected_phase)
        .fillna(False)
    )

    checks["phase_label_standard_innings"] = (
        phase_match[standard]
    )

# ------------------------------------------------------------
# 3. Report
# ------------------------------------------------------------

results = []

for check_name, mask in checks.items():

    evaluated = len(mask)
    passed = int(mask.fillna(False).sum())
    failed = evaluated - passed

    results.append({
        "check": check_name,
        "evaluated_deliveries": evaluated,
        "passed": passed,
        "failed": failed,
        "failure_pct": round(
            100 * failed / evaluated, 4
        ) if evaluated else 0,
        "status": (
            "PASS" if failed == 0 else "INVESTIGATE"
        )
    })

print("=" * 70)
print("STEP 2 — PRE-DELIVERY MATCH-STATE VALIDATION")
print("=" * 70)

display(pd.DataFrame(results))

# ------------------------------------------------------------
# 4. Show any inconsistent records
# ------------------------------------------------------------

for check_name, mask in checks.items():

    failures = mask.index[~mask.fillna(False)]

    if len(failures) > 0:

        print(
            f"\nEXAMPLES — {check_name}: "
            f"{len(failures):,} failures"
        )

        columns = [
            "match_id",
            "innings",
            "legal_balls_before",
            "legal_balls_after",
            "wickets_before",
            "wickets_after",
            "balls_remaining_before",
            "balls_remaining_after",
            "match_phase"
        ]

        display(
            df.loc[
                failures[:5],
                columns
            ]
        )

print("\nIntegrity audit complete.")
print("No source data or model results modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 4
# RECENT-WICKET TEMPORAL LEAKAGE DIAGNOSTIC
# Read-only validation. Existing models remain unchanged.
# ============================================================

import pandas as pd
from IPython.display import display

df = bowling_legal_scope

required = [
    "match_id",
    "innings",
    "legal_balls_before",
    "wickets_before",
    "wickets_after",
    "wickets_last_6_balls",
    "wickets_last_12_balls"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Work on a separate audit copy
audit = (
    df[required]
    .copy()
    .sort_values(
        ["match_id", "innings", "legal_balls_before"],
        kind="mergesort"
    )
)

numeric_cols = [
    "legal_balls_before",
    "wickets_before",
    "wickets_after",
    "wickets_last_6_balls",
    "wickets_last_12_balls"
]

for col in numeric_cols:
    audit[col] = pd.to_numeric(
        audit[col],
        errors="coerce"
    )

groups = audit.groupby(
    ["match_id", "innings"],
    sort=False
)

results = []

for window in [6, 12]:

    reported = audit[f"wickets_last_{window}_balls"]

    # Wickets during the preceding N legal-ball intervals,
    # excluding the current delivery.
    previous_state = (
        groups["wickets_before"]
        .shift(window)
        .fillna(0)
    )

    expected_before = (
        audit["wickets_before"] - previous_state
    )

    # Alternative: latest N deliveries including
    # the current delivery's result.
    current_window_start = (
        groups["wickets_before"]
        .shift(window - 1)
        .fillna(0)
    )

    expected_including_current = (
        audit["wickets_after"]
        - current_window_start
    )

    before_match = reported.eq(expected_before)
    current_match = reported.eq(
        expected_including_current
    )

    # These rows distinguish the two interpretations.
    distinguishing = (
        expected_before != expected_including_current
    )

    decisive_count = int(distinguishing.sum())

    before_decisive = int(
        (before_match & distinguishing).sum()
    )

    current_decisive = int(
        (current_match & distinguishing).sum()
    )

    results.append({
        "window": window,
        "total_deliveries": len(audit),
        "matches_before_definition": int(
            before_match.sum()
        ),
        "matches_including_current": int(
            current_match.sum()
        ),
        "distinguishing_deliveries": decisive_count,
        "before_matches_on_distinguishing": before_decisive,
        "current_matches_on_distinguishing": current_decisive,
        "before_match_pct": round(
            100 * before_match.mean(), 3
        ),
        "including_current_match_pct": round(
            100 * current_match.mean(), 3
        )
    })

    # Display examples where the definitions disagree
    if window == 6:

        examples = audit.loc[
            distinguishing,
            [
                "match_id",
                "innings",
                "legal_balls_before",
                "wickets_before",
                "wickets_after",
                "wickets_last_6_balls"
            ]
        ].head(10).copy()

        examples["expected_before"] = (
            expected_before.loc[examples.index]
        )

        examples["expected_including_current"] = (
            expected_including_current.loc[examples.index]
        )

print("=" * 70)
print("STEP 2 — RECENT-WICKET TEMPORAL LEAKAGE AUDIT")
print("=" * 70)

display(pd.DataFrame(results))

print("\nEXAMPLE DELIVERIES — DIFFERENT INTERPRETATIONS")

display(examples)

print("\nAudit complete. No source data modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 5
# RECENT-WICKET EXCEPTION INVESTIGATION
# Read-only. No source modification.
# ============================================================

import pandas as pd
from IPython.display import display

source = bowling_legal_scope

cols = [
    "match_id",
    "innings",
    "legal_balls_before",
    "wickets_before",
    "wickets_after",
    "wickets_last_6_balls",
    "wickets_last_12_balls",
    "match_phase"
]

audit_df = (
    source[cols]
    .copy()
    .sort_values(
        ["match_id", "innings", "legal_balls_before"],
        kind="mergesort"
    )
)

groups = audit_df.groupby(
    ["match_id", "innings"],
    sort=False
)

exceptions = []

for window in [6, 12]:

    previous_state = (
        groups["wickets_before"]
        .shift(window)
        .fillna(0)
    )

    expected = (
        audit_df["wickets_before"]
        - previous_state
    )

    reported = audit_df[
        f"wickets_last_{window}_balls"
    ]

    mismatch = reported.ne(expected)

    subset = audit_df.loc[mismatch, [
        "match_id",
        "innings",
        "legal_balls_before",
        "match_phase"
    ]].copy()

    subset["window"] = window
    subset["reported"] = reported.loc[mismatch]
    subset["expected_before"] = expected.loc[mismatch]

    subset["difference"] = (
        subset["reported"]
        - subset["expected_before"]
    )

    exceptions.append(subset)

exception_df = pd.concat(
    exceptions,
    ignore_index=True
)

# ------------------------------------------------------------
# Check nonlegal stumpings from full event history
# ------------------------------------------------------------

events = player_events

wide_stumpings = events.loc[
    (
        ~events["is_legal_delivery"].eq(True)
    )
    &
    (
        events["extras_type"]
        .astype("string")
        .str.lower()
        .eq("wide")
        .fillna(False)
    )
    &
    (
        events["dismissal_type"]
        .astype("string")
        .str.lower()
        .eq("stump")
        .fillna(False)
    ),
    ["match_id", "innings"]
].drop_duplicates()

wide_stumpings["wide_stumping_in_innings"] = True

exception_df = exception_df.merge(
    wide_stumpings,
    on=["match_id", "innings"],
    how="left"
)

exception_df["wide_stumping_in_innings"] = (
    exception_df["wide_stumping_in_innings"]
    .fillna(False)
    .astype(bool)
)

print("=" * 70)
print("STEP 2 — RECENT-WICKET EXCEPTION AUDIT")
print("=" * 70)

print(
    "Unique match-innings with wide stumpings:",
    len(wide_stumpings)
)

print("\nEXCEPTION SUMMARY")

display(
    exception_df.groupby("window").agg(
        exceptions=("difference", "size"),
        affected_matches=("match_id", "nunique"),
        associated_with_wide_stumping=(
            "wide_stumping_in_innings", "sum"
        )
    ).reset_index()
)

print("\nDIFFERENCE DISTRIBUTION")

display(
    exception_df.groupby(
        ["window", "difference"],
        dropna=False
    ).size().reset_index(name="events")
)

print("\nEXCEPTION RECORDS")

display(
    exception_df.sort_values(
        ["window", "match_id", "innings",
         "legal_balls_before"]
    ).head(40)
)

print("\nNo original records modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 6
# ROLLING-WINDOW BOUNDARY INVESTIGATION
# Read-only validation
# ============================================================

import pandas as pd
from IPython.display import display

df = (
    bowling_legal_scope
    .sort_values(
        ["match_id", "innings", "legal_balls_before"],
        kind="mergesort"
    )
)

groups = df.groupby(
    ["match_id", "innings"],
    sort=False
)

results = []

for window in [6, 12]:

    reported = df[
        f"wickets_last_{window}_balls"
    ]

    wickets_before = df["wickets_before"]

    # Standard preceding-window interpretation
    standard_start = (
        groups["wickets_before"]
        .shift(window)
        .fillna(0)
    )

    expected_standard = (
        wickets_before - standard_start
    )

    # Alternative: includes one additional
    # older legal-ball interval at the left boundary
    extended_start = (
        groups["wickets_before"]
        .shift(window + 1)
        .fillna(0)
    )

    expected_extended = (
        wickets_before - extended_start
    )

    original_mismatch = (
        reported != expected_standard
    )

    explained_by_extended = (
        original_mismatch
        & reported.eq(expected_extended)
    )

    remaining = (
        original_mismatch
        & ~explained_by_extended
    )

    results.append({
        "window": window,
        "original_exceptions": int(
            original_mismatch.sum()
        ),
        "explained_by_older_boundary": int(
            explained_by_extended.sum()
        ),
        "still_unexplained": int(
            remaining.sum()
        ),
        "explained_pct": round(
            100 * explained_by_extended.sum()
            / max(1, original_mismatch.sum()),
            2
        )
    })

print("=" * 70)
print("STEP 2 — ROLLING-WINDOW BOUNDARY TEST")
print("=" * 70)

display(pd.DataFrame(results))

print("\nNo source data modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 7
# BATTING PRESSURE AND DELIVERY ACCOUNTING
# Read-only validation
# ============================================================

import pandas as pd
from IPython.display import display

df = batting_scope

required = [
    "match_id",
    "innings",
    "is_legal_delivery",
    "extras_type",
    "chase_consensus_high_rrr",
    "pressure_reliability_group",
    "rrr_parity"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

legal = df["is_legal_delivery"].eq(True)
nonlegal = ~legal

high_rrr = (
    df["chase_consensus_high_rrr"]
    .eq(True)
    .fillna(False)
)

# ------------------------------------------------------------
# 1. Delivery accounting
# ------------------------------------------------------------

print("=" * 70)
print("STEP 2 — BATTING PRESSURE VALIDATION")
print("=" * 70)

print(f"Total batting-scope events: {len(df):,}")
print(f"Legal deliveries: {int(legal.sum()):,}")
print(f"Nonlegal events: {int(nonlegal.sum()):,}")

print("\nNONLEGAL DELIVERY CLASSIFICATION")

display(
    df.loc[nonlegal, "extras_type"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("extras_type")
    .reset_index(name="events")
)

# ------------------------------------------------------------
# 2. Batter-ball proxy accounting
# ------------------------------------------------------------

if "batter_ball_proxy" in df.columns:

    print("\nBATTER-BALL PROXY VS DELIVERY LEGALITY")

    display(
        pd.crosstab(
            legal.rename("legal_delivery"),
            df["batter_ball_proxy"],
            margins=True
        )
    )

else:
    print("\nBatter-ball proxy column unavailable.")

# ------------------------------------------------------------
# 3. High-RRR pressure eligibility
# ------------------------------------------------------------

print("\nHIGH-RRR PRESSURE BY INNINGS")

display(
    pd.crosstab(
        df["innings"],
        high_rrr,
        rownames=["innings"],
        colnames=["high_rrr_pressure"],
        margins=True
    )
)

# ------------------------------------------------------------
# 4. High-RRR pressure reliability
# ------------------------------------------------------------

print("\nHIGH-RRR PRESSURE BY RELIABILITY GROUP")

display(
    pd.crosstab(
        df["pressure_reliability_group"]
        .fillna("Not applicable"),
        high_rrr,
        rownames=["reliability_group"],
        colnames=["high_rrr_pressure"],
        margins=True
    )
)

# ------------------------------------------------------------
# 5. Pressure sample coverage
# ------------------------------------------------------------

pressure_events = df.loc[high_rrr]

print("\nHIGH-RRR PRESSURE COVERAGE")

print(
    "Pressure events:",
    len(pressure_events)
)

print(
    "Unique matches:",
    pressure_events["match_id"].nunique()
)

if "striker_id" in pressure_events.columns:

    print(
        "Unique batters:",
        pressure_events["striker_id"].nunique()
    )

print("\nREQUIRED RUN RATE DISTRIBUTION")

display(
    pd.to_numeric(
        pressure_events["rrr_parity"],
        errors="coerce"
    ).describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAudit complete. No source data modified.")

In [ ]:
# ============================================================
# STEP 2 — CELL 8
# HIGH-RRR PRESSURE SEVERITY AUDIT
# Read-only. Existing models remain unchanged.
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

source = batting_scope

high_rrr_mask = (
    source["chase_consensus_high_rrr"]
    .eq(True)
    .fillna(False)
)

audit = source.loc[high_rrr_mask].copy()

audit["rrr_numeric"] = pd.to_numeric(
    audit["rrr_parity"],
    errors="coerce"
)

audit["balls_left_numeric"] = pd.to_numeric(
    audit["balls_remaining_before"],
    errors="coerce"
)

audit["wickets_left_numeric"] = pd.to_numeric(
    audit["wickets_remaining_before"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Define diagnostic severity bands
# ------------------------------------------------------------

bins = [12, 15, 20, 30, 40, 60, np.inf]

labels = [
    "12–15",
    "15–20",
    "20–30",
    "30–40",
    "40–60",
    "60+"
]

audit["rrr_band"] = pd.cut(
    audit["rrr_numeric"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

# ------------------------------------------------------------
# 2. Coverage and match-state characteristics
# ------------------------------------------------------------

summary = (
    audit.groupby("rrr_band", observed=True)
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        batters=("striker_id", "nunique"),
        median_rrr=("rrr_numeric", "median"),
        median_balls_left=(
            "balls_left_numeric", "median"
        ),
        median_wickets_left=(
            "wickets_left_numeric", "median"
        )
    )
    .reset_index()
)

summary["delivery_pct"] = (
    100 * summary["deliveries"] / len(audit)
).round(2)

print("=" * 70)
print("STEP 2 — HIGH-RRR PRESSURE SEVERITY AUDIT")
print("=" * 70)

print(f"Total high-RRR events: {len(audit):,}")

display(summary)

# ------------------------------------------------------------
# 3. Severity by match phase
# ------------------------------------------------------------

print("\nRRR SEVERITY BY MATCH PHASE")

display(
    pd.crosstab(
        audit["match_phase"],
        audit["rrr_band"],
        margins=True
    )
)

# ------------------------------------------------------------
# 4. Extreme RRR exposure
# ------------------------------------------------------------

extreme = audit["rrr_numeric"].ge(40)

print("\nEXTREME PRESSURE EXPOSURE")

print(
    "RRR >= 40 deliveries:",
    int(extreme.sum())
)

print(
    "Share of high-RRR deliveries (%):",
    round(100 * extreme.mean(), 2)
)

print(
    "Matches with RRR >= 40:",
    audit.loc[extreme, "match_id"].nunique()
)

# ------------------------------------------------------------
# 5. Current top-10 candidate exposure
# ------------------------------------------------------------

top_batters = [
    "Tilak Varma",
    "Sherfane Rutherford",
    "Shai Hope",
    "Dhruv Jurel",
    "Rahul Tewatia",
    "Ashutosh Sharma",
    "Sanju Samson",
    "Shimron Hetmyer",
    "Hardik Pandya",
    "Towhid Hridoy"
]

top_audit = audit.loc[
    audit["striker_name"].isin(top_batters)
].copy()

top_audit["extreme_rrr"] = (
    top_audit["rrr_numeric"].ge(40)
)

player_summary = (
    top_audit.groupby("striker_name")
    .agg(
        raw_pressure_events=("match_id", "size"),
        matches=("match_id", "nunique"),
        median_rrr=("rrr_numeric", "median"),
        extreme_events=("extreme_rrr", "sum")
    )
    .reset_index()
)

player_summary["extreme_pct"] = (
    100
    * player_summary["extreme_events"]
    / player_summary["raw_pressure_events"]
).round(2)

print("\nTOP-10 BATTERS: RAW HIGH-RRR EXPOSURE")

display(
    player_summary.sort_values(
        "extreme_pct",
        ascending=False
    )
)

print(
    "\nNote: Raw event counts are not equivalent "
    "to source-matched reliable profile counts."
)

print("\nAudit complete. No source data modified.")

In [ ]:
# ==========================================================
# STEP 2 — CELL 9A
# CORRECTED TILAK VARMA PRESSURE EXPOSURE VALIDATION
# READ-ONLY
# ==========================================================

import pandas as pd

print("=" * 65)
print("STEP 2 — CORRECTED BATTER IDENTITY VALIDATION")
print("=" * 65)

pressure_col = "chase_consensus_high_rrr"

assert pressure_col in batting_scope.columns, (
    f"Missing expected pressure column: {pressure_col}"
)

# Identify Tilak using the dataset's full name
tilak_mask = (
    batting_scope["striker_name"]
    .astype("string")
    .str.contains(
        r"tilak.*varma",
        case=False,
        na=False,
        regex=True
    )
)

tilak_rows = batting_scope.loc[tilak_mask].copy()

print("\nTILAK VARMA — SOURCE RECORDS")

display(
    tilak_rows.groupby(
        ["striker_id", "striker_name"],
        dropna=False
    ).agg(
        total_events=("match_id", "size"),
        unique_matches=("match_id", "nunique"),
        high_rrr_events=(pressure_col, "sum")
    ).reset_index()
)

# Validate the pressure flag against our checkpoint
high_rrr = batting_scope.loc[
    batting_scope[pressure_col].eq(True)
].copy()

print("\nHIGH-RRR CHECKPOINT")

print("Total pressure events:", len(high_rrr))
print("Unique matches:", high_rrr["match_id"].nunique())
print("Unique batters:", high_rrr["striker_id"].nunique())

assert len(high_rrr) == 18745, (
    "Pressure count differs from validated checkpoint."
)

# Tilak's exposure to extreme required run rates
tilak_pressure = tilak_rows.loc[
    tilak_rows[pressure_col].eq(True)
]

print("\nTILAK VARMA — PRESSURE SEVERITY")

if len(tilak_pressure) > 0:

    print("High-RRR deliveries:", len(tilak_pressure))

    print(
        "Pressure matches:",
        tilak_pressure["match_id"].nunique()
    )

    print(
        "Median RRR:",
        round(tilak_pressure["rrr_parity"].median(), 2)
    )

    extreme = tilak_pressure["rrr_parity"].ge(40).sum()

    print("RRR >= 40 deliveries:", extreme)

    print(
        "Extreme exposure (%):",
        round(100 * extreme / len(tilak_pressure), 2)
    )

else:
    print("No high-RRR events found for this identity.")

print("\nValidation complete. No source data modified.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 10
# BATTING PRESSURE SEVERITY SENSITIVITY TEST
# READ-ONLY — ORIGINAL MODELS UNCHANGED
# ==============================================================

import pandas as pd
import numpy as np
import re

print("=" * 70)
print("STEP 2 — BATTING PRESSURE SENSITIVITY TEST")
print("=" * 70)

# --------------------------------------------------------------
# 1. Shortlisted batters
# --------------------------------------------------------------

shortlist = [
    "Tilak Varma",
    "Sherfane Rutherford",
    "Shai Hope",
    "Dhruv Jurel",
    "Rahul Tewatia",
    "Ashutosh Sharma",
    "Sanju Samson",
    "Shimron Hetmyer",
    "Hardik Pandya",
    "Towhid Hridoy"
]

required_columns = [
    "striker_id",
    "striker_name",
    "match_id",
    "chase_consensus_high_rrr",
    "rrr_parity",
    "batter_runs",
    "batter_ball_proxy"
]

missing = [
    c for c in required_columns
    if c not in batting_scope.columns
]

assert not missing, f"Missing columns: {missing}"

# --------------------------------------------------------------
# 2. Identify players using source player IDs
# --------------------------------------------------------------

identities = batting_scope[
    ["striker_id", "striker_name"]
].drop_duplicates()

player_ids = {}

print("\nPLAYER IDENTITY MATCHING")

for name in shortlist:

    match = identities.loc[
        identities["striker_name"]
        .astype("string")
        .str.contains(
            re.escape(name),
            case=False,
            na=False
        )
    ]

    unique_ids = match["striker_id"].dropna().unique()

    if len(unique_ids) != 1:
        print(f"REVIEW REQUIRED: {name}")
        display(match)
        continue

    player_ids[name] = unique_ids[0]

    print(
        f"{name}: ID {unique_ids[0]} | "
        f"{match['striker_name'].iloc[0]}"
    )

assert len(player_ids) == 10, (
    "Not all shortlisted players matched uniquely."
)

# --------------------------------------------------------------
# 3. Isolate validated high-RRR batting events
# --------------------------------------------------------------

pressure = batting_scope.loc[
    batting_scope["chase_consensus_high_rrr"].eq(True),
    required_columns
].copy()

assert len(pressure) == 18745, (
    "Pressure sample differs from checkpoint."
)

pressure["rrr_parity"] = pd.to_numeric(
    pressure["rrr_parity"],
    errors="coerce"
)

pressure["batter_runs"] = pd.to_numeric(
    pressure["batter_runs"],
    errors="coerce"
)

assert pressure["rrr_parity"].notna().all()
assert pressure["batter_runs"].notna().all()
assert pressure["batter_ball_proxy"].eq(True).all()

# --------------------------------------------------------------
# 4. Three pressure definitions
# --------------------------------------------------------------

scenarios = {
    "Original_RRR_12_plus": None,
    "Exclude_RRR_40_plus": 40,
    "Exclude_RRR_30_plus": 30
}

results = []

for player_name, player_id in player_ids.items():

    player_data = pressure.loc[
        pressure["striker_id"].eq(player_id)
    ]

    for scenario, upper_limit in scenarios.items():

        if upper_limit is None:
            sample = player_data
        else:
            sample = player_data.loc[
                player_data["rrr_parity"] < upper_limit
            ]

        balls = len(sample)
        runs = sample["batter_runs"].sum()

        results.append({
            "player": player_name,
            "scenario": scenario,
            "balls": balls,
            "matches": sample["match_id"].nunique(),
            "runs": runs,
            "strike_rate": (
                100 * runs / balls
                if balls > 0 else np.nan
            )
        })

sensitivity = pd.DataFrame(results)

# --------------------------------------------------------------
# 5. Compare scenarios
# --------------------------------------------------------------

sr_table = sensitivity.pivot(
    index="player",
    columns="scenario",
    values="strike_rate"
)

ball_table = sensitivity.pivot(
    index="player",
    columns="scenario",
    values="balls"
)

comparison = pd.DataFrame({
    "player": sr_table.index,

    "original_balls": ball_table[
        "Original_RRR_12_plus"
    ].values,

    "SR_original": sr_table[
        "Original_RRR_12_plus"
    ].values,

    "balls_below_40": ball_table[
        "Exclude_RRR_40_plus"
    ].values,

    "SR_below_40": sr_table[
        "Exclude_RRR_40_plus"
    ].values,

    "balls_below_30": ball_table[
        "Exclude_RRR_30_plus"
    ].values,

    "SR_below_30": sr_table[
        "Exclude_RRR_30_plus"
    ].values
})

comparison["SR_change_below_40"] = (
    comparison["SR_below_40"]
    - comparison["SR_original"]
)

comparison["SR_change_below_30"] = (
    comparison["SR_below_30"]
    - comparison["SR_original"]
)

comparison["removed_at_40"] = (
    comparison["original_balls"]
    - comparison["balls_below_40"]
)

comparison["removed_at_30"] = (
    comparison["original_balls"]
    - comparison["balls_below_30"]
)

comparison["small_sample_below_30"] = (
    comparison["balls_below_30"] < 60
)

print("\nPLAYER-LEVEL SENSITIVITY")

display(
    comparison
    .sort_values(
        "SR_change_below_30",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(2)
)

# --------------------------------------------------------------
# 6. Summary
# --------------------------------------------------------------

print("\nSENSITIVITY SUMMARY")

print(
    "Players tested:",
    comparison["player"].nunique()
)

print(
    "Players with fewer than 60 balls after RRR < 30:",
    comparison["small_sample_below_30"].sum()
)

print(
    "Largest absolute SR change (RRR < 40):",
    round(
        comparison["SR_change_below_40"].abs().max(),
        2
    )
)

print(
    "Largest absolute SR change (RRR < 30):",
    round(
        comparison["SR_change_below_30"].abs().max(),
        2
    )
)

print("\nNo original datasets or models modified.")

In [ ]:
# =============================================================
# STEP 2 — CELL 11
# MODEL VS RAW PRESSURE SAMPLE RECONCILIATION
# READ-ONLY
# =============================================================

import pandas as pd

print("=" * 70)
print("STEP 2 — MODEL SAMPLE RECONCILIATION")
print("=" * 70)

# Counts reported by our existing batting model
reported_model_counts = {
    "Tilak Varma": 239,
    "Sherfane Rutherford": 148,
    "Shai Hope": 60,
    "Dhruv Jurel": 89,
    "Rahul Tewatia": 125,
    "Ashutosh Sharma": 87,
    "Sanju Samson": 94,
    "Shimron Hetmyer": 117,
    "Hardik Pandya": 189,
    "Towhid Hridoy": 83
}

# Use the verified player IDs from Cell 10
rows = []

for name, player_id in player_ids.items():

    raw_count = (
        batting_scope["striker_id"].eq(player_id)
        & batting_scope["chase_consensus_high_rrr"].eq(True)
    ).sum()

    model_count = reported_model_counts[name]

    rows.append({
        "player": name,
        "player_id": player_id,
        "raw_pressure_events": int(raw_count),
        "reported_model_balls": model_count,
        "difference": int(raw_count - model_count),
        "model_coverage_pct": round(
            100 * model_count / raw_count, 2
        ) if raw_count else None
    })

audit = pd.DataFrame(rows)

print("\nRAW VS REPORTED MODEL SAMPLE")

display(
    audit.sort_values(
        "difference",
        ascending=False
    )
)

# Identify columns potentially involved in eligibility filtering
keywords = [
    "source",
    "reliab",
    "compet",
    "valid",
    "eligib",
    "consensus",
    "pressure"
]

candidate_columns = [
    c for c in batting_scope.columns
    if any(k in c.lower() for k in keywords)
]

print("\nPOTENTIAL ELIGIBILITY / SOURCE COLUMNS")
print(candidate_columns)

# Inspect low-cardinality eligibility fields for Tilak
tilak = batting_scope.loc[
    batting_scope["striker_id"].eq(7897)
    & batting_scope["chase_consensus_high_rrr"].eq(True)
]

print("\nTILAK — POTENTIAL FILTER DISTRIBUTIONS")

for col in candidate_columns:

    cardinality = tilak[col].nunique(dropna=False)

    if 1 < cardinality <= 12:

        print(f"\n{col}")

        display(
            tilak[col]
            .value_counts(dropna=False)
            .head(12)
        )

# Discover relevant existing model/profile tables
print("\nPOTENTIAL EXISTING MODEL TABLES")

model_keywords = [
    "bat",
    "rank",
    "profile",
    "score",
    "final"
]

found = []

for name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    if not any(k in name.lower() for k in model_keywords):
        continue

    relevant_columns = [
        c for c in obj.columns
        if any(
            k in c.lower()
            for k in ["player", "striker", "pressure", "score", "rank"]
        )
    ]

    found.append({
        "table_name": name,
        "rows": len(obj),
        "columns": len(obj.columns),
        "relevant_fields": ", ".join(
            relevant_columns[:8]
        )
    })

model_inventory = pd.DataFrame(found)

if not model_inventory.empty:
    display(
        model_inventory
        .sort_values("rows")
        .head(30)
    )

print("\nReconciliation diagnostic complete.")
print("No original datasets or models modified.")

In [ ]:
# ================================================================
# STEP 2 — CELL 12
# ORIGINAL BATTING MODEL RECONCILIATION
# READ-ONLY — NO MODEL OR SOURCE MODIFICATIONS
# ================================================================

import pandas as pd

print("=" * 70)
print("STEP 2 — ORIGINAL BATTING MODEL RECONCILIATION")
print("=" * 70)

# ------------------------------------------------
# 1. Verified player IDs
# ------------------------------------------------

target_players = {
    7897: "Tilak Varma",
    6725: "Sherfane Rutherford",
    4369: "Shai Hope",
    7260: "Dhruv Jurel",
    3926: "Rahul Tewatia",
    8101: "Ashutosh Sharma",
    2258: "Sanju Samson",
    3297: "Shimron Hetmyer",
    3330: "Hardik Pandya",
    6651: "Towhid Hridoy"
}

# ------------------------------------------------
# 2. Inspect original model tables
# ------------------------------------------------

model_tables = [
    "bat_top10",
    "bat_primary",
    "bat_reliable",
    "batting_leader_profiles"
]

priority_columns = [
    "provisional_rank",
    "striker_id",
    "player_id",
    "player_name",
    "batting_pressure_type",
    "pressure_context",
    "pressure_balls",
    "batting_pressure_balls",
    "pressure_matches",
    "reliable_pressure_balls",
    "baseline_balls",
    "source_matches"
]

for table_name in model_tables:

    print("\n" + "=" * 65)
    print("TABLE:", table_name)
    print("=" * 65)

    df = globals().get(table_name)

    if not isinstance(df, pd.DataFrame):
        print("Table unavailable")
        continue

    print("Shape:", df.shape)

    print("\nALL COLUMN NAMES")
    print(df.columns.tolist())

    # Find player identity column
    id_col = next(
        (
            c for c in ["striker_id", "player_id"]
            if c in df.columns
        ),
        None
    )

    if id_col is None:
        print("No recognized player ID column.")
        continue

    selected = df.loc[
        df[id_col].isin(target_players.keys())
    ].copy()

    print("\nMATCHED SHORTLIST ROWS:", len(selected))

    # Identify important model metrics
    useful_columns = [
        c for c in priority_columns
        if c in selected.columns
    ]

    additional_columns = [
        c for c in selected.columns
        if (
            any(
                token in c.lower()
                for token in [
                    "ball",
                    "uplift",
                    "pressure",
                    "sample",
                    "confidence",
                    "reliab"
                ]
            )
            and c not in useful_columns
        )
    ]

    display_columns = (
        useful_columns + additional_columns
    )[:20]

    if len(selected) > 0:
        display(
            selected[display_columns].head(30)
        )

    if "batting_pressure_type" in df.columns:

        print("\nPRESSURE CONTEXT DISTRIBUTION")

        display(
            df["batting_pressure_type"]
            .value_counts(dropna=False)
        )

# ------------------------------------------------
# 3. Examine potential eligibility filters
# ------------------------------------------------

print("\n" + "=" * 70)
print("RAW HIGH-RRR ELIGIBILITY FILTERS")
print("=" * 70)

pressure_mask = (
    batting_scope["chase_consensus_high_rrr"].eq(True)
)

filter_columns = [
    "eligible_pressure_match",
    "usable_pressure_context",
    "pressure_reliability_group",
    "reliable_rotation"
]

for col in filter_columns:

    if col not in batting_scope.columns:
        continue

    print(f"\nFILTER: {col}")

    display(
        batting_scope.loc[pressure_mask, col]
        .value_counts(dropna=False)
        .rename_axis("filter_value")
        .reset_index(name="events")
    )

# ------------------------------------------------
# 4. Player-specific eligibility comparison
# ------------------------------------------------

print("\n" + "=" * 70)
print("TILAK AND TEWATIA — FILTER BREAKDOWN")
print("=" * 70)

for player_id in [7897, 3926]:

    player_name = target_players[player_id]

    mask = (
        pressure_mask
        & batting_scope["striker_id"].eq(player_id)
    )

    player_rows = batting_scope.loc[mask]

    print(f"\nPLAYER: {player_name}")
    print("Raw high-RRR events:", len(player_rows))

    for col in filter_columns:

        if col not in player_rows.columns:
            continue

        print(f"\n{col}")

        display(
            player_rows[col]
            .value_counts(dropna=False)
            .rename_axis("value")
            .reset_index(name="events")
        )

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("Original datasets and models unchanged.")
print("=" * 70)

In [ ]:
# ==========================================================
# STEP 2 — CELL 13
# RANKING AND MODEL CONSISTENCY CHECKPOINT
# READ-ONLY
# ==========================================================

import pandas as pd

print("=" * 70)
print("STEP 2 — RANKING AND MODEL CONSISTENCY")
print("=" * 70)

# ----------------------------------------------------------
# 1. Display the complete current top-10 ranking
# ----------------------------------------------------------

ranking_columns = [
    "provisional_rank",
    "striker_id",
    "player_name",
    "reliable_profiles",
    "pressure_contexts",
    "matched_balls",
    "pressure_matches",
    "pressure_sr_uplift",
    "adjusted_batting_uplift",
    "batting_selection_score",
    "batting_evidence_status",
    "best_sensitivity_rank",
    "worst_sensitivity_rank"
]

available = [
    c for c in ranking_columns
    if c in bat_top10.columns
]

print("\nCURRENT NOTEBOOK TOP 10")

display(
    bat_top10[available]
    .sort_values("provisional_rank")
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# 2. Reconcile raw and model pressure counts
# ----------------------------------------------------------

model = bat_primary.loc[
    bat_primary["batting_pressure_type"]
    .eq("High Chase RRR Only")
].copy()

raw = (
    batting_scope.loc[
        batting_scope["chase_consensus_high_rrr"].eq(True)
    ]
    .groupby("striker_id")
    .size()
    .rename("raw_high_rrr_balls")
    .reset_index()
)

model = model.merge(
    raw,
    on="striker_id",
    how="left",
    validate="one_to_one"
)

source_columns = [
    "source_balls_IPL",
    "source_balls_T20I",
    "source_balls_Other T20"
]

model["source_component_total"] = (
    model[source_columns].sum(axis=1)
)

model["raw_minus_model"] = (
    model["raw_high_rrr_balls"]
    - model["pressure_balls"]
)

model["model_minus_source"] = (
    model["pressure_balls"]
    - model["source_matched_balls"]
)

model["source_component_difference"] = (
    model["source_matched_balls"]
    - model["source_component_total"]
)

focus_ids = [
    7897, 6725, 4369, 7260, 3926,
    8101, 2258, 3297, 3330, 6651
]

print("\nSHORTLIST — PRESSURE SAMPLE RECONCILIATION")

display(
    model.loc[
        model["striker_id"].isin(focus_ids),
        [
            "striker_id",
            "player_name",
            "raw_high_rrr_balls",
            "pressure_balls",
            "source_matched_balls",
            "source_balls_IPL",
            "source_balls_T20I",
            "source_balls_Other T20",
            "raw_minus_model",
            "model_minus_source",
            "source_component_difference"
        ]
    ]
    .sort_values("raw_minus_model", ascending=False)
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# 3. Identify available competition-source mappings
# ----------------------------------------------------------

print("\nCOMPETITION / SOURCE MAPPING FIELDS")

for table_name in [
    "batting_scope",
    "scout_base",
    "matches",
    "competitions"
]:

    df = globals().get(table_name)

    if not isinstance(df, pd.DataFrame):
        continue

    mapping_columns = [
        c for c in df.columns
        if any(
            term in c.lower()
            for term in [
                "competition",
                "source",
                "match_id",
                "tournament"
            ]
        )
    ]

    print(f"{table_name}: {mapping_columns}")

print("\nCheckpoint complete. No source data modified.")

In [ ]:
# ==========================================================
# STEP 2 — CELL 14
# PRESSURE-CONTEXT OVERLAP RECONCILIATION
# READ-ONLY
# ==========================================================

import pandas as pd

print("=" * 70)
print("STEP 2 — PRESSURE-CONTEXT OVERLAP RECONCILIATION")
print("=" * 70)

player_names = {
    7897: "Tilak Varma",
    6725: "Sherfane Rutherford",
    4369: "Shai Hope",
    7260: "Dhruv Jurel",
    3926: "Rahul Tewatia",
    8101: "Ashutosh Sharma",
    2258: "Sanju Samson",
    3297: "Shimron Hetmyer",
    3330: "Hardik Pandya",
    6651: "Towhid Hridoy"
}

# ----------------------------------------------------------
# 1. Compare raw high RRR against available model contexts
# ----------------------------------------------------------

raw_counts = (
    batting_scope.loc[
        batting_scope["chase_consensus_high_rrr"].eq(True)
    ]
    .groupby("striker_id")
    .size()
    .rename("raw_high_rrr")
)

high_only = (
    bat_primary.loc[
        bat_primary["batting_pressure_type"]
        .eq("High Chase RRR Only")
    ]
    .set_index("striker_id")["pressure_balls"]
    .rename("high_chase_only")
)

both_reliable = (
    bat_reliable.loc[
        bat_reliable["batting_pressure_type"].eq("Both")
    ]
    .groupby("striker_id")["pressure_balls"]
    .sum()
    .rename("reliable_both")
)

audit = pd.DataFrame({
    "striker_id": list(player_names.keys()),
    "player_name": list(player_names.values())
})

audit = (
    audit
    .join(raw_counts, on="striker_id")
    .join(high_only, on="striker_id")
    .join(both_reliable, on="striker_id")
)

audit["reliable_both"] = (
    audit["reliable_both"].fillna(0)
)

audit["unexplained_after_reliable_both"] = (
    audit["raw_high_rrr"]
    - audit["high_chase_only"]
    - audit["reliable_both"]
)

print("\nOVERLAP RECONCILIATION")

display(
    audit.sort_values(
        "unexplained_after_reliable_both",
        ascending=False
    )
)

print(
    "\nNote: Missing 'Both' profiles may be present "
    "outside bat_reliable."
)

# ----------------------------------------------------------
# 2. Inspect existing context-level tables
# ----------------------------------------------------------

tables = [
    "batting_context_counts",
    "bat_contexts",
    "batting_rate_pivot",
    "bat_details"
]

for table_name in tables:

    df = globals().get(table_name)

    print("\n" + "=" * 60)
    print("TABLE:", table_name)

    if not isinstance(df, pd.DataFrame):
        print("Unavailable")
        continue

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "striker_id" not in df.columns:
        continue

    selected = df.loc[
        df["striker_id"].isin(player_names.keys())
    ]

    relevant = [
        c for c in selected.columns
        if (
            c == "striker_id"
            or c == "player_name"
            or any(
                k in c.lower()
                for k in [
                    "pressure",
                    "context",
                    "wicket",
                    "chase",
                    "ball",
                    "rrr",
                    "both"
                ]
            )
        )
    ]

    print("Matched rows:", len(selected))

    display(
        selected[relevant[:14]].head(25)
    )

# ----------------------------------------------------------
# 3. Identify event-level wicket pressure fields
# ----------------------------------------------------------

print("\nEVENT-LEVEL WICKET / CONTEXT FIELDS")

relevant_event_columns = [
    c for c in batting_scope.columns
    if any(
        k in c.lower()
        for k in [
            "wicket",
            "recent",
            "collapse",
            "last_6",
            "last_12",
            "context"
        ]
    )
]

print(relevant_event_columns)

print("\nAudit complete. Original data and models unchanged.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 15
# EVENT-LEVEL PRESSURE-CONTEXT RECONSTRUCTION
# READ-ONLY — NO MODEL MODIFICATIONS
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 2 — PRESSURE-CONTEXT RECONSTRUCTION")
print("=" * 70)

required = [
    "striker_id",
    "match_id",
    "batter_runs",
    "rrr_parity",
    "chase_consensus_high_rrr",
    "batting_wicket_burden",
    "elevated_recent_shock"
]

missing = [c for c in required if c not in batting_scope.columns]
assert not missing, f"Missing columns: {missing}"

# --------------------------------------------------------------
# 1. Inspect wicket-pressure flags
# --------------------------------------------------------------

print("\nWICKET PRESSURE FLAG DISTRIBUTIONS")

for col in ["batting_wicket_burden", "elevated_recent_shock"]:

    print(f"\n{col}")
    print("Data type:", batting_scope[col].dtype)

    display(
        batting_scope[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="events")
        .head(15)
    )

# Check that both candidate fields are binary
binary_flags = {}

for col in ["batting_wicket_burden", "elevated_recent_shock"]:

    nonmissing = batting_scope[col].dropna()

    values = set(nonmissing.unique().tolist())

    if values.issubset({True, False, 0, 1}):
        binary_flags[col] = (
            batting_scope[col].fillna(False).astype(bool)
        )
    else:
        print(
            f"\nWARNING: {col} is not binary. "
            "It will not be treated as a Boolean flag."
        )

# --------------------------------------------------------------
# 2. Construct candidate pressure definitions
# --------------------------------------------------------------

candidates = {}

if "batting_wicket_burden" in binary_flags:
    candidates["Wicket burden"] = (
        binary_flags["batting_wicket_burden"]
    )

if "elevated_recent_shock" in binary_flags:
    candidates["Recent wicket shock"] = (
        binary_flags["elevated_recent_shock"]
    )

if len(binary_flags) == 2:

    candidates["Burden OR shock"] = (
        binary_flags["batting_wicket_burden"]
        | binary_flags["elevated_recent_shock"]
    )

# --------------------------------------------------------------
# 3. Original high-RRR-only model values
# --------------------------------------------------------------

target_ids = [
    7897, 6725, 4369, 7260, 3926,
    8101, 2258, 3297, 3330, 6651
]

model = bat_primary.loc[
    bat_primary["batting_pressure_type"]
    .eq("High Chase RRR Only")
    & bat_primary["striker_id"].isin(target_ids),
    [
        "striker_id",
        "player_name",
        "pressure_balls",
        "actual_pressure_runs"
    ]
].copy()

assert model["striker_id"].is_unique

high_rrr_flag = (
    batting_scope["chase_consensus_high_rrr"].eq(True)
)

# --------------------------------------------------------------
# 4. Compare candidates with the original model
# --------------------------------------------------------------

candidate_results = []
candidate_details = {}

for definition, wicket_flag in candidates.items():

    high_only_mask = high_rrr_flag & ~wicket_flag

    sample = batting_scope.loc[
        high_only_mask,
        ["striker_id", "match_id", "batter_runs", "rrr_parity"]
    ].copy()

    sample["batter_runs"] = pd.to_numeric(
        sample["batter_runs"],
        errors="coerce"
    )

    observed = (
        sample.groupby("striker_id")
        .agg(
            reconstructed_balls=("match_id", "size"),
            reconstructed_runs=("batter_runs", "sum")
        )
        .reset_index()
    )

    comparison = model.merge(
        observed,
        on="striker_id",
        how="left",
        validate="one_to_one"
    )

    comparison[[
        "reconstructed_balls",
        "reconstructed_runs"
    ]] = comparison[[
        "reconstructed_balls",
        "reconstructed_runs"
    ]].fillna(0)

    comparison["ball_difference"] = (
        comparison["reconstructed_balls"]
        - comparison["pressure_balls"]
    )

    comparison["run_difference"] = (
        comparison["reconstructed_runs"]
        - comparison["actual_pressure_runs"]
    )

    ball_matches = comparison["ball_difference"].eq(0)

    run_matches = np.isclose(
        comparison["run_difference"],
        0,
        atol=1e-8
    )

    candidate_results.append({
        "definition": definition,
        "players_tested": len(comparison),
        "exact_ball_matches": int(ball_matches.sum()),
        "exact_run_matches": int(run_matches.sum()),
        "total_absolute_ball_difference": int(
            comparison["ball_difference"].abs().sum()
        ),
        "total_absolute_run_difference": round(
            comparison["run_difference"].abs().sum(), 2
        )
    })

    candidate_details[definition] = comparison

print("\nMODEL RECONSTRUCTION COMPARISON")

candidate_summary = pd.DataFrame(candidate_results)

if not candidate_summary.empty:

    display(
        candidate_summary.sort_values(
            [
                "exact_ball_matches",
                "exact_run_matches"
            ],
            ascending=False
        )
    )

    # Show the best-fitting candidate for diagnosis,
    # without declaring it the correct methodology.
    best_definition = (
        candidate_summary.sort_values(
            [
                "total_absolute_ball_difference",
                "total_absolute_run_difference"
            ]
        ).iloc[0]["definition"]
    )

    print("\nBEST-FITTING CANDIDATE:", best_definition)

    display(
        candidate_details[best_definition][
            [
                "player_name",
                "pressure_balls",
                "reconstructed_balls",
                "ball_difference",
                "actual_pressure_runs",
                "reconstructed_runs",
                "run_difference"
            ]
        ]
    )

else:
    print(
        "No binary wicket-pressure flags available. "
        "Further definition inspection is required."
    )

print("\nNo source data or model results modified.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 16
# REMAINING PRESSURE-SAMPLE DISCREPANCY INVESTIGATION
# READ-ONLY
# ==============================================================

import pandas as pd

print("=" * 70)
print("STEP 2 — REMAINING 24-DELIVERY INVESTIGATION")
print("=" * 70)

focus_ids = [7897, 6725, 7260, 6651]

# --------------------------------------------------------------
# 1. Examine original model eligibility information
# --------------------------------------------------------------

model_focus = bat_primary.loc[
    bat_primary["striker_id"].isin(focus_ids)
    & bat_primary["batting_pressure_type"]
        .eq("High Chase RRR Only")
].copy()

model_columns = [
    "striker_id",
    "player_name",
    "qualified_strata",
    "pressure_balls",
    "actual_pressure_runs",
    "source_matched_balls",
    "source_balls_IPL",
    "source_balls_T20I",
    "source_balls_Other T20",
    "source_count",
    "matched_sources",
    "source_coverage_pct",
    "source_sample_reliable",
    "source_reliability_reason"
]

print("\nORIGINAL MODEL ELIGIBILITY")

display(
    model_focus[
        [c for c in model_columns if c in model_focus.columns]
    ]
)

# --------------------------------------------------------------
# 2. Reconstruct high-RRR-only candidate events
# --------------------------------------------------------------

mask = (
    batting_scope["striker_id"].isin(focus_ids)
    & batting_scope["chase_consensus_high_rrr"].eq(True)
    & batting_scope["batting_wicket_burden"].eq(False)
)

candidate_events = batting_scope.loc[mask].copy()

print("\nRECONSTRUCTED CANDIDATE EVENTS")

display(
    candidate_events.groupby(
        ["striker_id", "striker_name"],
        dropna=False
    ).agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        runs=("batter_runs", "sum"),
        median_rrr=("rrr_parity", "median")
    ).reset_index()
)

# --------------------------------------------------------------
# 3. Inspect innings and match-phase distribution
# --------------------------------------------------------------

print("\nEVENTS BY INNINGS AND MATCH PHASE")

display(
    candidate_events.groupby(
        [
            "striker_id",
            "striker_name",
            "innings",
            "match_phase"
        ],
        dropna=False
    ).agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        runs=("batter_runs", "sum")
    ).reset_index()
)

# --------------------------------------------------------------
# 4. Check nonlegal deliveries
# --------------------------------------------------------------

print("\nDELIVERY LEGALITY / EXTRAS")

if "extras_type" in candidate_events.columns:

    display(
        candidate_events.groupby(
            ["striker_id", "extras_type"],
            dropna=False
        ).size()
        .rename("events")
        .reset_index()
    )

if "legal_delivery" in candidate_events.columns:

    display(
        pd.crosstab(
            candidate_events["striker_id"],
            candidate_events["legal_delivery"],
            margins=True
        )
    )

# --------------------------------------------------------------
# 5. Locate existing competition/source mapping tables
# --------------------------------------------------------------

print("\nAVAILABLE SOURCE-MAPPING TABLES")

source_tables = []

for name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    if not any(
        k in name.lower()
        for k in ["source", "compet", "mapping", "match"]
    ):
        continue

    mapping_fields = [
        c for c in obj.columns
        if any(
            k in c.lower()
            for k in [
                "match_id",
                "source",
                "compet",
                "league",
                "tournament"
            ]
        )
    ]

    source_tables.append({
        "table": name,
        "rows": len(obj),
        "mapping_fields": ", ".join(mapping_fields[:10])
    })

source_inventory = pd.DataFrame(source_tables)

if not source_inventory.empty:

    display(
        source_inventory
        .sort_values("table")
        .head(35)
    )

print("\nInvestigation complete.")
print("No original data or models modified.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 17
# FINAL BATTING PRESSURE RECONCILIATION
# AND MODEL-CONSISTENT SEVERITY SENSITIVITY
# READ-ONLY
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 2 — FINAL BATTING PRESSURE RECONCILIATION")
print("=" * 70)

# --------------------------------------------------------------
# 1. Reconstruct the model's high-chase-only population
# --------------------------------------------------------------

high_chase_only_mask = (
    batting_scope["chase_consensus_high_rrr"].eq(True)
    & batting_scope["batting_wicket_burden"].eq(False)
    & batting_scope["match_phase"].isin(
        ["Middle Overs", "Death Overs"]
    )
)

reconstructed_events = batting_scope.loc[
    high_chase_only_mask
].copy()

reconstructed_events["batter_runs"] = pd.to_numeric(
    reconstructed_events["batter_runs"],
    errors="coerce"
)

# --------------------------------------------------------------
# 2. Compare original model balls and runs
# --------------------------------------------------------------

model = bat_primary.loc[
    bat_primary["batting_pressure_type"]
    .eq("High Chase RRR Only"),
    [
        "striker_id",
        "player_name",
        "pressure_balls",
        "actual_pressure_runs"
    ]
].copy()

reconstructed = (
    reconstructed_events.groupby("striker_id")
    .agg(
        reconstructed_balls=("match_id", "size"),
        reconstructed_runs=("batter_runs", "sum")
    )
    .reset_index()
)

audit = model.merge(
    reconstructed,
    on="striker_id",
    how="left",
    validate="one_to_one"
)

audit[[
    "reconstructed_balls",
    "reconstructed_runs"
]] = audit[[
    "reconstructed_balls",
    "reconstructed_runs"
]].fillna(0)

audit["balls_match"] = (
    audit["pressure_balls"]
    .eq(audit["reconstructed_balls"])
)

audit["runs_match"] = np.isclose(
    audit["actual_pressure_runs"],
    audit["reconstructed_runs"]
)

audit["fully_matched"] = (
    audit["balls_match"] & audit["runs_match"]
)

print("\nMODEL RECONCILIATION SUMMARY")

print("High-chase-only profiles:", len(audit))
print("Exact ball matches:", audit["balls_match"].sum())
print("Exact run matches:", audit["runs_match"].sum())
print("Both match:", audit["fully_matched"].sum())

# --------------------------------------------------------------
# 3. Validate the original ten investigated players
# --------------------------------------------------------------

investigated_ids = [
    7897, 6725, 4369, 7260, 3926,
    8101, 2258, 3297, 3330, 6651
]

investigated = audit.loc[
    audit["striker_id"].isin(investigated_ids)
]

print("\nORIGINAL TEN-PLAYER RECONCILIATION")

display(
    investigated[
        [
            "player_name",
            "pressure_balls",
            "reconstructed_balls",
            "actual_pressure_runs",
            "reconstructed_runs",
            "fully_matched"
        ]
    ].sort_values("player_name")
)

print(
    "Original ten fully reconciled:",
    bool(investigated["fully_matched"].all())
    and len(investigated) == 10
)

# --------------------------------------------------------------
# 4. Identify current top-10 players with this profile
# --------------------------------------------------------------

current_top10 = bat_top10[
    ["provisional_rank", "striker_id", "player_name"]
].copy()

eligible = current_top10.merge(
    model[["striker_id"]],
    on="striker_id",
    how="inner",
    validate="one_to_one"
)

not_eligible = current_top10.loc[
    ~current_top10["striker_id"].isin(
        eligible["striker_id"]
    )
]

print("\nCURRENT TOP-10 HIGH-CHASE-ONLY COVERAGE")

print("Players with high-chase-only model:", len(eligible))

if not not_eligible.empty:
    print("Players without this model profile:")
    display(not_eligible)

# --------------------------------------------------------------
# 5. Model-consistent severity sensitivity
# --------------------------------------------------------------

scenarios = {
    "Original": None,
    "RRR_below_40": 40,
    "RRR_below_30": 30
}

results = []

for row in eligible.itertuples(index=False):

    player_events = reconstructed_events.loc[
        reconstructed_events["striker_id"].eq(row.striker_id)
    ]

    for scenario_name, upper_rrr in scenarios.items():

        if upper_rrr is None:
            subset = player_events
        else:
            subset = player_events.loc[
                player_events["rrr_parity"] < upper_rrr
            ]

        balls = len(subset)
        runs = subset["batter_runs"].sum()

        results.append({
            "rank": row.provisional_rank,
            "player": row.player_name,
            "scenario": scenario_name,
            "balls": balls,
            "matches": subset["match_id"].nunique(),
            "strike_rate": (
                100 * runs / balls
                if balls else np.nan
            )
        })

sensitivity = pd.DataFrame(results)

sr = sensitivity.pivot(
    index=["rank", "player"],
    columns="scenario",
    values="strike_rate"
)

balls = sensitivity.pivot(
    index=["rank", "player"],
    columns="scenario",
    values="balls"
)

summary = pd.DataFrame({
    "original_balls": balls["Original"],
    "SR_original": sr["Original"],
    "balls_below_40": balls["RRR_below_40"],
    "SR_below_40": sr["RRR_below_40"],
    "balls_below_30": balls["RRR_below_30"],
    "SR_below_30": sr["RRR_below_30"]
})

summary["change_below_40"] = (
    summary["SR_below_40"] - summary["SR_original"]
)

summary["change_below_30"] = (
    summary["SR_below_30"] - summary["SR_original"]
)

print("\nMODEL-CONSISTENT PRESSURE SENSITIVITY")

display(
    summary.reset_index()
    .sort_values("rank")
    .round(2)
)

print("\nValidation complete.")
print("Original datasets and models unchanged.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 18
# PRESSURE STRATUM ELIGIBILITY RECONSTRUCTION
# READ-ONLY
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 2 — PRESSURE STRATUM ELIGIBILITY")
print("=" * 70)

# --------------------------------------------------------------
# 1. Candidate high-chase-only deliveries
# --------------------------------------------------------------

mask = (
    batting_scope["chase_consensus_high_rrr"].eq(True)
    & batting_scope["batting_wicket_burden"].eq(False)
)

base = batting_scope.loc[mask].copy()

base["batter_runs"] = pd.to_numeric(
    base["batter_runs"],
    errors="coerce"
)

assert base["batter_runs"].notna().all()

# Count each player's balls within each phase
base["player_phase_balls"] = (
    base.groupby(
        ["striker_id", "match_phase"]
    )["match_id"]
    .transform("size")
)

# --------------------------------------------------------------
# 2. Original model reference
# --------------------------------------------------------------

reference = bat_primary.loc[
    bat_primary["batting_pressure_type"]
    .eq("High Chase RRR Only"),
    [
        "striker_id",
        "player_name",
        "qualified_strata",
        "pressure_balls",
        "actual_pressure_runs"
    ]
].copy()

assert reference["striker_id"].is_unique

# --------------------------------------------------------------
# 3. Test possible minimum sample thresholds
# --------------------------------------------------------------

thresholds = [1, 5, 8, 9, 10, 11, 12, 15, 20]

threshold_results = {}
summary_rows = []

for minimum in thresholds:

    eligible_events = base.loc[
        base["player_phase_balls"] >= minimum
    ]

    observed = (
        eligible_events.groupby("striker_id")
        .agg(
            calculated_balls=("match_id", "size"),
            calculated_runs=("batter_runs", "sum"),
            calculated_strata=("match_phase", "nunique")
        )
        .reset_index()
    )

    comparison = reference.merge(
        observed,
        on="striker_id",
        how="left",
        validate="one_to_one"
    )

    for col in [
        "calculated_balls",
        "calculated_runs",
        "calculated_strata"
    ]:
        comparison[col] = comparison[col].fillna(0)

    comparison["balls_match"] = (
        comparison["calculated_balls"]
        .eq(comparison["pressure_balls"])
    )

    comparison["runs_match"] = np.isclose(
        comparison["calculated_runs"],
        comparison["actual_pressure_runs"]
    )

    comparison["strata_match"] = (
        comparison["calculated_strata"]
        .eq(comparison["qualified_strata"])
    )

    comparison["fully_matched"] = (
        comparison["balls_match"]
        & comparison["runs_match"]
        & comparison["strata_match"]
    )

    threshold_results[minimum] = comparison

    summary_rows.append({
        "minimum_phase_balls": minimum,
        "profiles_tested": len(comparison),
        "exact_ball_matches": int(
            comparison["balls_match"].sum()
        ),
        "exact_run_matches": int(
            comparison["runs_match"].sum()
        ),
        "exact_strata_matches": int(
            comparison["strata_match"].sum()
        ),
        "fully_matched_profiles": int(
            comparison["fully_matched"].sum()
        ),
        "total_ball_difference": int(
            (
                comparison["calculated_balls"]
                - comparison["pressure_balls"]
            ).abs().sum()
        )
    })

# --------------------------------------------------------------
# 4. Compare proposed thresholds
# --------------------------------------------------------------

threshold_summary = pd.DataFrame(summary_rows)

print("\nMINIMUM-STRATUM THRESHOLD COMPARISON")

display(
    threshold_summary.sort_values(
        [
            "fully_matched_profiles",
            "total_ball_difference"
        ],
        ascending=[False, True]
    )
)

# --------------------------------------------------------------
# 5. Examine the proposed 10-ball threshold
# --------------------------------------------------------------

print("\nDETAILED VALIDATION — MINIMUM 10 BALLS")

check_10 = threshold_results[10].copy()

display(
    check_10[
        [
            "player_name",
            "qualified_strata",
            "calculated_strata",
            "pressure_balls",
            "calculated_balls",
            "actual_pressure_runs",
            "calculated_runs",
            "fully_matched"
        ]
    ]
    .loc[lambda df: ~df["fully_matched"]]
    .head(25)
)

print(
    "\nExact matches with 10-ball minimum:",
    int(check_10["fully_matched"].sum()),
    "out of",
    len(check_10)
)

print("\nNo original datasets or models modified.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 19
# DIRECT BATTING MODEL SOURCE VERIFICATION
# READ-ONLY
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 2 — DIRECT MODEL SOURCE VERIFICATION")
print("=" * 70)

tables = [
    "batting_source_events",
    "batting_pressure_by_source",
    "batting_matched_source"
]

focus_ids = [
    7897, 6725, 4369, 7260, 3926,
    8101, 2258, 3297, 3330, 6651,
    2633, 4260
]

# --------------------------------------------------------------
# 1. Inspect actual source tables
# --------------------------------------------------------------

for name in tables:

    df = globals().get(name)

    print("\n" + "=" * 65)
    print("TABLE:", name)

    if not isinstance(df, pd.DataFrame):
        print("Table unavailable")
        continue

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    if "striker_id" in df.columns:

        selected = df.loc[
            df["striker_id"].isin(focus_ids)
        ]

        print("Matched player rows:", len(selected))

        preview_columns = [
            c for c in [
                "striker_id",
                "player_name",
                "batting_pressure_type",
                "innings",
                "match_phase",
                "match_id",
                "source_class",
                "batter_runs",
                "pressure_balls",
                "source_pressure_balls",
                "source_expected_runs"
            ]
            if c in selected.columns
        ]

        display(
            selected[preview_columns].head(15)
        )

# --------------------------------------------------------------
# 2. Attempt direct reconstruction from source events
# --------------------------------------------------------------

events = globals().get("batting_source_events")

required = [
    "striker_id",
    "batting_pressure_type",
    "match_id"
]

if (
    isinstance(events, pd.DataFrame)
    and all(c in events.columns for c in required)
):

    source_high = events.loc[
        events["batting_pressure_type"]
        .eq("High Chase RRR Only")
    ].copy()

    observed = (
        source_high.groupby("striker_id")
        .agg(
            source_event_count=("match_id", "size"),
            source_matches=("match_id", "nunique")
        )
        .reset_index()
    )

    reference = bat_primary.loc[
        bat_primary["batting_pressure_type"]
        .eq("High Chase RRR Only"),
        [
            "striker_id",
            "player_name",
            "pressure_balls",
            "source_matched_balls"
        ]
    ].copy()

    comparison = reference.merge(
        observed,
        on="striker_id",
        how="left",
        validate="one_to_one"
    )

    comparison["source_event_count"] = (
        comparison["source_event_count"].fillna(0)
    )

    comparison["matches_pressure_balls"] = (
        comparison["source_event_count"]
        .eq(comparison["pressure_balls"])
    )

    comparison["matches_source_balls"] = (
        comparison["source_event_count"]
        .eq(comparison["source_matched_balls"])
    )

    print("\nDIRECT EVENT COUNT RECONCILIATION")

    print("Profiles evaluated:", len(comparison))

    print(
        "Matches pressure_balls:",
        int(comparison["matches_pressure_balls"].sum())
    )

    print(
        "Matches source_matched_balls:",
        int(comparison["matches_source_balls"].sum())
    )

    print("\nCURRENT TOP-10 AND ALTERNATIVES")

    display(
        comparison.loc[
            comparison["striker_id"].isin(focus_ids),
            [
                "player_name",
                "pressure_balls",
                "source_matched_balls",
                "source_event_count",
                "matches_pressure_balls",
                "matches_source_balls"
            ]
        ]
    )

else:

    print(
        "\nSource-event table requires schema inspection "
        "before direct aggregation."
    )

print("\nVerification complete. No original data modified.")

In [ ]:
# ==============================================================
# STEP 2 — CELL 20
# SOURCE-EVENT RUN VALIDATION AND FINAL RRR SENSITIVITY
# READ-ONLY
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 2 — SOURCE-EVENT FINAL VALIDATION")
print("=" * 70)

# --------------------------------------------------------------
# 1. Actual model-selected high-chase events
# --------------------------------------------------------------

source_events = batting_source_events.loc[
    batting_source_events["batting_pressure_type"]
    .eq("High Chase RRR Only")
].copy()

source_events["batter_runs"] = pd.to_numeric(
    source_events["batter_runs"],
    errors="coerce"
)

source_events["rrr_parity"] = pd.to_numeric(
    source_events["rrr_parity"],
    errors="coerce"
)

assert source_events["batter_runs"].notna().all()
assert source_events["rrr_parity"].notna().all()

# --------------------------------------------------------------
# 2. Validate against original model totals
# --------------------------------------------------------------

model = bat_primary.loc[
    bat_primary["batting_pressure_type"]
    .eq("High Chase RRR Only"),
    [
        "striker_id",
        "player_name",
        "pressure_balls",
        "actual_pressure_runs",
        "source_matched_balls"
    ]
].copy()

observed = (
    source_events.groupby("striker_id")
    .agg(
        event_balls=("match_id", "size"),
        event_runs=("batter_runs", "sum"),
        event_matches=("match_id", "nunique")
    )
    .reset_index()
)

validation = model.merge(
    observed,
    on="striker_id",
    how="left",
    validate="one_to_one"
)

validation["balls_match"] = (
    validation["pressure_balls"].eq(
        validation["event_balls"]
    )
)

validation["runs_match"] = np.isclose(
    validation["actual_pressure_runs"],
    validation["event_runs"]
)

print("\nALL-PROFILE SOURCE VALIDATION")

print("Profiles:", len(validation))
print("Ball matches:", validation["balls_match"].sum())
print("Run matches:", validation["runs_match"].sum())

print("\nANY SOURCE VALIDATION FAILURES")

display(
    validation.loc[
        ~(
            validation["balls_match"]
            & validation["runs_match"]
        )
    ]
)

# --------------------------------------------------------------
# 3. Show source-matching differences separately
# --------------------------------------------------------------

print("\nPRESSURE VS SOURCE-MATCHED SAMPLE DIFFERENCES")

display(
    validation.loc[
        validation["pressure_balls"]
        .ne(validation["source_matched_balls"]),
        [
            "player_name",
            "pressure_balls",
            "source_matched_balls",
            "event_balls"
        ]
    ]
)

assert validation["balls_match"].all(), (
    "Source-event ball validation failed."
)

assert validation["runs_match"].all(), (
    "Source-event run validation failed."
)

# --------------------------------------------------------------
# 4. Final severity sensitivity — actual current top 10
# --------------------------------------------------------------

scenarios = {
    "Original": None,
    "RRR_below_40": 40,
    "RRR_below_30": 30
}

results = []

for player in bat_top10.itertuples(index=False):

    player_events = source_events.loc[
        source_events["striker_id"].eq(player.striker_id)
    ]

    for scenario_name, upper_limit in scenarios.items():

        if upper_limit is None:
            subset = player_events
        else:
            subset = player_events.loc[
                player_events["rrr_parity"] < upper_limit
            ]

        balls = len(subset)
        runs = subset["batter_runs"].sum()

        results.append({
            "rank": player.provisional_rank,
            "player": player.player_name,
            "scenario": scenario_name,
            "balls": balls,
            "matches": subset["match_id"].nunique(),
            "strike_rate": (
                100 * runs / balls if balls else np.nan
            )
        })

sensitivity = pd.DataFrame(results)

sr = sensitivity.pivot(
    index=["rank", "player"],
    columns="scenario",
    values="strike_rate"
)

balls = sensitivity.pivot(
    index=["rank", "player"],
    columns="scenario",
    values="balls"
)

summary = pd.DataFrame({
    "original_balls": balls["Original"],
    "SR_original": sr["Original"],
    "balls_below_40": balls["RRR_below_40"],
    "SR_below_40": sr["RRR_below_40"],
    "balls_below_30": balls["RRR_below_30"],
    "SR_below_30": sr["RRR_below_30"]
})

summary["SR_change_below_40"] = (
    summary["SR_below_40"] - summary["SR_original"]
)

summary["SR_change_below_30"] = (
    summary["SR_below_30"] - summary["SR_original"]
)

summary["removed_at_30"] = (
    summary["original_balls"]
    - summary["balls_below_30"]
)

summary["small_sample_below_30"] = (
    summary["balls_below_30"] < 60
)

print("\nFINAL MODEL-EVENT RRR SENSITIVITY")

display(
    summary.reset_index()
    .sort_values("rank")
    .round(2)
)

print("\nAll original datasets and models unchanged.")

In [ ]:
# ==============================================================
# STEP 3 — CELL 1
# CURRENT BATTING AND BOWLING LEADERBOARD AUDIT
# READ-ONLY
# ==============================================================

import pandas as pd

print("=" * 70)
print("STEP 3 — CURRENT LEADERBOARD AUDIT")
print("=" * 70)

# --------------------------------------------------------------
# 1. Current batting top 10
# --------------------------------------------------------------

print("\nCURRENT BATTING TOP 10")

batting_columns = [
    "provisional_rank",
    "striker_id",
    "player_name",
    "reliable_profiles",
    "pressure_contexts",
    "matched_balls",
    "pressure_matches",
    "positive_interval_contexts",
    "lowest_context_lower_95",
    "adjusted_batting_uplift",
    "batting_selection_score",
    "batting_evidence_status",
    "best_sensitivity_rank",
    "worst_sensitivity_rank",
    "sensitivity_rank_spread"
]

available_batting = [
    c for c in batting_columns
    if c in bat_top10.columns
]

display(
    bat_top10[available_batting]
    .sort_values("provisional_rank")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 2. Discover the current bowling ranking table
# --------------------------------------------------------------

print("\nAVAILABLE BOWLING RANKING TABLES")

bowling_tables = []

for name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    lower_name = name.lower()

    if (
        ("bowl" in lower_name or "bow" in lower_name)
        and any(
            token in lower_name
            for token in ["top", "rank", "selection"]
        )
    ):

        bowling_tables.append({
            "table": name,
            "rows": len(obj),
            "columns": len(obj.columns)
        })

if bowling_tables:
    display(pd.DataFrame(bowling_tables))

# --------------------------------------------------------------
# 3. Select an existing bowling leaderboard
# --------------------------------------------------------------

preferred_tables = [
    "bowl_top10",
    "bowling_top10",
    "bow_top10",
    "bowling_final_top10",
    "bowling_provisional_rankings"
]

bowling_board = None
selected_table_name = None

for name in preferred_tables:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        bowling_board = obj.copy()
        selected_table_name = name
        break

print("\nSELECTED BOWLING TABLE:", selected_table_name)

if bowling_board is not None:

    print("\nAVAILABLE BOWLING COLUMNS")
    print(bowling_board.columns.tolist())

    if "provisional_rank" in bowling_board.columns:

        bowling_board = (
            bowling_board
            .sort_values("provisional_rank")
            .head(10)
        )

    else:
        bowling_board = bowling_board.head(10)

    priority_bowling_columns = [
        "provisional_rank",
        "bowler_id",
        "player_id",
        "player_name",
        "bowler_name",
        "reliable_profiles",
        "pressure_contexts",
        "matched_balls",
        "pressure_legal_balls",
        "pressure_matches",
        "bowling_selection_score",
        "bowling_evidence_status",
        "best_sensitivity_rank",
        "worst_sensitivity_rank",
        "sensitivity_rank_spread"
    ]

    available_bowling = [
        c for c in priority_bowling_columns
        if c in bowling_board.columns
    ]

    display(
        bowling_board[available_bowling]
        .reset_index(drop=True)
    )

else:

    print(
        "No preferred bowling top-10 table found. "
        "Review the discovery table above."
    )

print("\nStep 3 leaderboard audit complete.")
print("No original rankings or models modified.")

In [ ]:
# ==============================================================
# STEP 3 — CELL 2
# TOP-10 EVIDENCE STRENGTH AND ALTERNATIVE AUDIT
# READ-ONLY
# ==============================================================

import pandas as pd

print("=" * 70)
print("STEP 3 — EVIDENCE STRENGTH AND ALTERNATIVES")
print("=" * 70)

# --------------------------------------------------------------
# 1. Batting top-10 evidence
# --------------------------------------------------------------

batting_fields = [
    "provisional_rank",
    "player_name",
    "matched_balls",
    "pressure_matches",
    "reliable_profiles",
    "pressure_contexts",
    "positive_interval_contexts",
    "lowest_context_lower_95",
    "batting_selection_score",
    "batting_evidence_status",
    "sensitivity_rank_spread"
]

bat_audit = bat_top10[
    [c for c in batting_fields if c in bat_top10.columns]
].copy()

bat_audit["small_sample_under_60"] = (
    bat_audit["matched_balls"] < 60
)

bat_audit["no_positive_interval"] = (
    bat_audit["positive_interval_contexts"] == 0
)

bat_audit["rank_spread_over_3"] = (
    bat_audit["sensitivity_rank_spread"] > 3
)

print("\nBATTING TOP 10 — EVIDENCE AUDIT")

display(
    bat_audit.sort_values("provisional_rank")
)

# --------------------------------------------------------------
# 2. Bowling top-10 evidence
# --------------------------------------------------------------

bowling_fields = [
    "provisional_rank",
    "player_name",
    "matched_balls",
    "pressure_matches",
    "reliable_profiles",
    "pressure_contexts",
    "dot_uplift_pp",
    "boundary_suppression_pp",
    "wicket_uplift_pp",
    "positive_dot_intervals",
    "positive_boundary_intervals",
    "positive_wicket_intervals",
    "positive_interval_dimensions",
    "bowling_selection_score",
    "bowling_evidence_status",
    "sensitivity_rank_spread"
]

bowl_audit = bowl_top10[
    [c for c in bowling_fields if c in bowl_top10.columns]
].copy()

bowl_audit["small_sample_under_60"] = (
    bowl_audit["matched_balls"] < 60
)

bowl_audit["no_positive_interval_dimension"] = (
    bowl_audit["positive_interval_dimensions"] == 0
)

bowl_audit["rank_spread_over_3"] = (
    bowl_audit["sensitivity_rank_spread"] > 3
)

print("\nBOWLING TOP 10 — EVIDENCE AUDIT")

display(
    bowl_audit.sort_values("provisional_rank")
)

# --------------------------------------------------------------
# 3. Batting alternatives — positions 11 to 20
# --------------------------------------------------------------

print("\nBATTING ALTERNATIVES — RANKS 11–20")

bat_alternatives = (
    batting_selection_review.loc[
        batting_selection_review["provisional_rank"]
        .between(11, 20)
    ]
    .sort_values("provisional_rank")
)

display(
    bat_alternatives[
        [
            c for c in batting_fields
            if c in bat_alternatives.columns
        ]
    ]
)

# --------------------------------------------------------------
# 4. Bowling alternatives — positions 11 to 20
# --------------------------------------------------------------

print("\nBOWLING ALTERNATIVES — RANKS 11–20")

bowl_alternatives = (
    bowling_selection_review.loc[
        bowling_selection_review["provisional_rank"]
        .between(11, 20)
    ]
    .sort_values("provisional_rank")
)

display(
    bowl_alternatives[
        [
            c for c in bowling_fields
            if c in bowl_alternatives.columns
        ]
    ]
)

# --------------------------------------------------------------
# 5. Summary of evidence warnings
# --------------------------------------------------------------

print("\nEVIDENCE WARNING SUMMARY")

summary = pd.DataFrame([
    {
        "discipline": "Batting",
        "top10_players": len(bat_audit),
        "under_60_balls": int(
            bat_audit["small_sample_under_60"].sum()
        ),
        "without_positive_interval": int(
            bat_audit["no_positive_interval"].sum()
        ),
        "rank_spread_over_3": int(
            bat_audit["rank_spread_over_3"].sum()
        )
    },
    {
        "discipline": "Bowling",
        "top10_players": len(bowl_audit),
        "under_60_balls": int(
            bowl_audit["small_sample_under_60"].sum()
        ),
        "without_positive_interval": int(
            bowl_audit[
                "no_positive_interval_dimension"
            ].sum()
        ),
        "rank_spread_over_3": int(
            bowl_audit["rank_spread_over_3"].sum()
        )
    }
])

display(summary)

print("\nAudit complete. No rankings modified.")

In [ ]:
# ==============================================================
# STEP 3 — CELL 3
# CONTEXT-LEVEL EVIDENCE AND CHALLENGER REVIEW
# READ-ONLY
# ==============================================================

import pandas as pd

print("=" * 70)
print("STEP 3 — CONTEXT-LEVEL EVIDENCE REVIEW")
print("=" * 70)

# --------------------------------------------------------------
# 1. Batting candidate pool: ranks 1–20
# --------------------------------------------------------------

bat_candidates = batting_selection_review.loc[
    batting_selection_review["provisional_rank"]
    .between(1, 20)
].copy()

bat_ids = bat_candidates["striker_id"].unique()

# Individual reliable pressure profiles
bat_profiles = bat_reliable.loc[
    bat_reliable["striker_id"].isin(bat_ids)
].copy()

bat_columns = [
    "striker_id",
    "player_name",
    "batting_pressure_type",
    "pressure_balls",
    "source_matched_balls",
    "source_pressure_matches",
    "sr_source_point",
    "sr_source_lower_95",
    "sr_source_upper_95",
    "source_sample_reliable",
    "positive_interval"
]

print("\nBATTING — CONTEXT-SPECIFIC PERFORMANCE")

display(
    bat_profiles[
        [c for c in bat_columns if c in bat_profiles.columns]
    ]
    .sort_values(
        ["player_name", "batting_pressure_type"]
    )
    .reset_index(drop=True)
    .round(2)
)

# --------------------------------------------------------------
# 2. Direct comparison: Holder vs Towhid
# --------------------------------------------------------------

print("\nDIRECT CHALLENGER COMPARISON: HOLDER VS TOWHID")

holder_towhid = bat_profiles.loc[
    bat_profiles["striker_id"].isin([2633, 6651])
]

display(
    holder_towhid[
        [c for c in bat_columns if c in holder_towhid.columns]
    ]
    .round(2)
)

# --------------------------------------------------------------
# 3. Discover detailed bowling profile columns
# --------------------------------------------------------------

print("\nBOWLING PROFILE SCHEMA")

bowling_profiles = globals().get(
    "bowling_leader_profiles"
)

if isinstance(bowling_profiles, pd.DataFrame):

    print("Shape:", bowling_profiles.shape)
    print("Columns:")
    print(bowling_profiles.columns.tolist())

    # Select bowlers ranked 1–20
    bowl_candidates = bowling_selection_review.loc[
        bowling_selection_review["provisional_rank"]
        .between(1, 20)
    ]

    if "bowler_id" in bowling_profiles.columns:

        bowl_ids = bowl_candidates["bowler_id"].unique()

        selected_bowl_profiles = bowling_profiles.loc[
            bowling_profiles["bowler_id"].isin(bowl_ids)
        ]

        priority = [
            "bowler_id",
            "player_name",
            "bowling_pressure_type",
            "pressure_legal_balls",
            "source_matched_balls",
            "source_pressure_matches",
            "source_sample_reliable"
        ]

        interval_columns = [
            c for c in selected_bowl_profiles.columns
            if (
                "lower_95" in c.lower()
                or "positive_interval" in c.lower()
            )
        ]

        selected_columns = (
            [c for c in priority
             if c in selected_bowl_profiles.columns]
            + interval_columns[:9]
        )

        print("\nBOWLING — CONTEXT-SPECIFIC EVIDENCE")

        display(
            selected_bowl_profiles[selected_columns]
            .head(35)
            .round(3)
        )

else:

    print("Bowling leader profiles not available.")

print("\nEvidence review complete.")
print("No original rankings or models modified.")

In [ ]:
# ==============================================================
# STEP 3 — CELL 4
# EVIDENCE-AWARE CANDIDATE COMPARISON
# READ-ONLY
# ==============================================================

import pandas as pd

print("=" * 70)
print("STEP 3 — EVIDENCE-AWARE CANDIDATE COMPARISON")
print("=" * 70)

# --------------------------------------------------------------
# 1. Batting: count positive and negative context evidence
# --------------------------------------------------------------

bat_context = bat_reliable.copy()

bat_context["positive_point"] = (
    bat_context["sr_source_point"] > 0
)

bat_context["positive_interval"] = (
    bat_context["sr_source_lower_95"] > 0
)

bat_context["negative_interval"] = (
    bat_context["sr_source_upper_95"] < 0
)

bat_evidence = (
    bat_context.groupby("striker_id")
    .agg(
        reliable_contexts=("batting_pressure_type", "nunique"),
        positive_point_contexts=("positive_point", "sum"),
        positive_interval_contexts=("positive_interval", "sum"),
        negative_interval_contexts=("negative_interval", "sum"),
        weakest_lower_bound=("sr_source_lower_95", "min"),
        strongest_lower_bound=("sr_source_lower_95", "max")
    )
    .reset_index()
)

bat_candidates = (
    batting_selection_review.loc[
        batting_selection_review["provisional_rank"]
        .between(1, 20)
    ]
    .merge(
        bat_evidence,
        on="striker_id",
        how="left",
        validate="one_to_one",
        suffixes=("", "_context")
    )
)

print("\nBATTING CANDIDATE EVIDENCE — RANKS 1–20")

bat_show = [
    "provisional_rank",
    "player_name",
    "batting_selection_score",
    "matched_balls",
    "pressure_matches",
    "reliable_contexts",
    "positive_point_contexts",
    "positive_interval_contexts_context",
    "negative_interval_contexts",
    "weakest_lower_bound",
    "strongest_lower_bound",
    "sensitivity_rank_spread"
]

display(
    bat_candidates[
        [c for c in bat_show if c in bat_candidates.columns]
    ]
    .sort_values("provisional_rank")
    .round(2)
)

# --------------------------------------------------------------
# 2. Bowling: source-adjusted outcome evidence
# --------------------------------------------------------------

bowl_context = bowling_leader_profiles.copy()

positive_lower_columns = [
    "clean_dot_source_lower_95",
    "boundary_source_lower_95",
    "wicket_source_lower_95"
]

negative_upper_columns = [
    "clean_dot_source_upper_95",
    "boundary_source_upper_95",
    "wicket_source_upper_95"
]

bowl_context["positive_outcomes"] = (
    bowl_context[positive_lower_columns]
    .gt(0)
    .sum(axis=1)
)

bowl_context["negative_outcomes"] = (
    bowl_context[negative_upper_columns]
    .lt(0)
    .sum(axis=1)
)

bowl_evidence = (
    bowl_context.groupby("bowler_id")
    .agg(
        reliable_contexts=("bowling_pressure_type", "nunique"),
        positive_outcome_intervals=("positive_outcomes", "sum"),
        negative_outcome_intervals=("negative_outcomes", "sum"),
        source_pressure_matches=("source_pressure_matches", "max")
    )
    .reset_index()
)

bowl_candidates = (
    bowling_selection_review.loc[
        bowling_selection_review["provisional_rank"]
        .between(1, 20)
    ]
    .merge(
        bowl_evidence,
        on="bowler_id",
        how="left",
        validate="one_to_one",
        suffixes=("", "_context")
    )
)

print("\nBOWLING CANDIDATE EVIDENCE — RANKS 1–20")

bowl_show = [
    "provisional_rank",
    "player_name",
    "bowling_selection_score",
    "matched_balls",
    "pressure_matches",
    "reliable_contexts",
    "positive_outcome_intervals",
    "negative_outcome_intervals",
    "sensitivity_rank_spread"
]

display(
    bowl_candidates[
        [c for c in bowl_show if c in bowl_candidates.columns]
    ]
    .sort_values("provisional_rank")
    .round(3)
)

print("\nComparison complete.")
print("No original rankings or models modified.")

In [ ]:
# ==============================================================
# STEP 3 — CELL 5
# HOLDER VS HRIDOY — SOURCE-ADJUSTED RRR SENSITIVITY
# READ-ONLY
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 3 — HOLDER VS HRIDOY SENSITIVITY")
print("=" * 70)

player_ids = {
    2633: "Jason Holder",
    6651: "Towhid Hridoy"
}

context = "High Chase RRR Only"

join_keys = [
    "striker_id",
    "batting_pressure_type",
    "innings",
    "match_phase",
    "source_class"
]

# --------------------------------------------------------------
# 1. Original source-matched pressure events
# --------------------------------------------------------------

events = batting_source_events.loc[
    batting_source_events["striker_id"].isin(player_ids)
    & batting_source_events["batting_pressure_type"].eq(context)
].copy()

# --------------------------------------------------------------
# 2. Original source-specific reference rates
# --------------------------------------------------------------

reference = batting_matched_source.loc[
    batting_matched_source["striker_id"].isin(player_ids)
    & batting_matched_source["batting_pressure_type"].eq(context),
    join_keys + [
        "pressure_balls",
        "source_expected_runs"
    ]
].copy()

assert not reference.duplicated(join_keys).any(), (
    "Reference join keys are not unique."
)

# Expected runs per delivery within each matched stratum
reference["expected_per_ball"] = (
    reference["source_expected_runs"]
    / reference["pressure_balls"]
)

# --------------------------------------------------------------
# 3. Attach reference expectations to pressure deliveries
# --------------------------------------------------------------

matched = events.merge(
    reference[join_keys + ["expected_per_ball"]],
    on=join_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nSOURCE REFERENCE MATCH COVERAGE")

display(
    pd.crosstab(
        matched["striker_id"],
        matched["_merge"]
    )
)

assert matched["_merge"].eq("both").all(), (
    "Some pressure deliveries lack reference matches."
)

matched["batter_runs"] = pd.to_numeric(
    matched["batter_runs"],
    errors="coerce"
)

matched["rrr_parity"] = pd.to_numeric(
    matched["rrr_parity"],
    errors="coerce"
)

assert matched[
    ["batter_runs", "rrr_parity", "expected_per_ball"]
].notna().all().all()

# --------------------------------------------------------------
# 4. Verify reconstructed uplift against original model
# --------------------------------------------------------------

original = (
    matched.groupby("striker_id")
    .agg(
        balls=("match_id", "size"),
        runs=("batter_runs", "sum"),
        expected_runs=("expected_per_ball", "sum")
    )
    .reset_index()
)

original["reconstructed_uplift"] = (
    100
    * (original["runs"] - original["expected_runs"])
    / original["balls"]
)

model_reference = bat_primary.loc[
    bat_primary["striker_id"].isin(player_ids)
    & bat_primary["batting_pressure_type"].eq(context),
    [
        "striker_id",
        "sr_source_point"
    ]
]

validation = original.merge(
    model_reference,
    on="striker_id",
    validate="one_to_one"
)

validation["uplift_difference"] = (
    validation["reconstructed_uplift"]
    - validation["sr_source_point"]
)

print("\nORIGINAL MODEL RECONSTRUCTION")

display(validation.round(4))

# --------------------------------------------------------------
# 5. Calculate sensitivity using fixed source baselines
# --------------------------------------------------------------

scenarios = {
    "Original": None,
    "RRR_below_40": 40,
    "RRR_below_30": 30
}

results = []

for player_id, player_name in player_ids.items():

    player_data = matched.loc[
        matched["striker_id"].eq(player_id)
    ]

    for scenario, upper_limit in scenarios.items():

        if upper_limit is None:
            subset = player_data
        else:
            subset = player_data.loc[
                player_data["rrr_parity"] < upper_limit
            ]

        balls = len(subset)
        runs = subset["batter_runs"].sum()
        expected = subset["expected_per_ball"].sum()

        results.append({
            "player": player_name,
            "scenario": scenario,
            "balls": balls,
            "matches": subset["match_id"].nunique(),
            "observed_sr": (
                100 * runs / balls if balls else np.nan
            ),
            "baseline_sr": (
                100 * expected / balls if balls else np.nan
            ),
            "source_adjusted_uplift": (
                100 * (runs - expected) / balls
                if balls else np.nan
            )
        })

sensitivity = pd.DataFrame(results)

print("\nSOURCE-ADJUSTED PRESSURE SENSITIVITY")

display(
    sensitivity.sort_values(
        ["player", "scenario"]
    ).round(2)
)

print("\nNo source data or original models modified.")

In [ ]:
# ==============================================================
# STEP 3 — CELL 6
# TOP-10 SELECTION STABILITY ACROSS MODEL SCENARIOS
# READ-ONLY
# ==============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 3 — TOP-10 SELECTION STABILITY")
print("=" * 70)

# --------------------------------------------------------------
# 1. Batting scenario rankings
# --------------------------------------------------------------

batting_rank_columns = [
    "bat_rank_k60",
    "bat_rank_k120",
    "bat_rank_k200"
]

bat_scenarios = [
    c for c in batting_rank_columns
    if c in batting_selection_review.columns
]

assert bat_scenarios, "No batting scenarios found."

bat_stability = batting_selection_review.loc[
    batting_selection_review["provisional_rank"]
    .between(1, 20)
].copy()

bat_ranks = bat_stability[bat_scenarios].apply(
    pd.to_numeric,
    errors="coerce"
)

bat_stability["scenario_count"] = len(bat_scenarios)

bat_stability["top10_appearances"] = (
    bat_ranks.le(10).sum(axis=1)
)

bat_stability["best_scenario_rank"] = (
    bat_ranks.min(axis=1)
)

bat_stability["worst_scenario_rank"] = (
    bat_ranks.max(axis=1)
)

bat_stability["median_scenario_rank"] = (
    bat_ranks.median(axis=1)
)

bat_display = [
    "provisional_rank",
    "player_name",
    "matched_balls",
    "pressure_matches",
    "batting_selection_score",
    "top10_appearances",
    "scenario_count",
    "best_scenario_rank",
    "worst_scenario_rank",
    "median_scenario_rank"
]

print("\nBATTING — TOP-10 STABILITY")

display(
    bat_stability[bat_display]
    .sort_values("provisional_rank")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 2. Bowling scenario rankings
# --------------------------------------------------------------

bowling_rank_columns = [
    "bow_rank_balanced_k60",
    "bow_rank_balanced_k100",
    "bow_rank_balanced_k150",
    "bow_rank_wicket_focus_k60",
    "bow_rank_wicket_focus_k100",
    "bow_rank_wicket_focus_k150",
    "bow_rank_control_focus_k60",
    "bow_rank_control_focus_k100",
    "bow_rank_control_focus_k150"
]

bowl_scenarios = [
    c for c in bowling_rank_columns
    if c in bowling_selection_review.columns
]

assert bowl_scenarios, "No bowling scenarios found."

bowl_stability = bowling_selection_review.loc[
    bowling_selection_review["provisional_rank"]
    .between(1, 20)
].copy()

bowl_ranks = bowl_stability[bowl_scenarios].apply(
    pd.to_numeric,
    errors="coerce"
)

bowl_stability["scenario_count"] = len(bowl_scenarios)

bowl_stability["top10_appearances"] = (
    bowl_ranks.le(10).sum(axis=1)
)

bowl_stability["best_scenario_rank"] = (
    bowl_ranks.min(axis=1)
)

bowl_stability["worst_scenario_rank"] = (
    bowl_ranks.max(axis=1)
)

bowl_stability["median_scenario_rank"] = (
    bowl_ranks.median(axis=1)
)

bowl_display = [
    "provisional_rank",
    "player_name",
    "matched_balls",
    "pressure_matches",
    "bowling_selection_score",
    "top10_appearances",
    "scenario_count",
    "best_scenario_rank",
    "worst_scenario_rank",
    "median_scenario_rank"
]

print("\nBOWLING — TOP-10 STABILITY")

display(
    bowl_stability[bowl_display]
    .sort_values("provisional_rank")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 3. Challengers entering the top 10
# --------------------------------------------------------------

print("\nBATTING CHALLENGERS ENTERING TOP 10")

display(
    bat_stability.loc[
        (bat_stability["provisional_rank"] > 10)
        & (bat_stability["top10_appearances"] > 0),
        bat_display
    ].sort_values("provisional_rank")
)

print("\nBOWLING CHALLENGERS ENTERING TOP 10")

display(
    bowl_stability.loc[
        (bowl_stability["provisional_rank"] > 10)
        & (bowl_stability["top10_appearances"] > 0),
        bowl_display
    ].sort_values("provisional_rank")
)

print("\nStability audit complete.")
print("No original rankings modified.")

In [ ]:
# ==============================================================
# STEP 4 — CELL 1
# CREATE FINAL 20-PLAYER TABLEAU SUMMARY
# NEW OUTPUT FILE — ORIGINAL DATA UNCHANGED
# ==============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("STEP 4 — TABLEAU SUMMARY EXPORT")
print("=" * 70)

output_dir = Path("rr_round1_tableau_private")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Batting summary
# --------------------------------------------------------------

bat = bat_top10.sort_values(
    "provisional_rank"
).copy()

bat_scenarios = [
    "bat_rank_k60",
    "bat_rank_k120",
    "bat_rank_k200"
]

bat_top10_count = (
    bat[bat_scenarios].le(10).sum(axis=1)
)

bat_export = pd.DataFrame({
    "discipline": "Batting",
    "rank": bat["provisional_rank"],
    "player_id": bat["striker_id"],
    "player_name": bat["player_name"],
    "model_score": bat["batting_selection_score"],
    "matched_balls": bat["matched_balls"],
    "pressure_matches": bat["pressure_matches"],
    "reliable_profiles": bat["reliable_profiles"],
    "pressure_contexts": bat["pressure_contexts"],
    "evidence_status": bat["batting_evidence_status"],
    "positive_95_support": (
        bat["positive_interval_contexts"] > 0
    ),
    "scenario_top10_count": bat_top10_count,
    "scenario_total": 3,
    "best_scenario_rank": bat["best_sensitivity_rank"],
    "worst_scenario_rank": bat["worst_sensitivity_rank"],
    "batting_sr_uplift": bat["pressure_sr_uplift"],
    "bowling_dot_uplift_pp": np.nan,
    "bowling_boundary_suppression_pp": np.nan,
    "bowling_wicket_uplift_pp": np.nan
})

# --------------------------------------------------------------
# 2. Bowling summary
# --------------------------------------------------------------

bowl = bowl_top10.sort_values(
    "provisional_rank"
).copy()

bowl_scenarios = [
    "bow_rank_balanced_k60",
    "bow_rank_balanced_k100",
    "bow_rank_balanced_k150",
    "bow_rank_wicket_focus_k60",
    "bow_rank_wicket_focus_k100",
    "bow_rank_wicket_focus_k150",
    "bow_rank_control_focus_k60",
    "bow_rank_control_focus_k100",
    "bow_rank_control_focus_k150"
]

bowl_top10_count = (
    bowl[bowl_scenarios].le(10).sum(axis=1)
)

bowl_export = pd.DataFrame({
    "discipline": "Bowling",
    "rank": bowl["provisional_rank"],
    "player_id": bowl["bowler_id"],
    "player_name": bowl["player_name"],
    "model_score": bowl["bowling_selection_score"],
    "matched_balls": bowl["matched_balls"],
    "pressure_matches": bowl["pressure_matches"],
    "reliable_profiles": bowl["reliable_profiles"],
    "pressure_contexts": bowl["pressure_contexts"],
    "evidence_status": bowl["bowling_evidence_status"],
    "positive_95_support": (
        bowl["positive_interval_dimensions"] > 0
    ),
    "scenario_top10_count": bowl_top10_count,
    "scenario_total": 9,
    "best_scenario_rank": bowl["best_sensitivity_rank"],
    "worst_scenario_rank": bowl["worst_sensitivity_rank"],
    "batting_sr_uplift": np.nan,
    "bowling_dot_uplift_pp": bowl["dot_uplift_pp"],
    "bowling_boundary_suppression_pp": (
        bowl["boundary_suppression_pp"]
    ),
    "bowling_wicket_uplift_pp": bowl["wicket_uplift_pp"]
})

# --------------------------------------------------------------
# 3. Combine and validate
# --------------------------------------------------------------

final_summary = pd.concat(
    [bat_export, bowl_export],
    ignore_index=True
)

final_summary["scenario_stability_pct"] = (
    100
    * final_summary["scenario_top10_count"]
    / final_summary["scenario_total"]
)

final_summary["limited_sample_warning"] = (
    final_summary["matched_balls"] < 60
)

final_summary["no_positive_interval_warning"] = (
    ~final_summary["positive_95_support"]
)

assert len(final_summary) == 20
assert (final_summary["discipline"] == "Batting").sum() == 10
assert (final_summary["discipline"] == "Bowling").sum() == 10

assert not final_summary.duplicated(
    ["discipline", "rank"]
).any()

assert not final_summary[
    ["player_id", "player_name", "rank"]
].isna().any().any()

# --------------------------------------------------------------
# 4. Export private Tableau source
# --------------------------------------------------------------

output_path = (
    output_dir / "RR_Round1_Final20_Summary.csv"
)

final_summary.to_csv(
    output_path,
    index=False
)

print("\nEXPORTED:", output_path.resolve())

print("\nFINAL SUMMARY PREVIEW")

display(
    final_summary[
        [
            "discipline",
            "rank",
            "player_name",
            "matched_balls",
            "pressure_matches",
            "model_score",
            "positive_95_support",
            "scenario_stability_pct",
            "evidence_status"
        ]
    ].round(2)
)

print("\nValidation passed: 10 batters + 10 bowlers.")
print("Original datasets and models unchanged.")

In [ ]:
# ==============================================================
# STEP 4 — CELL 2
# EXPORT CONTEXT EVIDENCE AND SCENARIO STABILITY
# CREATES NEW FILES — ORIGINAL MODELS UNCHANGED
# ==============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 4 — CONTEXT AND SCENARIO EXPORT")
print("=" * 70)

output_dir = Path("rr_round1_tableau_private")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Updated player summary with shared key
# --------------------------------------------------------------

summary_v2 = final_summary.copy()

summary_v2["player_key"] = (
    summary_v2["discipline"]
    + "_"
    + summary_v2["player_id"].astype(str)
)

assert len(summary_v2) == 20
assert summary_v2["player_key"].is_unique

summary_path = (
    output_dir / "RR_Final20_Summary_v2.csv"
)

summary_v2.to_csv(summary_path, index=False)

# --------------------------------------------------------------
# 2. Batting context evidence
# --------------------------------------------------------------

bat_ids = set(bat_top10["striker_id"])

bat_context = bat_reliable.loc[
    bat_reliable["striker_id"].isin(bat_ids),
    [
        "striker_id",
        "player_name",
        "batting_pressure_type",
        "pressure_balls",
        "source_matched_balls",
        "source_pressure_matches",
        "sr_source_point",
        "sr_source_lower_95",
        "sr_source_upper_95",
        "source_sample_reliable"
    ]
].copy()

bat_context = bat_context.rename(columns={
    "striker_id": "player_id",
    "batting_pressure_type": "pressure_context",
    "pressure_balls": "context_balls",
    "source_pressure_matches": "context_matches",
    "sr_source_point": "batting_sr_uplift",
    "sr_source_lower_95": "batting_ci_lower_95",
    "sr_source_upper_95": "batting_ci_upper_95"
})

bat_context["discipline"] = "Batting"

bat_context["positive_95_context"] = (
    bat_context["batting_ci_lower_95"] > 0
)

bat_context["negative_95_context"] = (
    bat_context["batting_ci_upper_95"] < 0
)

# --------------------------------------------------------------
# 3. Bowling context evidence
# --------------------------------------------------------------

bowl_ids = set(bowl_top10["bowler_id"])

bowl_columns = [
    "bowler_id",
    "player_name",
    "bowling_pressure_type",
    "pressure_legal_balls",
    "source_matched_balls",
    "source_pressure_matches",
    "clean_dot_source_point",
    "clean_dot_source_lower_95",
    "clean_dot_source_upper_95",
    "boundary_source_point",
    "boundary_source_lower_95",
    "boundary_source_upper_95",
    "wicket_source_point",
    "wicket_source_lower_95",
    "wicket_source_upper_95",
    "source_sample_reliable"
]

bowl_context = bowling_leader_profiles.loc[
    bowling_leader_profiles["bowler_id"].isin(bowl_ids),
    bowl_columns
].copy()

bowl_context = bowl_context.rename(columns={
    "bowler_id": "player_id",
    "bowling_pressure_type": "pressure_context",
    "pressure_legal_balls": "context_balls",
    "source_pressure_matches": "context_matches",
    "clean_dot_source_point": "dot_uplift_pp",
    "clean_dot_source_lower_95": "dot_ci_lower_95",
    "clean_dot_source_upper_95": "dot_ci_upper_95",
    "boundary_source_point": "boundary_suppression_pp",
    "boundary_source_lower_95": "boundary_ci_lower_95",
    "boundary_source_upper_95": "boundary_ci_upper_95",
    "wicket_source_point": "wicket_uplift_pp",
    "wicket_source_lower_95": "wicket_ci_lower_95",
    "wicket_source_upper_95": "wicket_ci_upper_95"
})

bowl_context["discipline"] = "Bowling"

bowling_lower_columns = [
    "dot_ci_lower_95",
    "boundary_ci_lower_95",
    "wicket_ci_lower_95"
]

bowling_upper_columns = [
    "dot_ci_upper_95",
    "boundary_ci_upper_95",
    "wicket_ci_upper_95"
]

bowl_context["positive_95_context"] = (
    bowl_context[bowling_lower_columns]
    .gt(0).any(axis=1)
)

bowl_context["negative_95_context"] = (
    bowl_context[bowling_upper_columns]
    .lt(0).any(axis=1)
)

# --------------------------------------------------------------
# 4. Combine context profiles
# --------------------------------------------------------------

context_details = pd.concat(
    [bat_context, bowl_context],
    ignore_index=True
)

context_details["player_key"] = (
    context_details["discipline"]
    + "_"
    + context_details["player_id"].astype(str)
)

rank_lookup = summary_v2.set_index(
    "player_key"
)["rank"]

context_details["rank"] = (
    context_details["player_key"].map(rank_lookup)
)

assert context_details["rank"].notna().all()

assert set(context_details["player_key"]) == set(
    summary_v2["player_key"]
), "Some shortlisted players lack context evidence."

context_path = (
    output_dir / "RR_Pressure_Context_Details.csv"
)

context_details.to_csv(context_path, index=False)

# --------------------------------------------------------------
# 5. Long-format rank scenario dataset
# --------------------------------------------------------------

scenario_rows = []

batting_scenarios = {
    "Shrinkage k=60": "bat_rank_k60",
    "Shrinkage k=120": "bat_rank_k120",
    "Shrinkage k=200": "bat_rank_k200"
}

bowling_scenarios = {
    "Balanced k=60": "bow_rank_balanced_k60",
    "Balanced k=100": "bow_rank_balanced_k100",
    "Balanced k=150": "bow_rank_balanced_k150",
    "Wicket focus k=60": "bow_rank_wicket_focus_k60",
    "Wicket focus k=100": "bow_rank_wicket_focus_k100",
    "Wicket focus k=150": "bow_rank_wicket_focus_k150",
    "Control focus k=60": "bow_rank_control_focus_k60",
    "Control focus k=100": "bow_rank_control_focus_k100",
    "Control focus k=150": "bow_rank_control_focus_k150"
}

for row in bat_top10.to_dict("records"):

    for label, column in batting_scenarios.items():

        scenario_rows.append({
            "discipline": "Batting",
            "player_key": f"Batting_{row['striker_id']}",
            "player_name": row["player_name"],
            "original_rank": row["provisional_rank"],
            "scenario": label,
            "scenario_rank": row[column]
        })

for row in bowl_top10.to_dict("records"):

    for label, column in bowling_scenarios.items():

        scenario_rows.append({
            "discipline": "Bowling",
            "player_key": f"Bowling_{row['bowler_id']}",
            "player_name": row["player_name"],
            "original_rank": row["provisional_rank"],
            "scenario": label,
            "scenario_rank": row[column]
        })

scenario_details = pd.DataFrame(scenario_rows)

assert len(scenario_details) == 120

scenario_path = (
    output_dir / "RR_Rank_Scenario_Stability.csv"
)

scenario_details.to_csv(
    scenario_path,
    index=False
)

# --------------------------------------------------------------
# 6. Final export validation
# --------------------------------------------------------------

print("\nEXPORTED FILES")

for path in [
    summary_path,
    context_path,
    scenario_path
]:
    print(path.resolve())

print("\nDATASET SHAPES")

display(
    pd.DataFrame([
        {
            "dataset": "Player summary",
            "rows": len(summary_v2),
            "players": summary_v2["player_key"].nunique()
        },
        {
            "dataset": "Pressure contexts",
            "rows": len(context_details),
            "players": context_details["player_key"].nunique()
        },
        {
            "dataset": "Rank scenarios",
            "rows": len(scenario_details),
            "players": scenario_details["player_key"].nunique()
        }
    ])
)

print("\nPRESSURE CONTEXT COVERAGE")

display(
    context_details.groupby(
        ["discipline", "pressure_context"]
    ).agg(
        profiles=("player_key", "size"),
        players=("player_key", "nunique")
    ).reset_index()
)

print("\nAll validations passed.")
print("Original datasets and models unchanged.")